# TAS v7 — CPU preparation / A100 experiments

Upload this notebook to Colab. Use **CPU → Run all** first, then select your
**A100 80 GB → Run all**. `PHASE="auto"` selects preparation on CPU and the
study on GPU. Models/data/caches/results live in the same persistent Drive
folder across sessions and future notebooks. CPU/GPU environments are
recreated locally; the notebook kernel is left intact.

Bring **only this notebook**. It clones the fixed public baseline, verifies
and restores its packaged experiment modules, then imports them normally.
The editable implementation remains in the experiment branch; the
compressed package is generated from its exact commit. No separate source
upload is required. The code snapshot is also saved to Drive with its hashes.

Automatic flow: blocking P0 audit → 30-image smoke → 240-image screen →
predeclared shortlist → confirmation (3 seeds + greedy) → controls → P0-D.
Final remainder and P3/P4/P5 remain disabled by default. A session
budget saves progress; the next Run all continues completed stages.
Shortlisting requires the predeclared coverage guard and a nondominated
observed hallucination improvement. If none qualifies, the study saves its
comparison and runs baseline perception diagnostics, then stops dependent
stages without forcing a finalist.

Full A100 weights have not been tested by the local CPU suite. P0 rejects
baseline mismatches before bulk generation. No scientific gains are assumed.


In [ ]:
# Edit these settings once. Reuse PERSISTENT_ROOT in future idea notebooks.
REPO_URL = 'https://github.com/RachalHys/Tag-And-Segment-AGLA.git'
BASELINE_COMMIT = '624f74501bdafe9b7d6a05ae739d12b422ce8737'
RESEARCH_COMMIT = 'e9f4b0a5b0b993c24d9ae6d1edb19e58b3260761'
PERSISTENT_ROOT = "/content/drive/MyDrive/TAS-Shared"
LOCAL_ROOT = "/content/tas-v7-work"
PHASE = "auto"             # auto / prepare / run / score / check
STAGE = "flow"             # flow / audit / smoke / screen / confirm / controls / diagnostics / final / external
MAX_SESSION_MINUTES = 180  # planned stop; run again to resume without changing the experiment
SEEDS = [0, 1, 2]
REQUIRE_A100_80GB = True
AUTO_SHUTDOWN = True       # flush Drive, then release runtime after this session's work
RUN_CONFIRMATION = True
RUN_CONTROLS = True
RUN_DIAGNOSTICS = True
RUN_FINAL_REMAINDER = False
RESERVE_POPE_IMAGES = True
DATASET = "coco"           # AMBER/POPE require STAGE=external and explicit verified manifests below
EXTERNAL_MANIFEST = None
EXTERNAL_IMAGE_ROOT = None
METHODS = []               # empty = exact initial 14; external runs should select frozen finalists explicitly
BATCH_SIZES = [1, 2, 4, 8]  # measured once, correctness checked, then policy locked
ATTENTION_BACKENDS = ["eager", "sdpa"]
LAST_POSITION_LOGITS = True
SPLIT_SEED = 20261021
COVERAGE_TOLERANCE_PP = 2.0  # approved absolute percentage-point loss vs harmonized TAG; freeze before outputs

def release_runtime():
    if AUTO_SHUTDOWN:
        from google.colab import drive, runtime
        print("Saving pending Drive writes, then releasing the Colab runtime.")
        drive.flush_and_unmount()
        runtime.unassign()


In [ ]:
# @title Versioned experiment source package — generated, do not edit
SOURCE_PACKAGE = {'schema': 'tas-private-source-v1', 'base_commit': '624f74501bdafe9b7d6a05ae739d12b422ce8737', 'source_commit': 'e9f4b0a5b0b993c24d9ae6d1edb19e58b3260761', 'sha256': '48d4ab1d6faf669853cc7d55be79cecf153a7bcba0d5048a2254f61a6ef6582b', 'files': {'AGENTS.md': '7888f139b584e98115332f45d2c93cb787b9b3919236fb02a3e32adab6e30be0', 'docs/TAS_V7_COLAB.md': '7fcb695807d1c03e3cde1ce74797d760282fee19ffbf623d5e57eddb06109ac5', 'experiments/__init__.py': 'fe7a0225cf031bed0d2204dbb763b431d530abac99b01ea33faa4d0f6b244526', 'experiments/tas_v7/__init__.py': '5ff13022425f78d78413600e40c3941e750ee983755c9b17a5d564609400edf7', 'experiments/tas_v7/audit.py': 'f6ea192ba422b551601e3c558e16429b5b7670b57ef4cf42fc42c7428df42a3f', 'experiments/tas_v7/cli.py': 'b16c5fbd3f0893697b332af31203fa97b686537f83b4663e71ab3b40861455a0', 'experiments/tas_v7/diagnostics.py': '2a1a56e626656d6172bf0c9355a5db5b60519f0ac82350f5003cf3416cd1f77a', 'experiments/tas_v7/evaluation.py': 'f3f8cac6999331c5dc67c0708c628b0cc81a25bf6045224b0c124690cba7c2f0', 'experiments/tas_v7/external.py': '7e9d9eb0f91f1d424c1bf34bcdbd45587d16faa8d2148eace5dec068cd97bc56', 'experiments/tas_v7/methods.py': '2913cc58e41e1a726d7e1ac8c7b2711a5259afa5dff4fa7e6f0d3fbf4d7ad9f5', 'experiments/tas_v7/prepared.py': '81b5d30c97e731f8e05bab40b944835333ba937d041c4cc135e601540d93fecc', 'experiments/tas_v7/report.py': '8bd6aa025a3ef41255b24cfca247f5d9d1779a1a6b32d5b2f246bd39bf7a0994', 'experiments/tas_v7/study.py': 'b48e222fc337c42a8a1e29fef4efb2ec7f40af10cc9985915ab5d8bea563ad5a', 'experiments/tas_v7/study_diagnostics.py': 'fe252e3e95b583423d59157d01364560a557f70e2ca556868ea00bb174f9bca1', 'experiments/tas_v7/views.py': '9bffe95f8996c3462b7e82b3a51587b9629d5dfe6d45c7654a17989ffb160775', 'requirements/colab-amber.txt': '7ee7be8843a02ea16d1333e4271ca0b8174445aab6867efcd46e57739a95389a', 'requirements/colab-base.txt': '992b028d26a85bd15ccd29321a8192bbc81efb6696df38eb834820f06377a9a2', 'requirements/colab-perception.txt': '0e98ee13388aa50ad35045d981a278d347fb0c56f1634fca4b9e5567ca6aa03d', 'tas_runtime/__init__.py': 'ebfdfb5371eb85b8640ab449c246efc862a11ea70e142718bbef4c9f7b5deca5', 'tas_runtime/artifacts.py': '37e4bc9271a13ca334df828af92b58e425706d0164f9f30384c81e8dde7b2c37', 'tas_runtime/assets.py': '675315d0c9b6d195e7941829b0bc93d3101a34c8f5205d9f3aa6fb9e0eeb3538', 'tas_runtime/attention.py': '94585b4980179e09acba88e0b3492c2161d94904faa7783f6f5a8391c51854fc', 'tas_runtime/bootstrap.py': 'a679b0195a09af4f18afe5f5fc1bd948fb983490993225860ec3cb66c6f2f733', 'tas_runtime/config.py': 'a58903c262e490dc90a3b1cb9ded9b6bf1c18c018a1592f4c9c2bd19c6aedcf6', 'tas_runtime/data.py': '39cb6a954ddf713a622630c584087fc3f5e8e5cd4da0d9a2377b7a7aa1be2523', 'tas_runtime/decoding.py': 'a5a370f0345e5ed6fbdbb612f16b9b285ab287bbb6241dfeb39189663ebf88b5', 'tas_runtime/perception.py': 'bf7179deaa1bcc8421d6c6e989554399155c1e391fd16c0be57185e43871e025', 'tas_runtime/persistence.py': '9fef2d9912d19ae2f20964167e121f2874e92ae3e3e695f3c64e96cbf1956a1b', 'tas_runtime/profiling.py': 'a77bb6b706612bcc2ecdf45a12756146f6db0b67f5a9eabc58f17ad7b9d8c4d8', 'tas_runtime/runtime.py': '068acf5e27671c6b1131374ba167cc464c86f74816a2553beb1758bac11d497a', 'tests/conftest.py': '5b20fdc9ab1817a64ddada47c22d695378148741eada4f98b1a745a49f7c73d7', 'tests/test_artifacts.py': 'b21d651d6110626e3e99471230b5d7e9386b91f30ebaf035ef8e9c5427d9f506', 'tests/test_assets.py': '6b51775e20a03a2c9384bc9922285075bd960131ae9368bfde5f73fec98b6753', 'tests/test_audit.py': 'c6fde3cbbc8ad8394a0456390e4b4f350871eda9fc7efa1fd2f30eb50bbb446c', 'tests/test_bootstrap.py': '8d599f15e50306e449ff55168cb740c2ea4cd847220bead6554d32522fbf4d27', 'tests/test_data.py': 'c55b36a369a9da72cfe17fc8c88b478cde2ae3d466bbbca67b60013ba18e92e3', 'tests/test_diagnostics.py': 'e5cb5776067150a2b1b6aff62102f7e881d0d8d88ee6a53485265478b2e02874', 'tests/test_evaluation.py': '4a8e77ae0c9e4b2723fb1fd1a40f42e3d5c4e15075ea2cc2e2d54dfd350da98d', 'tests/test_notebook.py': '4c6ad80f1e0811eab4fee9b07069268d5bdfb479f124b4ed6b3467c686e00d1c', 'tests/test_notebook_delivery.py': '377d9a8761c98b6ba6bcb2752d6f7549bb89f49feb0b109678ffb3ef5fbe85e6', 'tests/test_persistence.py': 'ae9b71a14af81561373fb273e58b773f8ced8fb8e4189a1206742cdbc682ffa5', 'tests/test_prepared.py': '1cb769c861c2f06cff94d17bd5027354a94f0cc8baf631ed3b8cc1019b797a73', 'tests/test_report.py': '65f0088060033ed91f2649124f7c11c7554154eedd3312b16cd7f3992113f60f', 'tests/test_runtime.py': '3db34ecd2dc499a6d56257bf95be6587b5d877fbd5e4add76341b4a51c285321', 'tests/test_science.py': '7303f63cd2f15c550ce45d78c5b6e904fe148a3d58e59f103239d84bb9ef2ea0', 'tests/test_study.py': '6d6783c7f216802eda1b1768353a1f7f780718728f57dc5e3e18e068fb6e7c80', 'tools/build_colab_notebook.py': 'b13f20f986428b416afbb1add752dd04f7d7cb696e4188495bb8658f556268c2', 'tools/notebook_delivery.py': 'de704ff351f24c0ab3cf43a892e2aeeaef3d5d5bf87cd047f14ebe985886e8ef'}, 'archive_b85': 'P)h>@6aWAK2mk;8Apq0QNZ(%r001cl000R9002QpMNU*xE^TCOR84Q=HW0o0R}9cYTOgJa^bnxCTfoh>X|X|Tv`#KXVNulBVoi|@Kb&}fecwoS>@9k+M42=5-kUe`6)mIxhjpfHF`aUnXR}q$v|9X0`&;7Iz|josGMPl3*G1mBBje~V98-;p>!;iG-R}>JAFHAJ`<ws!`fWA)H78mga_=3PAy}3?DjOWF*>jSFO;%|gi_0ijH>6xg%^5(d<?sD%-MpJ+AiVQdwd8!}*880#^$|4`ZOp3x{IZDl!(ehnpO^zlxQ}LQoU*fq_sXu3yKb&%I<zXWHSPpr@Eu#~cw?NAWQih&=)0m#Yq^@u=Jc3&%xV%)i9ScF-&Hr)gu5GK`M(q^;La&W6&l+3h<KA`bRl^~AH5JnV9Ltcv4Z8T(gz@TE+&oyMZ4@LJ-$w~<LKB{0QICSTAG<&Ozx@q?P2w4{c*8ce5ixUeVu(BKtx6EYG(t53G#f9H%^)4C(-<QQ~}7E^9{%uKuJK&pXq|s1^h(Wdz+>-O+~9QU4c>x$p(n2a)}Ho%F@GwdRiQn7R`_RK%X`RRdmmK7rc?Q(CU8qC=Bn}aTJE4%$pU2?giWtMco!-yR{aP)~B0QIM94PS%cnx>6qi^ADRlOWFof5THU!&P<p4fFmLWvK3WVNxEPwF4Zy2d+?#Za5W>eXj0f_2jwAkHoJ00dT5B|&Joru|NYW<ZS%USdlUa1k9d{zao28f6Z&6o(>j%#9a7C#E8I4g^eq~*zfe=9xwbtvMaodXSmXE-O;gP1utdG_MR0TY#AKNfLw*lm%$p>l1;^rDA=Li!IvOxsY5;-{TnKsdXW=HTLpye7iA=5z`OcJMR92^BQFQ>1EaTav1+?GWwkm!Avu3+*|rskhXBu2q(bb?AOI!1m0Dh6k*IVE5wU$O)WsGlSA=gBOoha})7Z9^csND(cfmQkKhLEn#eRlo%A(Hg7(RjjN+07J(IFH`d3Nwt{!g1S&ai-VBvWY!H?pAw-+Dv5r&ng4PEi%Ocug@ru?C~wUi#{iU%qTx*|D32#p1%S~=KqH$o9;hv@6L$f$l)J~C1HM1d02OtAG}N>PB@Sc;Kc28P28FGJ6>zo~8a4p3#!&931%j*9>|anz0|XQR000O8001EXQ{c><VF~~M!V>@h6aWAKWN%}0FH}KOUsg9?Lr+XWLN0A&eOSSA+eQ+-<14z7Lu?fZP_i9oqjjoiS(d$#C6*;+YuB|U4uBys(EtNu29%i7{=@#^e#yS>0Z2NBz1S3qndyH0`t=)pPq*_M+PtPtd#$>nqQ(WCJNJ+zx0NPcnxJxv4^OS`Os$(hg)21~Pp+Zm{_89#Ki#~Z7S5`CnG&xa_u{8jVpZQ1mG0=#go+fcb)!2KbUDG6b&CkRF7dJ5l3P)zoY%D2;{S$fSN2x>ls@8k>l*Eofc;nI2`kz@x2B+6d2LOjNi`*>6CC~_e(WIjVIxv~aJ34iP}XkK<hrq2d2|<XqjibZj&e<XYn>`_68=2n^wK2x{(Jh|H)NXlO_Cg^bSKZza@@pc0QY|L`aUi)?Y7C65*{MeMF0jzM`w$>qa*72CLqEzIZ0{3NZ=RVAK~FwUV?}#*O@;wz3a?I%VR5R)>Yue9&xUauq0tsx7G;}mF}_OkO15#6wUg{a58c!RZz0=)#ZZWa(!1YFxbPZjU>xkEI)1KJWgeLQ{m@wnI<n%x``<q9nFttGx}*p7jKV_CVYPLZR~4hnm^N_@d3Za>Rc@D_#oYEOy?S|(qv!0mEmexCv}>TBm?4Uy1Z1s%}3Z`$r**kEKOdfQW`m!)GB~+3VC~Pz$D!;8fV~})eHj;>22_`16We%Ht=JG0#gMIj;+%@sF~8QTDKCjS2z@pd*cJss&g(d7gl}f0iY?h+GXp(mbta=kyIe7m1odSUI8y!b$Tc#`BoJgaG{8DLg)H*1yUZr{_ns4bNmxOjfjee)|n>wq_w@@mvBfTrAw}S7olJVgYTV=;Fo%(@DTaXms`sFa;*dRWvCEDQlb2wGa}#TZRbD*e@cC`GL7*SHhOg3gPc=q-ufwwh^90r(4wT@K4EXyQmwX0r@!_FKy-R&a?`8_Y6g@R8%xv5&aMw_kG8J*<^fPG=XbZ)_ctHzZr@%1{p!ta*Xt#%kS^CFVii+nD2E2$(I8Nig+^H_w6&AOG==R;K7M3EYgN<W+yMfPnGpgMJ?|&I`U6O|TA9Kqn|xW0&Dl2!DX*Jn&zJvtR)x^|pR;TYG3)ab3YM*%Tcas>gG~Mn&RL)3*5z3ZG7b;K-?DMncKbX{Zgl8dNlZWKq8AS8u5&1gvff1izDfUU&p-|^jNOmQwctI&9X0BFFMTUeFiGB`mmV07#f+xEkgwf?re`l^WNNk65QyWKRBK30>7UC31?bVxvy+#zi8P)Et>hQQYU)E09$iEnz2NzW!htp&t1FVg*0V_@nJfj#n7V8HU2bKcQg5_%EytK(Y3P6%f4`Vb-%)AQy1^-?@bv8Ut0BtgDZQ5(cbai6QCFTvZJ4suEM8=bm)YXggx+6FFD6N4%2Kmbqy8J5g0N!0vnt?3P(c+BIVZ>JheiSNX!ke)RSQsA4?W~JaFU>#mF}-iV4WU-AO=DtqzMR;VyE&bt5G7zLW%S?Xa@Xdo%d*V{N(`#+NX96q`ky6zMyBzv+Li^KhH1D?{BX!@!{(1{C=@`bCS-M6H>W%whua4b*d1EXX)(Zxs1l4?KnGfx#x6CI{PsH_}P%_68YqTp=2bah>YVF!K%{LK$TUob(pAz2QEVpoJH78Aya?VxjzKWWavCrJAfLtBMD<e52FO+L1-(eAvCoq4Kz#B*B)$H8KzsOTfi#yNOlwaK!^_su$xV%-*SwT7#T<-g%S}Xwi<|sWwj6y*gc5yUST=<>3o)*A9M4IcwC-e%+LP8ofI4zHKi;4xWhw^I-gBHV8!%<tobni{OS7Y<3G>eohEE!I7mJf(_q{KFoKD9lv@^A#%K+B$4-uIhfE%}U#ONBD#l;aL{9Epu5x1;U!+@c3EnSWoSf35=Hc&$9>AB!Kc?K(GA5wWF}V>_KS|cwp|d-TQA}0z0&}`=8UDeObdb9&SD|28&UK|W2GeM$YBUalVUz4dN|4!N;_Ba{uW;?qU?Ts8>q@Q{BU`Ye8&^Pg)z(J`hf;hVXwd1v$*hzCR?H9KEDR!Ovv%E<#ukDX(9Jqj&=ym3R364L7AR?z;p9XsrWnD&hfcMF06<QM<_G1A&a?uPenK-%G2}`ZSO_!4J%)-JFLSO4;4KM2Qv}*~jm)zK*`O+3<4I;WI;19r$s44lqsGW^O(djKa+W14qT_cQ9rqgKg8S1C{Xqsb!hmYj2FS1>Pm^~Coxw!prZzm05hEs?3$ZzBsi&6`Y|-NEO9-Pw$@~Sdy~mJ)MaEcEv2-Z<x<@X+iwg>!f!^iJNqCtv6&q7}I7IrYIydXyLCRdtG@yEvS%hb8v?3l2gl{obk<`ZM$2g(R_%<=PuV|+Vog`CQ;y!Ud%*br{Gdp)}GqF7A7J(WkOi)$GH7|lRh7ix4ut95WBx^aCxs*5EUj;m0SYXb1Z;l3B8@qhM<dk8kL6?3nG6!NLyAAWbH@3W&p?*J_K1_A&3s3jJa<BM;gK5G2ac~&{A>B8(4&F4*_$6PmGRXZBmO~?0{>aJ8hsL(@e@sk1vgPAcqK_xvs1^e&pzWi5l}nts>B%>qchKLIuA#e+onen8Hjr&%CI<+)qfkh%&LR1jK{6<Y``L^1c$Oajcwio*nYwj+r94SrrpJ_Fj^MAzQ4Fo)zu-%5*IO{nLt|<XfL^9AUZp4L&W1hcuF;l5uzDqsp;7tr<KoqTWd?$=97bQP?D8}@yZpEysZ4AzYK^!)&qhn?Ch21=$wNa%;%BcMiUb_cNXdJvd^I0=CHGVlpW~32*L*Ly(Q$U{J>y~<Q%Co8=4tO1+pU0*x^A7peNI6%Mm1}>CD1@qwglWc+4D*ArQeJIuX{|{{Whfo0~O1fO?{R18+giwN2cc!u;5@H!T5{d$!3?ubM&6FEf4nn&f|N4G<sF>wL)zy#NutXQeV?5)E1^Sc@L$CdJoN+e|mfVIa^#W&c|COEpOmw&fk1D!uj1&E{Jk*K_LeB(Q?;?<Zn0ESC^S6BpN4Fes{+lEXQC=d4~=nI!IaQL0ljd-xuUif~r;>-?|QOJ-e%C)DHIf*Jsyh4EhJZ&jMql1O5B)YJn^FnjJ$jNC7hDsM-1L1b-<2O`(FiOT$xGvKBKvqrzSC?R6N-xZf<7n8{Ugb$R>i{k!??{7uH$X2E4cY-NW2rdUUs*8GXBKK}z_{59GC66vUMH^%pLAeaX#!!I$I?XrAa`+tRtrqd`_^jBV_%c<epGH)i#AAbFXQ;8lqhSn;1fg|H@kfGro4>$3r*DJtJ+ZFLJXHSObs^LA(-UZ5w9&X2BjR%Ca%J^PKev4q9Cf_t9*98m15J;N5@2#EgZ_QCtGI?LIH!!?P3OzDlg;kLkBN?K@O8SP~?(i-`tXAFocv>VNgg#n?i^bh&oATa1jBmQI|0yfZ?mpmK2ng<dAl1QiJoM)fnZha*atEQD&%TM@G=Sy*15ir?1QY-O00;m803iVS)VpCv0000^0000N0001GcyMKMX>Db0baO9XUukY>bYEXCaCtS!!3}^g2nEo)QxFDl0he$EVt*>Jv<;BP@jdyjkC{)`B?-bi;cDnfQ?5rU;vls-JhfNKTjLMvSa51XN|23$7CSR<A5cpJ1QY-O00;m803iU4>Ph&n0002u0000U0001GcyMKMX>Db0baO9sVRK)0H!ojbX>MtBUtcb8c}0%94#F@H1bctQ$~B~=qM(Qn1rI?$bm)i=V#)R)+X>|FacHP#cC=dSW_7}_K;*;}+YZ*h9D5F#vXxLN)pP~Hi|HMwKSnpcBu^~O5G12@WPO4>;F(e%*-Ubj;ip%?Rt!7!X8Y`N0tJKh;S6W*k)kbWiVP=Y%20wEaUW6o;19R`^aYMaj)lRnFY`)it(B^;hx)kL*1P6*sgI}4eupJ=rZgWGT73afO9KQH000080000X08GL!jpPyl0HsI(02=@R0A+Y^WpZh4Wo~qHFLYsZUv@VyVRd9_bS`jt<y>2H<3@IV*RSZn4+d-ql455csNz<QvzoON$205Q@@~Rnp_*iqf&~ItT%@5@`tLcHZlD_%lF}qwsR|!7WTP*q&*eLpZj9%7-=+EbA<1^)<F#1dtMx-AWVR7f?3JvVQf-9#q}ENH<k@ABr^)&tiX_czk!M1#>qe&HNhQ1cx{5r{ySUhv`Ci2Fwy9xo9E)UM<Yf(WW_c}fmg?d{x7EpBasQ31<vNvBr7AO^R7EP+s_UGW>w7(YEvvX}G8`YtvQD-@NIOv0`94|4zgBrR?7dOze1p5yo&5^|pFd!$`cM#gAvIA)6|YK}0atY67HCkLxG2>&`4p%5E~)jC61tb0xKe5pUIfE+i!$FPskN}m9+NW9_A0C6z08uWs_IZw2Urc7li!ryYP47NeZFaTw{0qARu=p$ULE3O6AIv0Qza`{rar)&)q|b^$SouaQcs~rZIwh3!8AqkNu}|FI>1-%YaYfcStHjkHfn2Fk~5opy|qxR^K6^!3^k$nDipgC7;%`tmuUqZ_VQDlsi(NkA5>P&f66lz%!q9}m8p`nLd^9%QP-w93xs5@h*)a)=}qna6nSZ_5V;S-c8*%@3+w~?om<_lW~*;r#D&uf8{Fp){pSze^GzOCa$ls*N^K)q9@vTIUpGwU6_X8uuI7gI5jBCXl#$&b47!1Z+_dfU;V}C#a_hvz-Lz$1=j%M3v&<v&yX#Tm#(7|+{@$o;t$^^;lxx}pt#g`7iY0DTj(%vhqwiYv=sR$TzdR=DNq<GV)L*~ie;f^^mKNsbULn^HYUAm-huRA!KC_jx2Ns=rn(eM|!pve|OJt8w{G@Wf+DO4hKC>;}cMac?bGM{5LtZAkB$MfE2n}w3{Fu&W@@S=M2|r+CeJML*gNxy}WNRj=ngTJWQ)FPbU>3F};lheD`6R|@@o76b4g(#5mU?yy;mW@K=IY@(-xDrYnJlXNyw-GrBTyykwpK~7-voS_a#Rf*QaZnmuD{i;LjSzspRFl*-@p6$KR*2Q!++iVL2DjL0DAt5)?sGl{4ZtGDIjn<Rhi$?5r}`9Q%g&V?no=k1i1eWylja(yerGR^u71w)Gpbn39~`iG4Q?;n`8@9M;;6JDKD!Upc=^Mt(80E_4;0BJM<))M$>>jLvd|<C$PCt>l$37i5!wl{FW3Q`Jou28;53cudutBV&>?{HYv${f+_)m&f8uK9JY+MX)bF&Fw45F&O~N|uq&Uwhl0ELw1l|SYXp&8Repdz#Sa2^4vJL3eN;90`$!VI<8_)>%6I4tSu^lY$XbDa+__u^i?8Rv14B&25B<v<k~&XnH3ndPd%s5<C=9W3J{R7HpMUtHM?Oc(Z8kuKzCbkrgMLE{vOh!kaRn3#3v7%*1e4RjW`{G{qJ71Dnn{CafibTh2ToX%tCijd4XT$2@ea{uUcvKjNXegji@oSjn{h1idQhvq&vV=Cp{{^p=4n%FpAjph%bT%gcZlb?ngtztw%qkJM6s9d2K0!v=LS8-8EZQt@Jeni+twEiuCmzxp`eE`ZFd-1mibfF#-?C5n*`!*FtdXO2}nHd<W7}hB`XC{wh}u8VNu_!4Du9pDeFsliF0=NQr(J7J;InO0lb93TBa$w>F9!{d<P9x4UZpGnJGZ33Vas`a|Kaao}~v=4ZzC|IN3D|i7rVzCG|ak6|8S%2OwnFy22oVBR%$BKoYw?038UnF|UGgTJp&r_EkmVy#y*Zu9K-C$dU)a>tMbF!noDnHJ%EEY`FpzO#p2RS>MC6y-Yxv4C7eOIWwS~i8L1z2kf3EEA#Lp{AAvgd5#!~4Spwz_EEHdfL0%61s;U`6(Dj_)p7nnKe;bwlQB^>=Si=G*UvU1B#NoTOcm2F&O+xF;9#E!aC%eu)~r#fFuqlv0ObP|nEi|h0toLyk_+3iXv=fkwycZ|w0JCk^Q{2GGpZPKQVl~EOY1R-x8Jx^IRrT60oskSFRqsh4;m$=$e!I<t4*BB6>I^9c#y8F6*XR>ih_dYf}aYxVf7O~{wRNWhe5}_s6E@r*9mMT*+S-Q02AkYb^y)$@$QfB{`Eg`tqf9^M<wpAZvhCOE>{ZhOMxxq5a(9_Wgou#x6ALZj8So7!$V{V#sND@*5dBQB7qjLVj$d5I~^$fqXd(E|M8oD#)ug9cm+>alF{@^W6>>IJJ=;auE?9ZXlgsVvle@?w~o0NxN6Npwz(^gG_L@QOcIipAsE|66%ffIV&q%#YrYa_+2yUl3F9b%U?Eck5Kiqjt(iMquYoX<9jucf4F(zUZyZvdAYwF+u8;td^AQnAH6XPDeOZRX>^Xv##~58;VDciE2mVG)R>3g!oAR#%$i)1vIP#dRb#K#FDM04BM%~CW-y=NH!Jx?)Km#de>Loi7HiNNN39BjLBMtmQ`~^VD3<oVhh)EUN64wI2DS+iv!bpY?HxlpAPXNHlTI-UYjSvK|JQGg{_!4lj0bGTspdz=ban{s+ZXpY&EPD;w37RRItW+rg>PFA^{d+wGQdbGF94|oYKIKii5y>tCZ3n`?1j}l6NH~>OfH{n-OkT_~LDq0Fs{<@Md36gR^FiOPza?GQZo0ApX9rF=YLz#JA=*%Ftp0=E$@cgF@p~QZ+aChho5+1&L1mlrcgVZY_!76CEbbQcmM$>2n+H2`x9{1;KXauLohtPJB<2MU^IA<crhIpOb$4?M5jQ|I=ncC|RywU~+DB>Zw~hJ>VS+*mPg6=Q8}#O=jcndJn25x`D^>J7I%lGZ@zh`RAqjhPE)6T730@i##L0gJXN_O6$3FRuw|_BGsu;Ci;5!veXaM*rH$_V}>_HGhGvRx8*Ip<*!1%z;D;QZZ$+F6qemB|;#;G`2ITCLrL<vmKS8{(3zR}a=V3Y%Y!>0p03|rQ0>O-O2Aus*p%3=bhSg`wHQpF?!f#@Sio5xe9iG8sXj`ir#k;nJ=hJCvA9aBLeWNQoGli)(d7T5#_^0XVmN)S7Gu8w6^2kqA`rU~dwqH{fM+mIHg;x>>3smj!IfQs7J(h7#1aC>vE{P!RA>Qb5?6#EUz&|`T0I>d5|)FXr3TD4oV9`F9(b`L-{(4Hn$zi`rKB_IvAXggNV8$56}I>JY*zI&!nhwTfL(TI?(!Y?OE-scO_Wez8H^NvYTTg(TBB2cu7+f#9NhW%V(8OZ`6f5_nHQDgU*e*l%1t&R88f?)cHZWu-XP7LTgH1h#7e77n(9Ku@A4H)Ok#AuWvRv$(U?&XLSu=E8{A8lp`%BRaYU0LVjwb)WZ4hd4n(H98Az>VV?Ah<H$G=NRD9qF3*p@S|6DJ9ntXhV_svr=3Sp}f~B`AqJ~3BzYoCU?rOW8gGA6W}O;fPUJjj~ezF#(m7dCyo4=p&v8$a}EBj=pDRO09{};DuLKwNZWEIh8)F_#pMn7T9+bLQf>Rw!AXy@GZ>-muQzZuu1{xMBiALS;#^Z*MKf%RgCs#TwOWh`gAWC%WKk;2|8X$eq&Hqe*5`*<d=Yn_bPWdbPi(Ua#w3NPDG=#=PW|T|0|`XRRH&VAy=RQRd>9!-ag62cIDP?~frb~aW2$w_!8fI|4mV&rUM+CT)m+>|v`DSs{Y(vYfQBo9Yzr8)Q3VfTx0!IvgzJ|}t8;rBVcx~JGdi;BCJUB2w=o&NGQ}e!!wR|kq%eQ4=s6xJWEFP#W&RC@G&#_Hh{W9^#F2WoE=Fnaq6rSGB-Mn$9yXWy&I4!rdRl!>MY&yQP617w0IR$L!mvx)rLMJZn>4+=S8^kMe)m`L@rS?N{n%M%73#tRE-1_mjL2e-x$ZF+3nx87MF^aeFe|}=zwf{GG}kzVjJe#i7znv5Ae~t}THGw1G!I7N@}0Ek>C!%qe8xYFkW90_kC9{+buV;kb<;mk`9-FU8nJf|Yeu?;h9x1ckC*E2O#&<w@UKiSVci|dgchl8K`UxiqUZhPPw!ahy2q*kRR^ucvQIfUSON3U{NuAcu+{a0-;q99HRB6H4&U*(Y<TNoW!-wfix)RmUNJtxz9(UI)~Coz_c)sJ<(L1Vyk6()t<1HG&IakAt8g9cbs6GVLg*%g`rZKEXR=jv=uM_o(R`MZ>Ebg5k(G`j!6-fE1J~n)$3-0Tt1T`qmTq`SMUQNwK0yu(2^k$vXU(3&Bp2<1k)u`6;#7?mHaKY7s~OE}<^JtWE#F*(AYAwd76JO_@mgMVh2=yv?nq%_SU#Q0Ox?JfjsqsI+w?!$z!xMvF=Hu;uf@%BdJM~J`Z<$2{rudhGdLrZLSU{YLo?qfaHqWEg6!_;m@^+LTIo|Bihzcmp)I$L$JQ7+bp)KV8NwLY5UCSxY$7CVRR=pQ@hZacoaAI)GFFIxO=9xDrAZ6MN1AGRWFCD9ig<@|Gfe+%6?|**!xt*ryks1It3`|POVQ6-{+|WWG?Tt+3wxm>XO7ZTxsB-xmMV3n%|we`;OySg?XFF@t|xTeHcw_<G#)?x(;1^#fhUukAksynigLpdV#TGqC1<`?1G|%*{&9|`6a8(gD97Ru`%FB^il)}*xH>8?<&!QH+;W62+T+YyLBV>PH$LWTxoT2b9s~yPo^#l)Q-`=+LI?3W*|frWJQzy338x9W1qo3}So(m-S;E{C@J8;Hm;-Ke^HE3Cp~;I(Wl0CxbPmb~QvAd{bv6fE<|4noE3iuHCk(+crhb$u1xvdI#14X0Bm#&i6wJ(79mcI`o*_$nXWQ)e+xs#Q6)xL5fZ9G{6F_KSX?Yau4wV~(wu(ckkAp0W7R-dXxx2JZD!Y-0k+P|CkG<5Q-=Pbmdf^(yns|%3qI2GsXcA>yff0@PtWIUx3o8@VS-oSBh|Qdd@@GpeUP5s%tBy>J^s%(4E~~c?o)M{GD(Bp{Nw~C$H)S|7=<g42p4ubE-8itWPO@h7Y<Z%#*mC~&{W`&TWy;nTl(|-q(av1uF$-WqI*eFeyAjL<{x1ugx>Ljs{k{p09T{}Dm!4nF(HbXt91G3e`y@6Cf(b6eKnfG|b|y2Dv(nVb1H`fDWO5{^Ie2j5;H044Jnj*z8jBh2W_h%~Nv=X3)8$z03p;-I=8k?3AVQU^{OKFBFVpcW7r5rH%(PyWZP{_MX2J2!7JtK2%sQHL9lhIsUh>sV!#9>kYaFMK;D#!}rKPp3{gVU8%bnG0BFA1N%LN!uvE=*FXHP{H4phMs%jiDIE;wuS=`~Kzb9SBKJXJSSA|tvzp+{jg3NUS(rfPae`n5@{E15hR4^7?7dYUe-@wj-qbdw}*Jwbp;ijFd1;zgF@2BhEGIN6iI<aEE&ua8ZA=#?1z1*MZC%xg353rWYv1#iFyyNWSZfSDa#QJ;<VY8<nZyPFI`TtKLrv=-;oOU|yFoLfIRw~q2AKk+JG(St^=058v9=+)JhH#wlsEx(+Ht_&EYWB_{J$-~~EHg#1V=jfUB_N+iBIL9C+2t92WuRB0?uXjAR*!v=$acT1&PWz^YK32Z9juesFHt)wQ6G%<)hVfzI)c7<nz7T(v<({sz@A1Ad7-h(Zj*Q|GlWS0vWZ&#L^B1dTvr~06F&!SE@#Xl5U>TD<Jodyh)c4<Z$EX*d2`m4wGVDofJ_|110uve@hWD@E3<Ynlz>c=+svYGzWWW^Z1KZ4lugwGfLauJVm<m}=h8w0hTi`2@n<GG}SDNd8r#td19aUVYGQQ87a{kTluCK*cI!U||CnwOauwnN(zW2A61qOiO0MvqDdXhrP2QU~g$wRz4)T){rj`knCMn0xmW{$&<#lgs3MXHbcKeIs*tc5+83&neUWi`omAM+%usVx{3(x=2}q*f{}YreXu-a~Pn<}@1_B~>gRWs)Lp`9}gGYlzwx^&zHG>qvTQs>=K_)$%y&igNPJ)vfoD@cr4YN4#7;w?_JM_1*lrEkWR>DQN$anf2xJWkS2RX;a-#U_$$bDS8eg{Z}4zct7P<t;_uPEM#xu<MkC^bo%TA4`FMY-<I;}#ZrATzf$dM(`GB7cu<GA{;vp-UFI~qZ^dQ~GqLfax{TO6AAm_`wA=!|Eh+4VFC2B7+FEE=&ca@4EU-${V?|CZaLz_KX1zWbPE*_448P%pdoVzFF;Lx@EAnpq<w1H@*5V050FWknWD|be!f3jbgy$ReeVI4AdwkUmYnhL-Owhi}RT#D1CirR}?2VVCo2LBUN_y}{-n<z*xFXVqM^~>>r-=$6O0pH)(s{9&M_>FmP)h>@6aWAK2mk;8Apq7Ho1hE`001!*000>P003opaAk67ZDnqBb1!sZb6<8hFJo+JE^v93Sl@5sxDkH$U%{#`R?e1_y}g$`=pmP+o5P`d!R89I2m*nYXq$~B>PRYyvDp8;-|&Ye*-4A$A(pA(%y8zLZ-!2i<m&Dptka_7np>&b1^bKzWBY2)cv~@gjkii$*>0I=t=_Xv$=0&oh>ETE>`K*qoh3;!n{BjeSW#?x+iOu2Og6L;dRk?jb~BsBHNAyNBjWde8`Z{VW#W_R*PT|SFvH5;crzWhyIQWJjXQh|4p?pq-M3aYA}dw9k=rPAPsh4)OS9Rm5*yalyq$ZY^wW&tuT<E|a#oA@B5Q=*smi2GX?F3^Nf{w3GnH~nse2nX6z+{~Aommdfkn9;T&bn8g)`+gXn}~e?3?>Se~E0RmAo#3s^pV*Y%xffMm-6v|9H;MPG6Q<h_(n?Aobzne-T5BttwSb4qL5ibJ*}<BpgoT5?L7DbhRh|kzxxS)!unRTaFKZ#^yd+Kd_7RpiORKJdmnltok4#Y9rYB)Pd+-Lhi|iZ-rSf)e4soWV1KSyYn;K$X6`IJr2xPG@Ss>fljvOM%1#U{jvhCrQI)$F9T5}Bde9<+g2GXOESDkz6HOBb)U%Z2Y+tPS2q_c@@gYnPN`n7>*d|oo9or}^6!`Tzv1JT-~UNpe!agBo-R>@ct9<xwJ>7`)_t`VwrFJATVV>EpfIQigvz19LzKV9#V9z^poQrh!U{r2X~PM<vn!vh6-93qYDJ4uX{3uINLsim=cQK0uwY=~oUu?-Y-;sF>esa@pNJ^#&KJz?M9X+@73H<WDYdanjZjt0?V9R|RRJ|a8HBVyBWbV_Q%06<E|#~W{Z0@;kWJqdhYdBnrg)XeFMW5!MOnN!_toHHcM!L##zR|*BfT)Xm1|i8B4k>vjnK~s%hm1WuU{pxf_pMVGaAYom~(h*6?h4?tyray6$fvkOa!FdL4hwK;8xYLl#_rs!>ua3ZB$wbg|Y%T$pyooMWAJ8$XL~}N_3(nD4&M+`zX@j;^L4MD0`UNR8T@0bkK#3hD#{b$FAM|c(U^GWaX!m1Bn+DUiq{}aU8$<+vnxy(c0q}zn;zM<nW%_RV7OT@mg#YV9Hs<HM^wLI%@ox!XF^J4DTuaNKqqAH2*y`0t3*Cg5p|D`@u;|0iFzSDu9#NSG3?2xE5qesRwJDmUlq3oee6okgl-hMN2CRzviX1Qq*^-k;RsBFl7sRn`Yv*L%LCudY-bC3yewpcGcUix73p!9=%oI(g9v*CR=7;2xlWb*|t{ed2+ToJ6lCh%?d!=3caLkzh=~qlQe{Y{+@Z=3n+j0sPkZJp!gbKHFTtJ-w$i#lvXd~kkj~(1iDEcKTY7^H$Oo3A$bz}V#m!cd3+=f)XPCKaup8kBs;#Z6|a)Vu_3Q2bM_44hSuYRQ6;vAVV38B{ZsNdc*$jr9uf8aU&{V+tjuE65fp|HpbR|*p$i{FLkvv_f*~?-Wuy!F7Y_J2n@_`qIV8#>rs@T*q!Wp=Bbhi@oQR=SOm&~?cl{hY>GX}Ngo~gjv&6UUixUkLLnn(Mj=aE%86A-R28V_xl>D<)J=h%OfzyYeQVF;c@_;>v<8T~Y$xq@$Vi`fi%qJ={hifZd?L190sGcWLN3sBbr$NH8r@O@~$`o*zNV!0au%Di|m-QVn`qaf3oIE~Ed6FRC8moK4K|$1VD8c*>-5>R?$cs58FjPxQ={g|E_bGe8{NfFN$7D5i*5~2WLztN^M)moroiVWU_%MAP1D3~!>FYH}4I$wj`6#$Gc_7fq3-nlSj|~qJ1nSd$wE9*^0^Mf3qozBD@ceB|rs-tD5ASFQ2`Uye;h?u*g_0Cj)7Y4(|IVTLIsbyD85r4JoixxLuTj~h-uA@Eckay`RaI))QNzoNqEcm1q@$e~ud0HFE%Ri#^ydcDwfM7_=&ue-?{67;^K#jNo6$~Lxl<BXwS1oV`y{PVhoTSHUg-gIPx(X-bn{BmBZIRc$p6c#d6M0d=%XGFI=pm81|3s7N0p4njGd^vJh=vZma47y?C9!?fk9vr^cP#AE4m58|17BRe!&^)3&B12K@J+1qcmJ0-5{Nh@0*TvNXCRasTT#4D3giyS!xiI<DI*I{c-#Im)pzx+xwfF>m+p@<vekP>h`;MJIbZt4sBC&j5X}5A5(6$R(e{i$vx0H)t;Zn5eogwU-9qu=nII2Y@ZQwh3h`9_oBv>iz$#bJv#c@&7N9SeMzI~@%11}j_L)6JaXJ6|I<-1QB+6Q@tNn)H#P@!4g39o<$tsY^HmU<%?2i<8-_XQ2=nBU;P?yuyP|*B4z_OYMLd!d<{w>K-Hd$7`ERI9-=z>bwf8j~e8D`bb7Ez8bb@-}7Z>pydH8%kOjB^Y5h91q*{M5jJ!XN<Ie*`Xuj@B5`Nljwivb$M|H{57_9EnVhXJ$|_C@KZ10y&LVvsj<lY~M^?QnyeBvk^gQ_@}JJqKWFL5s?US{Mz}1@eRoRDrVyqv~fdPs8wK9zG1C$M9Q|JM$#14`z=leHCSIIS`MOE*nQ;dzuFEMEvh;g8ga95Va-CKa<C!e*9A9BzLq*q<GZ5-@wT?&z|o0kB%GARN|+LPGhp1yV<7do6dwLvS8BP2lEebW>fcOH)8eBkh~;%4RTu)EpO;|JIZWPPz4l4NUm!`>FhsHO9KQH000080000X0D?=fTdEHL0Oc_N03iSX0A+Y^WpZh4Wo~qHFLYsZUv@VyWNBe%Zf|pRX=8IPaCyC3ZFAc;68^4Vf$GDJl%p6sy-!Nixui*2_nJ5pH|^bMG#rS8BsLVO5Tq?N^?$#80YH$V6z8UMo{24yi`~UyU!Db!;c)nUkymArWhxOnRmN&prA02%yi(<@%JGX9GEc-R-K%7>D;0cz*Q=~nd8|Z|%5`4oDvfnG7(CDOqLOA%>Aw}a2KKk9hIfI;)vYR#iz<=Jq)N9+RApUlL{SPU*0NIJ;QNit*J-{6`9O=>6y@RZve=7KeX7$^Y4}jqd5$zmahL0=RC4Rh<Ef}NsRrFMHAo7e2lWKZp;9oTY9lML1r4$T@X6#-Wk#6Vr=T-@-b(!u*{zClt9bxP3<p=%0*`;Xp{W8@RhLVeJaCn|2K@pO7#C%!;wo2K<MdW_mQ7@?@4y6CKZJwfa5xyO%3>>`XjKCf6-6T5?uxPkWt>3`O(H89XHsjJjUjT60?=-k$yg1{o2{%iJghqGa6;z!8@a=;fw%!f;w%PsL|$)q2O+^yJ0mc>P1T)~wM=ta9wIc%U@-Wn6FmajKU6;ZrL5I>K##-~EQkc=uWr)>g-izmep2%^N|UL86{Ck_{pKsIakM||4^vUqyG+ejSs|-H^!!;6J*qPWmBT!dWhoEhPw}?M6}{V)1-K4MB>rM1hTs878lzQ*M3R6*#Nb?!hNS{B;%s;Ty<>%9zC_ckn4MsU$VgnlYVRO4qBKvdC>m*%t;TF2veHV7+!y>oM?#v=vtNq<JS$T<@d>T{#^Rfq_(zuvS;;ILp?8F-UZDnRv>yv_bAiDv+DzPdPe1%AWvZ3<RYD-VD$AlA4eg49`zS~#7`KVZ9w<W~J6zmh!*hQThy(oFU&7x@Fj7mlZ;$<@cn0J+g9Cm%TzpxbTbY3dRxAunR0)pn$p^+HkwQJ?41w#^<_%QfT~d>Gjog4SI9$3#BR^@=pEnOVk`~)qLqI9<<Nh~KA@ctw${Jz82^I84?jYs{19Yl7C(pC%FlvRIp1OdacY|{g2x4SI4k|!^WneKFi^+FhZ1F@%4y+bD|3j699?{@HB#>y-Y*`c;&nQhaP|d9dzW0*5+3)CePE8JKusJ$3a3D8k=dS3x=ZMEvehK=gF;tL-`PEBpEPPUCEA*VY)+>%Z;z;h%`mdVx{@geJdkPZmS1zsATPSl}M3xGaRcz^Di+kfNybOQNS{~EV-YhE*fy!5xIgc_9V=6|Z<yJYriInk*kx8MHFI95Ze>&+8oXwP(Im|8)Ck8+giZ`lD48P3#NBk!)p#J1w8<)g!a>IbVlU+80s@><;sQZFf4{Ghp$(!dl|9bp#^5Xj60*puX9z7<l00^);3BZ*lI9q^I*N9uNcG}g9i#o4DPULTpp5AI;1E48~%nAjxP9WbUT>lB@fC&ZFQ^~~+(-<O=>Ok2bByJyaE59jGjHG~4rnyQ&Az1S`Q?k@fU(P7H5I4`?P-5Xasu-`}JgbcYhr7c(sPt3g3aB~EDzho+uBfvFCYY~ajtPySsf5;gHZt9Mm>G)CW<kUd^ZWg|0By$F3`k)H25CxzbGe8!$ma)T5I5@H&mb*cE|<8cV_Oq3yBK#Ii)XX84(BHHh6nL<atI`hFo=_eLtiQ~+IdpLTV$$7<FV7y>T^!~bq)x+m>|Z$D4wKEnQ5HUK<2~<-=<pQo0;g9FDCTdi)N>@8F@yb)$ons-i-Ha&}R)uQY9n9X?$;VD(Y%iS6D6<y}7zcL%q4$cMRi{Ho3GB5QzyP-9Q)Ml4ZL9Lj*trP%HPa1?xaX!@0MxEqX@55YL$N7Q2N%ohCSDHpv}9k!JLn0(GJ|QuZ^IM}k+#%-53SjnA0tSOnLP;}<^g2zuY5uo+?+LNuL81z?HYf%}R6;Vm1^THpEyG@ZcA*kRg=mr288cv@p_0A^_LW1{hiI;}FTrvE3iTI}qZ#Np{P3(-HJzrfH?t^O}|HsFR08;H-2Wa#f^45z{@{_sS&(P9Z!+|zH^Mo1<B06WY{E>8!(R6|O5A!0ZO^zGtvdVpMk)`p%Z<rCX25Cm=!6?q_S&jzchH^N&uFzpUi8Cv+@vNIrs?C{jKh@x2j4xO<!55&5br37Qbh#DLh3pT(6Bjny8$}KE^bPJu<D#hYycz;bV(z{d80)&n|Av7=-*64;M1IVufhh8!9YDll3!PyLZb#vok0ouJv6vjCW2XJUP3d7M!ZGeHUx1-*|2J8Bu7K#AG<pcDrWVu!bi4}2NmjGjBjA|B*Mkci8gKh>ot-#6B{7q{H#BuSZXaz-wE#F5EYdj110HOEIqwOvq5MoVfyR1x-#$Q(w;vYh44f4O*XZ^v`qn@Yb!4f(QRM!fg3T&Hw?~XZI!4Z`SfUkCFirbDLi+7KNgdii`jn0A1R^ElBdHww7n^y)-*fz>{cPqRljN78$(d8@-{r>v<Yp>8W{jLiM-?A--!sg@a_7~1}C$CY?gyrD&Ra|5l7#Dh@wAw%p^fnLsb?b*@eF5kj4_mD?$7~A=WTbm&%)Od9Xy6MCIyIL#Pg`kcVwb-Y(~q240w?850ZHszwIxY<;*pt!{eju_%2r@{q3ii4kQ_a02eQ7CyNLR09bqN=>i9Lr#a);$?nxLq3e6KGDlQWE3+u4mRUFH(SmaIrW87i~=IcXE>0jyUAM%fRahJD7(xfJ6$5XlA`kzCgJZ9#r8m@7tu1C|LDGD@d+#tD=b9g|FXtbV~gDUu)Y2QW9tQLZ-({DW<km1OsBVr$|Rjx{Q#Td`3)PorrZyx@JIfXP9tTFAD=S6eg#f>ix=d&!kWg>xUQtZfTBnJHSsW*%j(&ELd*Vpe~nIjHND!7{@!P{1J_*S)!`oa#BbL=A81?ubunn3HKE}O=I0C!>6MVYE&qD<{m=(0I7ke}P2YYq`7gi{l5Rv<g3GsMU60RZY#A-BtPT^F@BmTz{->~57^G_%*yr=8go!Db9Rv^HVE^y9$akKnyKe>1{b!7Xl*;%T3l*(rvoeyJ731QCZ>ai_|W=ld}%dzyegxYyM#vY(mXJ!S>N+B>G;W(D=M(qc(RpHWprcSV_K7CD|@F2;l7_Q`BQHS)&@F)!P_DjO!OcSzyWp6PFm5p9FZN|>$O(in*Oiodxf@)M31?MxRrH-N~oTS)ZA?~9Dws3;@n8fJ-Ln0edZ6*d0Q2(X&@B23fNOK&FEA_s_bg4(&}L>$k=)2z-l^jkEhmya>zMD(faVkR<n>}dqa2aM|CvDtYfg?vxpN3BH{?f!hoVfjE2PFrN|Y|+QoHLM++C>m=#g~fHTEij)|@RZUWpZX&{VmMlL(%kPl!marzE7obHN2|2d71VpksyU>CM}er+c88h1Zcct7NWzP;OIq(#If2G$3l(1cq;e24x!IsPIRq_)|5}-<_V)QNAKpLzc>=O<m$(Ad_4?hnPh7S#=j`C-@#g43nCIcD&MA@O#+??g2R>qGpXhp8(;m0?9g)Pv*)46p!GOiQHO`iwU7^9gx2m}t;-O)*A0JC%x5sug-|_wfy{3cHXJ2ShQ*20<H6MeJXy5``FfVhk42HNZV!5m{2^B)_Q@g>Q?GS)LvRy~ISXEoO=cNs41xM(G-qa9*ZD*59GxpQbSTm1@4Bo<EUNB&Q*m%DiWjiQ(pT-pG6}=ceF=ritKxx@7M7lAO@h5uwk*2}i?lw@fM?3MIxB$d29tq$I46`i^VhTEpjk5G3kd4P7?j!h`eWZE$%sw%*PsHrgNoJpVn3+V)v~dQ`KN_L7e@8KBpeF(JclqqX9!`^=M0|%Ra(9Th+;Qwh%R_|sO&s)Y0d6}oIi_QX?l1-(TpO@{Dt3b&=<%*B5L#v6QFQ4AXc3=R&~RkHNJ82#CjbLt-n_x9PzGPE(yXGZ1?F7v7sP6&P7^Rb(D|Q&&a%#h!h;r0V1)PRI#kRzv9xGlB_F%JGncMNlj?wMw=hf+BB~0^f$@eKNFAq0gAEd%LZ;^a18xh-GSvXLXzLk)gdbUz{0OO$PAA(tRYns~-JlBHIS716Zn7s&c#1c?et-1?9ui)^_-XPZet1E@U;i-qFueCunH%+tVOZNg+?idM?AS%l9-X;sTv#9IwA5TILmyZbb!i=e4@1+^E_-u~#}5d&&8frd(P56i7JR`5Khhl=F5PwlV|VF6<_E(9v!`W0zgW<4cLT3^b2(i+e9Pv_ySjIjU27Qm1&0UN(Seareb<zQXPqFP^J*_lGQK#<sCMK-a;iHV{Cw}gI0^}x%ZP#NYPi%kE$gW9x0?v#OzkWD^3r|r0Mp3$<W0o;j7(@gx9G@giE^3MCDYL-L$zpDh1H)sT^BHu4(#Y$(_+&EU7M08wGhe7``m-@jpW8O3#v62km}p3(URXM_53n*y@JIKJ<1_U?ka2Ps13yn5xoDmX!SwE*6mXhLB^6#yLT)Dv4QBVV=y0OSd47BOfyKG!SVe|_U4`wBvZ3Y5JVgV(QZ>pt#HXAkFQ_Cuy{f`?VLwoK{^X12Mw0Iskg|>QyhUTWbIeKzIy*3!78v6s!)}7Jr!HX)F9Sp9)@911=9tGxanIPCFiIZoi$+vf-WCyhC8z6$_N3j#B8I%ONeu`444yrU9^}_t3`;lPYEnQz>!4CxT>ihVd*pWv@smhwgps3B`cPSw^a7kgs11{C|gdAOgtW>dV8uMeHD7{Vq;rhP9y4bWu*@>n=6oHH3Oc#iy*XA42tcA8bD9q-qW>Ceo3HrK6$dRcSGC~x@y>X-b2g5TRh8~8?WYN3<ymxkmr1f&?4x4jT6C5s2t`Bpag)%I#<>sdiL+Jtns2gWApNt?`EC(U2>h{w`tzY>8!=y`9T2ztf7o~ZNbpLjmXYW!CYXZV{h*+THKD;qZ4(T#%g91<ZnJF_#FaTK*dLoMsrXdQM9*~6Y*q>fWEa_fw&waz?c5P|D%=_reFpjJI+nbfOhgWqaRR6@h4n^TdQ(3xeRa)-U|)+O8ucmEoub*_JPIn%_fN14WgV_BmjDN>v7Zq2ln_aXr6E@9zZ_U&GDT5$}hd7c)eO>Y2NxT1wDYZd$jtP`53R%CNW;+kl%T00E>amC`UOLa@pHLtV=xPp74K9fU;ARTYFjNU~k!>&IH9bMWJzjmHJY``tE-@K-gz#42XRL4A&d;UlU9|YNs)~*e%n5Gx@f`PIA;WFFfnmcRctnP)h>@6aWAK2mk;8Apo&{AV+*5004Pq001BW003opaAk67ZDnqBb1!sZb6<8hFJ*RNY;|FDX>V>WaCy~TdvoK)k^di`0%58WQcF<}dtdS?ckA^!*^0fp<+W3ll6fo;7?Q9+42*{;uA-{@8u#_?liaVnXMn+j(mJWSs@o(YFpr*|o_^0@G8&CuRZ(09%S8FF|MKeX4=1nRzkYvydiH$ke|Z1ljUS|up9OKQB0t^+YjqqZRZ*(kFUp{di!u(2ndfm=f2ET|?+X9BxCmaQsXF;fkR(+Yr)a6uld6o9;w0Pv+AP~M<R~bElVYFhbiXZT<!yQ4ZOSq$E>2F?ak;6MvrunOhT-`$-~V&2D}X8Us@(W-;g=ib|Mc$d`**%xd2tFH(@^<gP^z`gV^#2=dijeAOMk0U0#o>L>I1HytCh-?w9mK(1H6{5doR>CD#!f&rNSKjNG0(S1*J;%epLYS+jk#-eD&@%Q7qMEAfR=c$V7qLyZ1l(ufG5K%@5w|_kVly0|5uATg`)90UJy?OH`>Y{J=Ma4U$4xgb7yD>Da5%qRKLzmnxe2C`PX=*LnqF1b9h(>Q?T#+p@q5O_&H3gk=>Zezwu2Ugtr!!9xCs@={-`H2#D&Cak~v@vr_)=h4Czf93zEZp*I|z5HY0e+Wu2LwX#kJiY;s6mhz)k|0Mr4_Ms9k&;#UurD{j_6wgd^%*<mV@=jr&BTInyXPR#hakKrnuCH~t_oe{SRJfG!PMG<5wRRtxDfbZ@a3MN<`sB_y1&zYP?dTMLWO?aU*}#!K0ngCG|@p+_?sZhRO;_GF=U|tNZ<mBuoPI%9$?N_I{pTX0sf|zCri-fWE;fkiMk1r3JgNLu5Ll>WbdtX?k_Rpe2+C!Q5;%6ds`NMl~x5WNRSsG{x--8UtzX;tZI6ksx>gaq3ISGIw`Nz{@>qxoD#<*5aM)=E(u21f)ut9!G7fz3NV2Rgf_>R=mFefw4$<YaIFfj++eYnY7^YV!0{$7Vp0h^%u`pT51~!hGy<>MY88hu5Dmhx%7bu!^6lBo8;wSux61X_Uo2KtS><Z6fXd+Ic6i+LOxY$VHjp9v`b(ix`<yEQ6oTq1G(sv>ky-^+QWE8)&8kYnQtPB{k>u4P48o0pkp(3UV{5@^8>uMw8BNHPzlH`NB{;=OK&<)cdoTo=O#P4Q*NPNFjeA;cNr4r9ni<`<`1t;(A0Ygf{^+w6*ob-S;hvwvT_c0StDnAk`{SE$48X<5zq~qs`V4T#o{xW{<>}Ma^V73XJyYT7+4<8bdVYGkS}k9ks_5y9<tkjAKYQ^kTD=HgE~97X)b`oSaQX85MWnubcK&?iO}Yd9{oVU_fB*htf2e0Ko<9BJ`SOcrD|LRheE#w*eDVD3>Dlt-v*0CO)bi<za}}*tXD?S@M8Wy#mr)o#51zj~efsqDe05GkdB4Y^fbruHoR;dyC~spd{G7y57SdHDqAz=9Wd>*;AQLKw0L75CsLGO<>;h>=vJ5AX`w~JEKtudm#rbvd75FdDE&uIwl|(3kO^M5@Q~($>a%EP7BuBK<lw@lIDTWeGNeM&bj7))tpcoZt^O?6W++!$7Tre7CfI%SEah^>5rQS~=q<G(H{<)5!=tHO}{!uG5fe7bQKT|p*#tkBf2rz*1%}q-fHfpxq6O>$92L-IbnjVXoOgb9GQlmi|#bbp!n~K0wMG{1ND%&wmOBw)J!q{vHLmGOe<ab@6L74|PDxr>BO~gQTAirx&K9Lc!xD2^0OM*Qka~<<fO2Y(sBxONa|JJ3JPKGW#CiI_**J(ZR)>vRAMh6p8sSrn7a+IVpi6l&#GDa_<L`ljHoeP~rj2UcY8X4fkQI=M_AYn)mEQdU3s3s<BxDON17^7@rUa*R%S}K(WvI1ol@rYrt%?ywmt^fv%>847B1RI@ki<=lK3!OayuVne`X!UEA+V^do#!?23zgAut016vOy^N{B4b6gD@k5~^{;6&-cz{J(!Wt8T%c=j1QW+7i$_QF;8*^zAZ?(L7Sa6-2abZg}qwaI5!AqVkYV;!Ib?dFpROG=~1Y@L%#w?RKHJm~PsIre+o$97E%!5_Q9o$~1SXKp*#k4F`z)v0*Gj-^p%*pKRJ#cS~`6n8BSd-;thTiUiJrhA|GQoL8G|A%dy2_|+p6`iT6|>n2ZZo&=z_KMRT}+Jxog#=6DVE9eBU$myp7?I3Oi__370F`=eW+MKR^XvMj2sHDgMGluQU+TnBa$ec!Xn}Urn^wm^+tiSp-gfXs*<O&6C@RIma-iQnIT?Tt<Cu12EbT51o>K-4mJLHMB)-fi5Vz~SLSI&g18Z$Emg3UQf6y;T*B!~ln{km&?AwZ)C~ZwX?pNjlq&6;tt=qoK|-}UYqzXlzz_83(IWVpeFV~=+&p;Pk)U|c;MeRs!Q+N3!Om|;v+N*NG&jvI6N>aMVU~e^iib5VKh{f@A9TM~#BFfYLstHfC|U_cMkY0sT`>sgI$oP<3yxdMAY1Z2=wTJh@bOwvBPgUCE5_qmbl^9V?oA+!QY2uZ7IGy8JFTNkg{*?qIw!oKt>8&V3Ml_tVc`fsWt!9MS8>5Qcv*e=#E9&mT&SbWw%E}msLL<J{i`G(+%;){e5-}Bj4Ckzv))=43Hwewa~UYxsG#7suoz-+@ET<?tFw?NN){s}?t@asNU-ag^aGh3nOl%A#i9i3ELKF~Ak?gv@R!tf!eSW9dSn&Y1@LZQ5aE$ht2ENwJq&i-TgWH9wI+ots{-{%(iI~xNr~x6Y)QE-mwPYLYl59+v=U$mLNOx+06K4~0RLsVNrUPW>vBkH0sS4(%*%48Vx9sxgJHq~E}6{a@A90gHQ9zdkHs46hz5Xvng@(LhF?>=NFxMg)3dG0Qp)$tH7V19pfajhT$S+?#%3)cwl81DGIcR)YQNEfr6Qk*?GKE6%EOx~xJfWSBuQ0qz4yR%1(P*lYnkq82}5Oy5r1Pw#1HVGd9E0oREQ?w&4NUOkMxuyxq!GOO!jc}B6cB`kdx5-Xpg5rMA$BAh^>8Gvno+Lm^0#^kaY?e=ayDY-?GhLHRT&b3Z){J*eGZ!X89GQRtvSJ{wKAp$VN~&0^cRukhZ`YvsaKtfrkcp;K4P6CW7gJ7-no2uT#Wpc=D?4GFOmi4_%{Wz%n4hlvPX>kW{V0fXoN2;*#*jA0KaCQNbMwn09#b<uR$&Ygl^3Epf3_XmlMHYzDmAN%9H`Oc22ZGzt3afprN#SOCgyK-o4g>jVy{*EmxcHeVTKo)G`bBSzTTN33XRo_A1o(hZ`KB5cHpU(@{L7sifg?d`DO7zr+w7giaCzSrJ$3ZfIEP($;AA3_Ffo}#M=V_F7eC4q5@|9A+*0Fac$j&<>}%EfD{mh1w<vA0DVa0Nelq?bT6@J|T#jLF@B^nqhb5)?0sq3R&XX~mLLtvCFH1yc#yk}G0B%M0cp4@?SEMvkPZ2Y1ke$N4SR*kXn>py`URF7zrO*$RyZfGAnBg8}0#tre7w;u(>8TtSE<F9S6Z(37W98k5e?V9LmYzrazs1gz!<5_IjAI%b}$@|Z0poS3a9cimPb!3_m0>}-@J+?^YCqhj`0OR)Y;%+9rWgcN&8kd>PKzHL^q{zh02R?uMoRF(3Os8ru**0834d)1xb8e$4wuHg144_aPWDTK&SDoVUL@<;8P`Jo2pv1GxlD?vD@mC}^NAL!TH#$-@nO+3vO1uJo*7<+y)KR#hb4M4AeXzms8-GLlLZB6MtA_kr#c6{nzr9KgJVtQT(|M1$oA!#ncxG13bAQn-y_eE*&Gd+657v$Cpm%_s2hygGzLZf9~6&p^V03HO!e@zX(Q}C;GdLc3OX!O1v<eD_5WQ!;Z6se4&nv~OPwAjfpVjoUYMihUd2J1cJaR{UdRVn6VraiV%^zl|QAH{L0ZcEBJ9A_racXMVsM{4CS7GS?}v5=^r{=vtDvOStMImC-Pwc`K7e}|R9aPN4b++5$GRt$J@QNxc{o$MsqG_~}PQvzvHUfax0D|hL&+BTprJI$Q&Tr?jT(o$-s(pj9YC~svkaC4%qjxErat#&R!XD*)gbGyl5P@RUv2VfYqxx24xC@l$Tp~P?}Au%i9MsoyJj3@1BQ_4dX0EC`1@{+UhM4FFBQ-3sRPt_TFR%H=*a(qWLzc6sFPUnmUl@T6NdOvAnAJ!S6OCtLxl|pBZTfn8`c+&LDHhEAmrdKVTMol<6nhw=)aN4b^qt0$cD{$#nx?#+rjuO=Ld)g4ETjNGHgS`k~x1ts9^m<vJqgT;*7u_n?!#G&S4o7d^uJ_QbY`lkVjWOWe;>LjV7qc-ZeH^T+?hR9`>mCUmi`<*B_;KChjtv}GEyou1C%jcOS~<jWw3kB^bgJfUnQ7SnP986eM6kS>M}RTy+FAjT1cFnl?#K>>NWCTKbHKntact0|0J--)czV|@vMza#?gJ~xo_e>?IPN{E=6UuIsFp5WFZwFY5Uo~es}Utx{X2noKe{j!gx4_Qv~0hstK&K#TVn!#owoJwZWO}$L=h<S$ZKYwY0*Z%kZ2iB8O@u;{VjvmTzvsR#N7B{FOt1nF6CFMsO6T59U*o{=EwiYa*e<ro~LyupppX0+_OG+j1AjYv8748oY`j9yc##AF|CMRt337ZIx8GO235W=&4C_I+~(^$?8sY%Xyb0~bXkTr&Q-LqUeiK6u|=^7&YwQ3`ws5PJO;|{T^t0~-g)VOOLY#ug6)q60GWNanH02Fn*NQuTM|$F(a-5<0>>tVPdRSe^~r3bZX@{0MLC{yiD<{R_{gr;1zW>7^HmUT*zG7cD)oQS;8)sj^uPPo+0tCG$NSy4UC|^LkNKXLZ!GVVo^}_f%xzegbsMkwEWv`$5)hI@#|PwWL48u-*qjtFzIf4BU~*_+QH0=i7wEA8y6+tqaA4dUNvXFq$J^kxz`9;Od;0W?XOmiT3>xrX7vw(`U5DJ%Ur>%zTx$RLPkvb;s=kuHsF?b**=#=b4-~`{eh#*9^dgT=j#kep9aGh$<>Iw{+p$4%mQd~+{##?lwh94#H1!)v(?u*4!%#rzs%aae=(NFyH}hlv-zY3)6pD@)YwRClAb&I?9+XyBv44SnPyF+Fo8kL7%=p|uG>+R15jr%s)zz^VJ&}|7SvYRnEBWo<AoF~3>7TcXRZ^(OfSbu30(e#z|K9I}n?<I0(?7JH!yifc+mjq(0RA|0BPVMWhya2G@8FKuXf%awl%c#S^%M2Hb=FOUJ|!%QKK_=oxSF7ULO^nua|Q^o63+bHr9rXPV(#^cN;q<tX6yxxZ09h4gmb7rAxqsCHQxQ+biidzLei(oA&UG<)UcGWjB7%x0SD8V+O`#8^)MbLjGahF`AQdeFedE$Hv$+U=7SxCB2Lw$$!@ITTRz1Cy&GPOPeNLgj-EKai8NyGx~=<PneA!bhA;fU?#m+BC{Tg|_{KBXgt6K0C-2F$p-1wZyC&+#ou*y|BD=DI(+WHVBD>w)khBZ!m2W~R*d0R0$t4PuI?@6*PUQd~c6uo!Aj}Fsg*Q&e&}vr|UZ{YSuz{m!38MS38mE!fMc_A)oxpHI!&cnaht-64f|!<*Z+*=rklkpoR$l|N6?fR6*NN>6Byp3aZU6*Pox7)eGNR!HOYipTJ3(0xCx+o1t4B<4?B~>1w@}rR^Q-xD2LE!~OX27xpNJu~V$v%eKf)ADuFmFN4N>=nzO*Cut&c_5+Miwn2{6n!pg=u~kp+rE`-oCIEK2R;eGllwHA5DKj-3f&*s;G+vgl2o>wLT#iJdd2?+oGmZCq^W@WTaSF9Yb_$!!hC8qD~FB6%Hl2`0U0+2pboje}{rv~yjZjz>paZNs3JyJ*wLI@Fv(eq+xi)k);<bPy3elQ@jaWbdZ|9T&8Reoye{%a%)873SDqoZADVZ@-}<zXb1{2AzN~U>sAfhmU8WEuFXI)54%nT5Xpq=c9H7sYE%T2v>9y(4k@)@PvWU_|9QeNPm*60P@H!>C7F#gmP$>Q1Jcl<D!r=d^8lw=xir}s-0__$Yv%$JB%FjnO*e$p?BVw&);romyZ+WmSm46yc=#a-u;58Qe4yr2gz6V|G|e3=S^!XZJ*Wn$QgyEyNDz7v4<7OJD+zF>~;IZ>?5>x6^0tO6YX@AxIH0|mrL7-KP*-~*)(F7(~dCdv8AS?+cwsE_3oR2@XBGxwpZ54Vwjm?QuYZLyR#m*I*69M?6@yX7R<jUv(5UmgYDme)MHyG0gZhlg<JPJm};qM+KBxHdleF56{;i-n9qTjbJ`U;q3kBZX=48x6qPP4I<%uq@3k<HO`xD&e%eDf3$l<@$$$e?1BOx~T*k!F<7_e`bs>bym|`s+oA<P&pz{cj7E8_t4JS1XnAf^;+8U8l5Hbq>OlAn&(=q2hphzx9Rk=ETF`CffU)tYK1|lv_PF?be<{2F=A4|iwkL@HCT3a!VH8&~@=n}xIkz*z1{XvNyfMmN}ji_KWABu9gsYrKAY^>{jF=(U*4CYTM=~rVu8jSB~fZyf!qkfFl%z@Z+++Z6=qxocR{g4hqR|Ih0Bax{6nm+tGNSj(dFa270&5%R%sWpR@`;Lg)`e8b?g=!W=QKKymbQo3&SGHS!=FayE_4HjG`8W-l)`oD4gXCyW-JRUUKRF*J%3$9C9&s_X{36KC2SH4{9*->ez;q*UO<dbUa@bEERJYE2OxC=qn+&R*2khqmk=8N9RMA4DrKh#Ui#f<2pT~VIi1%JkI_mGMhIShU(HxRWw#L1df2bQCa1{Rk$k5{qcTk2}TC^9{wM9xIuX!)V>yys-Y!WWIlQ_ISy~Wsy&+fV%w#e~3#+{s?8TYvBvOL(#!kc=7abXV=((_$heyex88rh)4;V(oc7q!n(T;3J@vEI0cz~t)U>~wx#=j;1TZG~ysb%tfTfMY;}hZ`L=$*f#&7mn8!#SZde;?M{+iauM|^KO9&L<Zv!KI<W^F8vWAO^V<~|9qTsbacH7@^vv3jPO|2X}Oc_Oc~A@*{{pLY|6hJ-9NI3qn}uj)KRfte@`in4Uc`Yr>u}83mUjwips7@*2@ckvTYu@;(jOKlX}$`cRaxTj8yli<Nr48?b1Kkye9c&{nSk!xHRr$Jn5rH$-+)+NW{2hdk1L>C~+iTl@6pthrF$p^W*A6*A4B(GXu-^SGBj^=UaCF?kuQ#e@7qW0UlXD&VH5J=g%7Jud|%>V@)`f56d`PEQxT+GhaTUK5bI0hP5v11lmEB?e%P%?elrAU;VXmcL$(vzejYMH{k7SPM!A~SU(CahyVjEcw4UCA3!@i4m5M)Kr1u`C@t%{oIc<SG`W37!riGjusV~0EeQEG8nmpoV;<MA6c%O0`3gaWK{fq(XV$DiIhIQJ4)x+ZoZxHtzZ0$2Zooz3c@D<2PNviAXGbX7CE1a={mDO_47G~UN?M$7*3is?Q#Z8bzy|nqcG~GHd({ieP$uOloqOktDmB~?HkOC(9!aAzx5rQ;L;HT((^O1)XLGEN*XvPN^^>b6Qm|X_ogoa#n*Kz#Ml_jpZR;fGxKz3>H=+j&#9Yl*aT*0lGR{XoFRy;y&7X|_{o?A6|MOq}eLUya$<fctQLRHKU0fY<*~ji|SgowfHUW%pur9j*YmIWC$3d#qin4o3#_Gb}X_@=Rde|?LyN3_t^L*C{FdtUW`J}K_uh(i{`w~`2nzIUR6T^n8zZoir4%mqGRFz=iI_lFyZ5)gS8XX<o4Z6vUKOa6FtR`K=pEPk3yEU0dHsBoW1fE2n#_g$*Z;`FP|8YNkB>0iB8!U>$d%K;io@q~v!HdvUE+lP9LWkSd>zuAEP}XUMTA)wqim`O;OM&cXpzGZL(OYcT-}@M{NE~Kc4(?|xJEp0<k3tN6cm_2~*ExI3(j8_sX>{A(JM4qL)0V5PqxY2`Y17+WqTs7-R_^J33rYmgUJ4Ys@o6OKom$kS&~X~bJX#s(T7da5PY4XxzFGhsHUD7$`r;H<S}mZynr`oD9_CX}_5mOigc9lMZwLfz%lC5TGiYHRoK;UaMoqj)4$#!CFquiQ7k7P4G(6C~BO#4j;G%5`4<QO0mv%w8K0|W>&Z>DHuzUj^L*Gtycmm#X86<rNpyyJ!^J&+8=Cp=gj!vT?cVLA$0j^&LZ4wyA02LGV1Zt)oB1a7j7(v%9_*0|KR0e#U2l<Kqoea32u2<r@2feyTbvzRnmsZthQuUL{^}?*X)qeCLkpb-unby8qxxDmGACY5s`mGh~9i8M@ygonaq*v;V2U(A=Sn~O%C7*9vGARkG*6sB*h;A~qXs+RbTU6}mQzSSctj6(MZ=1iN3w)ZJZ&q<$l*e?L5^4Vv`nWDQksEq|5Z|A%_s4ux4vi>!xlt4T@`}$MsauK|Wjy=I%nU~TY;tuvpXKy{U#P~TDP=K?ezzwpsjux~Ke92(Xq4)aFwA~z&u>XL_tYhs8iETV$<GKZ&m1+{gG&vO*cLH`#$L0=Ll#enYtlk8rIs%zMP?0>V5yQ;HJ6y$n;?&aw6uw*R+8wSHmrPqlW8YcMp4J55@R~DMyg6n&W4Z549s>)xun!QtZd1OOT^tWd;<~RQ;2GBRJx>wa{hFX?;A{c;$(-)+)8Rqo%zI`f^`*v^r)RdS}j??`$vzqyfx@CG1l&-X?F~@y;~<sMqhRbUd^h`dj|vxE0IQg_#TGIZ$1=YvZg1Nj^@KflBV7@UqNBd^vHc@@CczR$f}px=i7Fe;|{pMjlg4qkLZG4FIjFnDeHo{G@{dNpvWYi5Cvy^d<e1OfBsw?AuvSCf24ZC5uqg+TD|@AyZ1l5dHw3+n~BY!droLUIq^}+;@uuum_p=tHJbQOI=S;vHqeRDHJ#GYfG!q!xS|zrjs=(5)Gz5S1pKaudSkbl32;QkK!`+wM0`Yc%qMw3C5+97r)37UKgQoDfXbHFJlDw!t9|GzvUdB_X`go6cf7NsBSF=5%j)=D44_kG?=_<Xc?`cyZNgxnbv+--lG$gcUXVwnJh-mQ8A{LnBZnkMcCMIaQ~Jb|w$wqty6qseGC#E9)_}Kp`QW?^7u#NzsUK|*^FgPi%%>iyM*Dz2<*4)P81^L9h;QOxooc#6lD~YIHulxrk!Ox?`Blmn&?)zev*sHe=N2`m+87i1^h2K7UI0*Cry}+ug(gvNUWN>v$l7q#(ewe^C&r5|fVv{W&ug#ZTRN3SD$Zy>{4!opWf;NiRnyft5uH~36lcd!+9~`U-e96Yu=Xj)J}9fV;LiO{!a+fK@~D9{DxZ#|ZUefP0XkRw#sbT<TB*GDH46HQtv-D!TdAVPL>HM9uEE4o&HbLs)fAX#!QLfjHJYHPV?s<OBu-sSO^FQ4m&3u9N<Hf&rUI353n}XFw?1y2WMhx%bVvCX_fV)a9E_-cQy7bCOGe&9`>90B!K1NvM@I~X_F61?jjwbe6TUL9^J&dz7lt^l!R{VIGpbRBHnOmw3tnk^5gm+VU)b9Z4Z(T`{T;W(-Ta;oG?=XwoYg|9Nc<H#sr^91Hb-=7<m|_^NfAygsEL`3Tvod<*VHbST9+6h$OLsdrAlA3FCKGfL-12fo9v6vbbTc^RExfewx@=AnF>nY1SZirKRq?78LrEUwh}^R;!$!SXD^>hEs@OjIqkW?ko*sFF>j`z2oZGA9lmeP=`n*nU29eq^|XneMX=52ZYKZj_pko^=A${v`wBLP#*;($R{OAY``p7`01OCp0i|3b3xo}!={j4UvS_1%7miQFv%?nn?9K7b6y_RRW0C!oE*g?H6p2$4GZ64E<P@&mR0lfE&5eTA31NN0tifw9r+5`fav}bqx+NU>z)5|iw@xAP!ubE@pPioC<E^K7Z5D>=9!%URzHXJ_{OOXw5CNV3q~=biXA`49jpN*9#O7>mm+u}J`)fKoYi?&Yp@w^Bs`Z>n<eXrqVTOiH=nUf2o<pQn7wbxQ%H)teo<xnvxQqO>hUv)lxHlDNz)5eDza1{g@NH!+Zt#GznjA0{g8OG^{MG22nyU;sg}E3K>-^4PMgc9JuZ@$u`ut#7Q%FpZDjQ-vWQ*IL`}cf=gHC+iC}V{A_GS_em+OH<@fF{1oS<R-fyh$PU6}Rd-KjZL-F5eyW6qa$e~2b*@8d9}wDY-lJC%Atx6a%W_izb)GRddzrp@6@#RvCkU#K2w;_Mock^p-`6WGOUo%;~2$}x-0Uv@ti6mvG5n4TU^P8}HKKyvS)L5FgE&A`LC<;O;^^OUSj3@m9lAHIdgAVpq(9S5}su$hCw4d+MqPaoO|eQ?|O;Voik+vU+L#!t;n=YDP1?d<O4x1Gsouau$R)IS4##M{sFU|+Po*+zr1)*H~44vR7X^iQ)HrA&+Qn7r<W)2IG9{uh<|OkJW4+xTZ+PQ2c>V59n)@rMti@{m_;3A_@CopZF_7pXS7^~YHJCY43CWioKlj8eS8ZCqTQniHQqOLD%s|J<un)B7>$m#Z@;*GmPg!U6IjWk>7Du_9K}EZ+TvPgI0rNa<!k=g1DZgj+FG1Q!F6f;y>RfWWc5oVqoqb9<^WhxfqCM3?vK?aX`>u|WBlbgBKAJ}Co-AAawDO}R4KmD^P5HGGR39b0pcQn}T>-YMj(0O+$qUD0KA1-;aVT^rL1<H3^17u3Nwu<ipG^ZbRY|G;nZX$7#H!4FjHI!?9!PWi^Q7RT`o0b|f_3~)7Yq7Mx`>2{b5<bUW|JxcHxUK2d#4PTrM@<r3%CPT-|s|(C(&SzNsKfot&-kT$bO!b+%Er~19hPy+J(K^;|yu`~^-Lk2h%t}oWw!QetxLE@FAs>ny-$t24d9X7Ux2n0*Xc&hp3&WOlZ{#Kx#`?7i*asiWm{b31d<KjgPmpp*=fD+(LpAe5JBu<r@4C^0ljGo#7LA|9&i=w`ta^{j3Xhwu3FAegcXg9QcPcmWdQ;bK;#O^qH=SatUz`>-Jr{=gIQu0$?q+B>T|^W&xj3QRq`!G;!#Uf~VRyH!iT#>03e<MMH61lk(4tCXzQTq<e#QS!ASU`bk$A{{kKf8YDL&^$KTTT6jNUNQ<-uUo`(IE?0|XQR000O8001EX{*FwaR|o(Agcbk*9smFUWq5F9a%pX4Zgg`mbYXK}b~i6&cywiQZeeULaCwbbOLOG55x(nJAk2jmbE%abCytysY!p>_tJb?)UON|AVL>D~f<+QE07@FG^xyq89()W(_UfP^fkvaBUw1P}lFP@wk)^bZSG>1E8&)gLR9$0{H%vTYt;55wgV1}{@=n&mSi^W%u{YoU=kf<;N~LAD%abHIIjOa3Sy9x39keJ4Cfi<VFz-5Lxs|FjCnsU8RGkI;M&87gdgw~4RAb_T{{cllSlNuLDn2RWFYLL!1y#K5BVHqk7M{fJdtoBMC;T?zJ#U5Sc`1U}mYYHkot3S~xwf+Ar5%sF2@986E1fZJRV&Nl7o)n2nOpwl*Z(ML*@!i#id!C;G>TV6G^sn>Wb9sOS?`Nz+CsK`Ct#+NO4QD{qTua~&}pyLFQT-CR?22j3|p1SScOBfbH>fs-)O_AL6^d8->Xh+&QBQr;G7HIG|b)#_*Bw~9TWGw8K8+~57OQ;32ouZog{z1fA^6Y0imocaY@O`TT$gFj+yMFO*goRR|o{QYt#)+aJ_@II(J1QcD&p(FG$Wl4SnD2=}LRcEd=l}=a_ZT3#^upa1kB%=HfRE9=m&0I^~>@YQXVII`MBrSKhW<-yxJLkw-nfh~D7Dc<+TKtFf(%jfEvQIT@_E7wN{)&CD6eXG?!Tows+DgqOG$oi*DZ^&m2D>q6bRSAS&mPh>wWz!Of)Q<k{gI;G%A4)3IB$Jx1{IN)OBmF)V#<`h}M4i#N<rPae#BCF)uWjEf)Q6Yk^Jz))}bt&jK>@QpQY=m|18<LB6`_vKAA0-pEpr@>*&>JQVLxO{;C0cqsWB=Z=iG<$HUcdgkyMML}F>r-r93s`3h)W}uDK}m2-0Y!eCpw{FfqM~Ah;>3hWduG$aLbCU-2-(5>%|P#AgI`3;q8C(WMu5EJIn0pPBM8mzFjS3c_PD}v(dyEOX8(*xdgPrP0981;^fR12GJ_2bWG|yvEMduX$9=CqCI#2J^#(f<Jybl7kmuC*~<u$T`drh$ijZ_RW-dRLAJ!!oK_10F9%~q6Ot}X3Ia11Ttzfrl7-v+&8@12Myy%-b<dR8r}S?#%8(Np3Z08b-0W#n|Kn-?$}R|WkV1%oX77Lj1>%->J5f<)U+h^gd(lWZo5~U4SI?ye9JI(n2@jflCSl$+J)|6~Oy20BaKM(f{L#qY#P*xBe|@=0C-S+1IS9+U!t6WM?ORhkz*B$1?(D0x*Z%~=(PJ-jnt3u1IXvAS)~bmYohrC73N9g6ZH?74EIFkxY&<eKJlISDY<Ml)t!8~_+RN>uou6uj)hGt*BVpr^ek%IwBeRBJCEXaZ1;L*?OZhjKU$(M=hwZ*_1#<g3|7O|^WevOw0;+AYOQub$`r>Z;)$4ic^jh3YHAFzp%ZDmJEy@k>ltNqOpDY2)+Ye#O>iq;*%D|_D{_|2Xd5Ws*)b0SzK}z+<*a|nK{+1d3&7z>?vP)+X;knjSqnl#_L|vkgyfdYiJ?Y@D$#=q%fnqgDkJgcx>fWdM18gk{l0ThThG_N92mdFJZiH%pFG966x%lqQ+aG`z$%R6nLl56M{J{GznQ7c|TT)TeA~%BT5>>HI{{I9%&*wjzFVd@@tNi+l4Ngw?XRL1c&TMh!`=8$Z`1ak~_m`8=QP4b7)Fx_#me&2#3gfEGhgdIsxtaQ<3YCSb92$vg0P0d}r3HFq0~C)YnU$6))@CNVf_B%0AOJ#}I$`sjRzshjZN^q%XvlR><Jp*O1?pLeGKvxtR;CzFN;SA0s^X!f%(5pL^(cJj3d&gvn+9y39T-!0I`Fj6jheV6lX<q&?q*f_Xw?PD+U`r>%GLx2Aa#Udj!8$hSNM~?;H&4E8C*A6AdU?c^K;)_=0#j7iaBkmdaD1)*mIx`{w6wh*tvWgo8{1yPHk=GLhJ_htqS5juR4^Y$*Tp`;h<Ltq8Lz^%et@>Mhpxp<4lqoJ5>9OLh4o_XnfMRnI@lLb?mO)aMD9j-(myeYPmzWIxe|?jol^b;#cx<0*W(FY#I;u-6-U`^uylpe}rJi8pF#tuo{cAp&eXbNd3Z*YjSq+@h7SW)Wx6r$qbE$GBlku2<BR9V^=-nI3;h7jd<xxYGZaofkT_U4HV%u_)0wxs>V}S2`wbot6}D4qiwL*L({=vdOVyWj*p!wU6MKe%i36kE4^!xdUsa|iR;#bR4?p>o^)L`$8NWn#Z1#63FN%_o<b`cC}UpC5T+RwOU%0|aDkd7sxLbY&+n7Xfu~l5^JSSAnmc+2&IgrlIZI7^JuQlvMUHV~+ftv@%>`b58X4THfym=aL<rytX@|Z)V^`OcIyvuq(N)AK<~Ajz8}~e!Z!(nLkJjx8s8FIhOP?nQFk;la3V4|;Pk!!9OO{BzHkb{dhIa`>cCCi{E)Xn2gN#XMr61U^^e)?OS<*ZBT-NMZ4JX`L*A9Q*8u8IIJYa;OxoRB8vLmOp<el9fO{|zIn;a%=Un&@wM~*7wp!ttA@}GHDLIvN=zy5r-+EWaivO0P4E_#0TL_Wd~IqCU`9j0$cK@&d!j>|7AZ;m!vGq^rPXfyOaZhc`yAbtGs@iL!fZV2oc@CU<p$f0u<!_TMDc?iQ*9-X_$bVE9^r4ikE;rVmE1I*2tEXwRivTTf!5CXX70K%4J?(=Hltqa;1a|cvDg`CLFBnR6K#v<R;P~$N+uj5DC)2S&#LXV%}zD1QpHDGq2LE56;N#lXiyC|P&_q_Pz-~!Xp6hWe^rIyzYdN3BfWVg-9?@&tv1QY-O00;m803iVR#)2p83;+OBDF6T+0001GcyMKMX>Db0baO9sVRK)0H!p2vbZBp6b1ras&06bo+cpyauD=3hKUC^bitM!08K=tBN#!JMY-bXuGd+)o1Cfw~GhZP=+j1`d`|SgK>*2dk=MM=5EOr-*-5++BbR6eHktq?Y2QxOwt%}96$aMn$C7Cd5C6-z8sN!*&sil4pSErxE-7i8tq$<;qip^v&_~a*<%7p|rnKDHdt3bS(pIv^O3!@?+lrX4Vkr!#2WJWA?Xkc}`az&aYzRL3?%SDn6av7x|j8BA^OMfkNoEAnzMQ%i(azE1xMJjzdJ)h&!C^bUHfmo>+g@9Prax(akB@4OGp*9<_NQyWB_0lxlh_ymZ;=~A(5U(`Jjnx9yml}vb=LX~z223#-TmY?#K~14^5ha0MY9Q9JNJHr>Q1@MgJHv%$%_K#rM38`4P?n}?5SjVw$-r@(!2rw}3C~*=XdBNfNn{N45~b#rCXhz@p#=5viYO1XZ`^vy1;djn14|J}v*zVylcHL7{Oa!ybMN)r`RhO5UA%F{IT!_I2LruqPn`?|{591c3<ked@`mgbv#YF7qX7+xb7j^^@F`WkT^yVUupvF)X?35U2vek?x`shlOeT{XnqqIONv)Hdz_BGEjmR)v%M-B-6KTXu98ab+yilewI-UGXqbnIjtzjna(Edb&Nu*Y-;a`|Xpu#r!F%92>u^PiimbEX_<{+~Gk1~m$gtRXhcix_UJiokn_s{w74zEgL10l!r{}vZXtbh#vtERJZd3)NbxCG}x-}Q8?jpq$>6)s0?d9wbJ91lk)I)LD+f6`&#Nd4=2dLzCAEiXU5d*jqh@Gp}(SK==j7HXblNj7v&MMOspn4vJPVmN9FLtG8MDYVHU#xsaoWbTZbN`)#O)^bNEm4{4ocyvIA)-3)xf5AEt%Lsym@B)yci1n93y$}VgwdS7vfI9v_*cN~@Y9COeG`7>?+R~a-v$#m{9E-0lYA1%6y;F|c9?g9j_6ChyZ5i_7B#UIIA63wu1wc|V>yAeHfuJK<{7d;}$81j_6$0@VO<9I#?X*kQj_Hc)7J_dFj&kXIj&I}SzPfk5HjMpU_RZ-ZL>XMwQ8@4CZ%$wT?YONtJ0#hu_>>;QY%nw$Y-;)FHRxg}bLbE(p-2QG{GXr)AM9^}7knT*!3RPRd;rXFIzDa#7J+-B^Cd>L8m@qqF&Tx1OR~G6vBRcck@j1)F=p2`Wjlya_Eh{jV+-S!@W)Y$#)3#q_B1N8L!0hUe3MOGL~Q~V=1^`1tbaXqk8ZYj1{qlV3c^cx153Al(^=riAq|L?i|9@&@VZikIEiCWi2`YlQ5&AzGj<O`cEiX{C^;OtPl(=w5lm6s`h^B@E?@mIe(RKbslofnhYViYFwi$hVZIa^6G`U06M*~RyB8q;?;y|;2pb;ur-#PtB0@>pGx8Haz<F&KlhYDzNy@H5%w6rxOl*DwAcQR4Y^v0vsS{@wV*Ij^i`3#d;os)>mPdC>%{k1bacfoDQ~s$C_R>fQu8(iFB@ZvfF_R*L+FwR>j${%KQ{{RO_^YQq=hI9Cc`hzQz|=DIIMvCAv^a#K;JVIpZ<rCJ^H)$vs0ZmAz?lFli~#P6A~OoADcI^#uZoOwdOSrVb593KP9ivlUnW=$Fm7rnh~PGN#bWJ><q906HtUdcGk4_nhRRUfGX)Q?4iTKa8GkNG*p}8h*VU?e>qBQgg+;7$g<9fF!fCiw_B$&t=7!`ZkJsCHP))Ie{-~z6c(|AR*s%ufC0P5G&5xSPEbY@bHAks<f)y-T29zJnpitSh@cG__<2@^Xa?oM^2a!MTUARZ&*|aiJHxb=u+q0u`j}<!aB|k3pdm4FV!3N0hT~HZr$ATLR2Mhz=Vgr#ov_EWtOUTeIelb<FRg#YTVA877coeI&*-X^D+m|PYBF_$Ly#G^ex@oi(R*L^xapYv0_p_E+l*AYd-a^9O;q(t-++~Ug28?sKO2$?0Y>TJ-1(bt#Dk~ci)LY@07W3(|`4LR`0FHTucRh5mD~ByxBhHLms4!>nL_k%?VKYg053xC_*c=^XQ=Ynauk>ndz(`@T%8N)nq(gji1yAIoZG95TkjR$+!Rze}@9=IG|KJoK*bJG;XhhyZvn6A6nn&FghR$N=Uy7;Kv{5^-(8lf7O9{?7ib{@<4n)`odsJECv@pNEc$n$+^rl<I^TY_9wodHnHs&^0Z8H9y8|mNLRW97a(QG;@|F@S8TOHCog$CCqXEpd4PZIC_ld@~YjU<P;Ur~F`xpA%EpEN>pK#tuetlatx+XI=r7uO4E{Iz?R_;OK%GTYo(!Qx&6r^0Xt5u0hUYGp<;zn$=rU$C=ZoxTw=gB75M_OWiz3u;b=8!&DpwM_ybJQiS(1$K4<vq`}wI`+dNz<wO^i;B=_Zc$*?3kx7o2?9+WNf4**1%o!F<7uPei4I0xkFyWYk0<3u`<~hzn8Y#U*s*WpT;}48w-)I<$9-eQYHVPYIoO!S+D9}hz0xl1x?}^j&Mj+hN?{;;t?=;MiTDa<1@Skuj7DWU;BgAdw1r-CO4Eu64U53<Vt|9ksVkZuFv61XaXN{?1F;KrTo$e{rgM)PdLYqD6CIl-V)vu!J-bO&Y}=*VJc_%>x!ncDJvOd%3GLnNK7nJ;`dh_+Zg$t5v}%&CWvXy%up@fzmg!6us5|YKbo^rnA$n;^htE$h=TF0sHySof1v&u~k!FActOb?Y$lIxXhz(<lKR}x@RqUnlkJH_2I9=@5Ek~0nLNe1+OJ^j0#2wNp4)<u$DHm(jb$!=4HOr$}T-As8z4&NvPR^qQ7$i{OxdcDdTRp+o8@bBvs(vQsA1YqKJQ@y8sljepfYFyS(&6TZ92ThMQu|uPhEN~lLdifROEN`_{;D*2n_PsFUwib7wbxYjBxR2<H{4eDR^;QZnd{2qF=-V)_F%Uj1@-@W#rd{^{9F5$>Y)b}LN|^2XX~+JNw6(_s}iDFyW77tkK1N6vM3rf=^5pFxbgr=R1Qu7X(SYo&*(*!nxvZ(LV1Dy4Oao&b2jO+cM4q_aFslRtD`<pmNm10$e=WA>f!SG8558`&yUL}omD_FnwX@0$bD?+M=9*N-t1mWH*2<SMz{NziSf}P2)d{^(G^`ao0!FS$inIAI(*tejl{F9NHW+52a{EOmo_2cAmlg)x-VOz3jdzsXf3)biuVO>uj>+~aM6Y-2Jji?fL@*~O(Y*0rsS+|d;||2lB7qn+Zdg)vuBiT$@{R>4g>(2jZ3!FZLXi~uENys?V@9gww~b^px+@kLAgH4wQ{Jb?TUsLG%~8TT%YO|#B0{8d$A5L2D_fS<7Mg4nZ!nSIn}t&gCx>1CQubx_#r?~+vCtsV>=B-(oqrBmag|lJ#?_Mj%P3jgtJ=q7Q9GRYy)<NmL0dZ@nfx8wSw$lFB|tNk!r3yH&@&$mEtqM%6^*vd^zYoT*<(F4oAjWxre2~u<G=I=AkWePCBKD+kCa{;<p5<dW{@{hxi+3Z$)!y*^epXvXu%}7lDHP2bclCt-nt0;Rxc5SY%*Byj$*7hFwhM7qDqt)VwEoSW{<7Q20QFg$vy{Siz#AqsFD<OG<$uuT@m5(Dx90K9<6|2NW=Gs}=-6WRyw-V{mP%S%f&HqLeIw-(o;c!m_!#wq=l{_5+v(sw(pW)9Hl{V1t4_WZ^GTa-$&+f}P40NLp-Y18`Y}kkRfgjRrek$?=w(FS)9hW&3IbBBHc6)V)_&AgJRL{j#J>@L$#_o#js{RvIZdxFAFkI0&GB_0rY)BH&NmO=`Y$?JI#HK+G!f%meq>;8DpXqXE`Wd*IHV54Cgdh5A+vJKJlgFZd<NsO%(o%(RAv8p>MP5Wj)62~XGfJiIkY8Z{C~>gnVltZV(QO=opzb8^}h*u1DAK9l8=^9kAwCSfd!b1%$Q>~FlXwj$kx37UXswS76W5-f-<3LOSs+2$Me+bjFqhG+D#*BCXX;;#JaVqE4$woh3KprTj`f1Tv`EgBjDSX(%H1qIUWGXN~W{(6N(2^81W>#bBjD4YR_2m*`}g>UGqt@@KB$e&-}!i_lp{OJl?l!|g^kf>v*!Lp+B0!Ls?W&nJ^wk`Cil$q`9Xj1!e=`{Kq-o@qFyZ2Y~k52V*+|0g0vHWHZe=O&L{cK#^bW);b?$~^xR>utYR%ZP9Rr}_qMZ4M%D1^pm<)=ZSq@)oln6_Npr7b3KQNx8Z>{LK(limwVYoy+I&uG7Y{^0T3s~#<>J)?j7ljS!+&W^>E$txe-@v^sDt=jl0w>|g|P)h>@6aWAK2mk;8ApoAXkg}Bq003<g0015U003opaAk67ZDnqBb1!sZb6<8hFK}{YaA9&~WG--drB`il+cpsX?q9*FA11A~+6~w+q%g1~>(XrL8U)*b!7v0`I@w$#Q6;G)Zjk@JBaeFXOWI)u64>Ou@wvO_$RG%=71Ue-gNM2hsW7a7HBYy%QeNG0&F%!;YsRaLiLz`AUlqVgQHd2d=>~Uo!wZ%JH;n=v2SG5I<Vuz-N%F?7B?&6lQW>;SNyCkhm7Yxe+J@^*Ay(b*swwL&L$_5;X09)<TK&tCub~w*Ql%U3%5a^irZS?0ID=GXqFT2~3V*9C(E!=La1APj+D6ohUw11vpqs*wPD~(pnQ+4J^A|3i%PJS^sr#KO!~hZ}Nh`IIDQGQKvIbd#QQL$trEP*ak*j|oHBOJ#F&8{3ExM2;3W1j;{?xbANi>;EGRV=e%UB{TAQX*1(UiHhq~>NbW&QC^_6{~>DFx5InKJLif;4|UiB6o!dto+&c`R$F!rpRZ6SP5{0!Y+F!0*}Iuf}Nm4wMOln~Sp`8XOlz_Rk#-jSRz%-MLxLi(<vo+xc(@49~dXi@@7!g@UECMovB6jO?^&z#xw1=5w*MrQ^|P%?y0n6D0&LI_k?L)M8z7%qUu$(|$Zy>XkLB4XY^}pMq5912?2viw7vI?qkJEIO!s#bxPmVBQpO(H4w2}DrSX!*)UO20I@Jo>M(j57+wTdZ!b>I|4u%f{W^OWEYZ=92TBA5^MZZ2xx8Ryqp_!@VOkVWnF3oo{n!X(8_qJ3=b&&ELonhFTv-#Iwtw+xM)B0d&K`**Qi-*wkZ|^#W46e%$elyS4>5d1b_#ctr`?HcjKMPMix`wbNak!;h0S(#V!lFvuqYnzzlAG|29+Z;uv4wE1+a47TBChk*-sw4ml+f@&#?|G!b3C1I3AMotIO*hT*5~e<Wm#}oylmFr?acm^WP37h?=>=hI4({76Vq{Cy2(SK~TCH&%_!qi}RMNW4+<ueg8vP)iKvxDZX{!$40KU1`w1e-oOKCc~DekTLBe_Xpp3HjvSQW`k>!>rVa;$G#!O8WdZHY6s!^3+1<CEU%aQDFVW*MDh+)vSL>11v&}bT;-fv-Hk9q8FrZGskk8!zUoK{U27_X2^L2Cj`_CWVCjooubeD&UyN_?-WDaD@^WYaE&_}VuaNWjd()C|bMxo7)WX@%8I@vE|PdFZ*qrB4@IWSoqiSHvbDjC3QO2r;KGXdERPJAFeB!SBSZVaZpTc>0=)3`gvF=$5>?mUj*3kLP$jrw2MOo(2J{WBeFw9^X3+pjwkV-N=Nfh!3+rm^|i3b0l^!QjOW8Q@i_Wc|jn4e}OpPse@TWIgPu+dg4mHlKIQ*szk^Qa$QnlJ3z^UPx}jozg^`yh3_;R-ZD_nfRO@l(n(o*pac4vLFRsZIAhBiUP$4qD3(heWI^V?N!ZIP;v{~5CPdpYz>=bPXaI4xgBy6r#4c2!Bb#n1A-_7@7q<~7?umAP0Vfp7cKvDMwT!?s8z~4T=dq2(M&hByQJo#Fra7vOSG!)?M&*DYejLW7@5lAYu(jS+>TK45vLR%o6lLW!T@h=eHwT$fqN0yv)H|XH14_18(a*jzx-Hw#)0ooqKAQluW-*z-QH$zpgn}MK!h=l&9m^y4slUThZtrr@Bu$tqm%y|z>f2QSHI)DOeR0k9Y$2{nv_CP;++8}>hKBRA6|)jjfLGSz;9!@QIy(U^V3aJ-6poz=Rf)q#P{0U3SUh4{NwnXg9mJ*_EBe`!kaCV==u%Ir0bCsrGRaO#@M#A4Dex)*(+Qj*H}{S`R8T_P64doIYR>mheLPv$0H5lCZ{iKoSi;ux9V{0*61nrG~n;15+55pb0JfDT-qz}(-i;qMlyxfXTt6YJir4^EWE;55x^HH;DsE*7Q<*<I_?1I!YVA<QL&_{JQ6&1Z*nTJd+3JG-kJu*?rqr&V2pI)aacVP3lg@2(zzOmbs#N5fAP1+kE}4MQFUh)!~(M%l1+cR@vnRba0jk7T)D8YfAKl`+Rj^iEoM3zR@?wZ4F!%SR~{&}ApH|>=jA}B$5S@Oa$!IE+fYL45!6j@!-WR`t83{(!!I_l*5NXWMybc`X<Owe_DR%XD?0Tqcaq<_B^PtAeE1qcudrc!BU6jc*0G!{`}*wRwb*u*87+670EY{HG?A!{k`BNTgS33%kKLZh5jq=5E{-(v@|m9f7f?$B1QY-O00;m803iSg*=J2K4FCYFEC2u;0001GcyMKMX>Db0baO9sVRK)0H!pH!aBp&SE^v9xT5XTxwh{jBU%{$h?5nMPNv>#iqX37yO>;nc7bHQ8c3}v#MBB<rq9Rhe>n`@ccZTnJv2X8CpxqB%i5w1x!{N*`Lq$>aOIEO27TY2tqN^&}7ECOOY-ueT+Cmp;ijqqrSjjS3@S4y%Ck-uHmXo5QI|dE3JP0P@D2isYZObc?rrS<-ElX3NX?QDvsOFL)12LPKwoKgH$Aa6ZipsralJU~aDJ++qZE07^yvU@cX@I&cHn#sK9Bet&UDX^26{K#ASdxmg?O?eTi)jn1rkQlSspM6WrN0SYFT<D843617EjzUYJ7hj1@Ry=Cih7scvBOfe(A}<OJ1SY4(FO;k6_ag|3Ei!=mu6)rBx}<R=Td;io8@dVo6TU2b}RnP<}J&3n~TMj8d+=!sF9dP-G+ZH18vvjDatr3FS;V<OA-mtcPHQ_s(H#g*>qAw&ofOgK#e3ZD>x~$6Ty*!6cOM2273cEwc4PF6txxW_I&L~St%HZRa>SgPRVM=Q;VZn|B-wwbJh|;@4>bV1z15XDKLkPfX%Ex?Vy&N>{!iOwHc`Z$~qS@`E}0@B%`(ACS*~TpvtC!k#gViZnp;=EXk&mNQRnZprlP%h&{~rdk1f48!EFsDM1l+c8F(s5wsC7XF_hh#`#0MV{#rjk<zh9(bCgR_MXS*3$NELZ;?JwrRvh!G!hMkxQ&#+sp)pS*Pr5QH*$KoqUI05{F2`>IEXO)a7nOq-H8iYOw4B)NLs>pu<g0g=pr!2xNCCY<qdMIutc#~Itpu3X#=Nd&$~8#ply|W_2=_*@~UTM<jhPDE7lLnFhB5`Er%1j<s>Mneq4Ae-Jqs4^C#PqQ#r4jxMH-nBZ48#>+^I{&3+`X%kU_bY;I_l<aKBpAh_x1pRN89nr~iz$xPq8{_L47f34wE`JDt+ZaN0b_ou`Nq!g&)$E0Y+K(rG2>-%C>xXOSPqj%uGfY~A493&G-MIYzRwnyE^j5}bNaUoNY@rET4S|Wt2jH5?uTDApW^e6p(P2T?a{wK0!6hkCLTP3VAr6o8m-s+&5L(G+31M^_4VfYFW8MIp-qVYbMX|1d3&qy%lmEZ#zdD!!knZT`GosJ#i8X^k;7uc~gHSd|1ieSJrbJ`=T;*CDJCT8ux!8W)nDs~+PkxTWW`v^k#92~FI499x)!A%eKUJw7~%Jo5n+VebGFSRFZS}r-D2GcO|yA`M+!oWzYreslI^Xr&44Xblx+gqWZ^{F?%o%)LChh@LlNpeN(<A7o)7BGT;eWF*d0vx~}LQ{M0x%asd!cV5SEtDDrDpe>Gnrl_Yg=CeOoAs|P=1>SP-4+1nU|(F^Uy@f#qR;mwKHM`jQ1Fo@Jk%LWAc$R>T?PdB%O0je283?}YcZ7XX<2p|@U_^f1ty3b1Q;fs4@yd&-2tWLPTH<y@l5f2YugG00~MmuoIDhAPi`(&Kj7lfu3=Br4{U7i)DV-KM?hI{02*EvfOK`}%K_Wf=;+A?AO*mHJ;(rnATKZi5m0D_Cqv0o%NErMBw(4ez5`?kx`oxr2d>e<uX_+IXZNh+4K7N_j>00ophOhA8sRmn3R#<=vHHCOHOm9pb#SO_$yjVP7$QAH!*L;CQ(({LW92W9B@&*KZ<C45!PbiaC(;5u`QHJ@vYWQ$?L3+oQUNR_8*o8HPfNBf)zBYweS5yVSjW*K6!JpP4HkkAm#3!l)}iU`To7AuC8$)x+hrp|#4`nKj`^N2s0~~MNfHwM@aE^gef;qLUpMa}<#83w@MvmM<1_+9w4rch3J^RYHG7nEJfopdeg$@R>#r0;wB87gD8^N%PRA4M=YvrStq&d*SpwzUAmLazN}L%QaOQdR_J=p`e=gA3sV9tHQ3l4n!+2)*+-Q1nV#E4Q>rDq~5oiW-MZ1rX_bBKL8I2$igz#{Bt-2Gbj;cDecJerKytNqa0KRC)opMK8?37$!LUeP9OTYO_{eG=}U*qrh7wT8_ey@7J|2pzs6^#MBR)76ArdYM3?W5{@L|?oOUK8u^aCx=7c$+?O<tR&rSix?}PnUs<V~^<Jm8{h5DLA`r(N@|^yL|t|KE)VS{M@kWq@>j*r>LnbGE&~H)yH~yq)QCDR;q8TvPm-8cFbp_Q<FHrFj$jWnTjrj;PscA;Q!cY0`VpdM`_c#{G64NqBkPPv`kjc;Y{`)Vcbw50=~ofzDvkzBJQln1zTNQ_IdWOLu#;ta-pZ}i+mf%?HaXiys3?;!n}FW7X)#K<f+3BN`#(k#y^E6LOGJYU_D}VGT{I*93YMlnA)uGj&J0MO<(Lu(N9RO=-aXICuHvrnH2wJ(l16QM*}!Cd4GU=L~t$*jXr<jfgWr#Si`Q^l<b&=4Qza**ONv&qBg=zS=T^7?T>X4svqVikGzY9em(2kD}}k%VSeMYnn(cRttgfYB^4_d;8*vTLAn`PIe^YS2EASY==U!Hg*|Z4Zy!h8QE3RGQ5KgGAWqK7y--l%+GY_%n<rY9wmM8>YY*^g#Q~sLIPdZUlMP&JNUwREX$C&UIm38DY&=<#_wPbQj2##*dDJ*Hq61W|>auhrc-cWbwl?qFqQl(RSR=fET}Oy#sv6ke@S>Kzjd9gpw{@^fXa~-8R2c`0!PY2TvKKM}zN~G)K9uz$!!WrKKE12U!!_O%<cx2(LfNxPZir283^IiO_XT@UB?gZ-9*P=(_E|4M%1T;PMw(}nm@?#+maN{%ed>?++}V$}G@pjPV<Kw{kOpkT+*XJV5>k1MI43VqOvJ7rtS*fcJ%M!*8SMluF_Yxm*d}EFj|5iDpTd&Asu^670r<DVJ)6^ulfG=ETg<bCf!+;aflUdqAb&{Io1MqIG2nz9dP+y<RTL5`UNMLw=894eG7X=Z9NgeGt&A4^HbHL0a!mA#-#geUu!~)7ua3^3sXd$t=Ii|<xM|OB2*^Ob*#vY_s|mrFYPf~o^=)MA=K&|t8JJ$|WOsn5#*Wfl-xz_-g@)dh;20UX2V|yoico1JEflFK*F1r2x996{jfVPpG3`^2-{RbQWlZo}_flokc&h&EUH$1<110}grd2d}er$kHg`bf%p)+ES;6fn*QvQSB&g!CrE_GU01x~)sw5$7s6Z8(nc~t9Cx>`@7tw?~;RCs4)YE$Z2mvOI2&YAWw{_1JyHC2^Jxu1$<ds-J_=n@~=0+g*;>SLILO=c28$kJG?uHC7ZSNh+U79|#(lesY-_SmRpBwoj`Zxl9_22JjI66a%7isa(_{CwDlT2G+W_zQ5G=qT|%cJ4%5xURUmgVwpeVHU}+ExHDkl&1Vny@o{~y!1L*;V}*#T!cQeVZ`vQn?tI&d{GhF4xTDQe{S7**-u7Km7!I2w_nv=nv!!+-L=Bm!tT|mTtrmCNx=cjO%<Vjy!q>!xBrX`G*Bg}@I>}qyq*`wS+=tQx*v(^!jm%jK-A&L@rAM(fNKnIl!L|ToE_-v&PyWO!;r#-qU0!+7^Nz1zmG3T@3uISoUli{Jqf}keu)5$Cxm+nKwMXAqei=F85PP2!BLw7i-oUKh3m1#)0^OFv@#biQg??jM&e^PpPsz&c-gIo_Ta~^JM@)@*GRwjc_L%O)1f2H=TuIhu@~(J7Bw=U-Uk!MzTFsG(2K2TawJ;TbdO$eqW>KkY#jB-bx)Hk-`#iRW5UI#MXO@c>W*9A%M!ukW_ZH>KF<zcW{=)3wFOL5Jbyx4FY^We*E-earYVow-5?S5KI}O4$hUo-hE494^a>doA|Oxwv>al0Gz|3+^c;x{xR9_b7-8`gLVLgvbNe1ba|^ND_g7aAb@eI(x>_HSg`cm4_}2h5y%XvY!{^g;KvNb~VZOBkvOZ#o-hO=hG58vfs1FA)>n&<4#x*<(6q3+tQ|vn430DS*2b-xq?V0tmzIbBLSI$2Tc3W1`Y33T9C^R2jDs_NW@?yJX>ifhU<7ktIv%0GoCLIdiyBpk=3$b#TN7Wa`YY1;XTGrtg6c}4%r)mTn*oKPrE_ttsDxHu$EAtg>8c`deVLv9jpi{_l!->`w!K>Pz|GaRnpz;}>1Vr;Own2!}hOih?myX<i{G<ky_@!Bhh$n|w19d~3#hU5{ET(_+VKHiWf~<25+D2H{cLiVm1+6wv4_bOO6M@@z_OE(E+YcjqIAHYIR)4OkFUfk#KK2b)kLcLvS3^!GzG3Q-jz2;QDih!_8LeQy3LgI=v8eR^qp8}c{IG?tOx(xsAj|oOYMARGOS9FOazVrwy3&|L#jfV9y;;&MV)bv&Xbz*t9f(rq#>?tnXKbc51_`2trcNbZlWQ3;Ny6`G86=|wRK;_UKZ!cIUHv8MeS)J;!t6g#O9KQH000080000X05s(=XT~M~0P&mv02=@R0A+Y^WpZh4Wo~qHFLYsZUv@Vyb98lNc`k5y#XW0t+c=Wn^((mCx;inTWhXnkv!{%%Dsdh=wV61TWVUWg<zmqyWpPcBT0ZP}T>kIZ4*&!}P@*z3clj`5kwBx-X!IMP@B5$9y4}WCdFrjRs%}oIvMmzt_Is~x+T`B5%bJaMm6xlVthn|*o_VWHy1J>o7#fvD>b?8;vsbl6lWo(P@B5R<x+=F`6s_B)t<os+vhA*{8fagXP26N<QBNk~SrRw#Dv#?rt>vSC)ny_7F4fa|(>7Tyf8kg>%&N@ubVWerI7zyW+q_A#Rl{v|akI&@EBXE-{1iwwaUDt2<EqKl@v2d9rV|hTy%dk%R8?7p^eAr1ZMKRKQGT$Bi?Ya8aUN~rdc)7^P5k`Ne~H#vo`#d4KisM;*4ecj>oZdNoE}Xk6QYRsIjw7$*Q++UPMc3@3}jEHZ@#Y59Zolx(=b5rm)<T1+NOzDsfDbYau<3<dYe{W?5$(KB7w2u>(nbNufEB4J9v<Ojkmj8tl9+V=GA3et<oq<QoypgpYE#iUqIfdD$7QUh<|zuqN4el-r{^~c>VCmKVh^atDsF;-FsQ#O?{0Be(3q>*Iin{-$hfO;;Y;5eKC;ikIRQ$YakR*U`hy_c3^|qbzWXg{U@{Cy&q_u6+X+lsip{KR;4*`;B6W;CBX@TOF&jzD{2-f+>3ad650x=pXc#ye7Y@@G(QEpU$;QiNc_ZC5&U->BQD`UCu$W}tIeK9Tj1af{sf*%ixoVoPFH!hlRu8@dDUk5=uToy%l+MeeSx=X+Rj4_>L!L&b&tA@a6j8G(_26QxZt#A#vFK?!Po^ZeUzjP;X2eZZm*HGTxeG5zgu7-<~$H1@zu#yT&J_<YjdC(Fs#^wBO1rsD<EOlPG@f7O=K~f2?dMeE9S*21zB2n4?gwr=iC>z5BGuxrVN_W``}S*>s^)Z;wqi8(3$JvCiI?!9x*|^_)@iL2qJX@JFQ+!{g=2wU*Uhw@jtKC?gD{v4pM)&L4tX2(`L2tI{AqTqih?GSfFY_I)mPbi+k_<=hrWJED||HgPs@$UC_zkpDtd%c^`fL_~!g`=q2S{k(Y520q14wEKNc=v50%QAzlG70gremfyz!rzH|3Lt)#TY#m6_%&!67I_`7lkq7j8C2*|n{d{wT>2>VQW0+I<n1CFV)x&gLeQ8@#t2kNt}(`p*1PP6R|2(>A~1pX1Lk$#1aQF%i@x$Q135+G@Gb>EPzJ)@`DS}da}J&}h(uim9Alw%q_XIYbO>!~d-p$onsfeW3Mg5xgDW`qXH<%K31^?!nceO?%rGPN&w8ve(ytMoQQ9p_)dBOI28!#4h94{p$V0&@nNf-sp8p<q6qP#H52Kg~3N|3!87OX~e$Ax6MouKDnsfQ#cCrbQr-lOx22wFlw;)bnT9%n#fUjKz7n@V4<+H;M4ySvdGgM|QB$i8mD}+tUTo$Tx~Iumq`h7P!akrn9cnlmrrr=vk80H&I&y*R#f);p>UE0_Kd-DkK0u*rf2Dojw2V$&){sl&|6pkov?skBqE7wbuSewSmyIsnCuo(lmkLu29lEOb5mafUlmV|23aITR(d5UcnBjQbw=d#jBLqz1gI^b53dB^R6b15qehu9Dbub^d6|60dMC049Le`9#_|3X?5Ei0Hf;kuF^G*C?ywQIP)iNs_&|-Xw12<{hu<h0YQ3_$-OI~g$I(XsMl#V_a4ki3(TGqW-mM`Bw2zW5Vp-gy1<k`^dvLi<b-wJ)*BW(6B9ooPqdATY@OC9+)TMUHFvuobi#;v?gRJ<1PDZ~)*F7QuJE_mQR&xJ-X<*FOHlN2u|jrSUWU@D1n5GrQd^WDH7U-Hwb+og#IV)6qrDsw_bZz9GhD2C%0R$q1fA2QUtxRc%O!DJ`Y6W<VkFeDt<M=yU?C6YdRl=kih9Me3TVED9KVgL49uy}YpWcEv(ig$IWB~APi6Ku7r+aZxW*lTQrbtH@Q<}m+U-FM|2RYP<UXy<HB6vh5Z_LQ<uuJ_7VnT*HS5|YyQy4-{o<r{xn5_h3@ugcBAK$*Y5&lp1<CGt-_kcc3>s8*WnN=zC=yD&BOg9cKP2V!2S_*l0aB6-tCI(@{`D@;Kw5(wdxU4E{U)3p*iKVPr635F^JmX59~mkWheXz*Z;Et?Y7^+TH4eJyp%I!ZLpx=mwqyTg)Gf=R`!lqLt{39}-HW(QV8~SoQeR?`Wbt)Tf}mRQ^H7^)&p=ax7Ph#s)2`Ygfenar2ck_BaUy;JG%{OEb-H++0}JY8*ER+Z`89~&Z45>>I<F_1f;hQ-rknpl{LL+Sx%8z!4a_MeV<%0Ogw!B)Yl5cz@H+W6*}CR#f^w&hF3?{4Wwn9X5Pc%)HbZN4r8UwyJZc9JHGdvWCy3S>RlpoogyB(jGP0YS{#1AS6xSl6t{tQ`20fY;s6Byit31Y`{kFI%Kw`e|h}LAQ)*wU4$b_Yeukx(kcx?fTP<RpmO-ZnU&`)A^z(v-!X6_2?Fp|=O8gJDsejg+-7QG;Obr^s2UV7w;7a45^G7-S;02zJ%f-IYLo9Fi!87lKzkhIVk?UPf5Vpg+pktPCQd=1k=X^0Ug5duV`u8y%k1p_m%h8T-_HfAa=%-P&ovme<t1q9LaZ$MT*Af!j=*4ZV7GtAJ0hYJ$~yD*LJ$_m4j5$|wh7NHmqIGXe;LPb!I^C}%8NSC?0`_4=4bcR4uZVP+U$@gjt1QX%#NxL%i@^VG~NjKPNM>BMGA%KIeh5aXFRV2&`&CqqN^hEQ`jIPQR>5P>__J5B-fMK_670<96F*yvBf}4MXYzysA#R8I~<)F4^c1!;2KpIA$iEz*MlP;$==WF0KW%~Y#nI&05u(N{173&FMKhad>v}Azg<OCzRC&JPezDAtmiju*Pq<Q!8=ic1W=DA5lJXTXRp+ld9mJHS=n5}zc0H^_a+VFSWkfz`m6F&bYzmI2}cn<*K5h%0<&!-l<af7mp251EeM5T_boqvZ?u}Db%ktyQz!h}TVc{iOEYOuXnao#2<IYnNiN>^#N19OMnth)2WI{T6~-Pm}Kh$<%NRJ7X;9cP4~AM;^6mlEGg=`c(V#(XALoW)!rmPEf%j3-c{K@&SLY(EG=!*UeD`=O(aBfM~M1HPvj2#N0D7${wuvh?aLY$Nu&<l^0>+4Tn4V+Y#E6KaN3|I9nj(-?H_73f&dT!Jt)r*CSEN3AwxsRs)5A)UK(+zQ(6t`yHC-eo7!Y5pb0nY%^rvN8*!H_+V2hzHngzGMhCQMFbx=2=Z~u@Q(Kr-W-*hR&Tcf&)#NVW`!<xag+TXS9CI%VJ;TTQ!X_{n?dVh)pPb&(I-O&^Dl?9QYKLD?wZ4gb^TLLh~W)MStvBTJ{DMaHIm0R+k)~u@S2%{YHIVG-19Ru>X0(CMgD#r0!H`sNey)z9;~?vB>|P3>!hyo{bag!-Re0qyuvqOf1{U;x=?E!Vb$W&{Wp5w7AVGg!tk8m%m4^Uw(PXu^Uibjt;`kEAr-MBEnhC4P^^vv@0@j460^&7CHdfL(bYAbWA%g{@dH=Z$H0^F5bR<|Ko=@3t#svzkGW6;q%*zPd~l+^x56wpBL{hqL)8^x%lzJ=PxhczhC(47&fie0^wi$^k!k8wppL7FtO}w#^INXPv?Jgv<336HvQ%_^7+Mww?Dp%UjO*X?9=ApCFS>EPSn0(QObl?=Z|R%4BW3rVI2-O|DnBMDL2>cczb{7TfugQk;~CJ)i_Q|St<(|nlNu_1Dh5%qTX?`@Cn&PiLR@%-PO_6J;qJ8yCwqRat)HYD(}eYz-P4(>Q^neWMqtJ<qT7BKbuZTKtB|i*u9gJqT>=^PX)*&v{?H8Zqo|<?aSq}%ivPR<KT_49!tWJphuy{Kf2VrfGyQea4Bb2uVCv{Q$@;<kkUP`8Kz`gVZrk@ItM9nzKOHy4ZkWY-c;;+F9^i)>~hH#st`S(p5o$uN;4AA(ca`f7EyY(_&;DAry+1rq~NJjO4q}=P{>_XmU!E?ZagsMsN4m7q&!=}Xy+H_7d&uJg$ru-XE;M*L|r9Jga)ALmMJ1)?!p(?XkBV;Pt)iN`fvA81yL8g@bWiM$|BIE@-k(~33~f;A>(0PqpSemAez=`z79QS_E;|gSzr`Pw@RR!EOc3j?&eFqx2vcGjVF2-aeO@rW+Id^4TKR=FQ02>4MA1Gf4iKeH>TaOiILHvC;9l>5{QoBo2k_<0k1I63K~N1_f!U`w(-|U7=Y0>D_YFYJn{Z=W`dDs;G#>Abd*s=2F}+chMjNnG`U6}$;co`EP)YZI*V#_)KZi00xA`2jfhO9pJ$W9`|pKG372!{EMV%{QNCzKV|a`KY#?g=LWP5{xCypG!ucyHViWNA(h7qBB^N)XJ;zSX5Lr2kbMTgW^Akm7sNW`97PKGnkt-q#3DgtCVtc#W0i!Of`C<3VInH=b_o{>)fPzVM7Ip;Iny6UvaCgaSqVs#K!V#X4HD#BiMj7@V95mo=tMN8%Hf5s1XoAR*cs#K^NTTs<)h6*Qt0TZ6%TXUszaco_lNiE@iBvGwA)-P3f+7IQ{i-)*n<pf`WmEu5;80@?=3=jREy{onS>VYi0Y?`7pQ?nVY_hbTo(1~+G-)>zZfO)hJ3FK3sikmU!gF5)dvG)^Edadm?*=42l5mTz%n#o8XCOL+bKB{eZ;7Uyi(m=tCPcvd>)E?kBEm~S3k>i+NNgTYA9~1~DkmEg;P0|mGuY&)CCwm<WHy+O)Rq*3zC|RlCCYFT(O{P~`;y{7RzzYO{|jLK;QjR**yIa_iAva|TiE)Em@e-oKnDGB5QZ<eu%?7?k=~`%3-8N!vcLdvBug}fDZtLi&}vU4fI-BG88AwKhKwUNtuag2N#AFn-`3ezs-Pv=Y^$J7cj$YWVwffo6ecPZ5cc2eI?;R$b^+MaWu1|j%ggJmsTVq{yBgT_1MIw4X)iVFQ;G#kF{Dd;`@L0_;9sXjs{8`2V0aeSWEAVjbZxCF7vF^@W{F16#7JntIwn?c@yhDJd@h^vFRKZs1C?u|t(SED-T5*dA+w+EN?k7dSvVU=on3Fi=EM%xq_j6ypZhW|BWz{$P|vOQlv*Hr$Tg1j*2WKaBKlsSERE%+7eTMJu-joRLH1zM@u4pk%s@<yQDFpsa{Jux$9jCM3`H{s+rN2s@;6tT6($nV!fMJK{wTU-7ufb2IG~{AxCS2-k+B(6VQFV{$<4Icid8UxSn{7B$|Kq^T=c*XJ<9bNN_!gE{K1LYVFttZb|1|=fxa$dW!hUfvQO7}dFNiTxNXX9jH!uu-K14iZ_68-#@Ghg541{~>g_@&JRoTHi<B*Xc?Z%%S6R4V73Pqj*)=YTN3gt23zIlOg_hoVyBKVX7kDtD0*=bS<dBCYTwL<;PtnKkqK|*Z@sg}Y>nHICSMpK)^+)erh9$MzZPuLb^0<LD<f9ELL%rI>#WgF~dX?c`UuP>>35DtB7aBB>xi>)W`c#@tIf4Q5V*2=QlNMO_mdE#0yNenfoL{hHN`MZ$1GXychM6JUm&Tf3&UWQ)>R(fx2O{l<<g5gT`^ee#deB1a$#sCx(~2PG`gwP387$||&n%Zngm&6OdUDL)TEqvl7Ctnf0I@bk&s$SbARCw+AsPo!VKz2PG=R%7-SyzI!$vUYcuI*uswy}Un~wv|M$ZS|$k5rlwH6X7^n9&G1Xt_PWkba6*PaP+littKFl*CZAv3%R`4t7RT#9utDJHB>ktTicuksAT$fWo1|J(A43jShbQ=iUeV4B!}bnIQrp^sEZ#Nj|rVUD3&qXzB}K;MfTGyrYB?#35!-Eh!!bTKjYsZTp7u7c7C4<X$zp8dtOpdwI)19hG{hNe`=300ncW@2BhJ|*j^Kr!UJ-MvyaTw16(q*^hnHq9;=OcPYag`yUR(UC=vS}ovRO?JK<UKJ5^Gj-!5V!v(-dSt@J9ho%d0cIBj!+@?ut&cyxmjwC|2R$mpVY1Rk^&Za*=9UGby=8djYYkuAm=h~TqgAaAE=U*8)KbT|MHd!yD((yiHlDU9BpVjmeo|Eox(g~qNKIN6Rg&d~^6J>c;aD$0of3o~WQl}A*V@W@^sOIM(3holAkYsO_LidW7(4tzj$S4hqFOj5(15c`MPbBSvvP-d2db=%Yhins$nq&f8JnH)L9%A*e|q;y&j=rkf7SQJY6F%7c=a6vX%BUW*$nFfrPgdgtK&Cp6Meh~gRme|2`oOW-v@Go7Y<#o?#)7@y1fF7R%u=9w(U$W<pLIvtfie8gZWH}1i=BM83G2_Vy>Cg__$7iY)Kup1+iB`NlVAvyc0Qhhw7N^T4ME34i#5bd|#t7w#M!YBHsPX&Tlc%sEH*}Jflf*D;dL)!<{Vcu})=Wri|h$KQLMHcvs`0fKAra;yA$B>@4)oj4A?WPVCS=>&Pvp#8mCC9tm2NGt=vEw5lPRz`%a1T!PWsjl&|Z->q_pVJq|E=oA+TNG7TQH{44{fpNc{A7?9**QU^25be5W3&zguF4hrC{-*=eQGCahb^G|9cwq2c7&q#@<zrB@vptg}CVRSYL12cI2~?y;Z+l3dwPRiBUDTfpddGsd&E_<AwBb0_rXP!93j)I;X+d^Gp!<z#KT@r|o$}x<NUu}{EY;)#Yec&&FPn*N-7x>+ruJX@;7ifLHl2N_o3Lv_?|E>ko!;QBt}_P+sm}N^WCl$|lgj@<C;K|i^R%LJ<#^S!;E992kR^`=IE$FsAs6F?2UcoLr#dQ8;0yw)lvU2N{l-zq|9f4<cO3;9_mLbxG*o3vzhpbYd^!K_QdJNq_{(v^lFPeT6Jh{{YKSnH;4JRa(Eu!{8>T@~Ure7r{|<=#{JS8SVR_WFzZukoX84~eLJ4M6(&{`1I0bm$4V)1K6By;uf6A2de_lR%50otM2RdYVz5qWl8F*heRe6&}-0TjRJl)^$E=!ut;#oN2Y-ppOJ8F#MUXqy>!f1hw$@WJGv-h5OL{nt%XCZAa0u^|tr$88f9ayz%MUsNoMhk?M^F_PGFmx(<noB80pY7sWw18o!1j73dH3B^2NA6K25Uk3d4!GyUP_tce?FW6k#M<r-mr?2}Lz#H8SVivWfw2j^@AUW(0jijT{*fgR$dIv~^}2GH4KpbM#R$uGwm@&{Zl&75jKh*Xaa#yylSLR$lzh}LJc_J(L<wo<$8>b!q@ud(a+}quYdU(!WFIL`>A$ih&sL1wlcLp}FJcl=ARCFqaqLU$tmxYRcm`6g#-p6x`x2#_mwruGEja<)Y3@B}z+I;)dS;zg6rf6ArUt5^ZdqUJ2N;k!Iv1Pc$=!n|<^i;NVV-)zC?sAxz?VPxp(cbRztLfxD#LbN-{VvH38>yg^Ch^pfJ~pG68O88P6S0<a}lacAOHTf&WuAzi~Lr4z?NppN9#1k1fq7LMLdbF;syw00^!0C3E~cvrz{qkn9GK~M!U<?ekz7NR{DeeR8cuhX&RkNA}xx$Gy{hHka>owHnxKkml51mIL$q);>ppN>)=F7Q4}`>(*WpfrtaS`j{A-=<}DTb*fSC|S4O?lO7c4bKG<oLHo2F9R@uyzYN6A7#{T6acQ7DNJR0adKD8uTUJfMm=>NY<x|d{$IEh*JQjtPNN+Pjvw9RXDq4Ay>ah|Zxm{{pb2U|EmwDhIl;<&J(AC8{(Wg6BWJUj-@ShO=K#2GV6c+j7CImL;OWoTylL#`tqAnwip7<Jrk^(BYs#w@^D*_5I-wF7{~it<&bDOnO~MG*X8YGq%#al&qZP%(MnEOC>@-BgNS)icn~nD0o5_6!5%H5pHz&~w4I_A5MngRfbh$)mj9V&O@h#q#bM?*e>>djMH5aM%u9HQPf>R@1trh=``xc?w4*0^wzzLRa3eQ8uUrU{UJ>o60yxC^m3NJC*MV3Y@{u-(74oBDkYj;$(?S8uNkaeh5R!1R)Z>NiRHVdXTMG;lqyg^D$nv$HzY#a`5dJQi~5je$-N|U~oir1|~~HcAvT}h~J9ZN)8y>$IFDOUZS#S+8S+ZV5suO*R>OGy_qP^oaSJg)Qx3b>UJiY6Y`jnB|Z41c7jyhmSLf}=r%Q$)@eW;=S#a`&10T=FXsgxVsuxK3VSa74li*Qnkgi4>-Fn!UHY=o5nU~jE+KU~N3Rm<yTgmcpi(g7U~_jc(I_5X(l2>ZMk8h5pcWr27`P?s4zmyUxFiF&939DP*6KO=x7QejDjT{gcu?J&boit9lc;p1v+!i&k`MrUdE4^&E<SGIAtRms_`05XfBy%nHE=hfj|EK{@>ofjVv3;_)gu&3K(&Uamoh%yOzsNI+)StO;f|j8!G9#a=7t+T<MvZ}WhR(AD~hr)beUn`v|y++Vuh9%<+ngt)Kgt#_6}hjuyWNIlcSEu*ib0TK|^8*44%M3JxvgUP&e|HKDj<3OzB?+?t8$f5s;Q3EE)P-jPi6Jyvu!;f|(&idk#ITc|LgUSI@i{ZmKAhrU_R;2tUnjy0C4tJi82o{V=t|I0xI&QMs~_8(0a>aXm$jcdtz{3edP#8vmMJ@*vpxD7b<Ll+o&##h@6Y8wa@m@L;m)%^g2IkpAk2>e05TodJ3xOR!H`n_+Qk-^C2~&`aGYV0cBOIIWXRS>b58e<5VEE>)*>wU0_&?ggdiz$A!ag2+o`iK4?eYBr^~6ciIs*oS|*nS;YIbW#Dt)9Plr96I&a<Lt`-{Py8JXlM^U{4g26t(1*QGIM|C-MEgd$N*<qi;BmKvFLXY$p_^QNY6l%r6C?-jEGlw^E}ST{jLYb-a8ZhVhAIQHjaXN%-o#g5vb^u0*>W5uoCPo3|0ylD>gaAp*<&r*wuP?K-cNQPg|w=eu9pWVnSt5&hC)v=x>WbkXRfmY_7exdQuuqC_A=N`C=3i-r@UV7#3!*gM-RjR3YM&CI=L<Q_mU&P2<%f9Ru#HDmi`Adc=ZtF!HM`tq*9zo>A>|31oswTFB#!g#FbO!@M+5BH{WccxP1G%nobr2dX0V(=KGsZ-Q}PmZL2xE!8pJ{F0&n@TeXKgF1%^;|%j09CI+L|LZKnZ?X>AzZ})<x&jj4FO*@<zx{EYRZx3jcqd(~djt}P$sntDWRh}5-fNyK<?>X%V4robA{aCrMHN<4-lPRxqZHR=A+5<#)L@fyoLw%}@~GA*g2%NKbf8H@qve6<h2L+8c-|vAj`To4J2*ANd^Gp(buoJ4KK<H|1%{I$<H=!{j)EkwW?0xZDX|y@?j*0^Six9oaE%zj9#4b)S4kY?bN>wx;a*LNsz;~}s8?C@Ldfb<)UK%i>n4p8uUvy7P5FYin9irRE38AE`M<Ha6LC3bfNAL8!Jz+e$0L#fgFuL@qF;x@!JO}4cd@4QF@~Lahs?pgu<v)U*VAw6JSI0X*crBEQ?AOq>n@)h0c3CDWw2Klc#QVuX7=HZHuRij&G>#wC}UTa97kgfju3Yh6`1@ZIEHAQkVmGYKaS|mfk*T&iq-w0YSIyqELBb@xpe%|fK5}NgKIuN0&Ki3s|HW=M;Zt92rwGnDXkwzwY~|9v6TD2!)P4Ie{XztX?!EDj^J+;Xq2&E9WzZT0FI2n`~2H5yuxGH8yzuz;yPfx7ZBE3Mk*IK7bU%2a_%Uq=|%Mlmi71>A=)S&WxQyB#v2=#s+@lu#*bsIH!hPHZ@6{#4ELPT|EB+UF745Z9sfOf@^Dmpa#H5r?a`em&J=pL6o_L|ISY|SuFj3T-R7aASJ>G|k4M3L{H<MiyyY~HuhQHXdCH^NXes(%@=bN>=O{QvCY#S-39#eP@7HxaU3kw%b~URhMs>Ub<K$+tKm34#!8^R0^xJw4uWILJ9d7#;=!ag>N4x`nL?1e#2zbv&1t+u+?ED!2@>hD?CZ`;=X^ca*WBwg4XnedLqr%V8`47$gF2ye`1ha@fnH5EkbMFBjJo@4BfcF%r{)WPW;+j+)vJg}GEd`BP9qy=7JP6^B-bZoyI@N*XR*nBDmnm?pZ{}TG{N!cz2%zeVpvK({-S<Lky!N=ktFS!j3(m$A17vkZAAMXABZT?|Y8P#DvHpqdJ=&Thw*UoO_N9yal2G1EDC^P%wfP!?{iFW61>0Y2*AF}PKKIp+6+IC9<36&)_T%Y;EEao|pW-mib{v`Qp$`q%rv(!0BCv+9OBh>*B`<7TcuTX+%c$vMs6(Y%raBc|%K*nf`GX6Jss_}LI5=(-cijbS&8`0eQA8px7}tISxoivhN56@*mltk<ERRlcMIN1+n(L`qw+PekS8<p155vcbWcvJ$ziO+BKEfyN?BwsvY%;9v#>zrGyLH0xOnF9EJI;lPXHd{ushdpqDYs+*CCUEsv@4T8=+%Ov#vP)(LocXLgCZ9I`<pl08E-+v#!ff2Yt=hYZNA!fsQOTe<(^X6Ud5_Gpg(8?g8jm@nNQq7?fo|?vPk+&=M02k!cUdRBX!dUePCTc1urmojX%`(P#;EArH%y96~qo)<>qq#*>#(Ab6ReR`Qya8c0mG()2Vic>?$wo)OnWrsDok?<M3z?A9Y!gIOp)Kvr)$aa5D2lleBwv+jj3|BQvH+aNt4`I&jckER8o_VKS*Ib9qUGykZX)Ro#E5QQ7x>{J2X3YCJD@c-e**U*&uwHf?>ZO~6<nuvZ6*2&}-VuMVqav+u8^PhKm`LJuV2mdb6CNWq*;^mG`u>mLp8YB|;5_NOdF*QX0>P*E5d3^wn_Z1hu7bZgscAgm$FU$SJOQU!QsD0=n&<@w+JP>!?kfwZd|!U-?6>Tk;McR=FP%MY(Fe$ufJH&s;WwKL@O8(=X|RA*s!e(~YUoB#O|U3_@|&!Lm<^x-+#kAG2LNkexxI-|Xq`WK&meCNxekD2Hb;oE_5rlOH-i>lZkZVOEo4(Q<_qF!n@F}#si-)npA1jp_86R%k=_#-M5UW2`f-vDcjYcy2(JJN^?x#}0DGjf9Zg6=QW_jOj&=YNe`9PkTIdXvJNp3S}wy=Qni%(K~_><^>jt&L@?DqF2v={w;l?`RvvQ4(sOoZU<pqj^(8(<<{reUc09??dZn$`NM?7DDU9o3|LKCUT*#?Pc&*)TViizC36+KKj9ks&|*Hb;KSohc3J6pH(+47oBNM?84!aX}<8}l|QfxP0|g)!K3a|wWwW%y}SLMWAN@pbs9Z=MwSRnM$yfeTiSIe-~d%R9(vm8J^m;H>(hyYF1<tK4${~pQMFM)?;1I}K(H;`7v5Of-kh<E|4cNji*hX-xsvY+>knL49l8$L^9>K(eZQ0814w+DyZeTL!{FkZeykUbxNS|RGCp(Qj$6NX7x2D=ri15HI(SQ+EK`gN21ZT11KeeANS?z2Pi{A91+?-Vr-1m|1)_nv*a7TJ`E1hndbGE~3U!|R)f^AP7uE|b@To#4QtoMQRpd?7+Qo+b8$#5*&eq{aWB!y%q05nh{rT$S$N9I(LeV|6?sQVK5BKlQ5tr{6tHz_z<@#ka?zU!koX)id)O=_G3^e#kn)#@el^>|y57k_f6^Nw5xhC{RJJKNkdOB9;F$6rzuo%0-jLo-eHWTT+nr(~(ce9DwAUrQa7Ebq|HDu=E_rynEnBH}>uVr5xduzLM15Af#ugL`StC#jabZfOE%T`zY)N*%O&_D+A4~uT(y^RL~^Bf?0SvA=@UNr{_AN`joj(<rDPc4ABoIH6!9zsmH1^yI=3`P#rFZL;8UAd%@B<$vRNF?$0I6}<ub{LijLbSQo_S?g4{%bQnroKtieHMukJMB~meFGPT5`X3f-}XL%yvL5r_|)HWK&R`9+<6jLD)wK=x0f#Z4#ZMKrmPD$bssV2&qV>tt=rs7Z-EqawLO0r9bI$>pw0abZ@I@W_hPtG&@d~m>4WO`^0s|k3RsDc&lOs;VV*NIg@&I0D9)2>pAKsAt8l7?n?>2$p$()r*3BO6@x;yMW+S$k$7n}4s}^Mh`s+Ho%2Or82lNl#qPy@vARRHjgMn?JB~Wq&#GnuC;6$ntzeiMFQT{UlRhj4QN(RB(0<Y(#b!15+#WYMSPiNj|`dU4{$E^}qS&cVC+TmvJE~V=u8xQX-PLr8`NVe~HlMfiXQ~%8wGQb=9((BtZONv~MGYH&FMch+u2N+7cjoqd0B5Bb7vWoYE_osdOU`!P}e|1WXEx*@HVWB*YYy5H)e#se}(<EKxbcq=sQK#=Lvzs?sHH@>3B&iM>N9RMvk<-x~=gqrZSNDVl*O}e&!G)&R1D=5ycz1@}C%?M@`?bQ$CAhR1fc3e@3!rA?jA+krUe0+?>-o_A!XsNBxnC}Jq-}UY6m{!%jr#bg8jce0gnO)vE}>FoVaRX0dzgR+_l~-x^Z#`ZxlB1}J4Iv_MHP4kiLYN6MwM)?k5@Vi#+=ro`1B{owj=R1OF*17A0~9MP%7A;w&djy*)<j05<@W0bD@I(YkhoG9i;7D4@kukbJ^LtyQ-ulJO0nU7LYzt1Mgr)anr~TPRQrtTRnCsPG+m`pXnKH`&K@#+Sh8<dmkM=F}}8He+eyJr&UGAP?oOGE_rc+wWqhK^Q%nX$>V9KTmatL^3LXeM&(r%TtbDy6kJ1B!?*8F-eHr3z9|RWS>`N#=j~<K5;n%_<_ho56ki+cx9(l1sJ`|TfeOvGSg8Tl)m>JQAYrTI^xrMOYwk((W!D8ZM{m}nE|^-$3Z|fOS1zV_eb?P$AOkmXt#g7I*zzsEXT(jpV%rSelFKDi<X9x_c87O(@tyq{*>sEN!I;}^{})h80|XQR000O8001EX4C;tMA`t)pqdx!uCjbBdWq5F9a%pX4Zgg`mbYXK}b~i6`baiBTUu0=vXKrtEbZKLAE^v9h8*7u>w(+}u1<QUR9g4nlrjJpo%p|d$x^?4B+;lpl(QqIVlGvjNh9J+Uc>LeHy8uXl51lSa>kmE(T<k6ui+y3~`~LeAD+-U5rD#}_i*hMbQSr>Hc%AYJM?BVMc>^Gsx6dDVwybIn4_;pGi<Xxu_o`a#xQrKzui8eO08Q$p1(WjmPCR(2sB11OQDz7$JyvtC<}jK86((`=eJ+6xD~Lv3%8I8A+ZFtaMRVk5ngQ&=n~YZ+@id;~Csrb%dtT@JJSE;eQG3iwSt|ioTfq{%ooLG}_Fl6>@<q)L2n7^)P4G4Usm*~qKv%JvHM}@$gd7>ETD9z4YWnv;8r~dv&Etjd`-{cC7AG%B_HEPFJW0I#REfH=<S7>mJ@y|dN(G;aqJSmhTQec!d)5|BmZwcOp=?jp8AVi9YToy6-WvEfCwAbPe8Xf?w;+uZj~Og%&(fwt%wR|IG(qB#{iBFLP^BnAbVYJx@)&vY$X@*6kI6nS_!w~pTbAYJ!QkBs(2p!px{ws&kT-HX%hDn}@B41#2QFd7kv*bj>1D-P&JLxJu*=<Qf`teF-juRopiHmsb5g2^MoBphL!EM}cWUcIeNIT3)nLs};+`jF!(om>Yu;7`OL-#MiFdCO-jNJ7m;9a;Ewh*S&5`BxTmF!zX2atXZ;m4CmZ5$oyK|Cf8UpwR6-2dDAo_Q*=ZsWq$C?!O$PDLd{}-S)UIND~$weFN^IC#HC2Uohg)3Ud!%gI!Z@e`qQG8+#`KdiAxXtRt*1+O(sK9Rl{8oeKr3L<C)q}&c0tflk1*-raa0fu@;nrIM&Cq*hM$R~LzESY3htycN@K!zEY^_;p==1*Vt){r^F&JhLNIyF`^w{UtdxNFsAk?yB|G5`1?=!%?G((}6?>#m4We3K=h<G>gjgk9_$-AIG>Jb=*{BVRF_>gDKQEvckZ*@*ZDjFe*UI$<@9&Y>#Mgl=4Kb$~Kl`^d@BeL=BSN>M%^d1b8QOv*wh`b_~ji!(PKw|}(+qI$w>R2Cmpq}^j_vd%J|A0M`10?Qhl;C(VPj=&{U~L=eZy#mgTB4e!<Ws2ZiT1845@fJ=puxkK$>v<~U<Z2;>J2>MnSz7g3%Wb#Qdt@}ZfKH;Gs~u<?QHl#)H#ReLLK$Udlq?cX7DJkZ{j~J^!mY1yyOk9@i?I|^F+y)jab4TuV6b~d^UEIxc=to@rSC&Q*ge`5ndiek;UkBh%>M;5`?f(%0P(lch4~z9hR|%#RuL5KKR*6NLZ}`^&^<7<O`;peCtExfq3A5f`AiHg0NfA6<BNVb6y1&Zh)hu6RyzOX)TyDu%hJ>qw%dJCpHx74jPoHn*d(+JCI3_iqHvg31!Q>L1eby3FOh*7k4fo51awAKpnIaOD@w|e8@l?H{l2vX!RStvCpI78VtC<-Fm-Uck6JFgsTzG(Td3If}Mmt2lXGE`yB{cpZ9KAq6MJN=+lmYwPtnA&cQ}udH~A>*v|0hxDCUhSP;dRYon}V9!a@kR$;Ui1p7j;CIG-NGJ0aOh!W5wAm+;trSigBQ$R_&QRf7y!zbw+H$fPgKU$$lDMdR$5k!vLF2RWMtlwbbIr8rKd0nv6E@NKa@YBi*miQMuLqr4M0`!A%dDb3{G=_-l;BRtu3s<+Y0iVN}c3e24BU&y-`lG%1I2s`%)AN)N)BZ|JZW`QS6JVX+Gjp`YA!gW^5zHr7^IgeIh%RPX77&j!u$Hh0*5NU9Ek4Oin{^026DxKMWYcD63@ISR01LcEE2D)DN1rqhA!7Uy_!J=e3VjC2LE0#5reGkTm4E}>4#v0;IQSMN-<=en?%+AVcn!Sb`k!?Rb{%Y@Y!Y!tKlQ2U*O^qTK@-q#yCVBR)OQ;njd@ZF(fC`W44GOD5xFg==bisbAS``xbNi<lgMG!Aq9#QP@ix%F7cm^|Oa?;^;+nH8Y4`)2$T5#9Pe76M3Bw45nx<U<z}i7^pkDT-72fx1O^1`L%4+CBHtJmoAvs!IvJo(%%mFT|Gt7Wgn@y#b7;0n443w-?E1)oCG3{3jMr&ojQpcvj=(q(MQ9i*O;QqCiB<dF!Dhw)XNG6mMkGzlm%jG}5djEGgULQY29{l?H-M{-E|9JEMJ^lLBu?zZ&e4^T}2KoZ0A-zj>ZH7LsKVEh?12Q_`iAca=MqXlTxK76sAQAxfe!e9NfbS##FwW9D_(F)g7PEto_NLO7*?VW9?Ttw9A07|Wrq=gZxV9KuDplxM4@_*ovXwVjscQs%%`(90r>XaFVgbOn%O1mk5bGw(H{!=HF^r7B$ld4s1BAR36Q-<E8&JTS>MyRVjQ9gix_p@2gp(VOJ<dc(xv?;8$F4y8=8snJq!eHfHq9wPI7VrLY}d&QmZdobT|r+(*@%xuq#T8=!<m}8RRwUSTg-){O_g`azGj3Rz*V$l%M1WOa16}eaPFQJ#g3(SFl|!E1wXL#oOEE8s6V+{+r<m?t5<1K87daV0^SGn?xa_32<I>{JwiBB48n0^L)E(&HUzW`qUayK{o3z|{+eXOQaE^R$}MOk9Y%&Fpt07<rq<ELgfbD;4ZYq{KumGoUuds&PNTAYzb)@d@nLv?@L$tb1?H1e1aQxz3-Qn><-!SrjBA8+It8;vI;$AYknSzBn{JM59tw}F)n*)6Zo`%R-i;);<8=}oF3m(~6Dr)}uy*|8r^O{atHpF)hnXk&oy|0d+h3e{O|ZE^NsO|gE#SINLq}7`x^z_rI-VAcGr<Yu;~Ewn@z|7M)Gt+xi6%S)L@_0<a_xWf?wfa(g#0=6$F00)bq-DnH?s?a-0g&*WTccT31N~aeT)K@(+On86Z-fZU^U?huyKIvWJP9hV&)@WpI|V0RowbmPhOgny0pT8<^79+(A<m)>&(0h(^3Yc$e6;iM+v@y)%O<TstTzU*zg0@(S{}Y7)zE35eNv8i{35A^_jA^J+9gBhQU`<=U`l7AuHz2p`kf=OnQbB)Y#k%<9R0a3I9ENHsPmca^UN<=~}%YIQN2qq7F=xC9m&!)>FY6lgR@fPwMFw57kWv&%IBzc1o1R`3tWt2QU;}LT1f!$inAnS&EyOxHd8OX^9G1fj%X@kCX~JomJ~xt94Q;G)Y@)Zhsd8DIG9N5y$0Qn_v>(bjT4+nfqwqI4ALilcQMb95B9`Wyd~EWfjAZ^e&(nn!-sZa2{gaPSgLXWkueoWE<1od0Duj8b~tfrGxb!2-eV~Y`td`FZEGJ_ylo0?$IZi-^@lO=yQ>+#ib=}CYFN}JN%E#&dUamWl4m!))4~zJep4TC9Y;;T2H0Ii>v-WDfukyl=9C44=yh*z?}dH#J9#`;*4pT))(U^x=Qyf5p_&30em<=oG%|0kSg8^tpO|LZJ)n*0!zO_Ewn)xR$Qj<K-`#dFk?NwKc5%Pu%L*AI=ZfdL~gG{vew}R4jkgMC`5wKvnZ!UQ**42j!Q~049~Mi!De*9D4Ifr2^!Z=FHJc5)FV>K)Kkp4Wua-ceW>$`ERe!+5$X!Oge3tbJc&#-HPFI{0=nXB>=Q*8j~gh}J&|?wIk~yn{sK!j-GcGk*f316G@Fxo?EK7*AqvDGAj>nkzQ)Qut}OFQ`;8p^Ot3KvOLdN^kbSe7+0p-rIkXIX#tYt1X$1=$7j_V0w1-@-Z>AaQQiUF+8u}GDL9nCZ6-6+5@vBFps?BKrc2wP(tXExxAU)Cd@opLE0^RT;N-T?E(s4mnv-7vJ!W-MQr;2adeqI)!ito<Xh){#h32}n@|67bbGa6?yM(I7BV2^fW{RR~@3;8VTgR0x&y5*70o3oL9ErFHi;E_}6W+sbS!w0tEXI=}L6!{$=mhH#$=F1ClTV<1FTeGh2l3=Z_2<GnvoLT%-HRm<faj<ns&BFCxoV0I?Vr{_}#Uy<AgsKo>iuz56JZ+ziO*Mj;sM%N*tE#1nRb(isXkKFHhZ&hh)HO)1-6qIjhMaCejTV;z;Q1?%XihY92vx*9ep^E%B~FRtJac1fS%ICQ@k=)aX4yu)`dE5x7_xe`u?D&x6$HB)>qbX<+tHP6FV#%lQd^zELEi@P9K2sgY)N-b)L>Bfk-mpgluf@Y*k!xdHSk~E_+jJ?#%}$vYiTrH8q>u~jn`au8DQOvY6`)sy<FP2+I`D)uP=7%#`5Lz`yXDtgk|7$y5SJD8}>)819#_e3_FTKtFOYUYe%a7DxLavjA0z;`RJ;Cg0m5KzW3R5wUM}84D1nc?%wBITtzggab0pG*J?S@;L4oZ{u64G$DMZD$@|3&Zqtbef2CKCv1tSDc5pFz;3Ya<RWf(D?%xci@CTl@&WuVFc{;?b#REWEGvh`i)C&@-niLJ{m;@n*m;5Mk)d?V{?~tkddhHE@I)jLCT${Gd(aLGIx$+iwF%{xiT_f&g!|-ldH|dB|1)(!8O7|u?*1%FytJA3KzMq_9e?_wFK(!q)c3?m^roJgSKKdIG8-MrC>Oxhs$y-=`PD7-lHYb-jSY2~l-6^^sVw&uzK62#F7&coNZnawU=Lm7x8|yA?&>5(99o#P%Z8)Q%Zsf*kKiiVItB1=B7jw=Sh`~v9->a578v4EWBSc8pmSRlnQXfBcS%6;w;YT*AD*%=2tG91Gto+Lu47W!_GVj%`#-_d!q3?Mb5PhRQ0Ai-4#p{vp&LCCM909pH#j&ebedDv?%<Uml9>Df+D?R4iL#iP%(*xgOYlA_i3<64d?V%&H+RJ`%I~G}5r~OLDHq?xWZrVD=dv?c@S=ajdcI;YTzc8+q=G)(rSaFmy9=PF?#iUKXXNtBPg^yi%E0@Lw$5fjSKB0I5|J@I7z6C$)zx(=c%fI5o*YyAEZ<jy%VSh7}XX9##!y1SR^t=hk<o(ogs`t$~3u;xtsu&M9w{G4Mx21&X{AV5v(g|=4t~BC|-{+}yAL{HS9*1?WkfRuD<2T}tV>$#!b+h_&&xR~nC69vK=+uW*m!yD=K@bbMZY3WE0JKXZKMxe#vCfIEZ5O(+oa;7POyB8OLgxD86Lrj;1iD+{z@{lCxTQd9WV15(T=k)Kd>WttTa1F6{Qhg49w_!A&p>7pPYWVm8*U;GVzml62UT9*#y1xN^xlCzv19w%wnDqR4Y>0sL=?^3Yk!=tHLAm^d(}XI>ZsxeREtXmlKvRK$9n<vop^ZPC#oQzo_9rC`_p(n&0UsrCPCKtJx^lE9u^DIpX*2Gz!68s+3<Syu35Wa0IBd!TStg$&BO8S^<C{5FI-zUPMch39`%|M4q<~~;9^{NG#5pu;t2jfk@?SDw<w~+2p<i0^X$4P*N$bW5Bdoqt(A_dX8P59);JR>;gpME;;8Y{iqKY0cosbpjIAyOU_+^x@0(dsobdwyl&s&Sj?cP`X(&#RSTZ$vqQU9q^38Yu!u1Oh7c0C>d?+ObOeb|q^hN(&m~26?_ql|S$?8mrrm(*L>h1eih_}SN4|dV+8n(ZC-s88@OZ~0%iIw@D%VrAan{N}k5=onqOPPo|Il%H%dvo_?b%$2h5}&gj{_(0~%_LXB^N|<XQnU@G&3QJ}J~l8Le%IbI`fxpRJ}4i2e2(4H=KFB`KJa8LD1Pa6Xju*UHAn}3v9#d!C*&4Cniu^w#kDc{<*dm@=7#3~1yD-^1QY-O00;m803iS%eYBmW4FCXFDF6T)0001GcyMKMX>Db0baO9sVRK)0H!pT+Wp{HfaCxN~ZFAeU`MZ7vrhN#NQj(K)*PT-J=A=&Qotxxbr(MsN<3S`OaiIv6AT7ID_S^To03awzcGl=F5d}VPc<{W#!{PAkc3b3WUW?0D&&5ryb|TfKsjDWfb6pBqX5z)g`CGA4dZX&<PAqjLu1<dzH{VVMgO{(y@5G{LDkJK(5-;CfT!?%lSBel*z1eD$*NWd($}~kS-n_o}P2{D?Glj&|s?trFO$IMQzNNZR6=LLnC=-jc(p&f{b+wTNFM$}QS(rMn8)P6$p*Gw4PHd$4Fd1B^l}zu%*%ML8-FTsp(Rf?wR2c)92$YvAkprU0a-dSCb-PDUL0!qTo(zV=;b5?=^hPAfvZ<R&B?;_stE(DVmb#WGt{DvM+D6uEhR<Xz(?S{p!Z`$!<!K$auNbnv+mb8{bXwkdYGt$8-T|5@w+#CB{KCPVqnZbU!9P5;2tfW&<@~B@)M!9U;uknkQ0y{a%?1KKpj86`&jhGi%*C*lW_<JQc)4oEH&2EH;<VslyxNjfmN(Kcn8{ZgnN+y{nOGKD)&P4lIr)Z($i|plmI<6_naWzVFkeqj2xOtFOjVvsrIv+C>nA5xj8qrE(QIX^JPg?A*x-MvOm(KRag&$z-x(f{G=ZP0+UkN%#iNx<f+f`qM^=&$GyvvlyH7kj1^?x#Ea21;I8#eFcrfA`NauBuL`D_M5$h5^`BLD@#G4M71Q@sY*w<Pth6PzAC~Y_V)Y8CDC3B<1FS2OVSykyO8onz(lzLb8jdC~&xg$Rme3F|bil|7!NpLzNa+SSB@trt%NP-Z2)0kQ;z+s68d3UMH682N)H|n!Q>X}3<SnrkiZvL=*eJm#>MV+2p#EPx1J*L3yVsC`}h=CEPK73R^H(PW`m06oMGtrWb-7`Kok{!9C4tbKYZ_MH^`-bd3PKiCkCX}vGJDkx2Tk5>U&+x@smMhpH(g%JUSprD|Tl&4pD=-LH+e)w%Ror2OD%3`ms1ZsjHtKZc?w?KI@+A8acYo+O-jpDN3iBG_QJ%txABYDB;r+bU35Mr=53dwN{nEy6ehE<*5f*tVE7o{~|IHvwPs&VImAs3ES<5X%uDStc*T6ppd#m4%#Q6IFjM+ev0r~R`O;}p8;>X)xS=o>(Yb7(GK}&f7V#Dt0HQ2#g7a1lFqn9;-nhB;Wq)5aerU55u=m$y2DhGAb(u71rO_Puj{S0@$ui54BClu&ofc;5i8WNm!`E+fwKZiW=kzs~=d^5;+#23U{Xk8>Wp47OIo4kyy4zEvQ@#OtzB8wurPcR6U=pv*84gwe~B{;Z7TR<L4meZ5z6LEuB#!BSQmZXvf#flIZ3J$%!o4=HWQSP*Tv{^Pqk<d2TP3UpP`FKMA$8CbNX^&SP^Vh+Y7(b8`;{lmYz-8u?N&-wa;33XyhWF^I<2*y6<m0Aunu4+ocrAKxt;vVDhU%AyK~NbSPXMS3j`#S4*$00d3HSZ;JqL3F2NAFAAqb2$kQO1Xw8q7x=Uq~K!mJ`Z)X0K!5tZIN9R!Lup!AO(H&p)&K_}1OT>fx&nOvT|yf}Mtm0)!5K_3L|fVm{F2l)2l{Ke_j8H0Xuo9gB(mtO`C*9~4(jN>d`W_g3#+{oJqauXd>s`4W8XYrU7IbsA1niW{VoP>2mpTlZvNkNn$MUUF;Mo)pBW%`u>hTY!*Br*NhoWAx}f7@T>EnR`T&aFnS`TIS?biA8g2Fqs_(e4QL4jwZncX2+RRthwbppBdRox8z+lk}qg{grVi3Zz_>2Nk;bk9e(11;F@a9UimJH8@)w%JAOj4c|10H<#zHrf;7AbNu7<`Hxd-%pfFU-vyb!P}L)7bqd&Db~i3``T>rVgY<+Q;Mcrfi=XBvGocs%1~bM)Np4@r^g}EhNl5py08899^~57RVR0-Xff$BCM5JV?5Q&_-K@6kbNP*Vb6OfP#WTIB;aRv^~?cfr&umBQ1QpqiN5$qwTI&N}l@e`JW&T-;2_2;7HFf<8Q?XYHOgVwFm<qAE7j}e%-C(ev?s;WTB{;;FKO6Fm;PV~0<Ex>mZIJ=hbb>L-KfV2h~TWn6y*R!YZN5SzV{4y4(PHw#u&&?uooQ%6a`!ufqi|pQ`adW8Hr+D3c-!}URm16S)3_y3(z{?U=c!;oEbsCrN!|+bI*b)m=)yZ*36L<K@Th|Uh`^+DU4OAyR{q_-EG@T=bUZc{7=kP_){1_jpmD{I1b$Vrco$yK+3WMB+;Q}&Hm>7d3MR8;}>&4DJ8ww(5fc*RrX3sc|VPlLqfWNof1K6Kz5bwu;qoMp*7>5|`L(YuKN6;%(gGZ_`h%ZzcU!MNzeAm`XEDg`l>_`A=_Gfq^SfBbf#j&&D*!`qoj@Ur;%288S`3k(g04`fq<fXDl3!$)*JB~>r;z?K`7pmZ`LoHX<8oCONrG((X3eO9DtIP<Z@2MM~8MVUJh<i?1VC1q<45ZgfE@0QP6<FvrwFwp7wSv|VIu?%jVz*Xp@aB5@O9Qf<?D0-rI^ZP!8KIG*=wz+()f!te`nAimdOh+1$e;X!Rb(ejw?W#r4?Kr!;r8R@k^1C-dsxd(+L@A3bq#hD`IQ+C4r5V5RZQ(ej6JPvih7RO$2L!vV;lZF`edc*2$4Nrr#;7}%+#_>Rm(A^+N1@e1P5efT~{I00jh38?S9;p%MQ!_e~;}dg58vR?+$v1g5tN8)G1UH&~?8gSAAsc?0F1bHgrK?WmA#j$e<eIEV`Y_+3x7{^$&LPO!R)r(O8YRs=#p}4s5Wu_-69;-zHCaSC9(q8-qGam|LQo8qAuLRDvtq&0`uQ<_R&6pr{1r#re<*dV?$nkHz_lnd;>dEQ36n<{rk5-c7^x(x7wbu*<i3QRp3RR^n9TneCwgQP^P<l2Qh0NHz82cLAAit4g9)Kt8L)4lBWSzMg5jBldaP0jvd^2!nxZSz3Ir-W(4C133h^U=RZAD}4)Ug7q&AD6br^qt)G)#*}Kc0Q6q;;2|G1;PYM-;3|km5@576!5X=saA5^=*?>$1h+G3FF9$x86s`my3j=%Q{=zNNxubYx9o-jTXe7$eL)VO4@&dq3o}a&X2mjhA`T#9&AwY}*<t*s=XfqEGAl%;EIS5n;U}F@|NNZXep;3GsI=@a-wt9HP+n<M?-EP?`DYKKQe|*#-`e-Ize|iQ1%lPrIH|za@%dyEQ(y?cQEUx#&e&5+Y0e{Zm{H^yhAKeJ|p7A*uQY_fEKq53QAD=9Lf<XZosw>4l0c1!c(TG?-c5kK(*-b0=lM0Bi2vhqG`~}y8G~T+vwR4leMOt6Q?J;x|b=Zw!?=y4v(8}8J==O&r)!D^yx;@`|eRg_z_Vd+V4}eiLfdVe{=g;!HAPWCk=Of-<=z+X(ylNp$JbcaKUCtNHZNl&ZS$$5KW-A>5tobH6*>!x%3Ic6hu*mpb2V)xKV#4v<eU`_)z_i`w>H9qGuXM>vd44N1!<qFDRq1EKW_P?ts&v+*3d$A8;pT3GQ6*0i0dE8*0Wm0Owor=OIzVTrJ*Yhh6jVvYCJu@gx~}zRTA?bi9}w<tFQCO`2Z_d_h@EbVj2Q?x4;(n@?X*x!a4e(SIdNYyP^V8&;yb5HAm64ix4iCZw^X}i7t`0;4l)@#NO_nk-TFEXXpXv9XS|EKh$5mY#sLX2or<TB-I>uiBtoycq*$K`0LAXvmJR*n!43{|BPvcT54We$fpJ2Baq$rRF}WsNC?`}z67t3mLQ>(uplt*@j6tM2G4F^LJQiC9*a+&SJ^!&if3Nev2<$t19$kOt=y&a@#>7B;C7eMZC;O`#<M-r7eh3xuO_(D-8ijC&A-I%-wg>+XqkDhgZ|r9W{)i$NdH4%S>benny&R`Sz8!PRFm`nV{*6X}=gYtK4v;#<VCTYtvL3<s3>dt3TS;#3Ztw6?!u}3HNMt5)q{i%Q2fG(vEY5sp3t&jA0f@ia4q#lnL0U=d#1~+xa|o|VY+Fmmx_9HrCpQW&_wY>ZaoIK$ILm~tFVS|PSJ9hex4Koiq6LPUyi%pAWUZ@X_qu1daQ<UtKz|JdKS0_Q%~on51(OklPHB^vfHDQ0RR+d@eGZLq+=%{a4c!pgl&1MQByk+;5HVm7Q~IT$<kVfz6F9&?L_e?HAV>0*cMhRBUo?adtax5FF7gk`UM3?)cY6R*r@G2Gi)^6_X3sHdzyI<Bs^gc>4g{2yZ?G14dS81+b2XPQ>z!`Xs4Vu{puHZ=EypcKzTze%NDEj7bYIz<nWl7gW&66>M6R9Vzhk2fVq!K4D$Rp-u0e@bZ;-?k9Gv9`bu(75Jz(Gh<=sIh3%tppY6i#BJ5GA-l^zp@YS>mbG(P3`=J%@R_apoa{4l-JI|Ff3=BUjqzk!C6;bO-vWW9FtW<-NXi~bcMG6Qj-C)SYr?YzQ7Cs6f5Tz4e!9+2TFdLKsEd4yenz>2&^J~JEagIy50nac?l8|QCW^E&OLw4u&D!Fxy{PKQqST>l49O9KQH000080000X0Dt90I-vjn0MP&d02}}S0CHt<b!l>CZDnqBb1!3WY++(8VQpe%axQdubZw163c@fHME5;Kz;zO&h@f<##ibOyKuMDy8~Du+NlMM_O<lVghT$>qz=n_=d4-Iub=uW(d)^-}mNZ3(EJU&n;mVvh5TJ(76=epjS~0VJ=m(baYKmx5P`Y$oE6(E7%sGd2k@D^GyiP*XBH<H|Qnkq+$rmz!Nk%|W36eClc_d{d_;I(78%!g<`F?o(JMvIk?#TmJzKCtc<_l0u0|XQR000O8001EXvX8%%O921?@c{q;8vp<Ra%FLKX>w(4Wo~qHFJo_PVPY*}VRL0Jba-?vP`gsYFc9qZ6`G-ru>45EFe(@*Va5d%l(Kw@HIXI7IwzQ~=VVjf-tOMsKJYi_t<eZQZx<Yb=arBJ2Y<6u@FRG9=AU6Q1GkA=pHMqT+yqfiq;d;Cg^9nmUyo*%r4$S&?Gs@mOi8I)RE;PZh6{N6yF!JK85-vE+4}QI8`uqVm+>;=+8A(<y;rCfk|lit1fRktuSLTW1p{v&S}+??6wJ?KT$GZc+>4sE*12$}F)IodBJpp#xIUSK5nhO6&EjH$2^fMS_cCM5JG>}Wi6=Ua@PnNTYe0FVbqom$mW<tQ$((T17+pYTyG>TIsC}<j!%|c%rWvhMYA+;3X_xPpkM}%LZZ^SnhJ-7(%4EFuV-C1KkaN`LqH|HSuCwogL{{o3Dx&%>H?3)I@;^28f^+=;<<JPa!0rQpTYDo8`vXu*0|XQR000O8001EX3YRvZz5oCK00961ApigXa%FLKX>w(4Wo~qHFJo_PVPY+CWpZO>aCB*JZZ33qbal*4Ys4TFh2gz_rBv@#qj5lJnN^_!h3>jDencVB(3r*i`)0c7s=M=^#d9qE+b#EA{BG??1(iUKzwf!{wpq(5P~@bSb+s0#c|<I=ZTlk?MUr~DTdPh0$z1wV?tR-YjGa3l6ZLInZHv_vc71>Qpy&PKFn~raM9_%njR%woxKzp{GZ-8>qKrcP=UuaFwbc15yE?O5o5~VDUvIy5(x6;0UdBl$j;9U5zyMJy!N&o|lqNL<PJRGTO9KQH000080000X0OeApZ(slb0BryO02crN0CZt<UvhPBbZKp6FJE72ZfSI1UoLQY4ava`gD?;Tz`v_lT>%z(M`<DrSOykz!km%MB<&~j9T9o+_B>ZYZDXe`#67g?Q2G;kUab$cg0ZXP%FB@pVf?V8w!AvdwzA?&E&K2&A|a&oZ`Woi;RTuEBTf>2P)h>@6aWAK2mk;8Apkq}{&yA+004(I000;O004Ahb6;|GZggpFWiMfJbZKT`V{~&aaCyC3ZI9fx5&pivg4LHw$)3&$+9FtYiyE#Av}v3mPSX#UgRGUPTl%z;B2sr=U+=&78NTRUduuzWfy66`9L@~q<(VPvdET2toabxy@`rz4ltq4I6|bd`iWdoIRV@xei90S?R@SWMRb8e{B6c}v{5?jB122>eSF3M1ubA3%7AG{G<vhLNH7g7K^5^e<FIiqDan4d9Z$tJIUiKF{fGlWM%`O*m&*iG%_gF3|>r^sPFmX6EDyC(VeN)^@q9CN3wvQz%N_yfs1D~p_lRf5CWv#-M=XtADR+k4BMOmYonn#iS$l{_XRjfo=$koceOJfx$c`PN^u&=PdJ{LRp_)eCE`+bPj-u*76S*OA~40ntC&>N!op~^(g+h=IdOj5^66gO_pi{iLu-^CTZTdh_p&sdbiMOg?a(Z7rHhHq$4z%Kqme{WU{e?->jJ`^%)AI+Qq5~t9WVnT{Ssu;>QU)IC`4-6gpqu;XafM6_Bktj>btvIDb0{1A@97->kRP~cHt;D^_7#0^tf5h@Fhk8IT&~}L@G7>|)o2u82h2-ofi1w<k%i8x|mPICRnp$r<c2BQI6ktt_RBXqIF5XEm7)$VgVVnLw2yAD|!kW=z%ScF1(hnY&rS(-)<@g&muY#$<u4Y#nEONa}Jj=^ib^A?r3WLZ*0RsM5RYpxe(_5>j*JW{G2D73$?BJ^`(cZ#D3ZEC*5u|%Tz-xEa7!JG>BsjU_jNnl5n(fLGfkza|F*{X(9>1E~$jtk9A)AVvg{LeT?5cH-2eN1@Y8XXDe1PXaf>Gm7t#Tvs=dn+LL&J^bgvNwvbEu@>GCB6hVBLO_;431l+Wu5S;_f}KVsKg3a_f6*Z_T`o*PF&66XH`W6Ct*Lk8|0ZiZET?LyDsRVJZy8s;7jp@hp`4`0GFZ$)D3i5F#)mp!!~;vWsuLAPD#TeJXCaRK8o)%#SFb`4sdvv=aNT4ghNtsv{{rfD`O8V#y!a7woHNU%S7CycyaUHR8?~*1b@BO;;}zgk{AG->Y{XOk840k><QJA957EyiOj9*M1%!c4@q^Bg2}<soyaez#2Vo6gvz<VN)Rlax*ySYteq6swKh(A4C%Ej*80}#ZXAyF#T1lfZm#=X)H9*$=SXQVK;EI@Nk=o+BXl@xZs5dAW?a%AI+4M%cK?+3~jA*PFctG5tLp20Ub%d{i^1fc)!iO@Ify_Uh){iFtN<Gx3D3r>SL!Py(Og#vs7<Je@X9tVPoiz-Gcj?A`$5#nERaN4Geksi58Q`BJtfANK%x3(2c?fpp-mDtZBtwvrd%=Y4D@tAirRa7F4TIcJoH&fMHFMi{iF7qHhjF!y16yf9*}v?H@U3xBHgR)O-*zXML1?$-G|*Z|bX4KJpnx>hfOB@d3p2AKDxiBStqxiU50v3z5ObFQHuxoHS)p<(d`z4p0x-z+yk-RF16ku7ogg9uodql&;mV5!3U+ccm!C(t=SZs{x|rJvlb<Kasr+iyE}B7d3Jf7&BSp$EF~}r5}P;*cV{k4x_v!K=6RXtl(NepmkFrLt)h&&r9F}z)J{Q0@v2VyrU&`>%Oi&yo{4QZx{KQ;4U&^N<<65UUd<tY0UwwQcAHE5^7z?z#$bZpZ)XA_pjN3tC$>wgr#JvJ$I!ke%m4zq84KSHUn-?s)t4dAiFOP@eQ|O3cJnk1aTnY4}jMCw@D(%;R&yF>Xck50zYeZ!%Hkr-g74a#_tUK2sSDSgAjVvyqXxWd&i-=l=7`6h=_sZ=p~6u5QO5b1dI1HsX-=x`_|;MN+h>AH!1Dy+t6uiWuq8^X+$U_ihRlQjBK`4eJk!cDGdneyNjUoJD7exCx*7mlU>@SHTs)+%OH1+@-U`MP-MsErJZ-R`T80UnfG9xAH$;h&{ILxD1RVE2MQcdJ(!?3X5*mrm4;dvaor7ZW;HViTGg;^tN*eXV=Aw$Qmomc`l?p_i0Va=TO*1*vR=#83^wnB&i*v=M0f|U?|9mS;;?p@B4NHQdw3iqd`eww2S44C4u%hx29KkP!c5jB1(Cu>dPBdxOD#eB_{%Tt^N{GgOgah}X7rq1ogpa7V66iAL>*`9j&?sGY*C^7(hvPAs$VA2w@^D_2;(zOc@Br)8~3%*hkO^>+=&!%gaqv{KCBzEj2Ucq<ch?*2CJG<hiS>PsL0wH)4$<r=ojF<9V4XHJsIC9`{rq6I9%)BZD`!KlLzA^ZP%9h6Zwaew|x-bzQ#5`JO-1g6v#S0LW1{xKr~tv*mtut5$~ql&>z}aq8RaCn|NW&Pz6TQ*47cbXr8VNOyy~8;8tK!?4Oq<-Ym^v_U()(b>dlPTAd~p%=wx;;V;7^J<FE&f>mYJ<gqUIv`<mlZ)?&7fbfPtcUcOMUk-Ai_XNlN(5xf=z(Ds%UP$%`ID<)^1Q&<c+b~bX^1^J*#obrldN>TUyg}Q5h)>7pjDdkODP%O*+3Aq#$a=dukJI3JHH^^LCcaP~YS&~LAML>ijTv=JfY;gpfGr3$DEGcuO;v5nn*}w0LYea^rH&Wcomwh#CWX%D6g$?zWqij!M1(61;<FXZntky_yD3w*I#CLpR}P(163v!HA6FWkERQB7l7lsr2xgF#Zi-z|qp^ow*znoml=BeDX|uEOsW0caiD<Z|1)new>~vnnsV&;32v?g)Tqk?fNmCWJaYi?rh>-N{;IYGosx$>u5T<-o`;Ey$KtoxlUKh0j_Pgyka8c;!4V@3fZK#^YR3bSOzBaRj=0bmF;gabw0uvpCJj5zN;i}sckKU|!qcx2PS32-s|8`7KzBz**i=)Qbq74R=2b~R=L>?w(b@WfCV}oe*B<<&Nb~M>C4W_Lf7|dk!<D7*%faMg}CYtLWnxj&2sG%8HKOX<;y{j<c0C+c72~4$K7~qic{HQBPoA{0Du>~N69bo$W1i#zj`$sODT>VR*LA3?=8$iUIU+A_|J!(=I@11m!MQVPU3KsC3OIv=wLdXVUZ;}MzJZo}C5BODwWPlWX-jyFwDzl`9)sYdhoX6B5L9~IeSQqRn+a009u8l@6+Ostf2e7MSctQbs8hz3Uo!ZBWHPoay2t{xR<$rF<r9+8o$rRB)P`zI^5)-<S_DGGo?n&OHgt#+IChCg9w!$Y`*R*s}A(OaH9mN-U-a6*Rj>om0E_j~yYMQ31ashg~qdxhQs!6eTO;ParIo^Ovy+tdFD|8iBWOP($S=6U^m(+8(B-4{zT}o_H!n+2>(_I4Ob(AzEV)UTl<+wM=X4E-o=eYJk`yyBv_%SF&&uqimpzLsO7Y6H)HEk_SXSp7oOLd(<m&|j+L&_~)&pj*MrFjb6n)BQlaO>z|;DGCeVGiU@C7i>n2xm^Hh184@y=2)AreF~%{$BYSq{2|QZGNQS=*0evf+X{P<XIzi(W)DU2o-cdUF+B*`zH7>+>Xan)d&Yl7;@EbCOZMwtpp>Zaq(=<1T1IT0&;ISFEYJgI*Gg*L#-{lx*m<DGEAyX50y<P^|~`=k_$2YX($wC$7&SSYr~lr^i5;L5%#kcW={bpk=*bEV=&DfD}A|)=<oLkn2fx81_Wmr0+v3*(uiDf6#di04w%flL1e7-#-GJDBpH33$16KbzMXuTfYVm~@_q;}iSj(~fcq!6vN>QpW*O+{-gygCOkaBu`gHKev=(Ms4YZVPG7P{>yYejI$0ab-3Ngiime~ayINvEV>9eEi(BOhTz`$}h2a;ZSlK<M!1sGJ|;1JcD-+5!O+z6`8>|s#VB&Kn7zV-77QsX~L(=}!DDL`LV!AS3#^A4H2M<;`oZIL}ss#rC0-VUx!SGQmfUU^H0>zpL;9zQ`^0{h^_tagT!t&i9!XX>kA;L`&}0<YOKI@<c`6IdifdbGDQp5m~wXwW4F8^Px|S`@LR(QDFbRO&ulxy`wzrYLQ}M@Y3eQtx`)1F=2SZBiP$N2mt!tqI%1sl7#7Q`u~N;&+^^>1@sZYVVr_vvo^PITT1kQ~;84ioesxvQJ5W2FmEr#!GV&+0%o|_0}9v6j0&79!wH?JfVfIVw(G`%ZV8Pk$K40Tyw$G3mSFtK&Po8{Vyu$u2Z-Z3)&wZ`_kk!TTVVUGfK{HeIw4vaS9<Rp-UPkM=iC{&Tpli&bnRb4ox+v&#f1R8UMQ)`QEk3whMDy99fuk)yAIdQ$XT8EA8*Vvo-1G&8t`6Mz3D~?WB&)r4B>*|G5_}P&9g->njcs>ZE~apwT2a&UB6@ZOtB@nz^_T*z_<6n*{SsL$T%DT{=kTn5VmPzX`8nMs#aKLy=MF?ie%iWyBSAkdGv@eKONcxorr{D189gZE8IyV?UiNyzi+qQr=$b`R&zV>SeUI1fT0(M^n2aSY0!aZ7EEZx1wVD!o&z$1pTx2g<5=gjBz5>Iue8K^JnMPPMP-eXtv!cGriY%u<vIgz>|jUh>x7|ti0yE=xp6(aN7#n@qQkE&JsAczK5jmgZFUf(jFPl7`HMU%HE72_Gfe{KILf7{Tz+3ySuTwIQr^>`eqJIwiGRl9L=?v8FPn;dNj-#$Y#zu<{C3PVzf2c_0%H<nQy7gj?cX$c;Y_T%-y%&>@wjK34M^r-A0Tv_$YIahRvny^ODLqJWCs&U1x?82G+2rPKd;Ft(`$ep@2n${6W;xl{qX%XEF26B(F|6AH}(JtpByDVFk51js%I4vME&8>H+4QTnir;%41zbQqY|qqY*e&t@I7#GxPu<^{sI|DR2>)ZZF$gLC@Hqw}vGB7%}qH{ic5u`cR=o?o}xT9WZzw0V%rWu8d^Ytwwcc{lL|}OnVxUfbTqVR?^)C$mMx`^0m{de(G+7dHqwCUPhw3j^^luyVcu+2WA?k`ecAwJ9U%ONr%*L%s&tsUmMl^GV`yAw#0PTmhbo+q*F84b~K;NP9z1f2o8U31YraEXASy3mf^>c>+R6sQmtLgGTo{w7_r_qG$)2kUswMFP)h>@6aWAK2mk;8Apok$K`T}n007oj000#L004Ahb6;|GZggpFWiMfKb7gdME^v9xT<enCxRw5|r$CrLl1HL=zbMgEc^yyIRpX4y_DrhM(<Ord*kW2uDkRnIo=*SW2iQm4C)slVkRo+!dF+(kuCgQ&H~`M|JHR0g!+4YBx!_|`icLk6l^|>>*!8-|@|y7CE?*UtSL80Mm&K+g+0D8r>ny(_S)Phg<V=t?W!LmdROVptTGW)&nvy$Oky}w_X(l+KS2WA3n$$}{iZsnwMpxvf;9>=%u8S(Gi!u|{v=l2rD<DMvF0!j-JxGi4hSn9Kr63jDWHnF|AJ;1&QLpaDq$&tq)uPO4Ey(*Hra!Fc$LX8<_eIH;gcWO`Dm&*zJ)n$bT;w%ft?t{E32u`2FfGdJ>{gJ!zWJGyo4n3$fDbe$n@ZrUD_mr~tZ??LEEnd$FpR+<EsGnnSfrbJQ;NkxGl$R|hNE>><keuHuPte{TxCi7v#!@BTV*hv=F&f_B5%GIm7<M<La&DK8~8aUZ(y)Dz_gF}v*}Ta=3BMg)Y+>21ruq$*WzZKW~;U(uCn>QDOWI%Dax`acdo5zS?#tf#lJS9YAOGgDZ+Bn_5E7vhr0DV2Qj?>o}+G!$tw^Re$&h|DJWA4%&*$^`oc?`$(Yc(xXIW8nKMR-6nVzzYJuz=4E{)d*sM^8ZnFxx4V>o$C!%z?+T2uQ6g}wLEl%kM)Nr~2RjH6+T75xnU;p%%mp?Avy?OcK9hsAd0f9dQW_(MHnf@s+p|UPA4i}A8wpr4m&L|MXhLc;{^e16rjJwU%ys*9JMK*Uri!sMZxz9XqCyCEf+ld3mv8;eG9y_kr?T;&|pz}q03I85?zuJa5y%9JNEAlkEGLc~&T5VI1AGSjMt7g;ba&vVBoB@UllNHlrs{+fciq)-{p1X|c6Vs@c)bRtHNov~;7g6L0*S15O+LoVM+)aa+$086>Xj#;=Y%fl&*z-~^@vV@%)U(4V7GCTeq@MkTys+)S?%#bJe+nzFL=MV|cD(pTxO)DKwhR`Y?^=E+Xh0+2spas{b%MmDoF~9@hlVs@QA%l&hG9s3$7XTra6fR<Fis+ViouTeIs9GU*XXAgG7?NI__HTPD!({pl6_T9|DL&Rc@BD+@{|THOI_Qu;*iHNr;$xvKMJ_-(Kw`>*&gV-MTO<Ne(Ezyod6gfK#vnVkK-^1T+jETFUxx88)m)p39M(9NMcVUZc6<yq8<xeKL}GtFd8HlWobY?4=llEjvrD+EuKbEn&K5MLPwlnz3Y3969$fNe=*kMddw{6;^SBMEuT8Cj<!c3uo`@JC>VGTa5jD%a4U^D3vDa%0C>4xERra--GoPpPtz!1PT&O=I2zYJ!E%^*@L7sptp%dZMO|EryjtAQ^;c)~CroL}<Is*H*G*F=pezFK;R+TB9s>)Vh}y2tg#*q7Jc>nAKTbT$ibBxy#7-?C;v_u5W6%984$EZ#rP7p|^EEkp##FW(jfL+AH07S>fLaC#5EV`m`QSg2gvU|p!?2zk(#UdbngUxRA;L7VBO7$xJIPUS(vJH@nEI_CkFzJ#@x-ZR+n&z?F8naH0y~cFG>t-AL_W7X%VBN=!>4f&aVls83d=m#X53<7=vnCr28UjJl)IPetG;5i5d1KXFV?{GYznE^dUBi9lNBwm#02s#wzR#8>juXFJrg$gx7c<V^%)1Z7tz3XSr`k54<fQ6#%RL0{dEETlgn#0;oS9{kL-Oru>s>y?3M;@%sgtjA-E$BAm6r<Fm#hd*gOp**N#~T5yOW(05p*@2-Mao1A~C`Suq$ipe!KwtBTLGu>SyqKV@=TZga<XBd{i*={TJEUcey`urzjj$e$u`c5akJf<g4P6E6-J=PtOvIF6kFFzG4C>Ds{;1w7vdm(Q71$C(9|3#utBfa${J%;8>|K*UAp4mclj(ge6FSsn^IauPdXUgEkAAbLBpPGKC#dhHk-USyx^(-PprQIGbHf|E&MZDN{8e*KG15?YC4jx3DA#N~h>AV>6KKXekXSnfDc1oP)UwR`~0R_J_PM|Nygt>DjH#`*p$h+=Ai-q1LXz+0t^auBBz0>T9o^Su->mz&rC3TSMxAWSUF1<?l~=pVD<=#>BR{rGcCN-1qcQ&9tiEXwL@GW4v;Y}x9%VH7|Ra6K+ki^X6+F~9&PVhJG8AOW-*J9Z4&FqoU;r#5gCplkpa9RMe%aF>lP(6ao<`!<5Zm*es8iwPE`K_DCvQ|1clCg2f)#SwFTpeZ0sB%ZJtLkPxTDo!G}1<(ulZ99~UQ%=kC{BI&H`*JM)U1?bofF6U+x<NogZgUTkG+-->Q=Ym>8pVMHa5@TO-*Eu&#4+ay^<xnRjvET$p4?pXzf4;8wHf>LOu8t>vuR*)Uqq?pfodjB;)+<LA@#rkBu<#5z7y~`4J_ZHNo)bY2|@}gEFjYePUQ@??Z&}3kd*xgID5WJ@FH#_JhD;|z?aYi$BV4Aqd2i#OT=!Ff@;Pgz^0U2)DB|{{B>Z1qwoN)pJ1>bJD<bd+X9f8S%0j*CPz<BqT^UY!61T#J`<4Pc#uJ}*s?%(Q&+fd?1d2*5ZR&>_kqD3_`aPwvCjZ(TW&;8nZ4uMz7?H|1n-Mgv5N$^&~CG;OS-zRGgeJ#Rf)Qq+(xErTE7d+b(Rl78>}!$k~j%`F98m607|xq8HemHjD(dsEW-Ugp9j7pC>V1>6YzL|!dZH9!_N+4*Y=*)fvc=uZW0jx&9t-5saj`ylss(?dZ%jTQ{>ofR88)Ra>dQ{K|y!b@`?73$0T!8BtT_R(VOG)Nkc0D&M{8_yadd{EQ`lrjKT|CC-UtmO@Y7M`Al&IkedKm$Ax^wb;0sE4^x}FJOz&l8KCdjE|0*u#*j17$hIO-S<XG-hCE=-DRFsVG?C*3Pm8Ohccv2ArV=IiD*%hNl-`RaEpo8te=KQ!HGM7l_U`%X>ABpQ-pOKtYPx0l8FD?<_^41Z0bSGr+{00sdKTo>erWr#?XsBDIPybKXo&unLnDXTQ6g|L%%PzR>|-8S67DS~G=_+les-GmOZI;9UlZ3k!h;>LsLXo30l&#i`rD=iWPT;gT+}k#R^Yyxe(tKKPt2+lpuQac_RnwJI0Sd(JT2|V!Qk~z|Mlv}mkZFu#oL#^yn6TQr@xW;C)n)Z%};M$_PT;yLF`FlI}|>JEQmT_d=QnAz=>Vvy8)neCyo+hFc?5?L>8D+Ex?metW6oBfM=wt%MqD;FKa<&a=J|e7@}7iB0G#^lPo2}c9l(gGpiOffdILw#c<T#E=pN4(Y9LCvaY&!X?{P%izLg*gJFz`VaoqZ<B>oRCe6J*1vS8Eww(seDnWjstBsJg9>di5ImZ=~rY3|mY7NGQAAcwxjdnf8ufG~N3f{n38MF51ml>lJ^`^`vpNG$O*P;c4ok2iMWb1l}#X2)xqa!}9WwFwX@Ql?LAha=7!dyzx^)HcslfM<YXk;mAfhduhiB(q%oR%bR%C&w6tZpi+6yQ_yt;_rE!m~Ur8X{N%)RC_&&zj3}BU9FP4fUXADJYlTZ76b9aB#_UV^gP-$QW(A_f)B&XI_bVXf%s9Mr1y3S<q{Y1v=Xf(k^9VoG(mtfmcp+xWSU8MPBtgHCNk?9WCof63NKx;v*Bl<Ka*5bdk|kK{`FxfF!z#fI`x8Z!pE8k%Qt0xK?SO%8Mv<fS({>sTA_+GxeeTU2_{rgGlg0-v&0!2v*=7xOv0<qaD{}aRs2anzbcY7pQ5MW7+2NP#0Jp7T>Ai=0qbeWNERfXC#G%;`P{4H{_D<7mTtcOfe~n72N!Rt|}n6<CqM?cq^(6=qznnHr51)MgU_eVO8ZN3-e?ss+O`=FwJthk``(XB*phXz9U&hasfoZjAXqcH_3m$fBy!rQ-*>SoD?}wLSM83Jil5NSbK;?UupIcOZ6&KQhSxc!~l#SvFs`@;0AdM7(|Ta=xFMh9v`WfV!eWdgA=S}1XCde^j|~##3DjT@C7nORiIU}bda`%Xsu<-sogD+CQ5lKl~*#MF&I)68%Sl8tZ|F`PvpagF5Bnt%Z>Q(fyj&pR!ve>1&2j(Ch%WPJGqA?!&NMLgHa_pVA(aHX$=d82oI3<8l^7Ea<j&wA|;Db$XZazM#+dATmrPi+z4Bljc~P^m)LaPs}?ljhx{^xFRjZ~O-;mqjn~$f(AHo!{DkK=UX%i>H(_ZNx!Hg)F-j$Zh?@ietrW}FtN;A=$I93pxB}k6uz#3q_+!ZRBAH>w>bp(nx~XcZ@1Z)$`OEm%Y8b{q+X+nz{@M6rHnWY#C!?Ox6g+l{ZLzvC12LdpzthV=ZN^c-fUxtJK-jvfmpF;xu)$Ab?O4Vn@J3?-4wL(E5&j<&*BN!@0bG%Eq&Wcxq+4j%Q3?}PN-E#5OpOa`64Oa~`S9?dMtyvIH1tGveeHh4eU(GQjH~$i*jmI^JlX@1KNA~uVu*IJH!ped&%`>(%eR`6)DO@NSzC<IsBd3LDlhU}T*;uWXMzqJF%$-T{Jy=4N{!jd%{9-;q54rtE5t~jK?o|Y<<F6hFM6Ok=!e4F@c+1javLd0`OK7j8kCA3kILO)tdYE<fY2DML4H#hL;Bi!vjz{3PI!zF;j$;+gKTW6Xjsq$U9<`rbq+;N<=%<5cpblBb-Jk8RLkjLj5|TEJNIFI!x2sizVv1UsfAF1we1Orr!z@RPliUBK)|OJ(98L1XNqVSY`MwNlaUO3*>FW~5>98jwL~>Tk31u`<uo5)8;N1;sIFd@xmhcIBhk@b3s{9*U1u5lWEvhEp~bT+j7D=g+){j0zdQWXeD4FTp<*+RESXo7Eup#AsL>vF8a9({y2xpD#>CDbmTgbqjk>qhK^>D@q^s_4bCEt1PGHYYHFtk)pMrLu+*Op$o5~$qT1_w-Pi_yMWwd~G*4pf;+*B?5cz)Rhu?JeU;B~f#T#s(|jRxBo1}uS$4VN?QI>9z$kP=3}-hm3r!uIo&ZHCfA_6@P4%B>X>Nwn9chZ=8{y7gFFC%TO|8SfK3tXiWd2mK>?-hqcI95=wQ0|S*4q$_02-BRRTL?%7AhJs3ORh-((y;9?$RmewEH=8`5>`X1d;iyR)Ws+Q(8Z2sN>pohV4X}J7$-`!CXp%h^t!c@hy`Jhy+qsh%DgY#Jt0i~Ur<XNK4~P_c>0x*9lR4tStna*Jt243;HeIbjO{_FtJd6#R<?V#D@1^v2Mrqn^@>P~!Yv0uMOWoz}n#qa+ZTXDMjJ%W!L)6?$s#>r{Ykb~D?Sx=bG1M^U6!VKTtwi0C@Xm@3h*DX&)gyz(!HOHe2DR7%d@zUxVyY}87lJoWX?3Cf0SwVZ4?V$yU;+P@pdO;E+NWstoZ!y@{wL3`06?LCe*gU4M4oK(4a*>x)29^EL*80;e=bdX@>1&_(vZ(|W%d1mE{}(zw_3vCn60OOVaPPlxP*12hD@a9531ke#MtYpABNiIK>C9DR(%d7`qQ-M`g0#yNa(Xe0e(sdq<@c_ynL=dB`kuxbzS7u5gdXlv_Xk9%}jOWU}F@yZa*iE6&xmse>#6(NZsyI*XvUoHZN4GiQ*?h$k9=XGTo4rI@!n3){^l`He~oK|86vq|4;G%fB*Vy1cUXac$Cob2_ayWbLtoXSkq8mG_T5HvmV;u_P`oWDOEpR={?_edU;ag->Llx6$9~EN<g_e8aEg$KCRj3@JV`yS1MLJLbr}DqxDbL5mLvL@YwosKekPmTJI$HbM%xR)aT<7#gVw-R_0oV=OnGFahfWfpoy_{1j>IJcPCPe^U{sh06r<<=y@NNu-ui|@$}>1+P|CQZopGrH2HM3sg}dz%>cZl)qT!}O?$*w#c*`EM`L4pH{vNj@S|*YM3x|v9df&YzFqMj=F$@T%}1Z6dnb8oyy}F_8azmq3JCiBb%4eSHACem+UZHRWSr@8PT*&$wWHmji~}Bk(oN*A%S{g9yfbh4D05Y3s}*jFlyXm&o8<3hkzc9JL(Rwo^qQ@dQrr}`_?&oAFGYEW?36Bt+!dP@C)Yx(<wnlEeD=4lNKLbqc_;wyVv1wv?HI;5fZ}els)BL*+b~c2p1(8>r^yMXa%7;U<cY@Kf&M#c+e7(sI%CpLdCpC6dI<ZRB_R%McKY|$+T4K!{?a^w-&qA2K6Fy^hTXIAqte=u@pSTa0@IIoAK>mh>29K5Z(*}^w?L`CfGxr%m`t$UWoH%czSA6-4tK}B=hhW{nOW!!KV||>l-szT9UQWjpY!V+lwhL$je?(G&G(Qqq94BpuX!u?@f$cT(=vS~u}|+U$QJJx_+EsPqN$J?9I4Jp(KfWHwQp3Qa+_?Q*r%vCFs4a2z9TmIb9<<>M^z@R)e5W?DCR@w*)!d7Od{>#Px{J^(WGGD2uS2{!h;_1v6)CUBM%^)NBG#v^ilc55s?r7Z?EffHF@3*xn&9CE`H0_YFJvbC3w~SvosFO&T^T!96e~e!Q+7vCeyrlKpux3JXWJ07t#WnlQ7%S!TZ@WciW^Ts2f?<!m=!V_63$+U*H@r+Y_*tYB?a0tIdj*`fy0161V}-u0p?2-QS>;$9lg4l=sEk7uqeE%~4Bdz0E##=hA4m&AHpvWl`Wtx;=H>oJI-ln>WBCLonAIP#e1H`-x3mElxZN-*N{iU0e=Yc&1+;Vg4gq|A5<DTH?|}sKc=S?jmM0>bi+Pk?yd!(S&G7HXM=fdr})b6%>r;YHT!0QWpjE&hMLLc3h(j5G_l#*XyE|J3Cu2S_zTwHEUu#-*TcsmvWB>!6nXrclWKW^w_Blk&&a>2{TL3aj+g^YV4r48|e+z_9F-e&*4@JFxZlO|2?sTUfb&|m&=1NVZK|u`r-BaA$H$$JhB}MACJSV@wUU;mw)a2`pc`gkF&w=#${<M81$d`IvH@BXfjRRwlpkoHa$)}?qjgM`S&{DB2Zwna}mS_-$jEjpV2z2Rb_u7{c&*&lXBh{T)$0?4>|2vCYiQMa0nHuE?e`(4dbT$qArGL>Z5bl)-YUDjIP0F_1x|0<sr&zoJKEs-ea|Dv_B<nUr+I9YjWSsx8(QBNc5ys)9*XJesFHKuw@4T7~D6bt5tCa>)Xg|908X;T0D^5wB~NAf`Q>s?Jg>L^{Z>TjO(9(Ha8{>cb;}<gX4At!1cO4YUbNO`A`)lj56FGZnR@gqdmXXtO#Y>u6cIQ;y^PoIvvnPt;A<Wuia!c%QxbH8GYgrZhKM($`7uzMkI@{rYLE74<^~XN(tyy_<eW)g^M)%I5$ik04D~fc;&n|oxyR>w6{0hM$5wv64Q1r_cyw8Zud0+X|6Z*;ayc7N@Zu1jqH%+&|V%K?RPqoXsb6*0pwoaW4|K#=deSinF86}8acci+4$sUMY~bav^}~}(aF?WA-2ZXKTXCXrPTq><9<<P%EU?UytkB&a08la=)P^)YuF;A#wm0sq|>gpjYi1$92jWMLM1B07QgM|U-Xsdz}{z{SU9#W0r;L-3f3=lY$0)*L9{MrE#E<>UI|zIshe|FWDP1n@YP`Q<#aI5TZjXYUNt3QOR?P?{qUhV4fyaurd6^6N`odvP8LCQ-VP>D3?Wa`03Uo~E+!IFq!}aX5EVIW0&z(JC}W8f_!^i<CSQQpRMF}o?v@!-$ui`Pz)igYQwn@0MdwSvYkcV)<bzmY1&#q2QGjcy3+VKs011#^UiT}lG)|YeUj}l}#5F7j>!~2r_c6N^KeU$ExbuE9iQ3i8CNN99F5AiL?V%xpXhf~*s@&$rOX_D+2rMJHMt|Mcs5RwEO-*j~ROPSC!>7E(Q|k4AN<$RlkYw8GTolQf&RiMGZYeIFQ9ZULKE2!0R@&QpOWCdj^#mGr7Jh*(E(Zq?yOf)Hau@4Yr;6rO(zw_@LU7rs5^0oTu4I1z`{CSFePXT&sb>Z*cZy^h%(a&|;GK)@^8=SV;qP#v>n`aFS><%OA1Jm<n6`5mA1b<W{9vvq>1vAu#1LCGjSyxXXwrDGdGlqzNw1Ky`(u107)yyQbaB}5a6X3h=*=VejVP2|wN0jd`CM3+UA7PQ$5mRZ^a+*m*)t?C#(^}T$2np3SUW+JRtqe7`#2mu4!Wf^TQZ>9L#<5Pe!9q@@v&I8)~}&aui|PRA(PdO>IXDXUmmsp%(GPOV#9nJg5%C1?QG{Ij>D$W2wKUhLKFG<xOXzI=dFjHG=u4<9^3i%0%+4n$4}W+uK8H-Y<tWO*XcLM{5t_|TYV?1o|UI36~)tlma#2+R1A3W=I8(WpZ}Gw9F@^P?Y;aicQ21BHF53NXPv!@Rw|vcj(%;~N#2f5m3{1}uIBsp_`iK@r5R(#12z7ytJIgHer}-(U!_%LTbW)oN_c5zfUzpgi^swL15ir?1QY-O00;m803iTEBmPWR1polB4gdfc0001VVRK({b#8QNZDlWEbaZ8IbZKvHE^v9pS4(f)HW0q+S4`E5)|D08Z7()dz<mS=>I4Do<Prn~SK_YNl}MSS>}-Sl_YQ}8de=VWQXOn*hQoP%^Vm3!-ze+2D#U563tq$RpYN}Mm%Metz(yNzm4H?%B}%w6Tv?<w!dUnv`}tS6sd>Y*D7y29jQ&RyPHL5c(T<xvh-NLyQmQR*RYEPcyx7CvAHj8PEx?L~D<=yJ+z2qjafw>8=$(e&Z{ECrb$k8x)wb4aw2C_?l)oBegQ5+7@bGD&(WTr-6ofx`QK6aloj~2SP+M2QdDK~f8tNLv7ipb0W+(Rc3Y1o-_P+DFi>gyQ+neA$mIhC^wJfB=V3g9y2LTPYyDW<1IEpq#H^A7YbDa^4d8@R+qe|&un2n;I=Ju_?uc#CoV76>I!w5cdQzjAMV@175>8+AwDHOwOIAH^6bennI`8%ydV6@swqRZ0G$=o`I-mwQ>cVfoSoohSC#`|pkn%A~xVt`oTLpLX$ob1rZEgG`AB*Ng1Hbr$n$dt-9oucgfbXj;!XDCi?2Ad4CC2E`11RVOnj4=8fqu93fo&^Xo^pqit5K^KQ9KCypg7(d!Q-E;w@yrUXHgcP_m{5Zm^%`6YKjd&e9vK+HIsf3{&Zxj;j-y(>6=s>O;W?74;aTQv$(yI`31^YHCGWt`C3{TYk~}4FpseU^&0SHYaL=$nQecoToyly)TR|s79px~&PO-UXzS5J0)M`AEJxKAGER~R$>KaxFj*f?K|ISL;%tQ$&7$cFKr*IJ%;V(w!(2(8t8!|@v`%NP8L!%!4SEC3<cq`g<=1XJ=E8i`BPXMey@r3v3>~l!{r_+nibXTFh3DVz1hV%uA#h3(;BAHysN3A~vpH>rm(mr*^YC4%tq@%WUQ6#IO)(;)QPD{>A1spVmepGOmBnKXnu2l*c&irBu=#a|iMbA-`-j>4HU@X?8M*%qzj(B^=q71S2hR%RI_Oc*$1OZ#4yVfp|Hc5@~e^Ks7xT`Qww!)@bbj}A&D88<7L~2#<VZF!Jf`blcV0{|KWeS~Yap2(*sW!q0Tnw;d6g#1CFcT_xXR*WQK(k^<&lV@Ab0(o_1~IkI;ifnZXIE3EhJ+GQ5vrFjb<jyA&9eY8_zfrfAb>Mu)06SG^OtVxZ<!56?@DHI^$}QR+WGP=yJFUErvfLu`r+c_w$q(WR_Ppk&u}O)&+t%5Kii)h8>P`L8p_)IEIXUX(y|biJp4;%O<?pZygY3e3sW*gu&?@KJ!$A+@?Q}yo=->}`dm<fRS|O27ev4ASTPGD>tU48&`$?cU4nHcg_8++m$24)L|Bg++`4WbC7K8vYRx@BZ^F*dudjgI3a!Cij;pQT@*aiUR*see-2QO6rQMICRTzL0$c)Tyb3Gz^gQ&4!*ZUgN{73Z0ert=r%{B!tv8F%zy#8g3Hp|^0{(wx9rDX$UuW*^fb=q%6uWN3r>v5YcbkoXuTxGp#gY_s4uCwvM;4XG+-@WtRb}TwZZwa-Wid(9|KECnDj#r@x1u6O)mNPW3=6kdg-5qYvh+;3JgZe?e#LImCeO}ePp1tXV1*6E&3KpAGyjm703XcLAV+BUYGKO!8H-5LqwJoMxk70|K7fdO+!(IDGMuuk-e^9n-{gR3JnH(dP@bx7tzFiGJf<k5k_>SainX`i+9@#AYbR5||EqX?9?9TaEE=P_|7cb*LKSi>)0t=<|p-L9NN7Ctr)zbtGwSnz5RKwx`q3b0VpCCB^k&Yd-2Q<2Wo41^S^dC@50|XQR000O8001EXf2fp?s|^4EHzxo97ytkObYXK}a&>NWX>DaMVsCGBb98cHa4v9p&01}9+qe<_&R>D@K8Q>x*s_x}=aijsbGbItOycH}o9TEw9EgM@oGF4KNZE?(|K44^kf0<x>GWn=#g0k5EEfCvETAw9Kk~YfMVE_t$%*Rb@RoC>2(LDxk=2@4?JH984R1)p>ykkyc`ZvepD0qvmd~ZUCfB^Fcu9lc(~1wRor1BVg>kGE8%{2^?MhZeR^^r~WJ45hyPB}7AcassdP~W9S;8`kvnF2!xhy#0w><A!BvcVb)~sSnUN};xbaw1>N^V5Ef?>6&Dj4-i0vR%A`8BT!3d{*b*hY#%S&pZ$R?C|9NUUqwv?Q0+q7*r>tO>b&<||e$MYRM{iW9L07HANLVGt}Dxh7e*=-RH~Sw<`!R)G*;)3Q=Q;M$sH%^Jns`)^8C?pZ2Bsb=k}6mv)B0$$w^wdz_?_HVtS`j)M->`>T{STNj$Te_8Nk!QGKLULBgO607}R!prDAk0plzsMG%<Uw%p&rg5*@IL$X&BtHfet1t(5~7Id>_r%S{?%_jrZ4HqNf>;5^N-JOKfd`1+U^1ZpRjFMr9i!F-l*`5gwN>db9xdcwpW5K#8=w~l)b*HTi07`p3##R^qFhE5M?QEbo()VMvwd9WqXaSr}RfU>*@2`c3#T4rT&s0(^{$)C*21@Q1AuGny!j+xdvM*67uB9^$n;-#b=s8-H57<!lxu8_#4vSq^P1bt0UDmAUrcIjuWydJGDwbH64%j%m%EvsXWb4T&EPOJvcpI@%&m(_jqs|cAKb?vb8)J7hve1skH-ovl96#3cF2cRu>Czn3;wja9agdi1P-%^wOE^+PZ6_%VAnV!s*n`2(J=?vNRi{z`dxWxTg!t*?|;1SF}-ld3JI|(v(~xPYD^U<!a!@6VtG&VZQ=D1m8~y?wcGDpfZIZ(!#>54)vJ1s93XwA#N~?g0|3%4xJdl^XyQ{Z$bJhiqCw$4~Yps+AKyb_!*0*7B%cGv<!q=U=Tl+yoG1m3YG!vr0Lxtg8T4lM0k)RFV*+4vDbnh@OkR-cW_KNU55|!=-nLd>Ua16%_&4|TyqLEoXJeJT|GR-3F=_K6W+nj0|MP2cu2&c?g#B2EGsg05<h16o^}5r_4J_4ynh>5Gi{#HH%vg%_}EpgSo1ecBbzAvT=CnQ=Pl%@ZbN!I?|mv9hm?U+8z5ju7_T!?K++TKHd;ffOtASkpqil%B|3#=HE(iWV<xg*03yM`+D#y(-i3MHp&>8f&;2gfzp}D(JnLEOPJwrE0+R@2aGYdWMYm0L#)^D4OfNF4p!?E!cf-GQqT!gc)lrUVc8a-`wzsYI72tDc0|RrK9A$M4X+3&`<S<tg9trhs(4+#xSToEREmKo76+p#wGxIbY)*5oasBw<;Mzjbwkjx3BsnJ~Bvk}_9Ok(<RJb>jnFL?urx`6>&n<Gm*+p{p;<L?>9+#!CT6<hN&KdndD9BashPpEb+!Fzvqy{LfBj^NEOmp(VF6a{N}=Haf$2q!U|lP7z|m;THk(peK!IV2Ob7KBh98cHl)0R%+Xyk)QoOEj>tgR~eE;z=_h8Z<zghdL5+@%A0`06s9jjDP?#?&btsWSh%ewo;ZVa6Xvg;<S{kP!Wbyh*O)(*;O0>hU!?E-d*$U**<I3dTRvTM!N8T!gX}-1Ev&5{AK8*t|K9lX-Zs6?1tj{mTo4hg~Y&<rqO9KJB|~-*G<sEpjqZ7Wj16@o=)P+W0*V9);LM_M+O)k-F|VI3Hb8_bTlHJcLk$DWo*Mli7`I{Mkar~_&fsMHh$yY=3Pr(fBxy50NU=Vej1(Z5O%>17l5^ACmxr6UqHU3rWJHtRM9MXmb^?(Pm+RfM9$L*4w@uIyREq$xBz9gvlsEsib5?!C0ZVt*Q%n6uFBCqSxFT@VqRp0Y%^04wHa$$jIW^LFA4qwDdenEph}$_VhTG(P&AjfT-l{zhJtW;d3j6|_&LF!tHi#|@aO7k%sCKHxoq|*&>s`}a(}G9BjW^A){0hZ6=^;aLoA`sV+zi}fx|x^2fi}Zw#rwaIDp~6y>A4kXarlX&<L#0btSk~xot(R(7orFb`)EKJC&jab&2)5=gAxE+ZoV&Nl)psV^Hq{Ljr@lwIdeb(JH!|n3ys-^Nc>3=xt2S_G~#Z@;Tc#pJY-)0SF}<?8<oEeH`oR&13@wEIopcgToLEbFyJD9z&6En(u%U6Yj=s-bP&owL@o|12kn&On4wKY7hzKko4BQ8XkKvW12=u&<lJ<5Dw^@AX18o1AM-mVlXt(A8ZpVT|f4lm~Sx07=awf1Mg+!(g3L7ZNb6H(8<az$r}!71*-Q_f-3@yhKidv+Asho`N0N{42*di%t=H`dxh@t;b4h|BY>A{nBIZC!Y*OcvfSv1wvSwf-kvaBUl#%t&l=3QwtCZ5MkcSlTF_HYYTy=BMjB!r(jUZ06pPX0PN%^_7Ulu5??Qea=+pCoXXuo=c(QP50eI$gc09eq`S;Vi@u@)M$OG`5om|};(HeUP8UsC3f*o0GWKApQCcTe_A+)m0Q#8B2UbEx-Voht;5Ij)8was>1FMS6b*|ou4@$8hQX7W&?-Qr*pc>&_J&U6>%O~d2FgUSi{@QZ$k$DEoFw^z4>Z#>JkyFsg1K;GoVE8xoJRy=@eSDX|)FTnr{XAAT|F*F=bBnQeOs5#Q%kijpUAC$W;OE0?(R0<oAv?+b9+LK*KJZS#PP*>ezA#M@z<=yOL7WP$0?-nM8VJ#BuSHa`Z3rqyxsAC_fUp<z`nla`QjB8GvO43xrr>A#j3%iBq!iV;0YJmMigJJnxR#@qQA?&i?2dr-tNPAR8Rq)$s*OYtQt=hI$XGcecys0qGQ`oA`kg{1G-K@%^1FT+odE$Sshj7~bE~4^x1=e?{to(s|fQs_`EqVR!?Zrfqk}p}lHOJH-IGb`5sjrtA=7tVPC`$g)i4yXWIehkrp|9G5XiY*F9fZy0hHJRYSwZ9i!X2RGhL;Sp9R3d#wt&Qg?;f?eUd7LXkv;>2_^<|w1uVx}bgHi8F%2F*PTD_8+EwHuheMAX3_YaOhZJk)RC21fHZ^+|2j}k_!FL+M&I!Pn#&BX}%)+2Uh*G~pPFXaZf92nWD|Jw{CyQp<PIUFKrhho)8MJr6KM=Nwb=dRWj&Ir=4+Xz_+;Kf45a_qn>bTd!qat(o8U`BvN9b{wb$<YX+GINdmux-fO)u|#KsVsW549=YOaaxnDpZ>hX3j4icoTxhTVN;$mbQt#%Yp=p{<(%wtPe$c{MhHl0W<#phQfW&`hXv3kv<X;AKTnvR`R|5*?TCkhV3g-m368gb$Jm@c$J}wvKv0n%H<?}<!VMVXk0Cx(m&GYlYR)$(uXdW6FX>fWe3eB2lnHRNIAmv3?@f?HKN5ZI*S6ezh>Z=qz~wr8Wg}EU6i~6D*<>8(!mQz1%9H<#>Hnp9`_bx1%by;!QR>EV@1e+TxLi`Ed!U~`*)xIp8a(G={&r`r~@Hm1n&9<ckqksQGJYs=07CCaXW=gg9@VKWT0S69P9LWsvFi_b&Xdb<{@j@d^p9`G9T#o^Pb|w8`%_<zC8&UC>QT`T~vi#0ih+7fDne4mfo6GVu2ZQxF{Mj@laNP@;5?u$edT=y2&7MR%94Qqd3-3)OGYVh=!wg_dQy0^mJ`vU+9vzk-3G8@oqp%&NJjkuK>or^K&D=@`}8Fcj4sI-`wfDXRHIjTE_5yWP3ZQ3{N99rVtyBaD1szMYY4bIU@$)-`&Slw0u2+{Q9i8m$RehU#01aL0gG32hA~F3hbXe{~@(3>|6zYLoB<ZahfLcEq)=uN(Pq%d62bG3Q8ml`wB3w5tfMprl88v&pk;E($Z?j@+w877LA3}+ZV{`@+yuG+Jv5FEjuQEN_&z!j)nG~xjXPEJlDZS;uQ~2QLr(>@F*J~c}_~PT(#~h$WNJe;a<_sWOOPsJdAEUz5ac+@B>u|G<JKsHL<{!4rAR|nqXVDJA;Uo_Aw$Mkb<~9cQT63z$Z8=c>ph31K+fXg^q25_T7rMLe|$FnwfhmtN%{YKah_cQWb&*fIa<cU}z&7M49RD#$Y07w0A<Hkc$Pv9y+@98gQ3Ja_BJ5n}9<bR1e(P4EJ5$&5eTG!p*H0ZI-2+N8uCt72t-o%=?`~m<R8<ATjRTaM>?Z?8dRteUtNrGsj5pJO3IxfrtBCoH;WRNjyG%;Oe4q;<z6HMcZbxJK{DMtMs>O8rZ7)h7CP$mK}!03*8eH+#JCoPS3JJ=2;edGbt;IjM*s>05U}QA$~(c3fO8I`B$W1*|0wA$~M)_j}k&9g$c2Etnd#A4@?MrOlUBjqu`cA`_V8W(`Ewu7|;W?a6#W(4YBP3)=>MJ_@kqA?uPZ5z9dOQ9ays#wqfVtw^r~}(oENXkbsK%Fo|yzR@3m*OD?<T{XSkB&2gwrH|odOn)>+56Nch}^JzWNRBigv62-?6bVlR|3}+c$pk|pqo60h@v@8qHEc`&T3;qpIO9KQH000080000X0EVf*T)PJV0MQo!02KfL0CZt<UvhPBbZKp6FJo_RW@%?GaCx0rU6b3k4t@8pV09mAcWkZgrb%X#ooSmUoj%-Vr)m3=>1b$)ve}hLm887c<mSKkfYg_5ZM=K^vMT@tLEr!c7)8<R4^<&k>E_G|BY7b|@=YP|;ha=k%W`d)+X<E%{ZS~kf5yaz3X-Kzju)&l+UZmmaTG<934}_PBzf&>Ba(#4veL#eu9SA%Nv-T;($F&Qcv^64g>6ZHTdjhMio0DQH|_dsd`>2l?+0t*?*D19?CN(@i`m4J*v~=<<2QVzRW7%e6NX>OKO{=rCQjc7WiOdjj$N_U^QGrHQC74FmS3{G(A+cQhN-Kh8YkZbMgL>JY9+v<Kc1QC#PYH#pl+je0T!9)S(Oo~v@q|IS!ckB@@n-oYIwS=T*vrH*wo04KoF#bL-<z2K`_;7pOiWioorF?Jx}(lz&C1}R2sp(>jeDrMJe1)XA3JtwjkkPt;i9%N?VsmC0&wCttj$Y@ErYQ&WI8pdXc?kD+Y)lz-);RFXFw$UcyiPa&HlrR<J*LQH$5cXfutDl$Eu03}=NI(#gGe5ggyqrDD_NeDzlx%}(s<+zq&8jU9bR$(rREYYw>di{6n!qyaPHpIRDWJIjV}iP;}e)a`%uh8Osz#Mv77_^ydp`#Uw=!-yfg<V6po{s^A+@~rJq5NsXgmH4nICAOR_9`jj;L-tjR!AJWD6~PFAHmg%gI6B6g>($ek{lGGr=fa>+^nhmy9<huw#Y>S5%EuWO^<eg<jsp47rvIUAT~URBCBOapGYcEMz<F$4iazLmw~Mgbg{jrTRya;BQ_3sBOM7DeV~W)~S7&#_It5%=X&wBQ7_CDo@QgyIyy$6^(HE!!BzfVuUF@GNZne1y##jfV`}$Jo+dwB<P~AID3jYUAI3Nn6s{!Np;Llkc$A3X;8X`iT?)a#BJ%2f8t2tW-v29m=FHcG_gomEVr6{r~iVv>~m*hNk+T2|oFKTv|+ma`q-k0KKynN!hce>CCypa!i>rkh0WpPFD8(HlIHr%DV#LACi{CIS>IeRi^&x0x4k<&@xhNmc9_joN5h=GabEV5a}n*<aeL|VIH@#nQ`LIAJ0B-9w_F7#G9J92%<Lc61DZ2e@w@__AF&%4{t2xnx^SO>Q%UfCU1iM=p)E$`cgiTdXfihH+cxqiA@u2;ctz#<O8t1*%FkqF8J@{7CPfYovt{Ez*OD*$>#uvYHSz*NI990(^a@Q+HxK>*i`{fFsWAcpY@1!a$HcDI`Gjnzf%1m#mo-O83O7AUqM4<z3zTz4`(-1FIYA;O3W!3kYZEfStPVG>)y<)`65Mex9rrffiekAyS2u;bj`ntKFtsd&+`{1wxn36_DClw8SN*e0=0+mTMtgJV!d#zfPirgX8`wh;G;p)?A!7r<*Z;uT(Q;4;YQ8t%9EDG6-I?GGEGriTe7IN?Cb$l(dos3JbIO9aY8t83DpQ+7YDx$#BQU&ClMJ{=VGP(tU2G+t-Y9TPoFPKYUb0d-7mU^7DYSUIK-P;$)j_&6rEF(1%BpSdM-+=@2}>C|2zpL)l99&5Y8b<+7P+$X4bKK}F%!6Q1iZPe;%I~Y<k4<MpOhLpr#*#4{bChJ?L5KCfoZg_Nu@WX$9eEr+S_1o*$tcI*tdG&Rx@9P)r)!SEZVJnp{bT#NR6;)B-w%{94pb~MA;hP<Z8==x&$<58#Zl^uf)#<7FyEP7zf<NKtl<?@&zLCbdg+VpU!oivypc1XejmTnl9VqFJE8H+`exjeVqT;YiG>sjG$YA6bj!KXTk@7tuqFe}t%yiS9I-Ak?0==W1!xFXQC;^4QZOW+L)JpzS3*WvRfvB@#&;gJos3y~6`izF1b9)c<cq~&cb7V$0Ir!*=j)Qr*6I?eAIBha!Y=I0rY%0!X*1N99?G&=UYw^l3{6p1Pes#@CpWwb*q22v^fzNQ#mYuhxSJq}Z>1tmde42YrBQaXi02d(8cY%JyJefYbM({xs=r3x|3z@M^QTsxZ2`9oJ<7@M_Zjx=(HwRrM0rl}C=I=WiYk>?n0Hahp5c!0&$BpaN@&(;xQXOR6Lv!uB-igP}k9XDb@z!527e7&Dv{1f)f&gyI{3{MU8-=<tXttVSdCt~VAMeA~km&1kK^;0ZT)cgUX{j`nu*Elt&CK&f#GUq^4?@<+vJUnvC-dl4jW<)XV(&*@HV$1LX7)kz56G_%Bmm%nP!o*%ga#7<Rkl%?!{OjSZ1&L9^2W>=ZFT1rH#)mV#<AoYY3=^MH=Hp=r|(R)qKMcXu<9>)+dcv)T=uM4e>&#3%_X<!;8S<pR^rIDQ3nGrx4mFk|Gj4gAa)8Vfr84kuyeMj6Al#nZyKdZA}~8%-yuN2^I~*Q)2OGTW`Fv8;Bxfd3g-q!r@h$xWOzRcxmW&vsF<eS{<9MnXxJaJ^mxDNH{bLUy}|Ufb|dcmb70hDaJZ2~_-Akl`&RViB8`qt0v9w2y<wLpFbeiUhRpPn{<J5}jsuRE!Sp6B_Xin+N*3Zv58kJ^0W0o-K+oebEwNL25TE)ru@O8=96&mq&1RE-0Z>Z=1QY-O00;m803iT&vjb~L7ytkUPyhfE0001VVRK({b#8QNZDlWHVRT_GaCyx<YjfK+w%_$DP<HN=%*vAOByCrz?v4|?Ti<CCPm^|LSF52Z60%rRq?Qlcs#gE~o(DhzBxSqZy)!o-VhK1n01gh$8*&`y$1J>+k$Cs<-A7TBRkRghw#xFd5J4J=Y*WTr8YDtKlrqN;3z>$ib&%hRb&$qOS(L?SFu2O)CddN>7I7ioUHyENrO9?AE>i&=WgJ#XkPk(giH*!dNod77i)4cEaaxo?TE;;MODVI#hR{h7)RTxT7g2VfCRq@PWt_-jB;M(T0%XG5Sjc4t-xzULm7A(82J#^W1h}1e9o$F}M-t{Lw>Wby%OFl+WDw?A0l1Px{IimIEQ@g=WdxgZ9A_|C=Gj{K{<12oT>3s>+Q8g^G|fudb}<;JwqJ`ZW%w{l!z#~Vwh<33^tjKfTn5oqmL(StGOWrh=MkHrTqW^BgI&Q_jW(AIQf@bKdZQuE((O=O!r26iL=MIKV1vDCy(6UjG6>6hUUwki-<fK@$nz{8Qd3Z7>p1jr?VW~%Tf!jC(l`tfe-#ufZYx&7>Fck3JdokQ)3X+ZY^J>Wcy;0bbn(mOr^}BYhV>!(cR=K2ETf?)f~E9<!E#uV|69Bn3<k)K{@b%p7yi#b{V1lwS(W9cn2g88%TcfnegmGoFGgXu9@CNH$1sb6GJu&!S$^X%hO-YJK7Ky?3=@A+h-}1v63*B-Li4i^%ZbywAUQog`DO&G>OuMTZ=Wwd0n*dgr~aRhzwuwcIvIQ^J}#GW7{gWh^imXX7>PW>D<HDPuQDvZD#XR-;ASMw!m>i3AI^Y+$cjNyWN^x1mPf#kz|g>p+fjGc_vf#XBFk5cMG$`d>h$>AuOs<t`T8_?xs0OVG?K5E!D;9W&M&?{`}xPu{=1JqeEjLN|LOAYwA!6>=1j!P<Dqcg!iUpW_~{)r;iq%>bn^OmKWOIk-(S3cd+`$vE~`x<nTwW=NxI!_oJM<pS8S5F+!N7|cN_+V+e;mA(}n90X&smg$m@7D1Y%%-FdzX@APmcfP!<5`4TziI=x8G}@-E0@kR20Pk7klQxF6lb<*Hg#1xTALE%6E>x1441c^=$k=@=1>cNTN*)nc77OO7!z75(RLvW@h2YN~x}XR4W(o`amp*0^vOgF+8K1`%+3fnd1bJurAAmjXoqUoaQ%{RHH1F8)QA-V;avEfNr0vnUSB8FUQss?X;Wnl;bv3n0L3PT%fft2Bzxcq3SA>VlZgBH+HMQ<cl3Zyiqrg;>Gj63IC9>eFhykbph`F^F4`&;a73lrBRL#fjI9z<==)<akK~MzFGY<F)`uS^(AKw31dwnQyz0uo>73(qV*xp>PqgXMxEFR2kxzAgLq?E!+$=_8<%5LfQh(T{@R|k>g1G&!>+cegx4bc1WrHWXGG|19eDr0m{Ce#)W1=7ZwXL49>)R?%eOS2E%=>EBLSg#g-~@FzGI3&>~JT0--L`2<PTbP+C=<QlDZapT%H@kA{5hsEK%r*eOm+-b9D7{x~b4@HLN7aStz;SBF=2U*8&~i&Gjoa~Q;#HyBw>gNn#uAaw&&<je=g>9`iLT4&WD7{WZIO;}t(Kj;Fqau>E)3@n)!Tf?+?p^pz7GO!wfskVQ4Uer8Mr7>6@xKeoo(!yW_RIh2UmJJ=S#mA<n$dOAhJ(6S{l;O(Foqt5Tlhge_M!#-uKrU&#-QA|8t2<|tqgQiJd@(g>@OyT?bbe0d!v=U=nVF~?)b{lpa0~RuFRUxTgq}4s&uh7_{~=FIjI|A0y@<ho28@Fmgy$P8EjF%KwRc)Wl&t}S98?gvr#1-T008>i>bsAla)nFc<K(aN7SKw@H><LNQs3KPMOJ}D?N@n1Ti979&i?Ve>`i&1xLQdRP-I|H!_6orwSkF}V?GowK&*gmJP{;FiV&nkBVkBuAsn;Dm*NMo76lqLIohqkT?QycBvGV+WC254z;*cP!(Y*0OynBQh)oJ)L=@OOc4V%!0K*d2W{8TS+0(7uPLp80hyvrpC#}0-8?$almhU<(L~Ep9u?a%t76&-s=&pV2&9MN5@bux&AqXTKg5ci+FwbD$<{ax=X#?~PR3M%@?L`$Qkx!;h?bb0-3>$Bv-3wM5jh0aynP*vPDIaQmp*#i6Fla9ypMHIEe0st?pmdX};0Y{714!3v!T6gzy8|``p*=yx1u!47R8cReqPT=zRA>(JRNuT-&I|d9ei8pB+xwuY{zYgkZ0H7Vc|TxopUWc5V-VBj3LREFmJu9%qf~2E%6t!A<M+9gzkxU$lf8=?xA--~WxXB08=sGJxrSy~#u$u~A^E3Z7SJ9<Q7u5ah-@j&!MjF$*Vo2LTwk-+KqCuqKqpNB&4}gt8W6(gw#vcFR5WTnz;S%%abTVYTj6TUn|*9@t&24{BB1wBY18I6Rldmz>G7336CtSj8xZcq>L$Ysjk5)QLE7V`h9g{WO7^~gWxWufE3Hl%<Od20<R<X(R~bwt0+Gsl6`#Nk(vm61CX%fvGZBI{UDkjo@_2clb0)F?w#k!V!_z7!3+$LGy-mR-9l|&O5MdI;YwQ8~rwRoI05pFCE0B4FpS=<2H6S=T8Ll@1_dZMRq|4ficRMBCaC<#5(AOqc;=+grWda*Edz1~e1Ok^Xx1c&kKaPAja^J{v+aq~{blb?Ya=Vf49oo$vPLML`1bHpcD83UX%7;nF_TY7o=xd?2Vkmi5@Qob~b`335ZI_s`09_w1w*q)1F5|n}qJfJ*j4$YdYWqf+*eBw<sd<0#E8(2Mw&ss5;EXN|530>;z?n*5vaNZO>=_N~k8b*qH5htC&`V8S@1|$^W-4`ecr5eTKdU$gHQ+Tu4B2j~6&N09b49#*eRza)a*gP2AD>~6ekYFEqPA>QJ=>$Jzo;j#X-85+A6adssx35^oAf5z^fi7q<&2n+YY2lG_JScvJ|`Nphe82+C9u<Fd}E=1Li>U~U4cl|CfSjSV;<d|SU`Ph*&$ZLX8dck8ypF}#y10ojvIQJi;S^GFo|h`DB1)x2Iv#>wbI8G>Bcr(OBrg&R<kjxRD#e~f)MGXgx?z#{-p>F5m`hTF&o$cppIiO;5ekJkpTU4yw@Zyg-HzjARm)B1f&g<3sAjD3};~aEqar68_7TiS3oqE(Hp%Y63`S>6f2oTM_E;>6)@k_!QE-w#$%3R9*f{whGmu`7>V_<aB9QHu|Y=bTlnl^KpMo>^mAUxA*(4ryQQz%E5CuZeCwm@Op;&EF-DFgW;7y4@zNeKB9w(GZHnNIq=gY1v4>Cusgw_8*GKnE`Vyki4M-bD)5kGfI-nK?=o%m<RizG?Ai3#jcduibS{a5qoNvvbEr>Uw>e2O%ZQEMGD<&)2aLbIe7w%!EsqM^1z&q=r9iQqsAVk1!wc$}4+JeX(7cA9COPFGgf+a*xLZfc`{J9|_MIm0bxCg5QZW{&1$LqMj7&17>+-o|=${DIHc3(3GStmm#27}YRg)Qqiktvsz3<kfBKanr%sn}|AXb{TlIm>NhKrO;h=T1ef-y&-vDB2LOaxoE#+QFE%FtD3BWxJBRq-oz0d(c7abCB7Khoc4$*{zr8fo4@U$eA<8GskW<A$3;khTj>Ilw3>4nHJgsi>ehYx=f3TvIuZHK^De0G5AC}ipF-h#E9i@GF2Ca6`0xwk%Pnh^=4)hV=r@-mDzI1=X5-<FI3OE&A41KV=(m46gg%J*F+=zlbTLJHkV5GPi+9b@|jcnMNSw0#DF8sKrlwEg3QIfqrN&mIzM7#(D7yyLzT}B2eB2Sti_8~$Q1mGimq7UAPgo*=%&@Sy_kaUc;>J|gS`yQz%GXHEfKMM@q!VV5wBs}#L(IV8kA75kSzPdmAL#Gt~rh#S-5nxupxDoGy>R63X;$p(mooIq^M}{M=MxI5_ZDk6mLi?mZdE%yl8~E9qZC|R7d(A2>TT}eCAs*BBq$uue(MoPHL4%WH!~{L@BMB{zZKN#lD?0wH37`i=QdBRT!=xYNDh22s>#7AP&YP)XdG10J|Xq!-6A)OF<8Iam_BZ0HfPBn>u!mIl2Lxw0J^!4Tn&gn%V0xI`Pnx>c2U6Z7MQ#DF?l6Xsi?+!RSUaq6L5I2vsY8@{ABzB*8|5tzv=!xQ-0TM*~RjRx-S$l7w6?C77R~jEu;6@kT6zIH9@%4Af|ATgNhh8A~voM~CAVcF01BBqJ@UY2cwiGaP2!J`^skno^b?3ptGcHUkGbe_=>giAv7MU*b7paxCVQ)}$}*0Uatw<aA`0VjB%@8ospGI?aIwT(E#GSEVxIYSXT!1j6Jp2{3Zuml+f4Ia!WNIu}}!EW%)eP825z6jvAlc*{7icM1C^O4+k@5~-ae>Om8gTHwe^*jB;8B-BU8&$CM%WqXb}y1~Tv+A}Mr?qH;R>ec0sx(tNKGw6U(B$E0#*T|N1pdzf3-k>-KU-|C3-hT#VJ9pNTtx^1W5DY`FI!E&ySl$L53V04Kakx$dF~94CkJpRGn;b;n&!JV5hl}F1OxYjbHS1_6pNSi1nm<XKQ=~*IaGSM;kE!RzL{YhX_v0rnKQiJ`Lwm_yZ<e2>ht7%bn&h2YCM<(Ml$wI^-oCc)dGO=bw@m`)TwP+8yHrQOPAke@Af>xQV}%iWd#0HOx1C#Nlau55zOkSz6JSSbtx$_Eu-K)sLt4=b#yhl4ByQB&%xCk~H~ci0%}H%8L=t1`)Kyy$G9i}0kQnFfNF!k+y4FKic@|Y62Bp^<AQPI@n9CbSad4AnfHE|r{vxU<)A#P`XR&GO0&6u?Ga<1PCog*$hx>5PrZrcGPI=v)PGN>J-k^Fj_BcXO;FBx$;5sA5r%aPUEx#IY9f6_isWA40WnfcxaaI-W$PB|jHzuR8sIW}BslYKo@m$jfS7AdbG7*?bDT`%|9UaWm6aO`-AVrz~P_+l>(kdx?BEytb7<A3II{AQTJ2o4tH&l-t{1!W1Dl)V}tj>O5CbV${O;HKj8p>`uTzNt^PE%$=_M~F6xrNo7b#D1^O^>xpv+P69Y9TTA+H}mC?60+&LF2)h9*1%6L$y|~$1Nc)9U>Wt<t8=19g^P$UX-LPYgC0Ur;Kq&?*ESzurmD(G_V*#)6JH7{})+fBf!tb+`*)I-{`tJmYK5UD_9WsS$-=)m#HOx%iTbm8#wzyQ)ZK6jmD#1#+53z&UD?XmBCHS*kr+JYAo%pxlx6sRGRwj7Ua(|!2AF5SO-HmZiOSi{_`;f6?GF#yOd?PLNsX<-^Edd3Eejs;39k{-~RQjLKEcSiqeUN@^3ULz{L@tXf@*cx{e-QU)K&1tQM}Jaz(I+o#c!-1H@HP>S9|f#m<L^d3HWM;VwZ&Vr@wwiYi=zP*iDooiX?8$mI342QZrr#gl-lR9Trzifsc8D4#A$jDmw+qO$ORBhO4}@@Hl!Ea}2K)LG|7v}-a?yQSK;p6RQw9=3`X*`L>lI7IMhk+qEgFW$7j?AiugN3W-Sh0->Zz8q8iVL0ff6;vTQ3hO2}WUTqfG$Slmlw4dowY9`rPPB~WdZLkVBPj&+5tzDLWWDdk6k9Pgbwd3{lj>~8!?-%^=_7N1l6`;{Et@6YWEt9V-v{9N+X+2Q#2)+UKx)ev^DG<6nBPcUG0~KX!~~?unk4Bucvjkt?`DIDO><C`dyg6@g3va+$hYbGzO`jT;ea@0_UKqPHsdKb9=Ft{mHj?KpYc#|z>vAAn09%|p31ZXI%{sIKYVU6oFer^(fL3Hn|5(_-Rc1L%%7mvzNgx4*h4}LK25s!k!Nj6ntP^e%fxzSBh$u(9K9F(cid>o`6^ULYMZE1+};S2QK->nG3Px;XH4T#85}$1A|05@+xKNul_S<vb^Oue8fgCjUC}z<+lW%tb)>uOlz4mKBkq^qc0vlbGy3;BV}#QbBz{j72W5IKdi{~K`1dM($G)_Aj7n_%?xeU5&;v~l*K^W+Rw-W&;}kQeYs2W=ZV!x3n$C@DNP?(QTy^%%jQS`Ge`?eOcDoJZe)@Ppi($4;dkb?+<9%v=(WS3;+K=Ma>Z?_VWjddl#4{sDVW)dFhDWNdup*x7Zgh^2O=$|s>~um&MJUvxfS#<RF`50KM~BjzFn;f<b&jqjdU(Q_8E>C+{Jp-SEg!|aM^v=dc2U};YjwR}8xY%Go<1Q8opb#RRFBe?YYSQ4OSn=eq{?5O0tuUd1^BGwX<(eSH(`cyy)R1AV+Ip(jI#UrP)j$HY{u+gitj;y)|lAw0f$Ew!Iy8ujZCE~hM`h;?c;!TjK+R68d%}&?)y=99Xx0{PP-=%6q;O<Zn7BG@Gum<{rXFRkH7(PiE=~58+CE0*L@mRi&M6*=^$LfFaArsKBnx0&{N_!jAAlc??60T<AQ7Ri#h<VCDPiCaP7xC1C<%b=2Hs;Wzio<`L(vj2ikgv;x|D?t;~4zkvT`(G0>W&l>}3ivJfAzl{+X5jc1R4Ox?`}2TU`oef~s-)!gY!AvjQFWybE%O0aif1~<7$)y}9Ys(xnZPi!bv%9X1W+*@qHzt$R5*`6j-fIH9}$cq>DLWbhSi{5?i2Tz*P)_K@w>TYn*Hm;Y`wRIm!3Dl#!o?&L1;1J-x&;M<dg^oU|`RDrf)MRP8lrH7;fTgV3&wlx(lu<V}F{I>WO7&U-Mc>vAJ%O7O+*0MKGMJFpw<1`UASHthS_nZR?pLr)10lUOV(5*eUl-T}IhoDbl8UI3AP06)aVq`N*f7RMhEbE~v;|g}_0=IBv=BP^TGy%$BDd0IN@0sp<JP?fdL5%`6QP`LCalXIv}&IjiIcg#;3wZ&PdYUku<@m1Kl64DB_0eBsm{wZq#Z6YqD$Elm-w$QkHtnPNeC;A)g-XIR@R(&ai$ZOm<lul*#n!^+aY~5yam;D2?%)CRIvOG7s)`YcHq|sNGqVv>J8l$e8y-tmK?5we9Kd0rCem$Ee89+0#NiI`yk9xFqmVk7*VcpE*Ib$MGD_y+l}B0RadCdepM*M5-Y7Y`UzgI=xaM(jbLfTu9_Nstly_PeH?2S>9^&cA(Co|(<vO&6<Iy^p<P7QrgvS&)dDmRdeS4mn{zjwOm~V$o)aHZ{MH*G+V{Wr?0sHfd8G$R;yPqi8A!X^P!)UAVbX8U)mW7dB4bnkMvU@0209L7KS+Nu)Z9Yt<_mRvXP3yEAly2-<`TGc89&s$&ZC)(4_Uqr5-g~wUy+?*1crX>K|RZ1t&f)^tIFCU=U-GY@)+k8rVj8$S@TdEjdr*OTvf$UpDHQi4O~mWUC>dy5m7yoCM0Znw0`e45+5>9a$s!2ZOZYQ(Yev<xFlG}M7@#Ij)EH1$`Y8r0nrqE7M6k{`yPW-wns>;dg!S=fa|vccn<Yfy-7?+8eGkEo!Wr;)*6a7sTFb|qFQsc%8Sr8Hi}r)+c;FUwh?5#kV;KDJmXRRh5l8eT374%oOZl<bIk*&=lt$~@tmyvu)g!OtPjBaT>_*^3zhdrqD`Hh>W!+hD6+@vmu$4eF&)~WU^U-PF2Ez{XWmy2>o%$2tK;hEDmA+f&l!!Jrc#+;@Hcace`~98U6H3^1e_ZbT0-NqQB-)CtHUQrYQSS}P7L-M1yST4X2v$D4qY*w;a39B8}$A1$5Vd9lkdZ=cjT@09D=Sv={SPI#|NDcZlAMp;0avMuO+BYJur{_TmqLWyE}(uBd!bRH&fr{dt@&4g?I?X9hIW)_HBpfKpmD@|9nqI1pG0d2R>4z)T%yz#9!O>(4oC=YX#==AA7%NJ;BQ})D}KIj?1XNgOYQ0-7~$@t9=7DuI-WuU8K<WlMbMPE;w5RdKUjD#~XDV6@Au@Qe99$>bDWGsFUj0Ze13}uU2f$Wtq=UMLAnKjcd5ekSc`DUqR^>zdG(;CY14Hr}UNY{l(>9O7ZkhrKn`Q5#1YEwZ^uXP!f-)?_}7Y`n42}4HMCfkOmkw0fxJ95noZ$SMx7AzZZm$Q&2GBFSyY3qnkXdHtvb%c|E_Nfe)Dm{|it{0|XQR000O8001EX6Bd>{SPTFFoh1MO7XSbNbYXK}a&>NWX>DaMWMyM-WNB_^E^v9ZT3eIbwiSNYuR!^bk_kocI@3%>u`*2)+wnB9J+>#ER-@rSB(!3dA{YYPixcO!_gnxHl(g$jJK2X_iU7|2@*QZSD0;h-ohrqqlTEP`>P8j4t#p&ZV_R26WrdM@cxo<$Y)VlzrD_%YYOH8gH}6!TyHdP)`8=D=UTQq0x@r&0vKHIk04cH6-M+7-sEpY3Rc(QY)xz%J^In&ItupabrLK(tNkl7)OND%9SMYAWmp|)H01MTna6~nt))$rKy+9wf0xjJrQC7xw)rO=;(L|>}glaD@R5p9JQ^KfP6_yTM0v+JK<<sg)b+-c93B3g-((*ANi!4Q>Zompwwb?9+qS<WQ>AlGFZEyQd<vDC_b!WjmjkXfmo7v30E2Wi1EsasezuHN&tE-Lw__NWCf3{#arfzRrw77f!t*mRgsZ}a|lr1g{^sVlSos%eQJH4&y;GDSypLV*b-mBScR;sOFcjZQ@GR9vE^hL_*=2e-By|O!9!s7xo29y8NgKg+pEJYM0V*WKAS<D3dwDPvrvV;}XA}jlSYhqf#pH)O;=S*^r?B-YJQHtk@d6FXC*fUQ8?vC*~9jK|sptYqbs%rU6)<%tMMzyk&U<<R1qZBwqizwmcgnoAlT-6}s3{919Zmlx0^F(HL^6}rkh@Cc>Y6@MdI7zayx=_Z($!hWC8i;PH3oyVZVpK6c7f+suFFpseh;zqpHv4<v^=x>Kf%ALSEZ=p#N@ny*e5a7rtCoU>ZMv5?d84j#3Q|M<0sEeRc}A;N?IE-!2Hrw!L0nqQ+@fyt_VnvZCiI(liK()rKeSltobnsUZM3do3FV_Dr435~{s2L254%<rE?JhXVQ2g}O<Ww5!%i<T^M;VLy3q9yLEnG#=EqkrU;O9kcM)wcC>@_Q3VhxaYT$LiapVg|CfOmN^Ssr@=2cVKJcp>Nx5;8~PqoFQ-)2*riYMY6DvE$?6=(2)wvg?_6R_HNoFYe7Mv0$f-K(cvr@J_M(Ok*8@>Qg+_D~{$8WG5%<cRrTVXB|ZZ1f_sJo^~Sx4r^5kTp>H1n6AZ>Pm(2aEiA#Kj~UE;@9Yhr_aCn_TQKyfhRtEM1aXE{zTu1M(&|{GC9<LB!|*j;0&HLDD)r%RNM}HTtUCdq6y8rv&JRLGu>Uwu6zsiRk5(#1{Pcavv=TE-ML^ib|LiUXDIf(Q#BWMM{)TvHk^-dl<8|%;(`RIp~0-^7f@QS$MA(lV=#Yt2L$@s#lC`aTti$Jx<LYzm@aaW&#9GG$;*7N_PV>}D#dc+61V8Q&^^@O;qB0|DaBlw-&TzT#!%~_%(10Xa8h-F@1g4UE%pa^<top#5V;J3Uib7y7Ph{n{u+eZSHgCEQ&6vif@O=HGSn%dH#HCy-#-)JE!b@`&QupBU=gFbE7;sf4BvSPeNH);@Cb}$RriQba?8(xup&ax@F5spfBu+N6s#gB(%ZJTp^Sl+1RKCn^UFv08`?OMdfn-b+(5syx1x}B0SJY(2g{nO#IAom02ok~*AR8MZll01C`smxMSr<_gQpCl9<;!PYEUFNp0Ou#mm9uIkLmSJxBvWs_WX;CkcKO-k5^nBWFtUfWcpcAgTLYgtAKOxSO}I-gIx^}3Yw@tLlGD!{;X?7<`SqnJH|eKh_HK!IgPc2+6vJdI?KnnV>KiW0g15=u}&FoP-RYe6#tH&QZeV0ubORg$Y@$?{oAnI{)PJDAs^I|A0C}Lb+0}_9sP54n)J0nN^0(tVmhdF7*6rzP!gkbsoS31MZCp!nVsFu;`N;@+EXkX6ysN}T;;`1YolVH%`RmC&!s$lC|8m|HZ-d<mf2$4_s5b<Hn(wE?U(a&c&_U>Kmy#-o?(zfOM0!LdY)))(X+4;LF3pWs&~V{bp_ZIYK#=j7K%nwR?tBdH;JI7C84l~&YKt@lidZN6EC*MOs+6hWMkIW=&i+_-bUex(`K2_`Clyp4s_}cW8j4DSJVP&SlK8jd0?lmvOFz9C1t2oNQ#Oh3prCb1)&vR2Ghet4&DM&B#_qoO4f0wuTyaz#L9AbN>*p<$!U;F(Eecyf}!_SJGD_5cw;`SM_fN7sp=4@=6jkGD1=={=N_%x6tuxvwkcEZ)_rRomJUD*uj2Fx3vRw!(d-rOa+44@q{TQ9as%LEJic~7pI1*ed3O>4r2{UAYWVG^=?Q3o=$(i2D4o$9c#M{y3qEHPD#8rT@idN)StJzvf;WH#z>^3b-Ox`X_`ua+hkvjU@n%6|_jj-=9_{Jp*9{_(Wd3ys7O^A~8ChO|YgEiLz-p1AfncU+yGnPt!Mz7ZGx5TTeQzu!*S^PhObR}u{LtnRWCB$Cs{vKP@0obj04xBIE9*Jq&Q$ab%ts}t#}YG$kVT<d3qW{+9w3ni^+(Lkc0$bV?Em2~jl*GKBtWun)RCS2qyXGTBW6&{_ShaD``L&>5<f(*-n@7od3iWe?-AhXP0RNzD{_lT9TT?vnwlgGs5PKwOQ;da6Im}9&K87cAS`3zm$-Za2gCb7YM9+y<?|gB0;AL2f=N&pJ^(6UYq&xghXdm>RtH)ew9DwDiIe0$J>T*$hN0;$kPNt;gP2UelUJMuP<zG{@Jg2Isw&h}8-7R2V=00yV=?5K@Z2xPd}7Ie62KNskH{bIcZO@0gjB>w)R3$*8!KD2b)sYh#u3sKeJP{G0Z)9HA%M4eIHX|a7$*iR7Th{b-&@&XSS;zDD+7aUd6)s%kQ|5qa;!H2grb~Y`pTO5VlXN9fk8V3vf3D<f`Tf6o{{*<THwkMVx!Z!xt=8l-D==(!V44xy^G`Z)uDu8xgvfm6ll%9!AJBB$+$k!c3^gwX%9xt(<T;k?}9ME*db((Z68`PrtBDI#DhzeWSmU~R2<v_jjJ@`Etvt#78k)v7`owY9Rd!s3j}+wI&Hi>Awj&dN~eU9VCFX;r$8c|qTq&lbY(PEJz9%TxpeqzvVo1IUOj-|$qm{A`O=NTa20kU_Z<(|@<56n0l^83F)!Syo44r&(l0@k@HZA1K=))Y@v*8!MgWB_RD4JHOoo|{Bt4b>+nfsPQv7;!8--c3;AC+l_6onIxbbch3lZ~+&mjB#+w~EhE#X4gmktT3nto4N;>yS5%!l!EdQt=q<Yn;yLhqe)L&~T0)?-oPkh&6*d^jFaC$~C`1Y^esYL<IiakpL{-|{3uAkjVO^4`JgX#PBLo+IxXz>tc4)~2Tmf;%)48R`#UF^ZWr5yVZy9-qdaEk*phXOK{+$6-;{EnW*x954bsDxYf%Ey%|ePo#_WWL%t_#DLU*hrsi7JYAw2j-aCCb#*yveyU-(#G~}0?mJ2E#UW$A;<@AXFy+%{hZoa>;=dhgey4$v2sVL39*bd>tmM~^i1G}1XfjCZ`Zf=0Hc9n~82lAx`gAk7AX&^#CL#61>Gno9X!ck_S;g-L_`4JUbD7oD$dh9mxjN1j0Yfaps!Whyx8p!M$p0HF5DGntizBq~B@V3#<~m*A4ypl2arp-;-XYYP4lu{aCK*EA)sW<#@^p+-j^-mz3riHE6kn|P@cLdvqH5WiyTBd<Vc1WCi*`WDDAf?US<&`!GPsvn0x!SO6^K*Y+~q<zqz1POaDKJm)9dkmfM586Jc^W#Oq1X{Nf%q=ivt`Me0#~}NKavE3>*ihA;5HjdV5>miO#tcouy%^^bmLCI((b|pgRmOL!ru?l(@;cS4}T6#1V>31!!t3m^%2PV7y5XpQfV#OCRI2@yms8b&X%h`!Dn%7b^U>f~vs(93WA|4yfslu9+GfvY@AJjIC^s!HZtoc1nrhhTRX+u9y0OF&HG%`=-Z-OdnHB7Fg%>F!HAiB-0+V{!<4L(?IC%v?XHe*-?#Vn1>35k&qD9`lBds;FZZ1|I6gql)s*AbpZl2*bY1joFKp{`h!{9%3_CKd0_20NtX0{AQ+qi!l{E!)ZhPKQK#fX^<ccoP30uue6Tj)N_PGeapd*I!L98CCr;<f>6{6E36kQ@aN)9d&RKIrDip;v{Nqh`52Oc!>CwGRdWS(C(olU*b`z5GLfA?D_X0jLX8#3HO9KQH000080000X0E>r>10e|j0Hz!O02u%P0CZt<UvhPBbZKp6FK}gYV`Xr3X>V>WaCyyHZI2tb5&pivg4lfMD%EYy=QbZy2#B55K>)kC+G&eII1to|l9=0FZphuU6vO|$&yf56V%KOtRDjPXa%MO)oOx!37lz@_XUVitCzXqoy?p)SJyZFU%u8*qOymdlub*E3^vxT+SE+pW;teZQAybu0_Fk1|W@Kf>K9ww!r8tOE#6j@4O4?HCoR#uYu33^wkypi<9b_pJi2a7d9OO|{X=&XN?eRyErhAb*GAlEYmnyM=!EY*O@=~NKQDw}2Y_&_pUZ$4mJiUS;g>dL|!pZhetUSvr%jJ_a4Al7f`nCASi{IHG5rtE(FghzB?p~Ti`lUo9X9<mrgbNO`kob|8Odm-aJ66{5Y3Ol{gD?z(;0Pg%^J7(3Msm(nR%lbgrd*eTWZIgBT$7~2TF9@7+NB_<SMn+=;B(9JqGpSQDb-OVWy@vH;_i>%@*{2|2!iiAA?|0-nzz!zzng&JlL`B{fz&uZ2xG+6Evnp>B2T29?CkYrX4NLz%q%w-zhMWJl!05r<uXTMBizez<+Y%Xk?>T`KC}z;2}!?jLw9(L>GxRJ@><AfuXVaw4{2Mq8-{k_Q(VrZhQobnsA0}A)uMcUH|FsMH%$K9C%Ha)>%VhX0l%E}p>+pcCwBoBC8=GrCy`dN=NRv<b?Z*H^(14@fjJte=^1z4hY#!7z0PL~1l~Oho3$TCyN#mV>IwlliNXq5oXJh=2hLTlO3qzitXUz-Gpzoa(BVlXavLVqL2PbA;D*kR3JHkC+xV{MsyY(@0m*s1n53#`*IWvfM&$OWO(sp7L@z;gZ!{cYwOPBbue51!>qTzgOQZ7BP!O`oi1bRJ)W{)d{HHeQVMv)du;f-`ZPtp6XZj#hP5@;~b6vA{BD*(|l17^$zr@Y+QrNf!V%}h9YXNY&1;Rhqn?dJ|BLf0?7sce!SXnNfgh~mlqn<NeXbEK{tFP)W#%MDPF_#TJoHSKVsLOu1|2q?<14ns*Rd5+#RVK=M4E`Sb_+AL(u+RhbLBG$DIR`kA&>x6c+Bh7Xw`#w+%=3}4_FuZFK!!9s-D1o66?==)B%N|ydC`?&zt-36OApnko4Q1ib5ohYsZQT`s76;6xAX3gInU*Lg!b(2YC&3UUTe(~S<%ER9p8`N*g*^YjPOKVJ`C^9B^VMM9LQjn>IAVx)x&bSj_uE;hcSyfy4c0RL)lAt^J$G{v~qFE4}`2+AF~czpMCo|g@;PhCoXm59XVR3q>5Hcre1ZbVPBij5pp!DwWPi9VYQkQ{S~A3sZcr0WRCNVb5-3+lxJm;8z|uHd*4}kAZ8*l8bImcvMK<is4YFI@2a9muar;}z|IK_5|hV5E3_oyNh@TKU}tqe+_R%Hw!A4koKUJYSr$*t^p_bEQu5|r=}S!X!>j5!!I4w4z!NQ4H~IIADz~phYI~8iyL}kDobYfv1BJWr2^9f?qt3!@@8}Lrn_CztInZV4e*M{f$Q@ZKvPp!{J+#n4J?pFc=XYxWtHTLuM%h`UM?xR8nS^;=_XCKmE1b&2NxBNxp}7XV0bp(as&nkQ>N{LqDtBEwaN5Hlv?0Q}Ivu%lVy{aXg>Qd&8Lm2G&QhJuXvW{G1G@GtvBzj73%T3eJ#<I)Aa3NJAs(D$$=h%K=wZ5{08PzhNCq-)K#2|>0ggb3rVMlG+G~(Lj61`cA1zz^`+9+fThxL?f;1$!4*whN!>-@w2PICq1|BBi=x9H!a5s(cdJ5rw3V{|bea0N}B$6v{hN1<kuJ|>rYPT{X#Sn#Nh|YBt8T7aDZNGo*Ibax+jq$E#QftxrXEWOs#vaq6kqHjH*X@`G2oP51+fc#@*Dj*MVKCLL?|8PEmKCJKZxX;7MT2->QHo7wf+550n*HZIPF=_I2hc4}5CiJl&k#=g>0#V2UdiwRnjv%BOXlCRU8wiJpbZHhrdjQ30=h6>vqts*<2xXi6MnZmb-9UZgrlh2j^mF(*}b;Xmj|i}1F6m;f+Hur9?UM+%Qa4lDq6*EB94Zd_;F;+L}z}2$=SrIW{&w%J7jAnJQShTY-sO7By9nk4&0}7_yD21+G{&{Ik-Ha)imxA;ewyt=?@DM$)6l|KrOVF_hAwskMg=<0A5j|G)`^-eV{NWj8Y(5ySP|QB#=(nCXJ@K<KZzo+26wT!ltg4wlZ;{5oKw+fkd8Xc8P{c>C&E|&|Q*Uov34<*&aHe12@9#&1@L<%IPrI(DAtDw3op#!iWlJbn}d0U_L}akDIW7!Q4w6!o`=wSHH+W6c)c`o!CB1bgOK;Ww-wmfX+4dZv_9($m}z^xfhSl6C!^%Uyi>`8|k3ZL0*I=s^|FTjfF0&%ky6=k*ZRSm)+y3A9YlIl!n04wZ-GvoN^X+QO^2jxvAp9J|gdtoomJ$fF?y7e4fo42;1V{&Uc_23}G2APLVLKtFoxZ9sPV(4J?mEL2ss!(})Iu{)60<uXRns=h7mAA>i%NJ8?dla8U9XwFgeS!d*!7MQu4Uh%k%Nr#)+i!`XZ^!c9(&+AHGDjYWQq#6{Wd?JCZs$fMP2Nro%*P5^aCWOYx*GTX6AQ^z&pXih~g_PLjGZo;^3Lc;~lx{3Vu_*FNqBq{J_{8n2J9;f6pqsZiSq(4#jrmsQsd`eKVQ+l)SRqEb*mG21sgQ=gy(0t<BPx5B3D|&#w!V@Xr3E}JC`3q_Mv=sya8CA3%;?#|l3SF3cZ)(3TJTlg1`u|e3j~kB#lP9{CcJfl^{WE#*4lOLo1kD{_zv*<md3x5XVWAt5>G64_aj-bbaQ~+P8Unh%5kynPV(xZ0U-g9zUcTl<&G8L}kGZ+yb2Eo~1M*Kl<dgc{r|$_$=;8kJyZ*lGomzrsVCVH8nD`9G7foVe0mwm6(OnFivRq_xGZQ6o{TdJv!D@o$v^w8{g)aVQI+?Klh&{L8dz+qRS6Mc<t^vEr(@k+L&p^JV%J<Q4e4M5gnn~aNb9$;l4XUHG&Z8#yLr*gwpDjD^D(qy_3|~w882lShO9KQH000080000X0JT3z7a9%#0NphJ02%-Q0CZt<UvhPBbZKp6FK}gYb7^yQWo~0-E^v9xT5WIR#u5ImU$J37M7l`yUD^f(4+?Nin>J{g8cnW1V;BTQE-l6ssd7pCe1`vfXLk0LT*~sb1?mrXA~`!dJ1@`7>~0iA-?w>Iv-`HYt(qHFRy7ywzUuevj(1gA@nX$(?a&mwVE0|s!`HrLN8X7_^t|br=-V#4;c*m2i$&SB2bQMg&<`C?Q&t_0ZP&A`Y1%&PtF{q~g?jgkXdC^z6*Tgg_4~To>0y`fs7b_r=&M>k^;w6Knn!*(y3ZdgdL*uK*7a4H<-O31mkS1e@6_u*c3s=8<*Of#AHT`+J*TI?b1~HY_q@2_>U-9=hbm7|c6yO#P1{tU!E~RAJ-rkA?91PNm7)ae#cHuw6ue|RP+`r}BI`53`=w}yF6U|2w*A`CrhKtVBpXoZWyg<M$64EOR->IZMI~-o*5&)^jx*6R(lq<i<$Km~GzHivyKAe0^;Pr0{?H*^$Fo9614;2tOwG1Z>8^OIHN6@ODR~wA$iX$5`~gNj%2)W;WCxzOdT<%C3;BIzsLQsKZ&=l^Wu!E;Ib=;$!jfWCcC=<ubpSUL8*<BBdJ}ykPEYdRI@|FYWby5?v~m)0$7^5!s!{7Tvyz!av;ry1C~xz2^C)>f#o%+KK=^UPQB`B*T^PP<lAKpZB%~2BhE*ztvaCKVBUkpS)Fk`^FcixbgiN6sn3suBS=E#+@I@AJ-GX(Onig%H_WVP?Tv-hR|D}1*<71d_8CA`3=yAQ<tz%7(7)7xIMD>vqxe{4GWG`3nUy7g1NJm;y1~_UX`N;PKG+TBx^$gD;tW4Jn?JPwFs|8<wAdk_WBG5}j-gW?FsRKFIc_K$B65tDvz>^ap$a{ek;!8P~FRYPc6?ZpvyIV%DqLr`bQjU!)L1D0(C{}(Zp8!kV^A2neGd)Je@8yr0ejRVr|D1jto15rq!V^|bd{K!ps868hfy-$~cYP24NVO96HI&9bam323d-#K1C3>Zh1oFLArNX0Bs!2tZA;U=0;bmjU&PPZdw_xU_%LtZtAITZj+hk%EvIeS40IKt1DQ!A(JV(GdD~jdhqs7E^@EZ(9J#IzcDf`Dd%+tR0pmG)F0QwsMDen08+pHFWX{;rR3fvSS!v#-QEtWKIj}NJ~=ty-Gnrp>nOI#J!5Oi3uj&P0$;2PGa7v?M^A|60MT-+}R1xXkO95Mwli4)6f2eO@<s!?n$rPNSufrC=1N_0Axri9t!1huA&seTtQrfMV}DCS8p5#XmQ(#W;JW=(`)ob5zg4?To30}qViI70t36p*faVI8o`D#86iV;Q|~L{@U9#5dVoMyv|TP(<0~P)N1+jfDQl$vZOQ7gO&DW^iZRn)&{t{D!ol9N=1~e4|M%ne%Ew6CZ8?CYM;oG`-mV)D1A1OlWC)D<4;jv%sL60Rser13&C|hlbFR4h~5#y|k$TUJFhnv0sjODc)V}#WXog6059!0DcwcN(5jZ|5Q@(Jt{<{Jxo$d?+Q`jlqw7%I@<aUxFK&Bs9d8MB7e>RbiTxc)uN-2E@Se5bKSuV)@fu^MWM-->V+&7z5q=TFhKSnNg#bodj^z79jrv!g#?y<r80n>keo2O-;v+7Lk}o6)erS0F@_tmY<IsbH7gpjX1`&t*qhg1>i>M%S^;7o>n!IsWorChjR3EykACCe2A*jez5!8K#!7w<HO>cU+kXD;5;~s-B0nowTe3aiWycGFzdL?tduT4Q=4OBjX@ia-ld4Rmre{!nlmKg+CJJ33RoYCU<62C+ivU&xI*SQ>Osm^b;T35yg{owxWiP^lMK_gJE6O7l(qd?64<1CQ<!_lkqgU9UK!^#QkHf~x=JT_H6;)~IGWgMRn&>z*w|xvskJf<?X`k&vA2W~xj9jFU337hyyS4$+8|ZFR-rS5vJSEc@Qhjy&b9MX{<43vaY>^gN3-k?+^hvlH<26*A;JM@C+(TKN=|X>Q2cZKWpKAb`0M0tH8ikXT7oZpz1khXMw@?(UmGl>v|NQm%+f$J5_2ex`F{2Y>xc^VRV2K9_h?)r_uvWuGG|xbHC8LI9Zb(RHg8zp~d1mo)k)JB!7z>v=z`f+RRvA)kwa}|c1?<Dg80bzH#$l}SO5axobAD(Wu5d3v>BIFbQjgjdEJdcfKBkU#D-ds^!~Nj55f~dPRQwN#2bRv9u8eY1Ly8#OjBrRNW$Vkz6}L=rx!798eQOF9G?*<{My14ZQj!=&y6bfs?a!vG4V7;tl?d)e8Y?s4RV3`uk@HE(6)sUGz7SBVU%u3GBUSY=pjL19Q9f7D9QIZJp!{-{(r=02MPQQ>-}*i}+7mRBshXqUAN)N%oCrdY#NJ2(ib8D$gbOQ!3fH0Th1#-N`Cd*Orki=9$&O+#H(zBLQ#Co1Wgr3Y)!&wKuwNmlPgjv%iCkB79rA<OQvN$TnH8iP8e<D~sKg=b^SxWwD_gr8<gqq0^zdy5HJf8S&1>xlt;lp+8Q>V=U`yhYilVKO#9G$tq#Y8^_D;cRNv5Tx*YuH5X=gOw^FtQNIE<h(+9=P-q2Rt1o4Yr5>kDvl?2X9BniAB4Y({UDffu7*ZN{*7SzV#mSp$<m%xtYIi+1h3+={h3tWsnum!$x-gy=d@Cp(%Vn1ncx<a!$=P}M5J&qAZaDFkU1S6#cG;oRg+YBdCL9_+a;jlH$+17@=?yrZtg!J3GIu7-vYbt$)o@Bw+cqUY_<^e>lO@P7@w0c{6E5hlc@7D^VHPy48q#yXM-x@VlnogQm@teNumaIAUHV|!>J9Y>y2z>llVn``tGp5Kb$KpIig))^#B`{f0bvwbDxPJhvIK|3p`0M_sM)Qt&3oSjV*O$nRC*Lq4u;g!rpoijkIF_8<`o0)`6$6<%nu*r6GdKhliFNu~xvUp$(H0+A;Tb2)pLG4V^86K3DCAclC1gootXL7fTb{K4%{y*BA^AL1Ypw4AS1*sM&&jeTHSq_O#m<zWT7&pWUdy^U99W}y^Ul3_*GE|sQd&Y>`IrI(q#_Q1+1)>Q6t<z>N>;ZsYcQB)erhZWr?|9udH+XsvnFr#h!!)mBM<Nq5v(O?^`F13lfe`jJjXEuauYyZ^Z$Onw5?r#q+M4Pi-07M@_g2@9Ab5%Y<t;3{8K7Q{Z6ck^$c~%~w+Z#uC!mu6SY1w5wqb8pSFb%ZGhAj-?VPOqtTy%;#CithTUm0M4i)~rWrLu50HhstK76i%5L>qAEq>)zw=gbQOpb7MXr3&uVW^s0ZgH?uPqBHVDL+@wpI;}he^)$h9?6^NH*vj6-h6fa<iLt_<uDSNeGj*rw)AUe1RB*eadyOGDQlY%w5a1T<Ua~%b28=@EOyG(m|Rv0$;3g&Fd>VCMCNalc?K)SKD%X3v<4j%NPyosO;2=Y?Ym}9*D<yJVv)lPf>}4#*Jn(E`@KAxW01@G3W^6-9S%dE?cmoDlLj8jgWtdid8>#nu0UGX7gy-QQA|6_?-*7f5K1b!9H8KbY=ti*3wFyNB%2@q{BEOnL2v^G%>z1$*<WdH$?eb<yuSF$KUll_h35)~8V=*2-0CVQoj>v-W`D!XEDt;4=J-)v8bDGNdqu35x=`7*WGOTzRiCCy!RyjLhMb&HzFM=g9-v&nnIE=qzH<9Q7#_>fU^%T?_ZZNpbnawrPh8mv(7)tRMh^7~g7M@Rc;f3<1O7m-EEtsrBlh9?KsOU3+DV~ci-l2r0H)lKQ^?|gMId)qam>w+Pfv^KvAj3iOt<GVJ)_2d--zLOls6aL3w7koG*dLOSyKSMujxtqARgyG;X8K^S5i2>g%E6?theG~$WbAF9Pfkfap2m#(&^;7k;qjGkyTWi<DO!0r%h5#FFA}-hQ}LC@}TEefTqoKEky~On$RGSW>ymXV0DD%f;r?c*;#hQs9}{9xHi0H;ws(eZjX&7(I8noe`-8wjYph9$G$U-vb&sAx67Y&(Cbtv>dXj}Eh*)hemZ=pulF=(kMlqESl^o2ztAyz#DCHlFFDG0>=C{wa%x2Q0?Ao((4{d6KbV`M;Gh{TJ$JG{fuqw${F6gb2Q=mmQ2%RJ1WW++kSq=``H7SM@a!{p49rc<xqAhko740!zd&&6B7tH_L&I~QzCSQJD6{A#ufoYVc6Y#RJ}2AZn+f8Z+l6LxdxS)?_H}RwU#2hEwY#)PL(Zpg;2FSC03XU%CfR`+ZL-5lPhIvP@K~<T@G^^gC^n;!L9(0<UA-*3;b|Drxw&PW=8~xaRRwTGqp_y<E+wgbKbLgHzTQr%fdFUubve-|mHJwNM1=IKWb72~q@mn9x_S#mi#nwUcn@9kmG=$J?OT_>zPMqFwV9jWbS2c!_^D}0ZBm<F^FHfJEFjf(1a@<`WW6$BdPmkuc5Vv2KO=<(btkK_*$n#ZWmz2=>-pn;=gO=q<DCn$BXzCj`8<1Z*00j7f#KiL9a{G&`t;qLnF>yt#+;s;X)MQIu^&OXwb`Ko7H%#7R^k`Nir#(`TnGtr`>eSEV5|CgaYlRNTMZP{QOoy<U6f|<nOka~R~`n<5xtvPp4{7Wn;Z7t-_k;biGC;^;^@Q;X%`uEW<d|93u}}6Tz>26V5@_mkCW+CE0DORpCL-!T_Ac4kh(cAF>?Ea7R%i2n2{~l&T7KVVVu`I>j23A4Nyx11QY-O00;m803iTLK5x-I0{{S@2mk;W0001VVRK({b#8QNZDlWTa&Km7Y-w(1E^v9JRZEZCHW0q+R}ef&fUQGwGvGsz-J*wJvp}2nuviEbHMUuckCK$LYOw#kLy?v#+d&J|7i*rJ`DXZ$Qu+t$d*X!z!c<2-eSP>yBrrEvkf6LH--JI9cr(4=!V@<tez01JznK?Wkya0?V^)BxDWyfRvs#j-*$o~OGz}5bXzMYx(te6?MUiyzuNQ4C!LyEecD9(F`JJ}%EW4pKR&#I|Ty(I@s(`YCli;v@(h90gLGYKNyte#MM9nTrH6J=w3)ip{6FqBtD66UMl!XXKLL#Cqao#3vRXfkDcL9l#h7`0)M}=RO`L?bqRSfDR&=4SPBvWDs&X+%|*D!u$dvIh+zQ0czgyXCwLXmPlolaqNb)`57hhQ(%mMIrd1?!gXM+gNkwTyG<!7>lIRt?gOx#C&$z5bSKh8_><M=$vD6d~mzQ}2%pJG}X}*KMvdBiCGet@}kPZ9wtU+Km1%cbJdEju#;Z^Cf+}BC5B$SDz38gse<n)tm#~IGB6uT0ckEC+&X@*Wl4wZU6o3uwcbar557I(BZP~H9r79HhS=8@Z}=&3*8G(SA=5V+A!-N2){gje4wu@65!naW(Qc|9(Q2vHcR^O`1{k>&!6XXVZW^~Wo$XMTYmX<>SvXg^9i{Ey5}fO=IU+-_gn0&K#<YA?BqYS8(YRp)CsnBQiNvL>%RuD<9cs~ZOq-IAx3LhcILO-omNU8fFTZu%}v@1RyU>}Vz+dy7SaOEixLSGw?~1?EdLYn8PJ#4Nf#-{T@-;egzDfea0cH0J(L++MfZl|?jjfEc_SHab86dzZ&Co(X25#n{BlZOT?M+D8N8$2a)RG)(4~q=pqms`2kCYMRD#rYY}(O-yEuD%PWmR=?gVjWSSt|m5kp>5jAVZXbkCAd+Fq^0`_*JD;4`MO(rIFy&HP5yQFGqBvdRf8`7GSi$I>y;L~u->*M#0o@6INTB%L;mCTG>@el&3}0$9^W=73k_0%Cp!k_j$?l*lQY`6lNz>^}U250{riKNrzrsW@gA7bF^M79NBqxM$Jmpj6W$ZyzMH$GaJns%M87`d<74P)h>@6aWAK2mk;8Appm+fGS}P0062i000&M004Ahb6;|GZggpFWiN7dZggpFWiD`e#TsjG+_>@k{tBM@p;DqKwy*7nEd-qFIM?9v!o^7oY!?JAad%bgb)vMk6Xd^lhNL7?qHMPXTEo~~k;CC|-ZPZvc^^tn@`|NAC2!xdFXtpHMa#;BhlJl{qRq-ZDXNs`B;~uT%-XCf1z}|h({-LDSqm$6yy38fT(jhcm#K(6&s(i_O;r#|cU{{xoKlh%b=9=6wyatv@vK&QEUC(t-?e#m-H)g*7>K$y%N4GgwFemK5q*Tm0Yh6g$wBWKv9{$U?$Cn@-qK-^YBDu&%9AR^V`|vfRddUl^b^p?SF2T$Ga<;^oE_O;)VlSGz#km(JY{vu8^W5Z1BSE*Zt$$j+YA`Ta#A<EW{o_tWE|isnB>)URq|*h(VgcASQT|EfT0BE+vGqF><CDI>i|v8%YA!@$U7!(KmrXXEYDvYS)QeTCS^qqS(@@vBCA=7Cwf^|LJEvDJOc(dK#Af!$po&hZh0D!U%n85hb7lMugbjuUdpp10KuE6Kdqt@v;bs^ia@C^c)kmjdW7UzNPtx>?Kh=)wa;1+Uo{;M6wmMv*F;LAVztzsa^V0-*RVkms-`$-f~u~qyVlrKu?DcH8MTKRr1IzkWVTzTV#H_RR6jQBki0mBkM(u{8;Bpz9OVT)aF+UNRiKtxt7_PKBo&|N4Zo)#=#Gm43MAPzrKPOC+AowC)aT}83;&Tgu$pgPoNWWevY({2M1$c{q|FZoc7mSlvOL%9RBB1H6i{ac+jC*Cb~LqE*<b`ZEoh$IaNk^*w)ZuU6~uK_<$=j^g*7VEtRQhrezZCLJMy8%9l!>%qUc(7opU7WpFC-K`eMf!IvC(cCpfvy+Jm&80yIL}P&7gICPjilxyhba1#g@Cmr^t@C5d1b4anzS8-r~NzZV2VVI!1IBuDoUffqyHi59F}$(wiQKVQ<T55HW#r*GcBy8QdfS_&uMqd~TfZ)gZ$*=75Xcx_jLWC~j3HD$`xcp-Vsa$(xjpb#vB<9|Udxon!M@n;ayjxShpFqoz#P11@Ba3bI-yIT2Ez*kC6>#czk;QT<93x@0iA3xh5utu8d7IlBKwb@l^OrR5*2<h;?X0`vIqK!tCsNey(eT4~iX2$9uDy#A<Zz`V&+IDr$l_vswnBfFL3}~aWD|qe)<cFA?uANML8mlF{Ut7xAfG5xJ!2FISZGKO{R@6x~cWdCY;(~*{a~4ad2~imR1vq$rI0PJW4#|sai1L;<0`-%&59f{KG*uj(YZnMJVs#DS&wW~D*|7aGGkm#&mf2=43Eu{z8id3GPzQTXPQf__LAZz(oJM>SK9O6&)Bqu>`rqJm+DD#jrG~-)DkYx!`;sO+(coTBEKT|(P=TTYH$R5=7b?7yuEgLCa?;7bqh=;~CI;^d4k)?tk?_e;Qymxp@;#_sC_gO_w=4JEdL%T%)@R$fbB7Ur%n}}JQ1!HOAA*BFThD17sOjUG!W9yVhCerhQK?nF#UV+701Baqfh!jj#7f7c%q=X_0U7&`{ObMEBQhDER1{w{I-3khFzy~B1@kFE6%RC9L$~y30SRI}l5asx(ur8I)`9>(MH37X(p}nfD(RqUmb8=EV49f7^Y&?(PRO2?e6aM0^@mPA`dA^;v`6y}o1fZ~45qEIMB3Dz6Kno-J8?2It+r>ChHDh+$;FR8i5yK7YJqakZBk9b9wRe!?mccDb*1>N<NPc4PyTC7)cpTXRISgR5zWAbht*LX{C}*JtDna5GtxgH$pJfWR;Z9U-?QYN3RXZ8(fB&^q)k<|vg+Y7PfE=~2g;?-5F9XbteR|}VeYEeL(GT3k3ohJkyJIfdWa0VfW|a4Bqcv$RoNH8vYH-JO-Rg20m}>C2oPY&Rb?s5<RS6=c^H3}-DRaRyn&K<Iv*+8Ryro?0~s}2)ipp}EK6>S0ReeQJPy*MEfXF-C{$5>KL##Q^8_;#tE7)`l;V*<AUUUf?qVyVhj<a{E0$`CgLL2*q{E6dhsZ&%yPVIrRpV-7B*Djzz@jyh4Us0|cR-<n_tn=3=jNa)X#vpWC{!3DEAKO&r%IYAi`<x6MA-|@?neCDfzcH@bXsM(RZxc}N4Vj{q-AH2A&)~cxKWUA!iA`e2}|dP(P!h&#mZlSJXrrns6T#SLk!eOx2b>IAQNh(sI3tj+&qI*m2Gg{BmzjY_v7nkhx5Fy!3kF7s{v^VPR7^y3IqhGL&2W?Aeee{wMtf|4=q9fdyUod@;#33%<Dt2R3lh^sM0}wQ0e_7muX^q5Fwd4IvsYrq)XZQ#Cux5XXdneA9dcf)zu)`)l=$e+c&(re0~1;?G=6Vj$XchrE0%$3F6b$`L9<G!(ajV@%-xIuT!wVMVxNuKq&7}>GUH`(Ofq8oHR>ax=<DVd_r~xUsufq>jcZ+zB5TM`z2VY>w*hJ5oLnH;w(q6?N3T(Ri$WOu~4UYR}iO~4$9911K^TlLBYty=U3-Aj4LC8@l4LgI=_Ai(F^Jt$iOZ>ex8&m!OE@+Q5kojVC|1*kTbN9hl7>!Bok6M583|UO_vP4WH@43M%Uo)4+U#(zzk5s9)Jef$Soq-;iT>E$Fs*5!4}KlswLCA0wumkMiA_%Vy6VkR%taGPpW_iNg4*UT*fv*G3G17nBa_xf*vIspz^+z-cGcQubh)YsHVMb5J8Px<wp>4Fn8jx&@!kQd6T@$rWNz69T;;gyGskSyMwIs<4*1Uw@9$&6mR0hVJA{@)J)B#UY-8Q217|Ebw>+cP@rb-N*t9t1AVOHZ=Sc-&)?^nr(iqqv?W9op&7KAcBE#^a(NG<-#0Y*3ZxHeRbUi022}qAT<gS(TC$E<+Ba(a&7%LpatD%`40AIHJ5j)gsR75^v=ppkq2=f#X69Ko3m(|%yWIpY_!Z@E!6JYL1+I!?y!Q}3c69~{4RcYpG$bBa|CrY22U9PrX3@i{OEQ}YdKRbpI%={N<T1Vuo4Q0K4&9v}OZ5rqz5&d>;{UXmflzHTe3IWS<ah>ci|O8S%D0&7xdvz93)2icFHuFlcEr;xadu-Tt5?TO-lUcuGRy&UIP}$~p3J!Oz}}B02+HT?K14rsl!A?wpQ6W8Z<^Zt-%=F^z^n=WNDhPNgN%%sSEZlSWo6ZTYVUKddC+nwMW8en&P)iLhFN?LN=KD~$!9Bn8Tcw~EGs9QxH1@{&${zImp5f~tCkJ)_siGh+kpSOkyaSQfF3z`IlBZj?_(l|Yn*f~7x0_%B+o6d=T5~iQ-mX+vYE3P4z~r;>U-ZxDUP~H4}ni$*z@IBQ$cK!4KRCm?;oWHUsBtjj0PJ}*b6n15GYGDozkUhQ<`Kc*atF;Ia51qYe9VBqznul%H6kq`q=E%t>;j9(N=X&p5S$_fcP6^2}+I)ep~IMtG|7ajp+gM%tEKnVw33OOh_;uPU4ty;L*@kvosB<`(cVJ;+4>1g35N-65uv-U*}i_&;#7|5qomi1M*!DmJ|7c0VrPDmjeA2p&-3al$WmL^1~-dXbllKA|KiV-Wm2<+h2VVR*o-@^w5NLY1S^55O_gcw!Uv@k@LP!bOofwj@z2qiRA$kCU2E-|Ndl;-OokdED9@K9Y_o1p^T`K-GLhEs3I}RN;+ZXk@b2p5?j@Mr}&$fR#as-keTYN9LExlFLKbNsp@{>yVPijQR(cr?}llXsx{*3n8=$F_26sT92NxxKhuN-w(@!Mbh>vtN^fgku?}oPgipHw=<fqfu2FeyX0Xca=;|&~nZUR5DwE8}Kw%_gMnk60%EoR>;K~u?t3u$iunjZ4o~UYrI_(iB)~S~J!<w6{rmhq{f)X&R?POHC34D79QAb#{qS5)2eT}{GYqKVJuc)B5Gqi>ZKTR@2<8`UUXk4I;O0Bz>#>lnIERPw#jpePHtD`75Bx}aKZo%$o$!~i`o&CY+WP+E$&M7EOd>WmYEzX+O^PwAKSA%(;=e>ti;>gM633}b3Uw(P2oA;RE<h%eAR4Iq_djr@lYw-TQYOr_Q_v|0Zm44y(x3EBb(ys8L$-ulbwsKx{xwGOY_5JLkSH75G=s$=&<#%}KlkJfH=qc5EjEQXfdCJ2WFLhYmZTyo=nFkf6S`DahRYbe0xuH$D*QO|RaU>kq{FB?4Su1zEe=*bw-*3azQsyZ(PRA=HinM~29T+0&osS=Z>opNg=lY>{AXEoxKt`E-lgfn<?Gi<?qy_F<>djKn-_Db!(I8ikzZ`J`8oWn(@Gi~0xALy>+@`I@N~Oi+Xu7y;*7&^uEbUl+E(6?1fEzc!#CzC}?j;Z1Ihq1Qfk8Z05{oxdrrYod>;`pOP+7NnDe**Ijp>*Q*n%dO-a}@A`c@xUd<I?*pRRe!;*;o)^Zn-`R9`3BZ|*&NrZ%1g|Nd`nD%o&HP3A`!UB>L@Yr%O9zk=1j08mQ<1QY-O00;m803iVW`fzs@0ssI!1ONaL0001VWpi|MFJo_RW^`q9bS`jtZBxr`q%aV?^A#2+vtm#(nPjDvkT!~x6S+mXT0OX#@!|)v%{=z&*TzgB(fWXJbyc~$-Ols;nK%Vf5{n>7)4<0wQHG-=ov}otKO5U`I*Ukz_IR-1h0&xvV%xha&+{x3YB1K5;p)-(+s+%?9<$CGm43;LYn>%jN;p9^9aTxthY=3Q=4}kI@Ktb=H4%C-x)X;oZJHQ<HTDHY2eNs~vaFTh9C;V|`XOA%l4@PO89quZeq|&qj$JJ443<-evMb3TE-2j`j%zSRA}946sT6{yT&S9E?n|=S7R4vEGo$s#NS!nbH4uBmA^yJG5f{Y(zKw#ZBUKBY(xfeCv!W!E!6LEfN2_Or^U(G|6K%@b?FYH59v5Af{Z1EAb>b4Fn8l)}zS6nO<fTB5Aelqt^yfo5u$>wl5?r;Lex`CVE$o>Sf3e=(m!I%aaE?0R58DNg<6;$Z0UlHIX}3i6sMxHOa7F9t?bjf9k2=m8J0fsbf7AAOWham$&f%3>!G#URiAjuVkDLpA2$aXCg|s)0dDDmE1+Q1t;2K|<4!{@uF5&B@RZ6kZArp#)Te6;^j#Kms(&3usM;*poF(n5E9EQ-NFEop8l>T-F--nzgm!iT@J1tg<`6ZpE!4H|9C*Rdu_zyXxF<A2l+b|J<99~N!G=inOl7yQidjYPVQoD(#Sxg?;KTt~p1QY-O00;m803iT+^z3H-1pok26aWAh0001VWpi|MFLY&dbYEd|bZKT`V{~&aaCyC0TW{Mo6n^)w5V(g*g`+h=*8&CdQ1k*6*t((U_F@DCB^}wUDN+?FCvM>XzH_LHCD~5Ww40wS>Ri8brSi%YK`Goy$s~|PX(|`|g^#oLzk{O8rGoCc*3vn!ZZ^w_(+SI~T02=lEGj2+kviMS31<QRUbn~Zs>)PzfAq2Z_C};DXr4ZSt(E&7vL&?dQ=yHPsZe|+>}q=U#Tq@IMYGu~Rl?d{!DlqfhMj>iJ-!hZqF0_jgFLX{{w_lcDsK<zM~^YNX(~}|gUjA!l+Yf&l@+AURNF8bvsK18<RHMpv7A_o0?@`zwF6PTHP`xX?QyC==(-F?oYwKZ_c8;7+iiITU(hJaMUlv5ZEF0dGmd1rL<oRygsQ<F)rY7-<=%a*g$nyOhroxoEMc#L#rZtAxS9t{;DdLS-LPmE&F^L<Y9vHH5)zV-5z`*ZYw1=)!#)XV(ei)D0iQJk`wvqARiy^<T&5C{K~=dj!i7u=O;+Bg<@2_DkdkxsriYsCrU6nueK9DmqR`m703VOk6r}<O8J9&-JCR6*!J9^<3o7Q)Eh|h0iomO2Bds+0Oc0723D=g6H!nS?!lf&EL|>#(Y}X50LnB8PVhKp``~t}fa4VA${rl&{ZUN}&<({qa06B;C81`vC?~RNmEHM^`sj?9D&QUE8n51Z5Q-A*X2iwg9>x8TAWlIqq%gOgbSqNLXJ$58Ix<_p)N{3jH;@3fDmN6T|9s`2q3)y#hH!$mSRYTYbIce8HZS@>VP?C7^1rv^taqNmJ;xw}Yy%*5<YQrdc1XH8^lAIv^^Fv%7fMpQ;5`4%gqz-`!$~sY49YhxYm=7oFax^w3wfPXU(Y8?vJVMQ>RJncnw9A%ZT^g00ZC)I<2BX!Mmoy!*nK&V_hvU#h%Qt{Na5Bu)*g=eydPxnVpb3_7JS=N>(82Gm(?JIjLj=ZU!;A(tL2V6~mQ&1HwlAixv{11rF{Vr;EDD4TR@mbdjA3ie;_C{hP?$Wq#a{EZ2AyIrI-j#rY_UIM?-^)jQafx4b?Kgraub_q_6SFh6fQ{`)U9_ux~Gnm#FFjR#Y<(9aBnGOXB5k`r*R^2&>R+)SJ{E>uFVu|4L9zD)q5Kr==!v_Cd-5I77tRZA?G<dV7b&5w-d~_oZGc5OU%OmG3QJrmr~OJJK*Y4&Sw&Lh}U>qnUGcFgr_1rN+nVV_u^>=noQ;r=U(*iCFD{4VFJcD8eQ{hgJ5Y<lNq*XjvH~h#=L1Zxe^JX+5q?C(%!i;ScD)2zj-xU8*j&J1bgtT4}J}pGO0)HzN!C^J~H9xilN^Wnim+DJC=kKBRp=;J2b}`GV(~n_1UHvxHKOdX2WB7aeAotT!(VLYl*m{88z$BF&@G&`f1sFH;M;HUROph*(lMCTR}zfI|=C+*0v}4c%EeGi5AbU7X9S7@+m(^w-hJNlCtMNgJa}<Yt<-`dP}F(uFG$DdBe))8gSn88tvevOxLQpAFKgN%W*pw*aF0~$ni2Uwz9oh>$)hnG=Fv3zlNI>%P#ofDZGcT4r_rLJY9_OdgMh&6oFpR2oOq*TBElbq3$m0V-N~q*LFZXx^!^65*NR{=vz_Wf$$)gH>d^=mtYFIEwxvgeegt8iLE7RBl{WN?7awG#-1cxES}-e#Wk94sL!tOPv-D8cbOQYCbv`OJ;WWvHUhpuWo#HC{{;=q6P&v-7%{G6&92#)NCluo0*Zt#FeCj=&vHaE5$PHaef{pDpCo3yg74REstU;FL6=1cU886T&7GsCM9(QJal*HS!MzC_nuWv5S+qc3b@ufKcQnOL^`i5HqSCl>l<R~2h+Qw-n~6KMhh&r%xKZO6`Jpa1LHu@@MQ8!$z+nRHjzwhmBi}0yY_KASQ7Ydu|Gfs!&Ubwxn}j~`M287PA7Lo4?P`>4F{)w0GC|chEaFEa)~C*FCi8%E`kcjiaCsRp&I=(m=d9uDKe*8;Ht&A{P)h>@6aWAK2mk;8ApqmRWT49k005RB000yK004Alb98erbY*jNUtx1|Wpr~caCy~P-E-Tx5r5ZT!NK%_OeIRn<eaJJG}CM1`<S_2Gl|=WswxH|K?yNQ@BqlN<MIFAU4W$cDLWb8LzRbE<nCfWf4d8btdPohd#3lP*iI59GjDH95o)e(x$=a2|2L-j&)3%<KJwoWT$`U+9;aMQz3aW=EEf3=Uw*PIlal4LmbsHxFoVJ?Ss&nQMPztZBqHUG&{WbYQYXG5KIFm}INcFf-@;5rHx&aM71hGO2q(J~&NNkr+=z^aOc{}|$mojihw{OdQWDH!MrI<SClgxCTJsa9Kk?x6_kX;<`jvkAaP{uf)IP986fM{8K`f7XDp^dyMIsSAc8?3i3#K@ALftdX5>EM@QJ@e7Bc_vJGMPjv(0D%}-8Go4{n`Qe^&JE#xU}qlwUik*dl|c<dEx=rKZ|$x!jDp2QsM``g^|n+&yBb8wj}0;t4!oV8xhT-J&$e*DRMJQ#GN@P{-2z9ue{}A(UPECUT6@p-kjdlavP`AZyP*K5tD{sw7ASQe>epjmQ<$3lKymc?Q5PUowtosc}x|{cN`8!!f+3H#+43t-1y`pz9GRFuMSyJ@I3bI86gIF+Lashxn+_-6yQL=_X<DYW4|LL)I};lpA`w#%T4ud2HzH&U|2}fRb-Hir<!|<-mAV$k$AO@-fHEMx!C1W@tKuPE(fL8hW1ae!Zi4fDy|EeYwo{ZT;MS!lhy3?Vlk+SqZvN*B-w>>*va@I?_46zT$rb$gbCT+LxFh@?%c=OL&|f%Oyj_tdCSLn>(S$guwe3}wzX2f@_c9ZWa??7{LXn!k6=XHNt*UYNREhy;ZX_0E4k~^sgO{;N=oI9W?XBw<2Kc&-YceddZ4NbsMJXd5z79t8fp=~PnNIWh70(&1WOmew5sx;CByZDT83}nbZ%BB@$NsGKt|qTJBk_4WWKsq2R<Yz)F8q5?N?jC6*wXx*!C{x^IEgZhwed&BIG}k$vDm$&eVH=`Z1`j#O)6TJ3>d7_F+*fp*{u;`3#Wkml3Psy$g9Z<k(@QV(K1jthz09jPl`op?GAZx^Em~*_4!6!Pu+)wwy3$8ShZ*Jp<x$h`?{Jzq@d3RrUhF-o;`Ez|>T`^459H!zkq}KNM|F4eeFrIpAE~S|bMiDyByPAvqYza%uM9$DEs^R5!Fl#_nMb+ZtxiJ(9_or{PxQq$cY%@HWC&cAHCo29&h?9W=0gqRnF+wW?g$c`3f|6#_uZk8K*ojS^h1mZ!&!bdd+!duth0`wISlZL)kgD=BmJ`+=oCDiC5xP>a<*KuJsXIA`B2Q|GcXJS|0WIt<Z!+G&AlN3l5kQ~HFJV?(4Sfm7AMaOVfXnmA{X&9v|KmhZZW56$zC!O#eZUt)!A%K);j>C-J_K84xR;bSBfmU-!-PWl6QdeA!TkOxDE=hu<$cKZ(QqrIiBhb_*aekEpA{V~`ndev4@ua}pbk>YD|hd40jF!$fYLJf4f@n8U>ff(5S*KkIaEbGhVVmUBaHMW7_sp9;JW$1@TgNN}k<)Bh>*{Z?HQR5>^l~I@8Z~?Yxv-mq8{7vTkoIog9pn9TctIy-~8AIh0s568<=PmD}(6i9!Yf$M7Iz2JveG^)ZN@yY0dbZraVgQ9|#jPNHVL9kv#ZgIs1-r-rXh1WxbIf6z3ccr=CL(7k#i`&qebEYQm*xouoSq+<&|q3?qwa2LPba|Bt!F&&Wzfk2H`R+b0|d_u_(-q^8^n(cHWwFQekX80Z_Mc<KKd6+5Bzx!s-{zZ$Fc$@xJYL6B4tn#&`SWS@!)Rc!E2Venu{>VRA}fPz!|{<mcXuGo)Ix4Wd35a(O~>syn01>@^dh0Luv!4x1=(M1b^VUFFasCkr}`CUbsu;w%Hy;e#=r3&t8!r0Kv=NfD7ve+d|YV3(Z(A5_}%9oNhS;5-p8;({sTi2wJ<fkn;Wpu)N=M2&9ChB8Ha)_CXN|md;Uwftmx_TcIIWkeFeKy;%%UVX|o;S$}QYLq=?)C?1@gh@ESo)3522<B;)9o$@Q2VDD*Xz&jB<wjsh^Uxu-WjNiObv=N2r6xJZ+mfJR5RCxhGZ{H{St-?)&(iSU}+`(w5SJ?W4&gcoI#ZSm_MB{>bl~C!oCL#v633Cr4Z+TYpS<tu}w3!WxG%iv{w5X(?QJ@sJu8W{?ycJD0jdz2Bev{@Gh`v!LEudoU4iF!*LN|<bJ_9-3at2Nq!vT1|h{B?=wj;bz4Dd@s8&>=?5DefjQ_)`B@-7tPO>qdqI^DRu_zwRqiXXADm)tNX97zBk^|U;wh_{o-J#(0gW)hae{%pH~o*x+`e7VGcn1{#C?dVx%_-mGP9*+sIo<P$&_ulQPpf;lA0oe14115FGvn@DrwI0=b^PGJWOWf+?;CQsr`vY3c&lw$zjRh|gtT3E&9kGI=cekwBBu9pi#`H05|Naql;H4Lrv!Qsl(@K4tj_H)keU^&+##W16bpeH8v(2^?k;&Cczx$1p3QBsZQW1F~L4Jex7kJ{6QsGW*AT`(sqZUru%ow~CUC8C@#oM=Uz3;#GmKSGZi~P8Ya}&++p7r>Q;GUblzcrjMn5`*Z<U*!y-Hg6$-uM6E7GcM@qt(VVFf_xYg$-BZ{UU|VMvq+V_6Dz+GiGcmR_oN@Z3<2=a0m**3zbGE55WPz(^bFQWi0FZG-yG$dLS7L`&ir=(~wH|SIX>&_SjB0)Br<-N0)-!EVj8kw_bkaJAP-OuW>Eqk^in2v%j!evU#|8^Y|xnc0>IM#$-^f5`r_+25<Mz@!&L5GT;=qaaZ&jlfdmf8kQc?`N^Yd7;DE^PmG&0#&5=4?x3kZz`7|NQwv5tEtMuH)Ex~e;F03qNXRpK+J|a_&bm06h{U7za)?s<Ga98BPD;yP&`!T)GkdT9KTt~p1QY-O00;m803iTnY*!xp2LJ$~761Sf0001VWpi|MFLY&dbYEe0WNCCRaCx;@TaVke5`OouVA&TrkQIBg-8R9|LDP13o1!NT()6LnFa%nnZDu7=C8^zYgZ%e1!;3DqH%S*b2KFLq_|0%G-waJDeKRMwtP}+^Tobe7gyp7XC6SF0mH3+%q+w=9*p`V}8=~0VtP%ITBx|O5DQd10O6g>>QL-X{!t#=74OpUTq=F5+DOt`Z{_(w*wUb5IQmljA8+ddt%)aTj{|<jk{+d->HwY7OnmyQS!jq9I-+3GOeZ!Tgcx`lIn9gp$O;}q96A<k~#_A$#L@7;7a#=M@@l4mO(K~6hHv{&nwgxZoB$xF@Yy-kIueoAI$ofaS9+TfqTkPG#@n-)w?EbZ6x9qK3n@lFwUF4mp_wVElufgN$#e~3TEp=vaDH8=r$-*x3@#2|XY*;Z|oR1}8h5qZQ{x`%3*9*QOS(dZ1%(4g~xQU73_h#yp;zx1Qs`}9Lk*J&2fCil|aa%GpWi{==*UaR*tl+sUxK{w1V)Eq4%^g!)eXMCo2~FNhQAZVPB5hPml)OufEQL1FbUGy)sc?yi8nIU3w(ng6BpY!L7<_`7Z9L&<nnx8VFX3cWlVP5DogotGC<sw!kKZ*4Ml_AdYgzNyW@waU8&UEsn<k2DS>Ez!nqc-owk;Pci!>eTN*YLq#(>I_RqKM0zgn*Lu~qeg^s3!*_h=J3F}&7NMaxA@zKhB1Y(9_4c}y-=(_YfWHXdl+y7UB>r5@)xtbnl<tD(Fz&3O?m&b+Pi4g%CbslrJxg_f%re(XDRy{HR*kIq-D-ty>d5JJlZy3+&#MHD@=W}buRI$P52ziA(SP@Ax(;jrfe6&2fZ$R+y1>5B=qPJw6o6W6&CYYqe@kQzWTp&xssV0u97Uf}1$=*jxbSq;d?@9g{E-oCnqw@xg6K<0;EW<PgA<O!K47Y~U3dU^fw$3L?ZJDxqT<LY|$$5`_DvE;|&C0cRAkcIJV)E5VOun?OK$23I-f10!UmT4l_?|E*>t>AZ>9-w{fo}8OK1*Su;>-`LrtD#r{KT}eg6pA?NimwOWcFdU`l4xXPlij5AfwZe<8*QDQ9j;sM(dod0j(Idxtxlt}c!0zVng+VEHOpZekXS@H;3_&4<M%vot#!0PxnfjTo~?Xg$KE6HLG=u{sbcB|$pI+TD6lG`njxoBD6V&^1&AwFY7XT2@D6V;ZV3hxs*8qE*oLHs`?9WCXAO{MzR$Xrp4H&vTW<T^j;Tu4P7LCrUHE#|?D;Y#&eE88{gW|+K6H3Uj~;HEHhvE1Q^q5FGDiNt_;d;#LH4joFBX$RX6`s?IF;d5gRh}D(nT^SV&e}?pxiiA+q}>IaSGTLo*oVn(P#Y9dzL&U)NZB@5oaYgB?0h~Y6C`Tx*{ob?FL^KbSMXHJIZ%FzgbcrSl5CpOnumfY;E6Lw)UW5Qp@ZcjhS;owZ~K3@<h(F-i36vq@5E%3ve*z?;}tKj(3M#cvm05bd}g6;dUsCoh-b6mh^It)o{7M-&f~!HAE_T9R)NSBRIBI=+e@9!lmU<WI6A&bl&RQV1Ws`5ga=ri4N5dx}vwiAuRiS0KQewdG{e;U$R5D4ch1<o$s&*dl&R++Gj8%<<QcvQfu6cQIUK~n?*UY59{J2#&`p%r9tN$Bafo&XCq(U35>u7m`e*A@ltMu(Or^FdYbdewc$2_vRo?FHg*p*G|-&wKw-A#I~X9c&AG45Pp}-&;SJw6X!@fWPAah#HH=i=ESpvSfMLlBmVC>An!AZ<DWDpbVHN?`kH!R7%P4~W4C+_?ob68!WFE9cu2l~5aA4ArnL#?qOBipXewgYezSvD#sGrwug>}o1b@=RK^*(YlApZ*nOeYuH+gJ`px*0w^A>_y%2YC4Tnp~Zen%_bWx7yxKAtT(TnXJn_u@{nryn<YjcXc@H4B4(Yfz}ZTxsz>Kz;ng$K2aO4z%Kv@5%)t3xf4LucpH`=xrK8lM~K0XOY5suHv$9Gx(n0E$!@lo;}q|H2Z%)PSi_gI1&rMVUVqc>jeR^qnjT)Ex<s3}drX(siH3pDz%<t7H4U;fK9NHXxeyBCB-I{Ib=0}6i1m;kTND!zp~&B-E|rpSg|mc^($}&dtB9LUh;M8|!u+TGw6pM_G2{0ONhw*Oqo_YcFg?N76z^_$Wrk}n+%ErL?G-4gurt`nE;H2HNgnin#oya%p$`Zj^eNj9Pj{HW)zGUfJ3ojqltwBa`>+e-%mp&Ilix$xXrPb*SC<k8=rE50h}Do`<z`S=;k~r{9i5`*#}e`vB=1Hj{ireSAnkz9V6!#iQS<G=NPqe&!dfHg1Gk(-OwL9Vs^*}#_|(yY&7VD-PXmVtsmV*5jXtC<N3A1IygQu40?2q08>CD4@+}L~4sw3hH7#F_mJK7H7$Jzu58=K({?=r<fH^)U&y#2K^Na7k{btp%tCQORO_;^S^y6ymxzFOAJ^uo@ugMP*zc*EIr?}6{7PF)UzbkU}=GhtX$Ak-3Ylv-U*loNG(CES*kL7lvh?1?%IOmbs)tkiytOIlJuvLNA3bNk&K1nVYiNA#%e8m}A=fz$jCV@0Emt{(Se);x4zrB9--^-u;4dn#UzqEmc<Q!8y6hN!RT9m@<!vTjR{|~HiqYzl|KTt~p1QY-O00;m803iUbd78y91ONcU2><{W0001VWpi|MFLY&dbYEg`Z*+5Xa$#^TaCv=IO>g5i5WV|Xusl(zqQuGW9s>9fwB1XKA}E5r7zRO0Ba1afswCxnVgGxFq@>7>cYR1Ga%MR5-kV`<O-nlA8zr|yww<w_JmBYISi2Xu7}wp42j>@caPh)%+iNddD7MCU=dI|*0G&0pRKR5gXyIRSau7;Z!UG3=kk;rH4;G6>1vLpV=i8GPucY%*?|SJPaJ=k#UiOvXLRZ|G-j=|Ji<oaa9*W5B76gA56C$fv2je`Z!n4AHGwJ{=D>`BE3b$TvqB%xJ>(ite#eJH`U^)3Jl!Gjq?|xvuo0&h_ey|qK0YVb)_+=G*Pj=T7VQq}28?ssvdRr{7i{%$Ox7r0OZ*18NMmNQ$VtHhuK(lCO;lEy52tYcz$%6S!G<=f285k_AkPaOCCR7i9SZnNnTljsf%YTGn#+)d9FH~e_a*Zl*Y=|oBprYac)=8te)U{dP-E0CJ>^3LM>-p^|LARG3qmK=c3QC1+Rpd9L1S(W}u$=|`OQa(Y5xR&OO>kR?3S6C491D5CGnBm-TLn3Z6d&=F{E-w<?!^ueLxkQ6?P_CNtYBc^Q*rybxS{7f2cinsDd`#gm1gVfBid%*VI@I}71yfc5<3cdhhL_ymDCWGUSX}J4@ZOMD=TZy;?1M%bJ7~UhZkIz&0>T-^`dd}FW2;5s`-vqejmR-w6(Ba)<~}#O@%kDEcw6A=o}-7o6lcjbDeiTGZkX7q1N*rT~LVgpi3Biuxi9J`9!A#!r0sf=}hz%J3z4i{_F8i{`>voJqngyX`|se2J|#bEYEN%93ra^$4;0qOIHjIC|-@jWY{EE1o_>AmamxWRoCmiFJ5IwvjW+Rqv%Iseu&izk1cR%o-`^t+)HZt?0c+|ATp%a;vl37+8k6lK@ucA(jBru6&xVxkIjsZ^24m!eI+d$=H2RjDm5a^RE+udJ#k?y(V|7%HDNT?6(EcYj?F@6Wm!;I#vzbU&(U}b0lr5#CflU9qBPx0L}!9=43d+h?8ohFijXp0j|bREbLp|4cv~nzb^*AzDH?TU1FfKUh?8)aD67lElIHK{p&jrv{3|#gXBMAXjE(5fY<&)@&CEEZik7L2bdn&Sb<+uh(+x#946`>?lFCx16zW6`hxQ6fp+nhI%go~D$uCP?6b2AdzLgbDd3GV}&gJCdV*eCYbE?BbLGI&#Lmc8;!KTw#;Wxl}BvH;&)}-y7k6kl7Mnr>uGImcyiN}r@jVIXj?=l(EI_XZ~u^R&u2ycZwql~B!jVbZ$r%nY<e-Vg5jL>V^D#FJ4ZgVvDXS0bVpu)r+a8F&9a6dBu0_npLc(M{-oNefPwxwkw^bV?_mCroD|D?eeVZ|99N?6ga*1Uqo>F_W?N{`;>(@Yot15ir?1QY-O00;m803iTP!x~{=2mk=L9RL6l0001VWpi|MFLY&dbYEm)bYU)VdA(WPkJ~m9fA?P@@Zlg8igwq{CApwRk?gibfI}M;y(gbR&=O@cD~l>gW#bzD-*1N07iHPIw@vGZU5Vrjhx7598A|LbsZ8)m%W{@0xeF?0wuM*+!cBaH*IDyy_^!%D!Q1z`6vl9EW{%P@O;ueQvEvC-M&v9ty5;k(8Tdgdsp5bcxf3bHZM~P`lvTzIYq^B80DkLQ6d6rrD(Q}uA_oz4Y-b9|GRnUgu1Z!ohHj#g6{ljyHt;uV7FUW_OmW8ucX7jt1CIgeXRbuP?{;!rfJHQ$&C-Ht9ozx!Zzx3AY9zteTc&yR*7C@B9%yd<Q-zurc{7MVd`(cVQZcm;7M+fPLo;|M6i<y*`!Mp!l#~W2&_bclCn?#Ek5;p?l!h50OASPaV>g>i73iuECJsO~fhdD8Bt|h&&aZD?lQ>YU-0<)w4sLFuC=NpMnHB!jHBQc9c5pKMIaR@SOK%u|hm#Tf5s{I%AO;81=Yp$vdKD^&;u&FaP7eM0x7RoG?3jG2HslQHgUHMl$ljcp-*U0p8d&h^biwgt1a;T%j=BRCdaG8m@%npFW>(82WtFuDD3XX$8&;-VlaUUs`(Oxw(TGzhs>nHEKn7-o;pOU7Mv=QM7mJ|Z@$Ka4ai68fay4nZ&WMe@Q2r*in%8C{cq8wEl$nGaowBBMG_5QA@j~-Uq5@~WkxKAJ(ncU~N7A?lE+NO&*E#OlnSt-dk1QJM>GIO%aC_RE>GJCJT7E$fz)*~XZ{uLTdV~V7k4-zeFR@OQd_WPd&#ye5oq2}UF$IAkdXT#ch?cd&=zwR4cy#C^Jy_U+a0jp^CtL<uB}9&~wRd!~Zj6r0F@Nh7$iW9xE&j0Bb|CLSH*;{g_wXMSPj2D=OZb0>@5fn}0SssAPCk^ElR%OdoRxJ2aNkkdjAzN>;G1vmADG(c0RzQNt;)%~7|t$xutRzf^h6CESkOKhOt=(ggUjS?x)>23{a4@Grq1>w2v6k}N&srrzUVFBFJg5j45$EVhsE5VYW3ESMnls;W-nxV&ryN6k*1l@pQI=aeGq0#9cNvyA&*m5VwsT^wZ;>U?Wwq~3)6bAvz;eOPr6vT<arIz+7Hu8$<MsRe6Tn)u8&y8zkoxg!n{Cn$#HOy4(v1Qk_dii6BCLum&*=y0O2AVk+=ZzJxpR~I)TAt?&M^Z2yjOoM#oVNp*2_LM+T_u9|$-v!TndkKiuOjJlfRQJU7F4gW>ny@WfFDJN~9>9@?eW3PAbYq9dK8_~G}O6=A_kN2d4=Tv(JFSPt5L6TASGPZItt&4<!eG@!l&3Od>F0D<G~L~*x$^EBL{XAHmz=IATRu+?-IpJcTJ{8&ZH>(%*l0GlQGz2<6<`;j{ikSx|bFk7KrDUgu!inG*q6&Mz(46rbIbl6lAS6z5tk!X?&D1_AIeF@NoHDBi}WiB?gvhj|`uzWxR6ns||th#6#&wW~!u9PzSF;tw3qtjTLV=UeLnOL$;I4#N;FEj_Va2%;reTR@i!vK8(v@rY=l<qux3P2#U#p_;UXriIoeq};)9eRD#s8k$td((5XO~Rz}fm=t(gMxsk20EvZ9FG4j$yb%*^;mH-&kIp<>N+s%Kxn3{h73>jjuizRq!Np^*#5CKl>Lftgf`F(o#@n!W3C=IcH#L<)2z+Da<b{gr`W7CJO+pImuC=Kws&<Qxv&uHF9H2*z<_%3vn+W$jhEKhtXr9|)yDwUEz<bOprf|I3#15B35qhR92y5|d3r_jvhQAowL+9w;Im#8T?TlE2?r|~#vUNNtsC(j6O`q;Y}=pJG44)#iCqtTv}1{g+%xo!G0C;{y9=rbCmrev9S;ogqK!6Jrv4vFnaH!O+=tc@SdK;sKnz=JS=5GyF0b3t8P3;=Jvw3nwO9?&8%1Y9t-C$Jd;^UYz3055I{*$*LHNO_c2Nngt)Yr=E<29KsS+OVV5|~~zE~e#{rE?<CA&BB<V%EB#*2+plMx{k-2QU=OH-u2q}T25X8!G^djvEf5`WN;g}LGZ!#4N(bIS|1=2#|^J^1g$%|iUVv(5jmvt@Gg_}NPN_|aBV(ADL`iTk@j?*`GduwDdJh;7Xl1TqOEaaO!gkR;@eJW>p{84yrF+fs0Tfxy2r43{Vf(kwxJs|yj^ZUTsn>7Q<Sk;&Q&(e_WeeSgc*8JanR_QrGN;*?`eP_LgvJ!!jV)CW(nGTW`WB4cdImHdN)>v!2^SS6(Pg{E7)lca0Dg&y7^RMUxe1zPCY&K+tXqO7`G(wToDGbjq$mUp5oAcMN#G)Bp|G?-|fM--##00<`P1JV`<FzuHJeQkSM7@etwpjmFTF!0$<z82WMc7L)2n2l~Q2Pw2C9%jU7cHwdkdZeXyD~1;{^QoX3a35d$Ji}j~r8eJl9gNsd2_kKb&qQIf=uT*D+a^V!v11iJV>q1LD?CH=>5muT^3cbAv%N)t<0|fRHfWBgGUKiqjs5~LPOxbxSCU)-YHDc29uv*)-5%F`y5;FTfJbTo=4mE&ywqKzv1NM8PgD;@uYV%@nd;+*f45(zKHxU9K~u^r9K<=RkckdS--`;r6NTiXC<?4gbpT!)*hnf0aHwAAmVHw4hmidG(|ho=@J$@Nf@DO{@uEbt&)oM)cP)ULo5-_IvB%IBJ+pZD31#@X#Iw&+s{Q^R0dW$jZ=RUE9cs3XjFk#r^p?&4`N}5sS1<o|XPs2gzn-9=#${(B52*d5MXCK#Ln+=TQQCZ|={_vtC^QcL15ir?1QY-O00;m803iUJux|eL0ssIy2LJ#W0001VWpi|MFLY&dbYEm?VP|e{b98BAb1rasja5yL+At8k=T}%cIZ_mrWwlVGJyq(dhpJbEEE5k|cN06c9Tpbx-#fMwFx?L|LP<QCd2jq?#>lSM#u4ejIIUZoWindS(Dg^cEm3{8GrA-FvDUQ?PGt1yxPA66{sZ%iEV{y>2O~RB&Q^|FwqI1R=k36q)M^U;_Qa+60S8%Y^<kKGgE%rFd84#-vc6CUx)L*R^%DrjmC`Q2GpVfOss<ZDJ&+lcL);4CYv|J?eqJoUzz(g;vP{5^IIu3o3^Y<TOj@QDFsH+VE5YDUw}Sx7YNKC-4Yol&EB&Ha%eT;4TC6gHS8qVbI<P;!A*R8_U#hQO!DvhAvLwr*BwLicDBt-$$L}AxaAWw9uGh1Y;O`!Po09yqdML?5gxKH`gLbXvZhm)*gd4SVp4d)#Z*;FM${shEugE&Dxyws3J@ocB)GptgV^|UHS3xZ*uucG_K#cns4vjWP7A&&`nzmbg@C-3`-1B@wO=2HDkG=aWcJbf5Xq7h9**m>O`uY}WnIJ{IF_*h3i%q!4Cia?q*3Q=sxi(UnYFd)hY_Xb3Fk9lQ&`o|WZ$^z4X+^BXxhHFi^WUxWD=^XuIOMqF_$5t9<eP%*v<aVxR8#&N@|aeYe!(_N+J>Tv4@J7~k-Y^)Pw&#a$8s(_jgJ`(MLM(LB<gj0Y)}wfb+{#KgQ9*c6N71iHFA%$`~=MM4#J5t&8k0&Y6k*qa-FwD95yrzQn_Vu<GG%#uz~T0GLTsu_?U<9_3Pw~^A}$>QHX0==zBR#E5LjOP(RHUm2ifuIMx$qQH<euzTJ+L*Qb@%(Z=YZFJiq|Ny5^rc{%W_VA@_mE5rBDQf&m7$NYOflEh3?36b!WTlosUx4kxgK4LCti7)36<;g8H{$wBj0u-U6I%Tr^@&<n_g}(%NB|KxS67nW+$d)d#;X|b^(PN*DAeC^(9()t|Q!_p3d<WqnUqycwv}W*U;4K|v%S*Y7>@QGD0|XQR000O8001EXa8}Du^9KL`X&C?j7ytkObY*jNb1!sdb97&2c42IFVRUJ4ZZ2?ntyyhv<2Dlh?q9+3zQ{POYUkR`?g9pyrU81mYf|iO3mgnXU?kdRBa13YCF@<}zuydBB+8caf?g0f7Bw^Eygf6+P)aXvE^o+Md^WA(ny|Vga=jLXU={iG=GQCoXDig5PKl^X-f;M+4N#B`ueoAI$eI+g<Qb)OG7-B*DntIzvhE(5z2VwSCTk^kgnw?h5<8HgGsAR#f1dGsR<%x$=SU_5elE96sDJa%qL6j!R%9XV>Xl=W%B{1_M)3_ZJTF*-g7O_VN)+0^uEeI!8`<!j)%t-ezaG?duNuWmQ6RT_+rU1&%!{hkhO7KmN~4WpjeDyLskmiKC)3Gf@`KYd^L9+&eZv=YC(E|tbUG>dnp94gWctC1P#h@hfttyPD9g%&!im{6dBe;$4dT2sQES8M0z9}NU+oJ1P_bhhuIUG|{7OX$f4@6>Oo{(~K6_lHJ#7IN+(;!HeYcK3o?)I6TC*L8uN3?UUZ#)fDB-(^vXq+>WN*(RvH}ee6UJv_buEnzlF`GVmbuq2?KIiDrTOvZPt(79is5{A#PIg)_|%W~^Kaw={JiFhgKOSk*n<YVB)6j27ZoQ$gBv%(fPBNrS}Jn+^Tj_ulH1mhUw-?~CvtuB$qN$ox<Vq^Dx!JA6g~v6zv=Bm!r34#?;%4soIuLTov1N>GAqre$zO<-*ZRZmhhDxQE^sW+KxB|xu0_cS3pUB<3#=cBvtdKf45bEp5VPe(Nx94DDW`NfM7{;3L*C?&VrR4Yd4}Lcr`dxN04RokHVLj|WxH#1(veT6$FhQ1UnVM0dr1Kfjxa7m;LLl2=cQ0dr&*5&!Ag3uM5CY2Rvktx{9nt37=xICAd_3`<&g9nhz~%*tzsHsvdp!pH*Lj~_~LdA6(Skpj$ejI^6LS&Kr}Dz5~m!~ngg)=S4-O9LP=L-u^{h8DGFB0S`=`m2$mfEEM^G12IMy~e~_xw3BBBks)WQN56l!>ObZ=?PG@wwoLgzVB?F=j5ix#5AWKi#u+<PlU9RWy_77edZA}TK1<ra8?>z$KyS6f-si6FrebRxuH>s^HV9;(ovnz>Br3*ZN3W`Vwbmbp9JVE^!s+!RhP=c3>*}wr-(2}~th0}Puem6pBRLc*@{$V%q@7Ju-hXnHxITqEz`&fW|A>4~ucJ}=UZ-=!3&jR%*G16e!J_wq@raqTO$j)Ms)LWGztOWVi*Bdk79A|<(BE|q5N4!xW?L?uZ4=0feor8#+Ii+)m&aS;d>^};dO4gg_q9z942yut8qPN(D9ZRy<dodbap#cM~lg(JN#1p1Y{Av)E9naGNaY&{|BVkO~H<0MKd$ja-caVN%3J`u4yMyl<v+sgCXMkzw5HR5yv!_>d%c`m^u)w0a{2uBVc=~mD{b>um0Q`biIBShqw-qozj}1`u8XH@YtLdp643>S)U~rOa=u}RSec=$N9z^1p907oc>r^6J@&^Gm)OWs$2>{dNcQ6ybQVQC9vO^4CQ0o%v?fGQupDb4ckZ$X{T0YddtCEPyJE5`0MVKAc*fkSX@4&PxvwrK5)BcO^?OCJ49Xk5``}ZLigI8as_AP*Y*Xxms>z`s19Cr-KGVV&+8cI{?dJ$*|oL$0@v195k!)DT&?F)D1kh(c3#c8boob*+%Z}Sp*EmLf@6n8;uRl;Uo(rFrzeF`LqhNsE?%XJ0;XB$G+G@L1I>GlAT1H$W@(dD)V9Aj$q-RNR1zd2Z6;UL$A%fW02=K=V=<Gj&*vs8puse$c{8)0)dEju%ME0SQwiT9dMjZ^by;`pYR935*E%9dQz;B8St^y&uC33`iCT6ijjb{5QS-`H(pLs0j_XMtT3U`g-Pue4&fyu!8`?UIHtzy)wn>imb!ycb&B9<=HaH+XlzjImNn=lqv({O0_5iu~w!7+wBQfn_zo+<sn9yy_{rhvt^mW<gmAc%<>Eq66eiLL1az=7fhjdRWJA)-_fUY${<;_P;}Uoid>lDT-FHV(*a9FG3^~pbMsJv<iz*W(G^^`LF<vq5dAb(hi!tX2b_i_acaR86H)<-k9wGhmY=90Dx1RBnM7=T)hVN0HN)-XEqv4i?HLLEr`Wj?Cvc3KyimikIAqG9~`xlJ18XnUZ}edf($eausO$uC|no0zTaMt`GR!R4cr2HRHEi7G*opdN>|yf(cI2>8-OygiCoTXG@Y$_Lj?W|#!6n72UZw7jnyLG7RJpSeoWJ6Iei=Gq5KqQY`yTUQ}nN$`I()cIJ0f7Uvp|eOqkTbFQ;w;lMy$jzy<(p^kV+~%<nZnA0(%fyQC$r49h!C=t-bh=Ru}R%{TKg&O+psd~g)e+E#3~PqBptD8#8Bd+b2a(|Bz?QaYNq$Kr7L|7Dn$AwntEwplYVB)sMb%MWkntHC*5v;9vk&g=-9uh||(jb-@0oUK-;EHZ!@ScmB(?=Q!rTLhdhjDXq1f{b%g+`MYpssc1Pgq$v#`}FYNc%O>PXRhx;3kR^al}JLbX84TWw3n@`<60CVs|zSF!!k&)(S<^u=dgPm&lrIi1TwHH18N%g@}j~?yPv^*L@%SO&+%KpsF4WcJ$4!wzx;UhkzC&Ve)ZAb4TSyeSc5g%-$S4k?LPuw%>N%yO9KQH000080000X05|U(hOY(y074T002TlM0CZ(@baO9sWpi|2Zf|sDVsCG2E^v9pSZ#0HHW2>qUqNsmDi5;O6zebq5HL4!m!OFg*iMJRGYDFuZ8Z`pkd%`w$baAQMY5bkY1)2RKDZL^g~z++9-f*NrH~p?rF)rs<ja+i1#RBTJIz44lA=Ibh+IX>D$7%nh@3777n&`FxJF*v%G%1c0!dk(ZJE5QH9rieBXV?cettUZ^?E5=A+ry$4wxj8DUolPBI``U%Zg}0R@p5x8y7{U16`D)q<RygLhx&L2hYi-f7nCt=ePKx1h?-t9;2LR%UXB|uWe;jElVj9rqs^l>*p<K#(UO43ZYSK-3A0g%^4y6$R8u<M<tbvYc)SuSoNRKP$nD52-1>rWu0Zp=76P0F;J(P<#!SK#8`>+2879mCD&*rl1hP&;7Br>dvAYcdH`uccEd_%f`=&MNnWKHU&9l4U`D8g1luK5P!7(VOHt{_x@l!HQcP2=Wncy74|it+4wqchH6tc#zCbu^EOS~cQ))95A}2jWP)uD(M!ad@*89$?Z+}w=H{;YC1~OY09Q}fDvvF45@g;sm@aj%QRIax;M04yx2G4jfsKZ70WHdZ>Ymkw$GHVK)w8H2wTH%@62kvo8^NK-?260g?XsX&)fC8El&2Dwjclr<?2-gZsxLI!EOjw@Ju@aR`7+yfFrCl-ww0q34c~v0FR$kfs5`9-1{WwNA5h=s1FgY!lf-IZ(?jP(DyJbmG#}Mu``d(VH#Go*4HAQ5RLR?~Y@y_wu<xaBg05GdqlgI_s`X-HC)y?7KnXH%#EP}jdrB5DJiW%amqnP8#XgV5Bj!ry3xP#SUdA8`lk0SudK*rI1tNsvwLGdBLn5sc3vn{mOz$JRHJ!qj7o93uNeAwoxmX0&rX}O1%f*}9}g8PTI)rP)%|C<S-y&rW%K$gG1Z;2k-tk|%qI|iXpatj0#L20;#@)Wd40mTOq{F1aP^DMCd418K@m|FY?m}2xZ`t>cK+?~d!GJc%eTG4~I;lQ}LEF#<ZX_n}~?%@?{DXe8F<U6QMeHYkmO%u#rUm3V(JY~0GgL=lRg2}dRWDo97WC`4LLew<Et_^ld4rn^E7!9nx+df35R0e$U3R_wm80p~N2J(RWdoFy)5iLu`)3zU{_eB7!n$E%ha2%Ppz?<x09Va<Fp|hNBA-8Zcb5v)$r)Gf;Y)^6leWqSeedGJECUnPRDez#j!EZWjF8`jLT#QeK)063FbR6TiU7DD2We~ymHj{!EaFgK<O@GIbeM|+@i>t}eh+GbjJ`MjIbs1kyPCpN4Bl6*De0*lKXD2V6Syr5s8)(<Sog4Sxd|eYB_w`zRwr1RdEhRNO5^5SD2dXD932g)?HyQj>X@3Q}IiRD#malEF<z9oCyT4$X$^Z3!@XQkIq5(+lUJSsD9_~N6Kwe4c^{G2-4Dp_wjz@bo?Wys3vrB;U$ZtR^EpUh4bY^hq^j5jJ>C}VHm^!^VRPrE1?-pIByc&~_r{m!nnT*bdr{m+%#1ulIk2F`T!)SOlyCBn(tJ(3zUt_1<+>=kWk1jrsCJ;E8U7W$g*tFE8i|$?Y7NGoxo7W=`F$-<AsdOrCcyo;eog*`hNv-i0K%;8Oxk#&=&0r}+Q&`K{xDKoL_*(GrPhJcEWDouc+-|K)!S)%v?g9w#T8MSdA}jI?ZJ(o5nvptw9AU7^E48tgWW@_naUHv%^w8~A^EtM9xZidOe4(@AIj0qe0%dF7;Y+fd0prx(9?hqjW<_A8(!!q0G}qp#LbrWqkA5FX)J6IZp7=ph@FQ6rKuaFFytR3_L9DF@d<ELNk|kT2tHmW2*qQj+1Wyp|QQf_p8EzjN_3GrAj6P1(d~mSXCG}}Wj9O?$Jm=I75nty>9}9rpL(h)V4mXQkH$(|6$>0$fo{~r0qM8?@OEX26IZMAK!bOW-3M$Q4pRK#h{)Lu=5z0H98~zGiB#Zv8s4EJQq@;w;)I3{nw67yn*O$@uCx~HR^W|^!fhHWTpA;)Ze#hQEgBeG?A1(e1?bIy4*jDY~6x+drC{3mPs<Ok+Gfw+G|J4gK9~?cfX017Qk6oOwkx7~<h9)kLBPoRpaFgizTM4-eV82n5m8dH1bl!N|gH`PF3<+PY;0w$&qTM~Vqg05R*(x^vLtBB^mSIO8W@z&Nl@!@nRnQJLgpX<9#rTDocX@aY*iLKEn15^g1#>?^8edNu(NxiA0Wm-|s<i>${{c`-0|XQR000O8001EXY(?Z&@&o_?=MDe>9{>OVbY*jNb1!sdb97&BZ**m1Z*OZ~WMyn=c4cyTE^v9JSKn{jHV}T-U%~N7$`pAi+HMG7VC|M+eF!>i-5zEb1TE1v8%Y#M%1IjJf8U)*%A(^W&FUAM<lT2a_}v{Xn@(v<R@{iUuM_E3TW+?sT!p1n;n!!Q+GMR&Lpp9zg!qF$U>Sr=zv{Fqg)w30i|p327D-rlw^kUNI3%l7ZHiV|u~O=aRic(Vp>G3nnh^NBfA`k~`{m>1<%duCu)#WBUh$1!2yEQmxCOARR81qTJJ4WFX~B%@bt!^so@7aq{Op`5*7C;oS|pWNlUOdz&K-(UmduAxQ#ROTEGqz8)jJR>I<7@)&Ejn7jg~@os4*zc2rVo(rw+ToOwvJ4u7hYHZltj$*u#&L1`L!hYQ-y)25BvL#jLooX$FA=4fMa^S7M5SpHj6J-2<$<mN(*@q~0%FWe#nHxkxjzR(jYbvh^-3=#Xf-ypK?|fTXntw~+>0`KatO>Mn&YC;oZ_=W;&eDj!yf^BNVv=RK*4ax2O!rh41;HXU&mbOXpap<3)t;VO*4#qSsIex=Ktl-JeVL1V(^pLFl=v%**{yVN^$EP39+BjidQqFZ=`$HPemY9ZIWx)s#{BbhY7e=Bq&TVR4L8E-4**=yLfv|9+9^_?R^vS-N1X=0m>VR0rtgz;eS3Z^sC8U(p90&CN4k@?&Q;qC!*4z_dB32nePxL$w>#*p1Xgyzw9rJ<bZVMQD*$aM7em^mPm)kDg&Z0fK_eZVn{zym%wo9d^*7<wyiz@Ti9yfdg)w)=5rY+v0L-7U=uxE>^U8rg@=6=R^DtOPHb3UoqEM<)06{T!{oj@tWt#q^FT9(EuxlYZ*z;AQTO9#wL^#_!WpY!|1ht#6<3DRP0*hn8#6rZwa%O(!%%ZmCK@E;qGWrSznLc|x-+LzJ^Y-EzGVNUXzq(0yy+I%m+4)^dZlIFT=|wY0*uP%5EiY29~Fl?1w9x#jH!e%7kwwJ#Uyd{s5pWT5h-F-I(T(Co9gVILrk{dEy>P_1CPKu18&fzgC^T7q{-FBi$9?44?6QKaF70M?GfcNE=oK}?W_(5t@bOgf3=usXl3((#Tx@RLRNc<4_TpnvKVX3Jl_d7FA-Oow4sY{g9_HxNi%Lt}FpZ|*qbmY(O)02rut*#6gC)`qRb8Z%bbsufIW)&TiyW?-;%|EuM*NBX@kx6qo&BoM#FRzJP*UWYQ6la!uOe5-hw#ScsJ9q<u0MQsG3uV_ZTA5YKDoj#5%|J*Aymj)9053hT1p|#RDjcmD{yU}6#hDKxeC~>03cz1F##u_<)9vr{VBZcGh>QNClyW;ban)~gC<KU>B2^IIx&W7tNd^tk*gf9KX2RKj)JE}LnphYngrsUl~YKV3Qhz8n#U;YWh;PL_|P98Tt(2G=;`05>bc)&j`K8w;CW;KT=xZ(Bi{=okRwQR)@(KD}(oCMEbMn*SR;7qFGJ!a;^M_+6ew;c9YeuNYMPN#c%`F|J1zq0!cr?bO?JX`}Mxe1)yQ_F9-R)+T(e9i}J&aP;2=pgeQUIBe!htwTkM4>Y7M&Lai$1UF+&dFUca(>uz2Ak6+Ga)T?GW3CxcQ4YLsKv$L{@oR@aB6$)N(^hSdjEo&{kt`MEnZr&K5O~czZ@sS;2R3=KTt~p1QY-O00;m803iU`Pxt{_1^@tM761Sl0001VWpi|MFLY&dbYE~~a&u{ObY*U1WiD`etyoKM<2De!`&SU`VdaG*cfH#l0{9U00TeA3IpowJAZUrU8B3%}Qck>q|9gioS(0VPZZ{W4BIm(x=9?L6sbq;NYSsl`AYN8d8FUYiqws&VGfbQKB<ix-A*zw6MoTMcs3xiwhL<d*%J3!4jE;Kl!i(ROlInPW#gyjSFp;q+k&F=h-Nyy<wfgo8&DLz<-Yf7#Dn<&HubA@hs(Mjyz3#kIjOK)kOtBJl(1K_|E4`LxG8&IYqns^~JrvPvs&bZ-%$gv&R$IPhnk>Q2nrur#MNSlZW^g2uNu}z_a48Z~R>UGUzV#k7-l92*YiNpau0S1jdzM^jyL^1Iq%E;6KjBGB;RG#<8hl|}rgrn$_@FDTYLmQSUeXoAx9HP;g5GdR^#&e9m9ia6zD!VVb`_g1q%6j6!;%)of@XHp@3hcte_G?Lr%?)~6RY#fc!In}FHE&&5V?Yh1Oj1_gZJsGkc$L=Obu0dJRYO@9DQ;!nq^Ytm*6q$;1Mh;MR$6FN@i$I4Fx@3VY*^nP&yvQ?T-f-o)Fg_3AYiBY4d`MRqADkGey`xH7ycv_R(YVG#Q53TXcy9ZVC2!29U$n^hI&Q$ii|hS>T0|8z!*dVAfn+1l8vOs9rd%dJ3nCat-P`qzozC*y$lS+jsa9`4>4sNzRu`rofyfHIXXPgk<YlZ2C;$Iqc+S2d8<%I^&l4E>3RMGZ@yoG#?<yG07&4?r}?3svs>?e#DcEIwHd}zC#%`Dc+wK^JHKaF)QTPS&regHq_L59u~&~H7N<DX~6AqyR7B!9esAu(L=UJrD-P1s$hoYr<u|0^#{|5qiP2Xc-NK?@*NAa{nZJ0=Ik#@iq(nJw=*ePb=epBmki(@m>AA>^D%wakPqXQbdQoDukrUhRn+-jR9g%@@Xfgge4_$lK_oN??{LUH=xS?jE$^-)VQf@q+;&?oO2wwkpV8*`T%^vosX$H)4GEtmbB~2i`ZNhc2RW-fm#aqm6)Dhj#9JEnq!sG+{Ngmg*}xN$OBhjX%oXzymxY~DbX^$TkM;j6sleudf2(Bt$I2S>>dMmSsHj2CjTTokxiy=3O|2P3D!$?Zz;zU#9G*)%oPx=u380sK{P8Julclc-<H^9f?FPvhe!LbUwBL1Ad*K3@Z7aj+4s%}9$)vp!{Q%#4Kv+z>Xf^^cL7zpZBkEqZ;EGeb%ij|dy{Gk2wBzz)Obq<w-2D7q8tx!5F6_Gwl8ahr>o~^^x^<dfP~+Zr!`e*j*U-w@opo9gjMSRhQ8`<KXdDtXu@zF*hQLfpmb|blr8M@3=0}z4lFQ+u7CER11}9DIq6dC`<a0;XDr3D1nSY5}8`5%<a}{QrBnVj)yvm!B3cj=@-XA{<1;&mQAKv0Fd<#AC)zpgN{f8q}F0A?n8Z=*q8a}MeVx;ly+y0~y5INzmU!T(Sxmq=f<Lu%VZ=#E5WmE4aWZIyMd?jpZ;aj6>7Tt^(0odIL1sfL)zFp!?tp?n=^LK-s4AHWy12+i*RER!y>sn*~7{Pvh>!a8ohEM7`LX^EiTKC53%W<>5#KW{%D2l4jK(b-1(w1%oVPfTKgs~S({*4ZO#|SKwy8R>yv$>d{(J(kOEOYdQh#Fb)j96$U_9{I4>1S6fdL*_Vq!Ou9Cbk@|0`8WIzy9^`C;9d6;SM5=rvfInSAX@5-JdUgXb!0!jt`zsD`-lF7`jNm@v0S;*4XBUJH6pmg6|cpsB*a7$QJ=|w&u~_f83*x682L<E$Ei=f-Z`f;y<7VOAWfBT)?kcGX&?aX@T5FryMmdA<({tAG4+cQKWYQ9u3b>A?2p7Qa=yLiXZXiaOubG(CKw;c;S&uf?x-OrkudZKv>TofYo78u&U>1NUDci_=sUmK{u;*HhPyJK5;ez3l3S(nl&2RkXBBg_7pnIU2uPKDFaRO?X;!ac48&Y>VDrI%UWq0aO&5RDJ=D7qT>zIArGd3dDBc5h4kfLr?Akk1eIc2E^9mMTF#}m7Npj|<L#6G4y1O6+&tO>VvpLWWXSKQ!#EZSmX&^i#vbbdOZ>QwG&ef+Rpt>pV!>PcV;Agiil4$f4-;GQjz{rKqkjQVO9KQH000080000X0NAes4V(r50J;?b02TlM0CZ(@baO9sWpi|2aB^jEVRB_;E^v9RSnF@wHW2^rzk<*(K<a2M&vi{<VC|mSvNp&bUyOjDCCcWa*F@T}Tl9b5@g-5V<+y8u#0kl}<K6F`Ob`TrbM}Y?%UL3MRie6Kf;B59kX$e{qj17YhUTm)m~1wPCQT&-g2kdrQBsw%sd5nqK`<I+O;w<jN}A+U2oS-Gx@sh9Se?^^jqGwk#Ukf3LoPRUNB9T)&e<DUFi|T8$5Aee8m<<otPR)g^^N1XF6g`!U2h~4(ye6GBnx{@LralWO~IP>w(oDQZ#DOAQzcA@s<9@pb<G-Huu_UxQbAVFV`Fd28jcWr&6Ye*iNgTvy#;NGs^srjN*s9h#E2lX4dE#sMT1v?2Wr}|w11c7&KQ-;#)-6~ZWnHp$)bP6m4Sz@TY>*uyMTj|LP-Jr0kX6zU@0Cj%GsPI8&b{wfj}XO$pQ;fnq?_LXq>X7N_jc&{GL=zN(5u+E-@&Quv*8QCBJQV?wDBSa+k;cb__Q;UC}$UHyVv})S+9xX1U$G(ojHb{w0`>XPU?*&9&z>LdNl+@=H)wQV#L2*oBHU6-mh9Z30{aYX~O2+hen9C{mUoLU_p~At8i778yO%ms-Z!C!}^!2eNiUF>F~h8wAwkeIOQKHmF5dI%mNZIzRh>5hP?yEWv{D`9kUwjDkm=%7r0orA3{N&Ni(hMp`T3=?aJ}o6_#bSx%*7Wq20FI+^N9u<#V4Gmtp?&Lm!(@wE~;QsRx32_!)%i<;C_F0AkkMr$y-tYcoL9PAClix(I@KaC=Nt~r4~BND49piM(J9VEahpvpXi;Tk<d=jcmxcKHoP8(3Vr1sr{aHj!Gw_SxlSgwgmkim6Z~Cj=2oUdnIHjpkFS2-iAo(bp)@s>F3U4?4|aL02r)_L$U!1Tp5kV^ADhV=$UkRo>A|UtFDE0$`U{7ccIC;m>BtMjp^gqICn-3;?VG(xtM7vO^W4XBe@fmYc~>G*|14=J^a@ach-$lM@U2j&#tV6K1T-jJz%cwoD^Ae_)~7w19OvJlRR#uqOrgR?|0C$?mc5P<?Y4o?vtV$iYUIS5%&#MSG}D1_<5<woExBAo$&%AF$_XS4bwkauG&;=}h6YS4BI6EPHpc2De+aNOXH!tt}LMUQj5pKs7muPcA$wY~9t~Qy)FWO{h2pT3xr}#w8?V&O;0oRCRBt5n<3mgVEH+i3-H~NXMG4E)6SzJ|&<B$#^bVL#iz6`WbOU0&!zOFD%^+Ht_c7O5oc=R|j@$zAwszw-e|^xckF{#<(8D`i(XWemo5vblNH)u=7Z0H~_kI64=wRGtx4Zl>q>_`rY0#N7Ix0t-cSW`-Xrpnx1Zxl40>3QG11}*B)}c#(y9_rw)5B=cdklYdUU?#h!|IpSt&cNR|-85DI28=-_X=OFOb42U+m;^{?Y!0%Mh$6y2;AvwxvIhdqh-fIzExCo%eDk<P?znF%ddFo_&trSG|?ENz|96OWIQPIguGyz&LQqmQ~8sDW(8VD^P^o+nHA0o>HQfKr}iY97{e7FD8CQ}Lmmt7*w7_0r0tF2i(Ej(NF)F|vXufy+e%)uaO11<X;x3I`G+UJIW=+C9WR7-e<PPd!*7+YikIw61fWz=)+~>QZIvrc0r&N`WoB*;lJ<g;GTpTzdL^s9yEE=3qZ=EB0{(Nozh(j`Z2s*bS9h*~HdzVjYd$!^;v9dmi`~TRev9ojx3!EnlETv0-!QUsj+ObUQ-Z5FRXgy$g*;hcEO(_~YxHU+22|rXj+8RPe4%e)jB1)!~%X^1CF&t>V+*_06mAe{&_ogi6Zi>R~Z)&)`@rC=9G>Af>|0xtb8h>}sZ(co;?Tf~^go9Z?$k=s(Xu+R)2wXbOYt3>CcEG794wY07#3(cwMcLOqA{EP(fgL3C_AYaBRUx$xF#;$2X;bdSZd2qQF^pl+qf)mt@@reWJxRtx^N=k5C|+<m2nHg~sfjUqFG{$HQ>g30&2(>i*x<R4YzcP?nzyCA!J_2!4SzmpzCbg0N4p~8@TpF?|_-SF_W|3!jlcX(uyde75C*dEr<oG7*Ku7CE8ItG(HY`w;0gUlMDd!>#z!iDWsj~s4TN@9DRYoEJ)>Bz}>cVJcny*DhY<&iV;5iLVs@l?lnvUVRX*|s8oN)io+?oKsv8cYLIMuPjca+@j<ea@u%xNI6qq5l9-O9KQH000080000X01)`JiSY#h0E`m=02BZK0CZ(@baO9sWpi|2a%FIDa&#_md8JrwZ`(Ey{_bBv@ROaXp6t$TQozHCE?Wz19nb;;Hp38TiMF{)q()M4vm*a}$2W<xWw&YbA(6y8-tnI2j(5Qrzt0)vnr&H*6fJ44nc#@<0%=w;zC)7kBvlH&ssu4!(3-+8u94WHN))t2qS1Au)f8hq8nLPtQlt7_Q>90vtrQiaA8RUE1xjkFiOTMtPbIZtUfLPXwv3aKDGi2HQ&mK=|7d~^l4+Vb1-GBsLpI=aMW?yoTeb@uz1K}~|H^(EjYb9CqMX#`MB|F;y(kiNm7rYQQAu`mnNDvKv?ZmY%UPNr<DfYvyucf)KqO<ua+aWy@}1tB?~@lJ1V57MMsoDYzL*~z#PN$(GwD|UZK%9IyQ(NPD{1t}xeFh?-`z?rg%-IeVGX~2`R?~W-oE+!_GcVP>?nuxnWJO|AS0+qRhKjt5}SWa$itbfCHEK!+1SJ~_Cq9~WkZMdKu=CYU%h_$=ADsYT#xM{`Vpm*V+j_lpznt~%88~s0kI4qf_Qk0#mpyKWL}(6V_1rY7vms}uF#@qo^UQS$RBAfVptpzvV)L?S#z&${lOE?@IF&}V56Zj@Ycjxwf@3GS&|yU3OSY)0&>jT<OM6>aokcYH&)ORty=qm%=mWp<TZ|*?-u5`NAPqW7@iJ_KRqWt8xsHJ2OQStetqqoRZXeE*W*kc`(t!gGA&AY;JLLovFb!A3fb#+Sz)&c<ir{+;rPbvdPwecOQ!+*TJg<&=F$fo`y|C-71C;L$Z$YgA?;tryDb9@3a=kxo4}23zm<veI&z=}slJb~>{BCK6DS&@Lmb2h3DYR5GO|&kY@qBx;mjcmtPKH~qbzMIqVqkFCe(rf&Ky(a2*e2U>zfYK-Vq*+<q2}%wB2lSfe9tuk^J7^dUX{Y)f}`}A%DC5_2sL-u!Hiz^xof0Q%M*Edi*CT8+t3HkWl|kRF^(rFIx7W_`Pz*lMwFJY`s2zh!|y3gKu{LoIVJLC){Ht+y*=^n*wm(m=Z;xycv|~NB^}Xaz|xW3Z(+l8v<@IP9Xw)q<hdl>!nw8IgW9|foKJuOi%sxDKx(b-YR%-fmw^|h*^_WeBpUmT?rY^^6>=(zQp+Ro1XQEOYO!TM|&`FkwHIUkTN$=CS-P;yZ~Ow@_YCI@IYzEpp@iL(pw(`PMXo;v+4Mmv@?60%dOU<N8S^^r1<M9{W&LFM}|BdjehcJ2Tk3FDMXV%aWw7S7~h!;1Y@fwCS4LVv%j~cAbN~B;U-xAjqxqSYSID4IB2t?yIgHHxhoey+E9RnLLSk)vB$EuJThNa4B9p$DhNY)G?@R++&5Y~CI9-8v53YSe@m1A3kR|?l__ds5BG}cqcc*$r2`;9J=#NawA(`)f2sZG+FhH)cK>#F1q$4DMb`ji(iiP4Ma^laF+0`pa>h6DfpRAm2|ct=<`&QF9ZvX;j;|*XTiqC8?umAw2-i-pG<rF>r59;7d@jr7>(E-9=-n*7P7myL!SOog*3aym;IwDww0*idg}rEGb|A8HN>87i(;9%Fb28$OD2vE9*C8n=d9L7*jbx@#h6R{zK-?cZ5NIg$t55hm{^u}%SU(;rrqEl#>7${#sQ1NCMKN?^YX%`AW#*@cTvU*SOhE{0!lZKguCp_vAXNOSh9$i}z3f-Vw|zUNvUW_JrxTkL%p}DaifQQU+U9onvamIOVC<cz7iWmR+Pqa;9BYM9!A|PS-dx|FIOz`iUFR=+6+$a5No^D;(Q3KyvqL+2xSdw`OsoOkRXR9E7!WMMZC!9U&h5WIf<)-Idm^);y@L0q<M;EdA4vwq+Y_G3r$h3=kq~Yb8sk~Ncf!y%=8^$IcNE9-4YZt8*3it@6ztC+y})_m?!m#*_Bd|nV0@Q_s&h=!VPspvv=FynPRQI5__(nPoSLjV^&w6qzMHcQZL$9vc64iJ@K4W4cN_@y#F8l%iEwXg&i_zL0|XQR000O8001EX-*hh!kp}<(Ar}Au6#xJLbY*jNb1!sdb97&Fb#8QNZDlTSd9_$=kK?ux{_bBvC|{(jR<(V*cR3dVnzXmq7KdEXCIt#ZAZUrU3Cof~Qu`L%fA0)QS)}dtZi3WU*ejCra9$q{6-ChxT;I2HceR$B6VYhCkxYxWA@@RWiQaM|Ijc!0nub^8^Jn&-@Ba9?W_w1&zN`73H`=Vvq9~e7*0SA`j_GYJR=A**n*0+UClmM3J!-D>@T6N=Zk-f==(rS!r!vhHy?dYSx!$&w8i3Zl;<RozLaUV6Il6kJqDo0uvtEgnsD*x{d!}|46LDnUXn@U_)_{$cIXJ~NJq*$nD|fu9PUR|Iwv}i$gIGKv@cAo_@-H3As?-dbg`BP=YsxKmgKMymSG1FSEglBt&5VTQvZ9LfDxD;!`?}v@Tk!cE``#*<lA3Q=`A8MR7|6+FQt>rJuj{pnS4@}Ne3mTCHpa!-o;5wI&1UhpiQ8ZIvT^g^1=Y%Uxtyis+mu|*r_+>Nr{rBxB$?zLXWHVXWR|2(2n?;PH}K=8-x|4VGn_cZFyz(+bxqfR)l04@NYk3(Osj;6!5$%KE5$}Mpe^|Bp2;fSgIjB7j;J8^op|82BKctP+*&jU!h{hSbu&_8&o*4;RyZYN=rLb%Yf{^QVzt=u7(~EO%(#+4IP)K6&#h6JjpA|vewq~%YrBaHZ0v{0DQ;HSU8#a%;jqjJWX>tEXO>-m@X|#Ivq8oilX%v?*r6m|^wIy2!5LGEOHIqVRXp}J%7&^MT==4NsNXa4R)T~(@n!?WlIIwdOl?`mm$QQWfy@Z`hJgQJS1=UP6KWbthpZk6FcgvsR+YuA?x4na<+;TT#YgFUQ>0}19+Ef>v<TE2&^*=7X$%@ZYg7kmyqp&B=PbIK7ti28N&MN8AQYwCfv9A_Wl}uxI1&}6<XUhUf!@)_Df~nqXHk)ib+q9Pw-immN3yN@5{s$SSl}Tq*aU>kKfqq9BI^{(t5`xF!NwkfS#>A)J&OJkefihlKSc`%_SH=8plUZ});^21rvbVJO<Ej^(_k&5FW>*;>TeOaZBXnDnP%6ARTebsy9#_6+gzt*m50(PDJ#nAZp-p{HZ@Nxu30|KZmi<SfUt!}zA-GD&EE!IGE2<?IC99ydF?&}YT8lRY~mz6EI9H#9(ysRP^_F%BvIAYcRWro;&{kG)~AFwY*q6rKlP6TP8LTB_)Lyp$zdLdfdg{M2mZ@ks_@Z>A^71yK#aju)jE3&|Alk9?dUF_o(^{TEIS_N`wmqB$~c|evj^Jnduj?_<+FJ>9dd9u_vVewi*YnrPScQD7DcApS|}Y42cPO|4Nf{butzVQk~o~W!AWKLF27~9;$8{_t~HtIr3c%uDn_0bejgJ*4V{dK5|2wy`Wh6#WKOWEtYoJ8&a?;+Jx+cA3LT+>HYGlFGyww9ZQ0g&^!ekb@BZ@N$kStq`BjY1{DGG}BIOZKpkQ_6a@6NV>)p(30nDK=pO=T!Ec}kZg3t~fT~X0g&}TGQ2y_e4f}_MW2@0Z9^-xAwaIb9!o?$59Q%e;KpJr}g`1aG)PvPi?uNN3Gy1kB)S04Th=FT;O@oMPjj3qW<;Jh+D*axJC6OX$se+NM8b_WPxWG$r9DKKQ&f;FNcPoj%iN>&~R;kt%lSK3(g#ZmP?-!LHh5@m~FkKu_ydFFF|1!_w7ugBIH#HG*IA6zSkF@qQwoy-6#6&otQzIm6}&aAbm4;YtAVe0$<)dn|&{KaaW99rHpL~=F?7CN;BI_Q;J6P>l-v>MraSsa3bRq<dAgTnVBA=~>)6y_yndd&Vy3%YRN(AX5=nqPJca(8Bx(P#%29m1apV0LI7zJ>tK)?h3oCNDPOpdlTnVsWI`a|AS!(CEfYG?;yOx`V4mK7t-C;H$yuxsB$yFkq^xV^saz!^v%rqRJ3c11G^Sy6u~tyTEYBMfqx?dYL{9-fJ!mW+8N&ynDGbev6&)wdstFn2%tLc?CpD-WwF;mLs6Em+RpyjnJRa+V)MystVd4JlLHa%)G|bUu`4;ThZTY-H33OE#ugj@7ARx9;1;Ob;mKIL24CAPCC5j>GN>Svb!E0FA?aaFc@v37w>wPf^pEnV7aI8-VX*FdZxU!=eaGhm26p6GHlDN2TEwM6mj`KIt76!bO%CH3LS5~PQo(>hlUs`OP+!3pf#>b{YpH-dGW>1t$iKwZ@kP+0?}{CEr!RPLZCgqvfA@Xz{>jZPeYQJ3AWZy%y`d*G4OS7-<}0{?$wOHVaV)$z+Adc@w#Ka=EJ|3z2op<&U>YE0*W8#3TG!1NqRDLO6d5x35Sv}7(=buVA#VKs{`-kzgswhjL2ArU1e<d${?c~xP&GD7k`hAy|(Yk>{<WroJU659~l3&x091&eSCEl&&><CN<Vhoz0m@DH(u}#?XaDF;<6p2@U>my^~cYw_<HV2^R%?M-hqATA~Iep%?&C&z5?F(w-9K2=EZAo8p}Cm{aX|%{KZFPugt5OpL@=q@Jc@14-JI16~Bwg{{T=+0|XQR000O8001EXXA}K~{to~ES~vgz6#xJLbY*jNb1!sdb97&GV`*h>V`VOId9@q+Z`;QCcmEXxfdOPnb9z{I>`lX(EJ-tTZP7h|O%RAPd6I6SNQR^wyIK0T?|a;lM~afAb~9jGJl=ixy<c~7UH7lCl3A@*Dt1)%AW9{&+F7Z`dRNNocTSc&+dPq}6K`bvp~w}y%}eLq4}W)5<&^T{PL(nVUDv(1Se5zKNklE;R8*C$9JMX-vW8hjDq?w|AGf03T$qP!w=JF=Q8`(`>wkFvty%YeE7r}b;;EKZZJz49j5m5m`B=zOZGnm^tVI<)+=Ph|>nyKo6*owD7Y_Wq&#GEvv3&EO5()nU8Wl+rJphcAl1a1@X(gjlu2r7#Yv2Zq7o~&`k;+!-u3h@?DncZpG+(Q_;;~Y0^9R{NRC-hMmbejBVk?_hk?bH;eG)~KihG$>QJ$qwJX%WuQYfPp5RV9LN&%R{&VXBfFYZ;UYCUzA0rm}az{@|#tF5dzdD7$4{Z6Gx6z5r8=BXxYwd36yIya)+=9&5laJdIeQk6;lMrEp2B8@a`9^{AU{wY!kj{qZfm14Z3t*Aa6!}CGOPdzL}DFv*nfh$Q=iYx&^Fc^Fr!`m9ffa`c<4ck?q;L&QArjbJR;uoo0i})$Z@Bc01npS88%N)2@<z=*%IpAMDMX>-YA%<RDBy!~>s;WzMzpIhq%pK3a(~<@w%AC*ifIsf{KfZtG-m&(Ewafsrw^)qAo4^@`6Z|t?F8vU|cz$3ld-dnbZ-&;5amf__To0{#`|k4ZL(6EVEBe!0_C-^Lwib1z7}erFl@V|m)veCs99Wp3@2Hh*Xg1n3@9Ti)v&g_PfczFb8mhzrZPu$`iCMpN=5uG!Qu1cxT6DecK8-t16Z`2+=iQs`yJ_d?s`E6npWb%%d)tBewgdBR2j*J?b7|9V+kuCa`C+vyNUg!8zD4(QWcnDU8~!7c+;88!`}*}i-NQ=&3!klLaXBW&vSOShk0BfPd=?Mj4r&=7m*IbhMv<nT2H8ROo3DTT%lF^C|JR#8xhHMp+xlpExoE2+RJ0M`YSCT9H7xxk)p}D`QI^+ij1jyp!Oo3-vW0?r4%)3v0$U9b5~q13y+*2*8}zg{>@$4g`hEyF*~YkMgja0zGy3}y?693H>O7r~<>fUG{jbm(orKmkAw=F`Iy#P)B^!_!NI-!s3Ym$N{5ksN3cN@nVGKP*MA05xXd3TQft=qH%L&f%CHYC>+h}%{yR<kyyz-F49Fe0P5mRnwdp?sw#VBx!9DZY<q7{oMOYsyE-JZV~En9hk_R~vL{sq&&V49a&NKP|K1nmtIwt94FX^NHr;E@&K7X0vbw{0=PG4V)a@Db@KV&AzO8Xv)C=5y}jB#s?0JPv2pjvtX`;P1?N0ZeedA~xe^fcyeMAdZ(BFJ59y7FI&^!nq?_cLg6mI=^*3f>YcYi;U}?00(l4l3z7S-sd`?fvT#qb;!09HCDtM1PxL_tLq)(<9Un8()s;)eqaIGG3j_ny{GdBi~dUocD(V?x;?b6KXTFwaw$KAg(wB4AmAcBS8t>eaK8(RPaKIT<}CfzE@dsEpJbUw3Np!3trduXVmOCxFhx#~q9iA$8E-_kmXYlpy=Jr1eJE~uKwdik^*X$2w*Vp3Vm<=KRr#t0jr1<Z#~dK;WLXN^kF}K7Hq@evL4xmkOUNGa1(_oc2F8^H+A}QK)bm|k?CKtQ%?>>+Xem3DA_jkqrjB}APWXc;3N7wZ#!!Qll?jw2^Hh{;gKz|;!Be!tEbSKuN2vZpdOQRTN}H@Sa0p!#b#QKQwd10EtfAnNr^Jr&2!28^#mnXs{8=soC#jzbIp^uCG#B;w`h<wmf;ki3!8l{1rav49#~NA1c`17esrfSKAZ=07+K_g@6wTUXI@uSws<g_`GUKWaD>Fl9e~b>Dw3p}PnBk4PDV{FGYO#(!NLfH4Cpf79eO47}BkFMu#UZ5+;L;LVo}WZa!|4>trb`Gnx3`c_jAszye}x`E=!2iYnTF#lNEI$`!r6Fyir|b(Zru_C4S{<Z*k$9E0{sHr6Vn-j6t-ZNU;5#+=h(jLVmm?A8bS*~P%qy&H}+y}G7v5Yb!&Wm$er|KvEQ#N?8zEq<PdP%aFd_?j!);Aey?$HfUe<UWHDZLkZEU~Y(9*2yKv9T(@K#0SI%T#Am(k?WB@r)Mr*O%YTu70lI)5UqERiWi37gCL<F)el|`#!IyuAdAJebNA=Np*Vi^t>Zzx<4g%mCtY^*TQ5DXZ0yH^9iP?R=`&En8-i_``yR-)1YmRggz6$Nj-UZQZ!6AqEvfb?yBw}iVdMom>yUey~_LbU}RF~$l@x3DmkTaC_;@5$_=_}j4Wm~&`}WB<RO_TSiwV^uy>2uug`zlAf*<5Tl^V9%kYXVuXiSs4jo!?xjGq?^H7E~d>(Hi{80N@NUa8915>a>sKrF?38YVPCl!0C@#?Z!D@YJ6zX>G?mO<jrb+6aXZ-<Y&&<gi_!C}31kTK)(R-Eu>z)EG3OU{);5%}7<Th^&t?@+QH8TT4I<sKhdFa+XgV3T*ur}S9X{w;ZqP8mB+-Tgzq}*EfcDxjqww~aEsXc*u>mkPT)8D$(=efU@xpA|OI$EYR1i2mNUvvMI-3}XSvED@tr~#^ELAOF(-5+lt*P&FyP0V5yX_`*Ic|9m8=oOcyS)7JY#MxsRs*;}>5!|Ik1}R66}v=XyhY|zk%$btI>#ON`7TRD`D85q(6I*rGshlrzAeC6L{%n=YLnO1g+8DvE0p-+^TDPWBz<`9M!g)c4_2!YCcXZbr4|_QIyn={Wgp5XRd4iJU04cLLEiK=F&G&0rY!T4AjF%wYjeZV4RBX8gMCMjR*%RGG@{+^BQK9+vWAd|zE#%^4;3Xkd)#jPYmdY$B#~0&ODG8-rCf}u6ywSai?NQ<>>Xe=>M|NAjSkNqH^os1@d{i%*gbaG$4%GB5U`K(n8b1$PDfYQFK=dts*dA`81PuY&cKSe4s)q*)lHQ!bUwen&2T#~aITd@hWs3@6tL>&Bg_O<1O9XN>#A(kvt0&bcSvZ@JMfYpy^bUX)bk%A-APhSOZg7t0=3@Q?#=0=a?-_eeIMy(qiC^AQ`9r;sj4hdG3@AF0d`j}eHQx2cYZ@D^kg;zR4bGn&fcl4eyJO}>QNFD7_Mp&fAAIrm%6i%Wu%^d1bq>Ys+y0vaZAUX<mS!$(a2w-Cj0gQC7rW}=zypp?eTU29sJMTAK$%lJI1zJ?{sl~52&n56g_x9NTA5_z;{;G0c&IqT|J9C=CBYL%T!TBY8VrMy636i%mE$0o1#P~chl=7toU2G<1;dyiFeL=?TrFj@AqqM*ilqWtc?|0TDmJsQc`OcAD?w{2_&&QY2bLdi%X@}SiyuNY;s?Xv=R^Wr4d}ju}Gt{_!$YYYg`^U2ZV-z*e$HZmSH0?hZi<m4=mnel}RWxo_MyvPHl92-78wOx5m;xoEXV%0hUROrka;nOZo>f6v__O^paL!{ivt|`bR(lOOk_XNTx*1sv-S6QLCO;R6=G@BEyUgMfT*KBL0r<wo*J}Daxe3@Ogret!Hea>FV!ulozjl4I`d4*Kv4#l8Lq{%TZov`;Vw~IUo7^BC)dh6L;LdxH6zru)Vf{IXX^%aKm9fp9`OEFZi}djz-)j8N3H<%BTOJSxJ#~(!hf})d9}3O3V|EpPN<i7mv)m=}q9wrm)7FO#^3ogP*38z`4G{PuCDgXScq6kk(!W;u}Cs1BM$1QJktGqR1INAbpjGd)di-U5cz)<>gkE%{`{q-@gBW!tW0T{6+WT5m?y%oL2t0$NaMk`Ekib2qeIexKAfg@Wf?aM0&!8?nTfW$j&~gq~7o}O9S`<75g#%90K)XmqIO*t-Y&>K1xGWU{*hCXmHYNA$^f+Al1W+dUg$5MO+r+Wk6UjFuO(paI)G>c>Lk?pbua_QJCK0#emt}WYkF$^rbCJ1PNah!>Ul=l|`6+@UmA+8ebW6cxf}=Hd}yAv7mYB(VA+mmvYJ#arR!#-K>BF*|qgXH%|*Fng!<}#lGA%P(xRO2Yw`KGC&%@cAx}p1uZb61DiAMgBTolK}>1T;Mpyvdwg#D<0vocrtVFtHiD;+9Zn~H??w}eY2os|>L)(`5d{Q$C^DAq)hR1e&{qOc6UzrYo`n@UQN^2sm@NSIWuEL}o$EK%INi>Wf=+`+lvv-O8VAnHTg*d8%hp5fsp*(f)sAwSc~FjtdU1!KI|@pq!oZ;$jUKDquw~x(?I!pC3n1pOg|A?sOc11fgXD-}2%x)LoOdp|HjPnz;Q;U6Li7M(z>_}IqiEA!DeUtXq;Ne@Biv1~=9_l-3fc)Z7k7*n{mZ|eTa2-5ndEEu=Z#6sQ}#+!Ht5n@)}TMfG2l6X!xiFi)xn{qtamoKuE*en_p!q9K3V`5BNOjqA4#t3$pIjA9i#vcoM#9J52pzf7KcA~0DZA9CMO-Ic>K0<>yR2o2R6Nd^BH-3$FzRgPd8Etb)nvu6m3k-!3=?q59t~-9mOC+PVh{<yU(6QSL0_|R_gA`<NuTHfSOrQLS!|hI{Bl!&j`W*$t7(EXZ9UTCtkF!Pv5q$5Gin6<t}rVXXIFwTw`aEFtjtQKEDhzT4ptO(hI{S;86dLx@WP%P%l$iY(!SOU;JMnxOPWdZf!wt#fF93Tmj`sww9T>Bf#y~10e#CuJ}Oap8wmy0#9#<<(*56e6K(5++Q?`fqBt!FT-XC5pH4^min0-$~k$n`Hq1t&qH57@D>KkB{~K~s+q%Vs3VnB^TpD)zQzFTkH{W=B~X0V?HOqJ)aX1%kF}(1k`I1?qcv3gSpbgB7cCqkakJ25Nw$13M9I1P`uo@4J6i?iVzzcj=iLUd-@P)gza4`2?W(6O6&^9Sp8++_IQU#3QLB|Kkw5&+lJ*Y0Y3J|XUNRSi+D3HKwZEiQN6w|X-&8Tpbo$p^$>H!e_WiTr=%#$oL%KOFm_RXg@Vw<2kiFMDZcy)bBlIO2=HN3|`d;l_>@$LB0WCQ6zU<x)q}{&gZ@OrK<B`<h_k_6Z5IAN$sqhR62j^zL4r7{y3-_-PcHrD4UG=fMXJ(;kAvkk`>Fqg|)ZAILZRuA~Vf$77auOW~a!VMKylQv<DK8T~qJb=aoxxW2*P6h-o;#w^7r-ZYIHSJ-bKBg=?wl>3^nIzOvCqd816Si(=4*yRR4VgF&pHWv>AXNRd~-5SsrQ)}gphJbQDZ33N!kEL8njg`O#+|mDa)SpEGn_BBiR$(D~rWeiE|kVIR6(=O9KQH000080000X0PW@mS1Aqv0F*5N022TJ0CZ(@baO9sWpi|2b98lNc`k5ywOVa&<G2z2?q4COC?KVZVki4{L0xojZ?9c+d$-s<6a@@}pe5SjMV3@iPT~vl-}?+nNt7%nx$Sm8Y;1Cd91iE@nW5#jEUMc3RTcSUT@_oe<n>i1SDrKzpYYX8)caDXX7~&KX5uHl6{_Tkm`v<=zT1|2k1H=Po1t=F3ssvnK7IJutngvWH^Q!5@m#HoYAdRCgLfZ4e9|PJsv;3e71hKLi@Q=(ax3y$#Wh##=5@Rk^;MC!tmZ3Q?U_uE@~C>3?%UH~16kzX@6wH^KMS5*i8S)Q5Q&u#Ow}&0u|}K}`C4uo*1w5dRJ;cFJ3VgM%$GxE$dQPW$Xf-sBcFxZW%VeJlByPYA{-_&RR2`|_KxH|WV3%8mX8^~;h)XeWHL!Iu9WvnaV>KBo2V`)9{!0!v6_aW?jw{Kd8^*HlBexAZ|?bjK2v}3Bkw%&W|23Y=%rGz_83cemN6D6k*y=I7I$^z@ib+s6p7@SSwzi$sdgeX63{=n)}-Ei?h#|$iqes2GTivwTbwW3QIRPzN^{};$;(6@+!j?DdVlmTrXI}AqpzOK^=`2$Gg${=w^}9YU6l_lJ&<|1t8s8O4;Hf}343@VEs3{ShS(RBVW*N*Bt<G3g=pf)Q`fS5LMK)j|E-tsI`(mae>Hv;ujL)4#DrGYMjh1Kl93lj-WGdhb`@*3bJsqDc70V>EuVMh`DrC@gdavNK~~@{SydGEd`Nq<bqP1EL(GT15p4M#%f)ST#upBo%6X}-ikgWVQSIk5XGRE8_EcNfxx=!8Fp!0ptP;39C`PFHY&ta*;l${Te5yHhV3c5{QeJaR^iGW-Ur+KpFKR71OJxNzO$`#26qU?3tRqlk9P#!V{|6<nDCn5mGy)a3+9Zvgkrm1mByuf9MYb5s&MzYG#miT&Lr@$9la7ZmIXbT@z7Kk)o|iGgXj*JzSOUkPS(R^sZt*adc`Y^w9DP3e2i=~(d>MJccTh66zX&5QrFcKbsypz#>9K^9!%td;t(w;UeYf*NT;?0Uf0|hF8xasww*;vs=4C1Jv@hcQfxx~@{Yyu4(Fg@?!=6b8B9bM};pQVW|JlJNLF4}*R8q;6@aik6>`D9)1|3X@;f81D)7k3*&BH)}o$Ww{-D!<%3hONH&55C+1$|mRXw$)0vKdm`vfIHYPuRdrd#tn9k>}I-NGwqJ4=#&q&fs~T`XASNE9(fEm2l4*(GE#gtKyDCxzLFP!oM6_NQn)lbuBqjTk_eo{R2*ny<RSlT|w)IQKL@*6hnba1f<!{K<yHSH5h~l#0HtGnj=dCV3=!<30C=Gd|Sy{FiJB4jl?NXl5!RZ%^nHYI6xgSf{uRC3_kk`n;ScFLv$mh)nSuk{vR4@PWU_pHPoRS%UfJC??IX_V(6sT>7|7@fs;B8(Pcm4C@{?LeI%rck}MOcjuCGjA}^^5Bx^H&CFJIc{QM0xcq`NTN)vV7-R(#?I$&%qGr?FG<19sn2pGmTMF2}2=B*eA<+^VT6kv54#!mFvF@P>bV{VY39kLcq)?>6L@w?Y(MKM9N-jg`KR*FibiK_`~Cdrw5TvnzX5TzY)x8G5`>onh&`3%#J+y_F^ymE-b(S466;xTcaq%MFnj0#pffnkvw`2vPi0Ra2rP9(d=jY=X&X?Vr!<chU^eBRSgLg*bU9zyy7$U4)MSnp61X2yY;U70}+>OhGtS7cB~>+py|1t#N%gz$?j?2#?bFDZ6j&ZbLnkSc~~wC^*+Lgo-=(5lhs=;Eg$7bEc<avG@P26l?&G_%eMUSGI~@Q`g9zj=!IFufUX5*wcE*%}oLT{Z6QM-EgUjZ@3~1~*q|l(5k0b$NucMEod}NLj{LC=;HfSA5ILN~AKW!6b_vap+?m&_DEt24}r*p^l0BO&(Yd*yyGs2cQhR#ojTagfMpuZQ>Y)q1M?#p;RG@)Dfc+RN%;b&$gJCh}8I=6j>$`y(aN(1+aC>l5$6|9`RQNx~!pUCchT@lAOs}^s(YT$z)UZ5?&XK&}M2+qGORk;d)^`+O!aW%-!}uK!7RK$&Ws1fbPGal2gB*(f>cb@NLNmytR=z&VviymQ-?3JZWoqJ*WxIM@E<u2+h<2o#X1sKm*Q*YPK6j`ONX)K6`_gHA6Ub(1A>c%bJZH#l}h?W23-xgpah*f*%JqADWm5-KL5nyH>S1ad(}8ueM0Or?x`%joPC|*HKs5*#ys(64WpnC<q!F2i{AUi|TtTQAj+m+({!L9Ic>Yo*b#z)u`ivpoq4aCR0idW_v3k90ZP<X}^X4?(c6ue74k6;Iy=z;Yq!N{(BiflbugoWM#-TmN!RmGTLJsn{_iz&7nP07|m1VcRdVjWZML=ld8B2TDi73FhESox_Z6i_fAk~KTE*Ya6z<^DPTZw!mAA~_vGaImOi;n3@(-M-s<yUzxP!I+~vP(?xe%N+nuUCU`G!ZPWx?f8AzqiGuPN=W^J|tfGdOLwc>T@d`mq(_RXx;E@{lEQ_<Jiv=%8tnh08XDUMOM%?W6Mk-TeZbfRc#vUljj7~HH`sHQ2_`mDy=NRFurG)C>1ZTR(<i|LXOD|P({p&y*vUGZ5?<QnpT8*1jd4YOPTesh{aQH4yE6sqo9_U~bNTx55JCPYAN+=_-hP6q4J61x_nR7~a#gHBs(4uIbto!v+!S6PSTzAkUIEN*+3!-v)gFKye~pstTps#N1FBaP&8x5}hK2r+CE(=_dnepfb-7ua?}S3%BD5cLMmdtC+VGfo$Rl4p-Zdy|GVroY!?M~X542_OhMri8nxz_9mwg{Bc6;<_prwkV_aT%p4h8($4ZP9-Qjk;sMw&PSI^Apz4cuzIKFmdLEw*JWV-M8^sj8(+?Jv^i_qpUNNfEnRnaJGVy~wr#0=+`F;v^L_g<mwsb2-J_{>I=0?%f72NIFs&OltADH<nuykQ!MD7+rcxRpt*(&Q0)I`w2iA^@*VTmqYs;|n<y7)bju3(RwUnjEWG>*_bUBPbH@UO%ajH-dy0Kgh<Z5MS_`1o3ZB-sZROB2h!Q=_A*SwB2tw%!?clc5KI-V0F>$j1Y@$D+*fPU1{&?SIsnoe0ZM4@zit%*Q;1(G?ur>SsQaRaM|Gf~laCsh00$@F^B2Aj6kSf8A5Xt_5vc6hu%HF?o=S|Xbzb*p8(Xq{cYK@_>1&5jkVkC&;sM0HD2Zzwb*!-NWEAq9ye)!BEm<-<V<g=FnK|IH6xmOwCXH4p7j{-2wByYxt!&?!~G-z!uaV+-j;s`+Bxp@_ASz_ll1AZC?t=yUo8wVP=VY|W(!1-b|F!|A8b?|*0uQV#?FeEa1eG+?!dk?(*0cdHab^5M%D8nG%OczMGs$#dMV@iKx))Md1L#j+M#6*%^#;~|*|#Vo5>w~e#OidC@oox`5q+dUpmPtF>ksoRg4ABF>9t^L6XofH~O6-b{*G%bJ|H_iJYO)K{i@knvf>J&&jxH!DBr_gu*m9q&+-n=agoI<?|wml{*Su?vBF=ZXV+Z*XWtgf3oBbxS74_XojyA0bOdXL3Yr;?6JM_d8Kj7<M+YmCk$pw6|ZG#JwLhORYs>f6>C-@xNW>{}=NZdt+-B0beo+@0cB*^Z9YShw3<cOHp~s@j!QRyhFqRsv1fl&UqUg1#y#HbNisCp7B2H}aGY++xRp+AIBQ3?>!hrZL2Uq=D><(lmtZ7J50>ONO*y3fI)m+CY$zqmBLSqc3=jq2$_%(iiWxD)J2@bA@fNqGAfjAJ=0mz2u7L8^RO6>JT=p07j(w7ss_}B|9=4gM9QrYqp;IWl;)0{OV=<4<G$K9(&9;2c7#LG%)bEM@7{>_TATEM_=jAzR@#+{2E=sU=v9<aG<T<sihIw?8<Y0wVu8DAGG72K^<qiTn7fH`1ht^?|25pp@3s$#&fdTT-5Y@!`7%+zM=r!a+#YrWS`buv~P-j^Lp0O9C<}MYC>U*O)1<5VK}HdX!Bs<pPrhH=-_n~Sfjvjc?Hv>a*Pg3pnA{<p%OzEHYUEmIG>JEBv1`>l_S6K%#dDOyc(w^+^X^XlG4MS-0rq0@Bz!HY(w-=Oy|5@SN>%o^SbS{Jys?_+Z9%!yIq(DzebI2oxpc>fvATvVVxDXtXPS?av#51(b!6?sai_~;a&;`x(}g_mk7=7y%NE;%C!|(W3Yf_@|f|oor7UH@pW~By%i{_tsX9Ek?;vVzDs58usFc=o7N~!$4+bcpiegf4tiyY=IVmgAH1LQZ12@q!qYPV{<}Qo)t<^yC~UO?;PZaLwCU-3odwX()F4VxIg23x2~^|?Fe#I%SE5p(4kz{Flm<J9B+3(rNTNdQ1i<nN{Q<`3ZQy?K7$$w<cNr%q)=+}cw<M#QSf=AoO=~j$`T6LaTwARW+B1IZ>ti$)+yFSUfhI$_J-U`gxU=b@5F?Lsgcs`qca4I<uHKA1^?MFmCjm5Dljwe>FsupNmyG*|$rGm9nVlawHhA2^ouAkZ{E`g6kBvGXIANE4YjyFSO=p9cZK0{{7~{g4tGOIjBW+C8O)L{7+dd3i%ZlO;S-(Hzw2nBmL>ud)!Jsjd!KltC)pj_H3a_LbzN^IPG=BZq^emn}K7BE=XI&MjHU{H8)ep?0|6|g}G?y1lWO74|Ue<P<j%%(_R;Qr8Aj`>r0Z>Z=1QY-O00;m803iS-uBFas7XSbaJpce50001VZ*OdKFJg6RY-C?!Z){;=Uv6)7Wnyn{Yc6nk&0Fho+eViDuczpNe=O-xpk#aEEFEqwEitw#x-Cg%CZl*^5NMK!Kmb7ll9_C(=GQ*J?ml6jWWRH|8*EUF?Ws!Dt}3y_zMek!8$Hi^BQsfsl@!%Vic%K2N~*lv9L~e&PR1h3D>={eyH`Txbs5o;b(myIB$>G0RI5A_>pZSgseI4#4h|M&z7|2SsH?h^K_HTKk(ZSSGaM9FNuH^L13OkOi?CGEe#WVVj4J!_N0n#xZ&lBWGLNKEcBGOYs)sTx^o2!Ot<q#}Z@b1vT(bZm1)dQu0m<M%;D=ZH_EAAB)8t;38@sW?i|FU|XgUr~t}ZUlX1!+69!94GQNCU$)poL!3W$?|#*@8R&vy@Gs-y$L{{aLC2eDj;d7Y%OzQo;5gb(U69eS&(D%7CgUnbS6p8HtTpM=pWOy6!)e-<teM_GJ0mCLovs>9Kn^O290o<8NN{<V95S3ex|*uJj4=GHTxHB8<+@u#7Iui258k2cLi)!yRZOuu@tMNXI~DZUHST8_&yFFW3)p#hjPwup@_n<p{emsE*VB2E?yS^8dAV@;;15<_wO?tniNQrBU57v~RIM{jm+<<)*z6f%pqkEfqsTA%30qs2kBDWo?Lo;}m+Jzlt$RTzg=h;_gGx@TU4fmtghek|!d7Cv89Wm0sy;tS#ZuzSO=&8P16g;5?$v%3c}i&YZiz~MI`*t!SqNhOgY&jLxf)#DBt3{r=m2~e{SEVFy|papDmaOW`=s&X@M53@gCY@tXp2a2BXcwcwV4o`lFWKoIN0A<X7LY4i_>3oV+Q)L)QQl%eR+N|-5GRdj{S9Q7{S#L2?*5J^!{P<ep`rY~NAHbf(5c^nDevna3$bnn}J}kY=Gx-S~=A*w$<O7|&$ji0&E0JC^nIP4<pU?BUD(XsM*X_HHiGiW-$Jx9QVzOg2@I=f;Q*r-N{Py?%5hvF-qJWkROSWR7FGj~lM?!umWSNjIRQi;gBF)1X#xzllk{4AjPVzLI`{D*qdHR$N`|UsfOHAraglYQpsaPbXs(NIDGoe7=s1h3ht+nb?zUr$Z@#b}Vn=js9zlF*hdSPAV-g~oEwIIcRbRQ`>9Iq90RmU5ecyoQ@iwlr4Red5xKZ2^4st-$2r#Odf6Ipul4OF<60<Tqxs_<TMT4GXLv^<SvDZ;4Cl>!V(k)7uowI9%6Liq$sAJ+<*-6v(9G3ww7NMnYgQo<RDMVd!=;^q}!-5~N#mYGZiaGS~nc??w;RldgjYY5L0$jY;H)5bp`>Z}q`3Qkmf;3D~eJBoUqCXpa3k|tnM0YjWDpz`@V0^e9vcsqntYa+Hpx4w^|oECb%f=qn5rYHD(+6=eKIH|&UDrsFRX}_kB<5CfD&hhgy%%at+hH$WTAgxleF<69)rE%e6?OPD6@&gcKET|6=1XY+Jq>%+at=7DX6H570os=^68BUUIp|Y@0tGuFVXmgwp-s1+aG80}rz(3%tR-ppR7$S_ohikkl5{oo{7>GIOct>=(J`!P#_cH@OKRQg-1iV`3can#my=cb30>C8(`9!=omQe~0fo)y^y(##Qw?ugcu~};zcAn#jGFDK0OYDy~UAv|#a~N#;aD8-ms;Ti!l7*?kJF_lmef_+D{i1*UXEO}5uZUQ&m~Y4gh7br}j4XTeG4ncJ%8D^VGM6ujHil(kVyvJt6iHTN5fLzzL@HI`u}Xt8weKQK8CqyYbzdLl;7`KPvJT6bVGhA%S)8x&9GuB#<#Q!Vq7+=Vw2o+o@B@}Bv7fjK!myxyQv6hhDKmUeOE{|q{TfiV;kd9&;EQX4sKKXJU`QJnQwVH@%PhwWA}b7#oC401B-?^pjpCt4Gw(omLvfT5JvoFxf*?SEVWV$qV2!ojgG`ny=CXU%17c3hQ9&*>We9H86|Ce`omA2%W-H~7qzN^dEWGPPfn7+$;TT*lLBaIS$`78QMI}ne_gjF8mn@gSz6h17*ANgs5EjBXgX9)1wgzKN91f76<&~_6oC<0-0!R~CWAJ)BnVwB&<I7nvxw@K>#2XC>e=~H{B%e1KUta|`lXJK?zgP$TPo*|o<msNDU0QFKmN?F@T{Rg`$D_%~TW6Kl33kCQKttj6iCckyeoS)Szu2(6_hAbUrz>Ja{n+!XlhOI(O;xB4?_VB1<mH`S#|Da)u!(9J=qG~4U`n;QlK2b!g@09)^CwU$y3-Oe8@=IENUiLiLStC1^-JF<mA+LZeXA(+Us@gXU53Rc=0E&~)WXDsDPMXq`fD&9Pp4;Bm%+u^<;`q71qjD5#JIFb5!139=fx|AJ0xx)1xeO(EgLRJ@RTiy&D+XdziT``r9*Fzdg8bzp1m_<`?s63$v7Yh2);Uk*&m8oS!=C5x|v-C)3-OX)2qK+nh6ILwOVNnZN$)ci^6z|B3$5w+CCap_D>K<kW1fua|vF#{N`+OF`D7s?!{(qHn}>V?v9_Hjow^dO=o8(kEXskyCg;#UyRN!PsbBXb9B(~=j6L_aD8<>4$dw><JR`m5x8#53>M{4i+i4k(Z%cWq)$7o%G6vkv@~Re1$@KCgD;G2kKyNK$pZR=bO0EKFB<=KHl7fe7o*FwZ^l!)lKj^8Bm-z!zL~xl&)!~XBEH-4K%WS?E~*Wtv^`ETM3eyR_{FP+)3i#3WvkQJx_&U@KZA5uDXQib>Bb2E3^F}E`x~R{mKdieUi8FQ@954o=v|Y+AkEVQ&zEqkD!4vM3zUovYZN&S-H`gcGsa^yBUM2B{(5xs?f6nZ%ahRBrHA-P#YO089z7pTXQ12kjF+EZy*Zn;Xg9q+Kbr-_-xz&%^z6&yqi4rFb8_|FctV(*U7h2P%M;N5nilyiScpc*Qo*>*Ri4(BF*6yNj>95{&j*O{mU}V^<vP!jpRpK}e+9fr`LndT-9!i5Aofw<G;jmWmRz-?Om?f5dA>}gACV((!@F1unmMqwhhw>pHx2Je^vMX)Z{s_aFjJ}}pjJO{oL~?1cVbb#&Sx19z}!KdttrUrv{%{9bq1rAEHg9RHd@UN?}_-=3Y;aycjVjWS=+%i0k-uvDHLkl&7O#Jw#nq!x<Mv_&*|08ga`{ZGMd|&g}N_0?PN~Z-TXt=KFwqXeK=csS7(2!tGf;HOoD>g(I&w52YF9`awu|09i2#TbR2)9vb8TAOU+7}hA>_5+LE8Od$kTX<N!k2S?Y*o8#<6r5tW?pfbeT~f}Lrd70a}JTGlpRD=p{xVrp})44Zc36kAR0WD}^Zpk<!JD<`1CGFma1qyss@xJc4QG=R_w?xFE{C^q5*EfFRSBif}E4ew#@=cq}mrJb-z#|#`D>gbWJ3BAjPdNv~TKgx%OAV#ush=<zXA4&uFoCI*sCVMfss;eZ07+Kk4NY8Xy+K2y+$25sm^RXu`3=8zc+u6nW!KdZ>U@LlB+;&_f^BgBM9$FZE%0zc|oUGmD{`xLX$_{xzh=n!Fg6S|B<ahkh;ER+uITSm`KXSh8LIW7(UEAWglR52)-RYfdY?pGYogJ66x9eSgG}$_lOSjPArbgI5k{nCwxSZq9d##O>)qw^im@=6ZgoFJmVCSX125UVB<GseU)@MV$5eb>+;^_8WkI<pxEagWlqMN;-LAI)#LDr9u1@N*@RC98oX2gdu%ChRf18nW=nwVmQSn{DAAF#S9Iig@k?z;;ej_XV{ap2<=g>O<dr5Ok^l2|}S9!7_NF$>#P(Oo=*76*%2`{{}HVLo(9St){~<E{e*+vYTusc<g5b<TL_>Hz%<eC`Yq*C*LM{Gu4V6ljfOESxd9*KPeYfa9{mx~=8oEvigGOOu=SwHO2b&wp&)NK%c8)}+$r{kIImlaQOn^14Nuom~0nn5Z*&LfYaM2}d1H_DmZk1Nw9S_{cy0lGVNrbz@A;5NEI8U%l{;5#v<!={_Op5?K7t7^K)4#5}n<9r0<$&oKC~k`m^MLK|Y>MgxPFSmmd38Nz*>m0_kRX$$cZFZ}0!_McIlAHf7Do59g33ATimhYw9IkHi>k!`i-*YoTE^hKQ-+0!ChQ#*kBDB1s#Ym`NB|+(7nb&HBTzh|b+<Lh?DRJU`16{Zy0P96C?;fsD?r2jnAsJXGt1oZ`bnUU;}H>LKxPZ!On(x$&zUHdZen^n}$gQun<qU&%0rI}NoPg$0*MwY@i*pNbP&>$5Up8|Oc)k_b}eT-?=p@g@veLEr!Zz|~;u{3p*dH}I%mN?e$+f`v36@eHVucJ#EWV!&dnF&Z}#Iwh)L=YnEIfg~(}2Q+w`p*NQ@D4q%9JqFyw{Y~J{OL$OoIfEcP?B<gm-GVv3MIRkunJct!!!mxrT_k|}Kz|GzSlnx?3A%^&?_AAd?#LG{A!r-@{=1+OGVP`G_v0|0MmrvT_N-xWp(j=+*!I21hx-CI9SRNV?)K=N{@n3;UibF!_+5J$ujq=e#mgi7JTlA^t(c8zo{<S@`=Ct@(o`PaynaQ&7NcBSRL*-8l!MbArGv0OVJoEsSY7zu=b?>|VNmyRV5d4l4q4ERf$0(6YRg_j!-jpGMbhPue*}(><H_|cS8G?9oKdgqCn_Kf-}wY$DgE0n1aWKtqKB+;ZAylVKcrJbc%0Dul$BR~1BGJdOLaMgYBUz(Od2@xd8A=8FVx<;V|{UP1?C6~YgVB^N$I#IgvCNzkl~*B4uQ~noaMt3<!xQ82E|R@hNOQzhqBuu>a*eFjm<W9u%ki!T!1i$SH($Gb>Os_XZ6JO_p`TGm*AMUl%yDXN7y*K`gVMI_BZ(Ja3Ck6^K*C&(~F_Epo|sfFRsr`Vr1xzmdnQl(984Lw*ftWNWNT~k+QV#x;^pz)%lgqvAqdS&n9}I>(6g9t6kbwr$A1+p+u$EwdTqd)}d3Z^|n_AB@pWkc7hKejnn03c6%3PcPPc;4LFOoeVNB~6%13Vvd4wnRZasY@r9u|c^ZLAeumRCaFSEbAz?)@QC*lDi!wUkakhz|3laBl7ZNQ_^JPK>-hJ67=k#^>Auv&9uuihNB0%<}Z*jg+Z9zy%c<@L?JBNZn1GmP6)jQeIUbBd^%FO~!k?x%2<eT+6CuM5R;e4N-+)0|cXK7~&D566Ao&+ipl59XgbOYI%Je+fgEhPn2o=U1Q;;8*;xxN5q{3B;C=^*HKZ(yXnM@4^~%M5I8)f+s@sJhFz&}TSsPj=O18~6@^EuRIPyAC+N?+vuK<Tb9h_GrC9+h^Y%wXO1agI}0q5LW>KU`9~n_^YFy2miC6HPAE|@o6M4g2MUx&KvCSXRzXj5Wv9UEsEnWVV>hWP$5<Q49Gx!b{~*N#hNq?@5A8!80&QU2xF&GMP)KKu^r>e_jI}DS4aE<QW~Gm1@Ky9xOR8@oQMp_<LuB1Kd#q>>a^5LPv|C@;WMLrIFkx)W$W4Df89=>9bcKX8>{OsdBSUt`Jic<@W~R~zx1QjxYGJ=ug9VA7Se7Hb9|TA<6q=GKg7c(x`$dH;{P5GI7yRFSfkGYcdxP1&@v1Ez8%5}D1E!?cP#>zX}ESDCG4xj?z6T8@Td^w{ZWDDwW-AAoW9&vg-uzO92H$?)yj6Dv-Kaqq?JrJ;`rrnfB)~}uc%QaUud8?W3NP!*2?VyHJyU&&B2RgF4sQJ<@q%Qg)G+Ul}?0FcDx;Va~73TAf_gJMQtC?kEoj}$(r-9XdyWa?y+x2)%cUElPkUK+0pS!Q*n0llmn4c(OH+CDSTb<XYL`;wdN2}aMeo7(j0|a?5y#L+*?7zHWAUkPvnEs1>qEC`qiq;>*WejAR_!UcN!d)RLXIR=f1d>rD>IzT)qJn?<6&F!2LIkJDk>O>h45{AM-gVilEaMBhzQ|qsDt)fURv3n;StQS{T3D7R?u7lGYRpa4&~Q!*Z#U+qOz=dWFaVPuA;Nx41;BI=j;Zb-Dyhf|!;niYKT#uJh@i6_7K>2b?be>_blc`Z#`}o6X=Ee$>mi@#$9lpNJ{-=rGSQO%+6r;U~_|7YoLK(-5#oK2(eh?X&-X(j%X*Ja!ZaPkVc(nuvbiB*$0Il*zDnE*ia}JKfx7G~55;KYa>qV&&M$g^TG!0(qspFp+?RoThY5F|}!_;bOaQ9b$0m?+12+fAG<?=M%^3rw=}bmEYa3Z~r7<+otjTN%foTE(4;qT(q+rQ?1MN6f_$cE9P`1^h_3a_DGI6Y|B?q9RG3oOI^S-J!I6&Qrg1o`S{J~<a?&=+tK9Wio&l`T_dwCYmlrekKkaxvIWN^TTs`QO>ycZe!9%z&aBqlwPU*xsTI!`CvQh*6WuY@m>?*lN@5FT(y7<boUIYBNQ)(E#VI;k%&k(@h74nq&m>%EYvwmEgdvB@&THei89QfAQN2c+_L_Ds&LA(V71y!2V~F6a;ggpY9)dfoe2JI~+S}+`-DH?#QCi0m{usH71#y_ut))2CUxW#Fj+yIqm4NG+rjtO@yQpACRhOC^GVz*lDk$BYMH`K?H7?{TyWyk0Ha{+rG;Hh|KmhQJzNRv8v~|3>czF*Du!?11O#iBunt!A~NV!z8s@CZzD}QGiSS3#NIDfWYl;SXi+~}4@r5UfOnCSSe<TQ9rve9XF`pYz*L(@SM-5x$4yc>K(9X^31kp6n_Q4L(5>2qb}3*o)Dx=i~;T6%}g)qRS?c~JZEZjHuHu&bmxyVaIQaJ+y2p3eB;hl~KRyGt3yIuy`t_&;P==K{5akm!WC0~Gl*$}h5I3nH`h6A-!d=#W=OxZr$<+23_;Cpy5lfp*!n>A+YtROp)(KDYy*x34Yp2m0a$*Sj#!4~F8iTM$oY9rs4=&1X)9#{QcB`WHf&=lMUvsqJ8_+#QH{fY{Q9ScZk{+>^d$jDFqcX?h3`(lY>)oh!{8hRlqzQ~K&pQ1$6krlxs~t|m9!_4KK?^D=t?6~Q}vL${qC-^v5bL9&3(VEeI^lh*Rdwi;>L!RaleMassn?msGILfxEdjSjCFZh>tv*!E}<LL`T&PVX!`pMO})Gcg@<9PF6xUY`34DX90Yz92sEea|$JQ~MTubwRh?!I^Q}drTh*pREgF{0+NS8=m7D!Cxb;XnH`xro+&78CQAX3%2~GL7lp?lkA?l9U1Wv?#k-ICWv7+3%E%XJo$~)FwM<vw|k1?sc|*et$Bi5=pU##=}yYRwk9>{YOYeSxg))?M`v5)K5DpXuhqpUH+#w7NAU*wdns;y&-t=Qi;F#7MI>Mz&FO5FW^*ns)3-}(opj%KSAg$j%_&gC3cPOLDtnS%v3t<Y*HoG_1Gp?Xf7hkGpS*k1>Oh0fz1;)2AEmW%Hkt<aF9ZFJlBC#Vb2@~Z<}#hi6Swc2W1K&|Y_3a@So31fFeOj(ay$8}`PLxujMd`&oy`(D(SCo^AC=3RvN_j0(`g)%JR1f<oJT>>b$0q;90#G<(&3a3w^e*}jjoX5ou(6i=rt;&zXu<tt)pBjPLT1D{EH5??E=Q|bKWfAf(`)nDfN)vgIA3TRkz3QoUco^%faldoAT|}Q@Qo{E0L6|y+7n@@u5RE`{ocxkoR^xI`vq?|GR0*{?&87#YC|jLAV`h3d2UB*^(cBYD!E)^A%6qv(Rpj8SeQ4X@{>d@3Tfg=N1`7eMY3OU)juB3-g)HOS|dD=2cGK(Y~QcIUe_#x({OT#C)yLerpkP&^FLcL$9tDhhKS)p4V21GlvJ}o4_PKLBQcx5D-HLfr-F0cOU#OP)h>@6aWAK2mk;8Apja9_Q8e;001N$000^Q004AvZ)|ffZf|sDVsCG2Uu0!$X?A6Dc`k5yr5W3D+cx%HUxD(8%u1~6I(Fi%GUGH(>zO9;)NUW@)p8&bvQSe53xK}V>woV#xR4+vIo@55M;3sCbHBn-6rI!jhF&w0i}jjo%?iTqYbN=cRhq0t(UeRPamyspD@K<59w6#wQSzKDsA46rSQ<sqXtb1KO|op+=ti<EBYa&8sR^wrp{eGgQlpXE!;6Opezl@%Rq};ja^b(JW>HI#Go}2FrgF(k=9ho*+AIy}nrcMj>7K)KG<tt>ahm<`<<qCLOEM$T;r_w$=-~BqZ&A=Cd%HL)4(aqYWk=t=E%x>o2mAXudvo;NQ8c>v^6M`@oMz`Imp@&=&3HuMCp!7@^z-FKx-O!z-6%w^zP&uT$bLHlGC!ZZ56(1InQSTzs`Oi(YL(xT?9YZv*4xue1q&fcTqL8>s9;Nw(-<s6p)+f>SPG%X&V<Ray%$9vm}YX#E3PzHOG{cQYZ95S_$?cgilJR8Q7jh~7{1_|E=q<fEDcA-ZpOTb5~+YLP_u?&Vt1Jm<t=!{EHGG_uULMQiAL9rw#vhwtLU2RXpB<dPHHL@XvVQk{z0ODKicn4(Hyq#ikX#5vwJ<elnpa@l2mEQ>)29vviw)X-5vZonJk4|Q#}LEg0+`rOkRZ!5O8BzFrT0^jaCENk56Xjj_Z~8(Nxr|it(FR{4wJqeg6PvJ^K<n>PbSVB1$weXU8p%zztcm^#aVoD|eb+Gi~=&oOGc0!<P;^<*MLvc<j04mD62t52CWWi)?PYEGbuv{6@=$ok}TW9G!N_!E#5|jnak+1&$@VZb~W%B80S2l@Do@jE4N6)~+1!imurl&1Yo=Q%jm_!;pnYQpso$duM~}Qjx3;0;CgroSgxEu<Qi1<g#AKSf*QRTK|D{V7#QWvr9hd@4Wtq+_qBMd*EDm*12n7N<kdW+8bx**@x4QKc8Hle&`@d2*z^u);bvaMgduXS0)7e4uUEhlY(k$okT*`ZbeJEW@~Vip{G<!0ipy<pTSPtqGB25sad@D_RVxm_RR0p{wB!=rqnZ?>4%zycmvTkZ-H@{tZ120ttH%=ioNOd@Zf-a|2^3|?7;`Zr+1QLMwBrkv4U@?t<R&e<dt_VS71`LqWiB8W4xcFD|TP-Ye4I4YScM%#>0@iMX&F>-D686QjlZTG&(kEeo{+*3p$)wubSNMx#Vp?vD!ID1NfjK1u#8#1gPFRgfk)lz}I97?igjj74tXPcZn!lym<|$wkD^rU{x*(=1>;QEF!9M&LfBR$we3FUkfPqj$0RNvHJ@)!aOW679eRTC_~_&7^Ahw8d-LQ>a);nA;b-V4-JN<^*@Qy1y*xRA!D*&;P+6MDy3-&Eypzhb^sY)LSb{{R5~*FE>~QUWmDzoI|RPxqUKPZ77wO_sXVP~RxwF|e^54Q>Qc~xh-DAPR+v&$4DqOD3FMG$O3ond(?+iZxF$F$6d;!kp^`%$RDcF<oIe0W<+`ZUU^~6XY77GS=E-H9-1JExD5|HVdB?Cr?4(S+&dlF_KNalwo+{XazA9M#p+ps~g$h@ag)30H?ybANEZFtUn2Eog4VkMxUK)tA*vC>rzM1U`Cjw@o)kBwLB}<o0SwiW~p&5y;=wvyayq!NDOrId@hUjk?I8YQpy#<SW23z)WDV$;wjeAi#9tNb%0zpdMER1IrBn)1VTl93GQVb0j_3}&?ZZ5(c*i|nq=ZRO?*Ww*@*P?J)V*VVz7nVEUKbcTf&6=#aGD`LQZi`{x1p2dNY}Xxxy^BJUfUi);bL$J3M#LScD?@iN^(a&`D{XW+g66nAg;XH!OygNXtL0%+=qt5K$UCwJE>n>H{?&Aj_I~V@i!H5sd$ifIw?|uA_YPyw!grglmG>J@8>0MYW~#ekF`PDbOd3ytrc8y3djuWU{vN0VGfk@&t}?oSuBp+!+J&G|nnniRG<TsUH$Yo=Am6s2*u0sJEKZP+-_EpXp*x6<dNSJGX9eAui2~CrIXCV+$2H908}2B@=Aq<my?1eip5tIm4B@)<S6eyf31*+Z$6%9{;<2$7CKB91B{^iBjS;JYJhXwe-=OV`Py{DyTJa^g)JAwZF(7i|mZwbdzqT@kLGgyXB74*QgIBNiU&w%CoUr?xu|fe-tf>W?U~MLbR>>ixLzXsYBcE^wEFEu~OM-;ZQ?xMzR-^PHJNx+QGTs<i40ylIeD`kHTPEJ2AECW%(fxAz<I_tfpkFIRmu*>gQ_S`uC-6Qq!ytFbP~bskcUN~OcjrD=J<ZHt!udtW^6rekF{$+_fl%&b6qqaw{wT#kI{`aDxb<e%U`B^@+c#f3p6A$b+2Onr6OT7SQ^~k(9__NNKy%k^cJBn;puQAMHF%!rQST^IJwZ7Qar`s|Y}>;`R~2(wn+GpDI0VlWUOR?*YV$LO%}#B!Y%CdA1HV9bdCe=w$0p@gUN5F>i-)&eN388PXD_tEE?pc)@M%1uU)_4l^-nfcG#;49>8gEE?9@eOFeiQ`Q*HSEFm<`^8lN%EFiGg}4b!AqnC6^p{d_Tc&RT(kg2xt2Q)9drnT+PTq2>Slau)q`dh%h@C7*jQ10_46-`FPfxgPO+prba%hbaRl@mnq$O!vJ!fX%#n*ujA#YTJJYza>S85<?ZCQyj@u+`?iC%}}<mQwW_Nt1Mi%LxoL~V<<n+AxzAf4~jwMePLw$f6?)LgnTth`jN7|RJIlJWl{8ObraYTG965xdZRvsDc{gen+nL@#NZtYM%+wzS8qFgMQ?G^(W|k~P=PHB4RCE7u-7;EoM{)z>Vg<w>5%$sP@G`Cr1HT`8h`@3#ta!Ul*|JoS|;sY527u%?JHBZz}B)Y!kt&5`<b~sZf}eYIVWBBF6V21kcIu5T#))v5d-~oI&p7KIpa0<5RWjV9uxB(<!9EhTLKY3UpgA-7&z48EFF%LPa#q(CjMW;1X|IoYZbe*F(55~i`o9zye(#c&6*%VHU6u@%5I;CTw)x^vKjW{K3#VWpN@+5e=EiV+WE2F0(3|!3N~UloKG;+x|z2(u+Ndz0e~Bdua5T)-CW(K#Eu1wDa)DlBlGBUA4e&SAwf&iIffiIaL?Q}8O(ZEviSmuKij7G;oF1J_iifofiy79R#0P?WUFD^DCtf{e*;iU0Rj{Q6aWAK2mk;8Apq0QNZ(%r001cl000R90000000000005+c00000K}SVSR8uZ(WKc^10u%!j000080000X08`-1onZ<90KyXh02BZK00000000000HlG20{{SIZ)0;WR6$c;RySWmPfS5VE^TB`O928D0~7!N00;m803iVS)VpCv0000^0000N00000000000001_ff)_}0A+Y^WpZh4Wo~qHFJE72ZfSI1UoLQYP)h*<6ay3h000O8001EXjp|AGt^fc4;s5{u9smFU0000000000q=A(V003opaAk67ZDnqBb1!sZb6<8hFJE72ZfSI1UoLQYP)h*<6ay3h000O8001EXOu{dX<Prb?rAPn(8vp<R0000000000q=A1A003opaAk67ZDnqBb1!sZb6<8hFJX0LX>=}dc~DCM0u%!j000080000X0M;3spbQ8A05KE*02u%P00000000000HlGOA^-qocyMKMX>Db0baO9sVRK)0H!ovsX)bViP)h*<6ay3h000O8001EXf=jSlst*7F<uL#NApigX0000000000q=DZm003opaAk67ZDnqBb1!sZb6<8hFJx(9XKrtEbZKLAE^v8JO928D0~7!N00;m803iUeejrDDApih*WdHym00000000000001_fyg=l0A+Y^WpZh4Wo~qHFLYsZUv@VyWp-g~bzyXAZ*DGdc~DCM0u%!j000080000X0RE0lpH~O~0E89*03HAU00000000000HlF}TmS%NcyMKMX>Db0baO9sVRK)0H!o#)bY*gGVQemNc~DCM0u%!j000080000X0Qts(C+rLW08}Xe0384T00000000000HlExWdHzWcyMKMX>Db0baO9sVRK)0H!p2vbZBp6b1rasP)h*<6ay3h000O8001EXp0<#(l?DI+Z4v+g9smFU0000000000q=7tg003opaAk67ZDnqBb1!sZb6<8hFK}{YaA9&~WG--dP)h*<6ay3h000O8001EX3E5{&Fbx0ztSkTk8~^|S0000000000q=5~2003opaAk67ZDnqBb1!sZb6<8hFLGsYZ*p`laCuNm0Rj{Q6aWAK2mk;8ApkVxFlWXl008lv000{R0000000000005+ccZL7}Wq5F9a%pX4Zgg`mbYXK}b~i6`baiBTE^v8JO928D0~7!N00;m803iSj>WD!i5dZ+AKL7wH00000000000001_fp)C`0A+Y^WpZh4Wo~qHFLYsZUv@Vyb98lNd0%8{VP|e{b98BAb1rasP)h*<6ay3h000O8001EXAbqr*r40Z8S1AAh8vp<R0000000000q=DVO003opaAk67ZDnqBb1!sZb6<8hFLr5VcXKXqc~DCM0u%!j000080000X0Dt90I-vjn0MP&d02}}S00000000000HlGt%m4s#WpQ<Ba%F90Zgg`mV{dF>Vl829Vr6nJba-@7O928D0~7!N00;m803iUfkH3^l0RRB;0RR9S00000000000001_ftJkx0CHt<b!l>CZDnqBb1!3WY++(8VqtS-E_8TwP)h*<6ay3h000O8001EX3YRvZz5oCK00961ApigX0000000000q=6dG0044jadl~OWo>0{baO9bZ){;=EpTOWV`Xr3X>V>Wba-@7O928D0~7!N00;m803iV7Ql)QT0001O0000N00000000000001_ffvvK0CZt<UvhPBbZKp6FJE72ZfSI1UoLQYP)h*<6ay3h000O8001EXJNEu}77qXbhcy5I7ytkO0000000000q=Br^004Ahb6;|GZggpFWiMfJbZKT`V{~&aaCuNm0Rj{Q6aWAK2mk;8Apok$K`T}n007oj000#L0000000000005+c_}>5kbYXK}a&>NWX>DaMVRLh3baO6nc~DCM0u%!j000080000X074`FOjiW}0I&`K02lxO00000000000HlF|`2YZPVRK({b#8QNZDlWEbaZ8IbZKvHE^v8JO928D0~7!N00;m803iT>sFaVZ4FCW)CjbB#00000000000001_fe!xw0CZt<UvhPBbZKp6FJf<RbaQlaVQ?;Rc~DCM0u%!j000080000X0EVf*T)PJV0MQo!02KfL00000000000HlHM3;_UiVRK({b#8QNZDlWGZ*FF3XD)DgP)h*<6ay3h000O8001EXcC!O(NEiSB22cP16951J0000000000q=DTO0RVJib6;|GZggpFWiMo5bYU)Vc~DCM0u%!j000080000X023CLJXj0>0G%ZO02crN00000000000HlFdEdc;@VRK({b#8QNZDlWHWn*t-X>MmOaCuNm0Rj{Q6aWAK2mk;8ApnbqjRPSG005>O000>P0000000000005+c<2eBUbYXK}a&>NWX>DaMaAk61WpH$9Z*DGdc~DCM0u%!j000080000X0JT3z7a9%#0NphJ02%-Q00000000000HlFCLjeGEVRK({b#8QNZDlWTWpZ<Ab97~HV`VOIc~DCM0u%!j000080000X07*V?(LDnI0G|i|02lxO00000000000HlG9Q2_vSVRK({b#8QNZDlWTa&Km7Y-w(1E^v8JO928D0~7!N00;m803iU!vVbaK4FCYDEC2u&00000000000001_fdEwj0CZt<UvhPBbZKp6FLHHmbZKp6E^v8JO928D0~7!N00;m803iVW`fzs@0ssI!1ONaL00000000000001_ft6wb0CZ(@baO9bZ*FFEWpi{caCuNm0Rj{Q6aWAK2mk;8Apm;x>}LK2002@H000*N0000000000005+c*<=9#bY*jNb1!sdb97%}a&&2CVPkZ2E^v8JO928D0~7!N00;m803iV5z+|Ay2mk<<9{>Op00000000000001_fedQ_0CZ(@baO9sWpi|2VRLh3baO6nc~DCM0u%!j000080000X0A*}f9{UFX0HPKE022TJ00000000000HlElbO8W#Wpi|MFLY&dbYEe0WNCCRaCuNm0Rj{Q6aWAK2mk;8Apov<n#C^!006`Z000*N0000000000005+cHGBa8bY*jNb1!sdb97%~Z*O#SbaG*EE^v8JO928D0~7!N00;m803iTP!x~{=2mk=L9RL6l00000000000001_fti2-0CZ(@baO9sWpi|2WMOn+E^v8JO928D0~7!N00;m803iUJux|eL0ssIy2LJ#W00000000000001_fhmar0CZ(@baO9sWpi|2WNBe%Zf|pRX=8IPaCuNm0Rj{Q6aWAK2mk;8Apmez%TV(N003zj000;O0000000000005+cR*V4vbY*jNb1!sdb97&2c42IFVRUJ4ZZ2?nP)h*<6ay3h000O8001EXH}4#VuLb}BLK6T0761SM0000000000q=A2w0RVJmb98erbY*jNUv6)7Wnyn{Yc6nkP)h*<6ay3h000O8001EXY(?Z&@&o_?=MDe>9{>OV0000000000q=90a0RVJmb98erbY*jNUv6)7Wnyn{YhPq#Y-x67a(OOrc~DCM0u%!j000080000X0NGFY0b2$D0A&^c02u%P00000000000HlGDpaB4MWpi|MFLY&dbYE~~a&u{ObY*U1WiD`eP)h*<6ay3h000O8001EX*slW(oCW{@x)lHb761SM0000000000q=6%+0RVJmb98erbY*jNUvP3|aA9&~WG--dP)h*<6ay3h000O8001EX5csr-@dW??j1vF=6aWAK0000000000q=ECT0RVJmb98erbY*jNUvgz|Z*p`laCuNm0Rj{Q6aWAK2mk;8ApqZWFA$Lj001Eu000#L0000000000005+c7PJ8XbY*jNb1!sdb97&Fb#8QNZDlTSc~DCM0u%!j000080000X0A~~ZhW-x#09rTz02KfL00000000000HlH1y8!@nWpi|MFLY&dbYF90X=QF>WiD`eP)h*<6ay3h000O8001EX?dAqoDGmStlq~=N6951J0000000000q=5^}0RVJmb98erbY*jNUvqSIWO*)dc~DCM0u%!j000080000X04A=b&S)0^01Z6=0384T00000000000HlFs+5rG`Z*OdKFJg6RY-C?!Z){;=Uv6)7Wnyn{Yc6nkP)h*<6ay3h000O8001EX8YA|>h6w-wBpd($8UO$Q0000000000q=5+X0RVJwZ)|ffZf|sDVsCG2Uu0!$X?A6Dc`k5yP)h{{00000FaR(BIt>5-#`^&P000'}

def restore_source_package(package, checkout, persistent, repo_url):
    """Notebook bootstrap; stdlib only, before installing scientific packages.

    This function is copied by the notebook generator so upload of the notebook
    alone is sufficient. Its authoritative source remains this Python module.
    Existing baseline files cannot be replaced by the source package.
    """
    import base64
    import hashlib
    import io
    import json
    import os
    import re
    import stat
    import subprocess
    import zipfile
    from pathlib import Path

    if package.get("schema") != "tas-private-source-v1" or any(
        re.fullmatch(r"[a-f0-9]{40}", package.get(key, "")) is None
        for key in ("base_commit", "source_commit")
    ):
        raise ValueError("Invalid versioned source package.")
    data = base64.b85decode(package["archive_b85"])
    if hashlib.sha256(data).hexdigest() != package["sha256"]:
        raise ValueError("Source archive checksum mismatch.")
    checkout, persistent = Path(checkout), Path(persistent)
    base = package["base_commit"]

    def owned_path(name):
        relative = Path(name)
        allowed = len(relative.parts) > 1 and relative.parts[0] in {
            "tas_runtime",
            "experiments",
            "requirements",
            "tests",
            "tools",
        }
        allowed |= name in {"AGENTS.md", "docs/TAS_V7_COLAB.md"}
        return (
            allowed
            and not relative.is_absolute()
            and ".." not in relative.parts
            and (checkout / relative).resolve().is_relative_to(checkout.resolve())
        )

    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        members = archive.infolist()
        if len(members) != len(package["files"]) or {
            item.filename for item in members
        } != set(package["files"]):
            raise ValueError("Source file manifest mismatch.")
        if sum(item.file_size for item in members) > 8 * 1024**2:
            raise ValueError("Source package exceeds its code-only size limit.")
        contents = {}
        for item in members:
            if not owned_path(item.filename) or stat.S_IFMT(
                item.external_attr >> 16
            ) not in {0, stat.S_IFREG}:
                raise ValueError("Unsafe source member.")
            content = archive.read(item)
            if hashlib.sha256(content).hexdigest() != package["files"][item.filename]:
                raise ValueError("Source member checksum mismatch.")
            contents[item.filename] = content
    if not (checkout / ".git").exists():
        subprocess.run(
            ["git", "clone", "--no-checkout", str(repo_url), str(checkout)], check=True
        )
    found = subprocess.run(
        ["git", "cat-file", "-e", base + "^{commit}"], cwd=checkout, capture_output=True
    )
    if found.returncode:
        subprocess.run(["git", "fetch", "origin", base], cwd=checkout, check=True)
    for name in contents:
        baseline = subprocess.run(
            ["git", "cat-file", "-e", base + ":" + name],
            cwd=checkout,
            capture_output=True,
        )
        if baseline.returncode == 0:
            raise ValueError(
                "Source package cannot overwrite a public baseline file: " + name
            )
    subprocess.run(["git", "checkout", "--detach", base], cwd=checkout, check=True)
    actual = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=checkout, text=True
    ).strip()
    if actual != base:
        raise ValueError("Public baseline checkout mismatch.")
    previous = checkout / ".tas-source.json"
    if previous.exists():
        old = json.loads(previous.read_text())
        for name in set(old.get("files", {})) - set(contents):
            baseline = subprocess.run(
                ["git", "cat-file", "-e", base + ":" + name],
                cwd=checkout,
                capture_output=True,
            )
            if not owned_path(name) or baseline.returncode == 0:
                raise ValueError("Unsafe obsolete source path.")
            (checkout / name).unlink(missing_ok=True)

    def save(path, content):
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + ".installing")
        temporary.write_bytes(content)
        os.replace(temporary, path)

    for name, content in contents.items():
        save(checkout / name, content)
    receipt = {key: value for key, value in package.items() if key != "archive_b85"}
    receipt_bytes = (json.dumps(receipt, indent=2, sort_keys=True) + "\n").encode()
    save(checkout / ".tas-source.json", receipt_bytes)
    stored = (
        persistent
        / "code"
        / ("source-" + package["source_commit"] + "-" + package["sha256"][:16])
    )
    if (
        not stored.with_suffix(".zip").exists()
        or hashlib.sha256(stored.with_suffix(".zip").read_bytes()).hexdigest()
        != package["sha256"]
    ):
        save(stored.with_suffix(".zip"), data)
    save(stored.with_suffix(".json"), receipt_bytes)
    return receipt


## Connect storage and restore the exact code

Drive authorization may prompt once per runtime. No credentials are saved
in configuration or outputs. Source restoration is automatic and verifies
both the archive and every file before installation. It cannot replace any
file from the frozen public baseline. The same caches work in CPU/GPU sessions.


In [ ]:
try:
    import os, sys, json, subprocess, pathlib, shutil, time
    from IPython.display import display, Markdown, HTML
    from google.colab import drive
    drive.mount("/content/drive")
    persistent = pathlib.Path(PERSISTENT_ROOT)
    persistent.mkdir(parents=True, exist_ok=True)
    checkout = pathlib.Path("/content/tas-v7-checkout")
    if (SOURCE_PACKAGE["base_commit"], SOURCE_PACKAGE["source_commit"]) != (BASELINE_COMMIT, RESEARCH_COMMIT):
        raise ValueError("Regenerate the notebook to change its fixed code versions.")
    source_receipt = restore_source_package(SOURCE_PACKAGE, checkout, persistent, REPO_URL)
    actual_commit = source_receipt["source_commit"]
    os.chdir(checkout)
    if str(checkout) not in sys.path:
        sys.path.insert(0, str(checkout))
    # A new idea notebook may reuse this VM with different experiment modules.
    for module_name in list(sys.modules):
        if module_name == "tas_runtime" or module_name.startswith("tas_runtime."):
            del sys.modules[module_name]
    import importlib
    importlib.invalidate_caches()
    print("Public baseline:", BASELINE_COMMIT)
    print("Experiment Python source:", actual_commit, "| verified files:", len(source_receipt["files"]))
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Isolated, pinned environment

Python 3.10.16 and matched PyTorch 2.4.1 / torchvision 0.19.1 CPU or CUDA
12.1 wheels run in subprocesses. The legacy Transformers 4.35.2 stack is
preserved. No notebook-kernel restart, system Torch replacement or CUDA
extension compilation is needed. GPU mode validates its actual CUDA path.


In [ ]:
try:
    query = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True) if shutil.which("nvidia-smi") else None
    has_gpu = query is not None and query.returncode == 0 and bool(query.stdout.strip())
    effective_phase = ("run" if has_gpu else "prepare") if PHASE == "auto" else PHASE
    mode = "gpu" if effective_phase == "run" else "cpu"
    print("Hardware:", query.stdout.strip() if has_gpu else "CPU")
    print("Phase:", effective_phase, "| environment:", mode)
    if effective_phase == "run" and has_gpu and REQUIRE_A100_80GB and ("A100" not in query.stdout or int(query.stdout.strip().splitlines()[0].split(",")[-1].strip().split()[0]) < 70000):
        raise RuntimeError("This configuration requires A100 80GB; hardware check stopped before environment/model setup.")
    if effective_phase == "run" and not has_gpu:
        raise RuntimeError("Select a GPU runtime, or use PHASE=prepare/score on CPU.")
    if effective_phase == "run" and not (persistent / "preparation.json").is_file():
        raise RuntimeError("Run this notebook once in a CPU runtime first. That downloads/persists models and data before A100 allocation.")
    from tas_runtime.bootstrap import ensure_environment
    python = ensure_environment(mode, "/content/tas-v7-envs", checkout)
    if DATASET == "amber":
        from tas_runtime.bootstrap import ensure_amber
        ensure_amber(python, "/content/tas-v7-envs", checkout)
    env = dict(os.environ, PYTHONHASHSEED="0", TOKENIZERS_PARALLELISM="false", MPLBACKEND="Agg",
               NLTK_DATA=str(persistent / "nlp"), YOLO_CONFIG_DIR=str(pathlib.Path(LOCAL_ROOT) / "ultralytics"))
    configuration = {
        "persistent_root":PERSISTENT_ROOT, "local_root":LOCAL_ROOT, "stage":STAGE,
        "seeds":SEEDS, "methods":METHODS, "batch_sizes":BATCH_SIZES,
        "attention_backends":ATTENTION_BACKENDS, "last_position_logits":LAST_POSITION_LOGITS,
        "max_session_minutes":MAX_SESSION_MINUTES, "require_a100":REQUIRE_A100_80GB,
        "run_confirmation":RUN_CONFIRMATION, "run_controls":RUN_CONTROLS,
        "run_diagnostics":RUN_DIAGNOSTICS, "run_final":RUN_FINAL_REMAINDER,
        "reserve_pope":RESERVE_POPE_IMAGES, "split_seed":SPLIT_SEED,
        "coverage_tolerance":COVERAGE_TOLERANCE_PP / 100.0,
        "dataset":DATASET, "external_manifest":EXTERNAL_MANIFEST, "external_image_root":EXTERNAL_IMAGE_ROOT,
        "auto_continue_after_smoke":True, "source_revision":actual_commit,
        "generation":{"max_new_tokens":180,"temperature":1.0,"top_p":1.0,"top_k":None,
                      "precision":"fp16","do_sample":True,"conv_mode":"llava_v1","prompt":"Describe this image."}
    }
    config_path = pathlib.Path("/content/tas-v7-config.json")
    config_path.write_text(json.dumps(configuration, indent=2))
    def run_phase(phase):
        subprocess.run([str(python), "-m", "experiments.tas_v7.cli", "--config",str(config_path),"--phase",phase], cwd=checkout,env=env,check=True)
    if DATASET == "amber":
        subprocess.run([str(python), "-m", "tas_runtime.bootstrap", "--root", "/content/tas-v7-envs", "--nlp-root", str(persistent / "nlp"), "--amber"], cwd=checkout, env=env, check=True)
    run_phase("plan")
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Prepare or run the study

First CPU session: approximately 17–18 GB of checkpoints plus annotations,
selected images and NLP resources; existing verified files are reused.
Only 30 train images are fetched, not all COCO train2017.

GPU sessions: lightweight preflight → local asset restoration → one model
load → baseline/view audit → correctness/throughput pilot → configured
stages. Per-image RNG makes retries/resume independent of job order. A
rejected optimization is recorded; baseline failures block larger runs.
Results use immutable chunks and perception caches use shards on Drive.


In [ ]:
try:
    phase_started = time.perf_counter()
    if effective_phase == "prepare":
        # Small NLP assets enable the offline scientific fixtures first.
        subprocess.run([str(python), "-m", "tas_runtime.bootstrap", "--root", "/content/tas-v7-envs", "--nlp-root", str(persistent / "nlp")], cwd=checkout, env=env, check=True)
        run_phase("check")
        run_phase("prepare")
        run_phase("score")
        display(Markdown("**CPU preparation complete.** Files are persistent. Switch to the A100 runtime and Run all again."))
    elif effective_phase == "run":
        run_phase("check")
        run_phase("run")
    elif effective_phase == "score":
        run_phase("score")
    elif effective_phase == "check":
        run_phase("check")
    else:
        raise ValueError("Unknown phase")
    print("Session phase seconds:", round(time.perf_counter() - phase_started, 1))
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Inspect saved reports

LEGACY and HARMONIZED are separate protocols; do not infer a matched
algorithm gain across them. CHAIR and annotation category recall do not
establish caption quality or preserved relations. Inspect ambiguous images,
captions, failures, coverage, length, timings and uncertainty together.
Reports include descriptive comparisons for paired captions with exactly
equal word counts. Batch latency and amortized seconds/image are separate.
CPU scoring/report regeneration is available with PHASE="score".


In [ ]:
try:
    import csv, html
    from IPython.display import Image as DisplayImage
    runs_root = persistent / "runs"
    for run_dir in sorted(runs_root.glob("*"))[-3:]:
        display(Markdown("### Run `" + run_dir.name + "`"))
        session = run_dir / "session.json"
        if session.exists():
            display(Markdown("```json\n" + session.read_text() + "\n```"))
        gallery = run_dir / "views.png"
        if gallery.exists():
            display(DisplayImage(filename=str(gallery)))
        for summary in sorted(run_dir.glob("*/summary.csv")):
            rows = list(csv.reader(summary.open()))
            table = "<table>" + "".join("<tr>" + "".join("<td>" + html.escape(value) + "</td>" for value in row) + "</tr>" for row in rows) + "</table>"
            display(Markdown("**" + summary.parent.name + "**"))
            display(HTML(table))
    print("Persistent artifacts:", persistent)
    print("Reuse this folder for later sessions/new idea notebooks. Do not delete frozen manifests or rewrite completed runs.")
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Release the session

With `AUTO_SHUTDOWN=True`, the final cell flushes Drive writes and releases
the runtime after preparation, evaluation or a planned budget stop. Errors
also invoke this cleanup before propagating. Reports remain on Drive.
Set it to False only when keeping the runtime for inspection.


In [ ]:
release_runtime()
